In [1]:
# inlegalbert_kg_rag_rrc.py  (KNOWLEDGE GRAPH RAG REVISION)
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#   +
#   Knowledge Graph (KG) with RST-based edges
#   +
#   Uncertainty-triggered KG Retrieval + Graph Attention Fusion
#
# NEW vs BASELINE (inlegalbert_bilstm_mha_crf_rrc_v2.py):
#   1. KnowledgeGraph class: stores sentence embeddings as nodes per label,
#      builds RST-style edges (cosine-similarity proxy) within and across roles.
#   2. UncertaintyEstimator: computes entropy H(p_i) from softmax probabilities.
#   3. KGRetriever: similarity search across all label subgraphs, top-K selection,
#      1-hop graph expansion, subgraph G_i construction.
#   4. GraphAttentionFusion: weighted aggregation of neighbour embeddings,
#      alpha_j ∝ similarity × edge_weight; fused vector v_i.
#   5. KGAugmentedModel wraps the base model, runs first-pass CRF, checks
#      uncertainty, conditionally retrieves KG context, fuses h* = h_i + v_i,
#      re-runs CRF emission head.
#   6. Two-phase training:
#        Phase A – train base model (same as v2) + build KG from train embeddings.
#        Phase B – fine-tune KGAugmentedModel end-to-end (KG frozen, fusion trained).
#   7. All anti-overfitting settings from v2 retained.

import os, json, random, time, math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60          # Phase A: base model training
NUM_EPOCHS_KG   = 20          # Phase B: KG-augmented fine-tuning
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

# BERT freeze / layer-wise LR decay
BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

# Sentence-level BiLSTM
SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

# Multi-Head Attention Pooling
MHA_HEADS    = 4
MHA_DROPOUT  = 0.1

# Context-enrichment BiLSTM
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

# Auxiliary loss
AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1

# Early stopping
ES_PATIENCE  = 10
ES_MIN_DELTA = 1e-4

WARMUP_RATIO = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD = 0.05

# ── KG-RAG specific ────────────────────────────────────────
KG_TOP_K            = 3      # top-K subgraphs to retrieve
KG_TOP_NODES        = 5      # nodes per subgraph to retrieve
KG_HOP              = 1      # graph expansion hops
UNCERTAINTY_THRESH  = 0.7    # entropy threshold (0-1 normalised) above which KG is used
RARE_ALWAYS_KG      = True   # always use KG for rare-class candidates
KG_FUSION_DIM       = 256    # sent_out_dim = SENT_LSTM_HIDDEN * 2

# RST edge similarity thresholds (proxy; real RST parser optional)
RST_INTRA_THRESH    = 0.6    # within same label subgraph
RST_CROSS_THRESH    = 0.5    # across different labels

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)

        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()

        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)

        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2        # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = self.sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = self.sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2          # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ) -> torch.Tensor:
        """Returns sentence-level embeddings: (B, T, sent_out_dim)."""
        B, T, L = input_ids.shape
        N = B * T

        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)

        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)

        return sent_vecs.view(B, T, -1)

    def get_emissions(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ):
        """Returns (sent_vecs, ctx_out, emissions) for external use."""
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs_drop = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        _, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")

            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )

            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    """
    Stores sentence embeddings as nodes organised per-label (subgraph).
    Edges are built using cosine-similarity as an RST proxy.

    Node structure:
        nodes[label_id] = list of {"emb": Tensor(D,), "text": str}

    Edge structure (per subgraph):
        intra_edges[label_id] = list of (i, j, weight)   # within same label
        cross_edges = list of (label_i, node_i, label_j, node_j, weight)
    """
    def __init__(self, emb_dim=KG_FUSION_DIM):
        self.emb_dim     = emb_dim
        self.nodes       = defaultdict(list)     # label_id -> list of node dicts
        self.intra_edges = defaultdict(list)     # label_id -> list of (i,j,w)
        self.cross_edges = []                    # (li,ni, lj,nj, w)
        self._stacked    = {}                    # label_id -> Tensor (N, D) on CPU

    # ── Population ──────────────────────────────────────────
    def add_nodes(self, embeddings: torch.Tensor, label_ids: list, texts: list = None):
        """Add sentence embeddings collected during training."""
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})
        self._stacked = {}  # invalidate cache

    def build_edges(self,
                    intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_edges_per_node=5,
                    max_cross_edges=2000):
        """
        Build edges:
          INTRA: consecutive + high-sim pairs within same-label subgraph (RST proxy).
          CROSS: top-sim pairs across labels (discourse bridges).
        """
        print("  Building KG edges ...")
        self._stacked = {}
        self.intra_edges = defaultdict(list)
        self.cross_edges = []

        # ── Intra-label edges ─────────────────────────────
        for lid, node_list in self.nodes.items():
            N = len(node_list)
            if N < 2:
                continue
            embs = torch.stack([n["emb"] for n in node_list])   # (N, D)
            embs_norm = F.normalize(embs, dim=-1)
            sim_mat = torch.mm(embs_norm, embs_norm.T)          # (N, N)

            # consecutive edges always added
            for i in range(N - 1):
                w = float(sim_mat[i, i + 1].item())
                self.intra_edges[lid].append((i, i + 1, max(0.0, w)))

            # high-sim non-consecutive edges
            for i in range(N):
                sims = sim_mat[i].clone()
                sims[max(0, i-1):i+2] = -1   # mask out consecutive
                count = 0
                while count < max_intra_edges_per_node:
                    j = int(sims.argmax().item())
                    if sims[j] < intra_thresh:
                        break
                    w = float(sims[j].item())
                    self.intra_edges[lid].append((i, j, w))
                    sims[j] = -1
                    count += 1

            self._stacked[lid] = embs   # cache

        # ── Cross-label edges (discourse bridges) ─────────
        label_ids = list(self.nodes.keys())
        cross_count = 0
        for a in range(len(label_ids)):
            if cross_count >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if cross_count >= max_cross_edges:
                    break
                la, lb = label_ids[a], label_ids[b]
                embs_a = self._get_stacked(la)
                embs_b = self._get_stacked(lb)
                if embs_a is None or embs_b is None:
                    continue
                na_norm = F.normalize(embs_a, dim=-1)
                nb_norm = F.normalize(embs_b, dim=-1)
                sim_mat = torch.mm(na_norm, nb_norm.T)          # (Na, Nb)
                high = (sim_mat >= cross_thresh).nonzero(as_tuple=False)
                for pair in high[:50]:
                    ni, nj = int(pair[0]), int(pair[1])
                    w = float(sim_mat[ni, nj].item())
                    self.cross_edges.append((la, ni, lb, nj, w))
                    cross_count += 1

        n_intra = sum(len(v) for v in self.intra_edges.values())
        print(f"  KG: {sum(len(v) for v in self.nodes.values())} nodes | "
              f"{n_intra} intra-edges | {len(self.cross_edges)} cross-edges")

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack([n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    def save(self, path):
        """Serialise KG to disk."""
        data = {
            "nodes": {str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                                for n in v]
                      for k, v in self.nodes.items()},
            "intra_edges": {str(k): v for k, v in self.intra_edges.items()},
            "cross_edges": self.cross_edges,
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved to {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        kg = cls(emb_dim=emb_dim)
        with open(path) as f:
            data = json.load(f)
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]), "text": n["text"]})
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges = [tuple(e) for e in data["cross_edges"]]
        print(f"  KG loaded from {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    """Normalised entropy: H(p) / log(C) ∈ [0,1]."""
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        """logits: (..., C) → entropy (...,) in [0,1]."""
        probs = F.softmax(logits, dim=-1)
        eps   = 1e-9
        H     = -(probs * (probs + eps).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor) -> torch.Tensor:
        """Returns boolean mask of same shape as logits[..., 0]."""
        return self.entropy(logits) > self.threshold

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    """
    Given a query embedding h_i:
      1. Similarity search: score each label subgraph.
      2. Select top-K subgraphs.
      3. Within each subgraph, find top-N nodes.
      4. 1-hop graph expansion via intra- and cross-edges.
      5. Return list of (embedding, weight) pairs = subgraph G_i.
    """
    def __init__(self, kg: KnowledgeGraph,
                 top_k=KG_TOP_K,
                 top_nodes=KG_TOP_NODES,
                 hop=KG_HOP):
        self.kg        = kg
        self.top_k     = top_k
        self.top_nodes = top_nodes
        self.hop       = hop

    def retrieve(self, h_i: torch.Tensor, rare_ids: list = None,
                 first_pass_label: int = None) -> list:
        """
        h_i: (D,) query embedding (CPU).
        Returns list of (emb: Tensor(D,), weight: float).
        """
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)   # (1, D)

        # Step 1: score each subgraph
        subgraph_scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None or embs.shape[0] == 0:
                continue
            embs_norm = F.normalize(embs, dim=-1)         # (N, D)
            sims = torch.mv(embs_norm, h_norm.squeeze(0)) # (N,)
            subgraph_scores[lid] = float(sims.max().item())

        # Step 2: top-K subgraphs
        sorted_sgs = sorted(subgraph_scores.items(), key=lambda x: -x[1])
        selected   = [lid for lid, _ in sorted_sgs[:self.top_k]]

        results = []   # list of (emb, weight)

        for lid in selected:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            embs_norm = F.normalize(embs, dim=-1)
            sims = torch.mv(embs_norm, h_norm.squeeze(0))

            # Step 3: top-N nodes in this subgraph
            k = min(self.top_nodes, embs.shape[0])
            top_idx = sims.topk(k).indices.tolist()
            top_sims = sims.topk(k).values.tolist()

            seed_set = set(top_idx)

            # Step 4: 1-hop expansion via intra-edges
            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)

                # cross-edges
                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        cross_embs = self.kg._get_stacked(lb)
                        if cross_embs is not None and nj < cross_embs.shape[0]:
                            results.append((cross_embs[nj], w))
                    elif lb == lid and nj in seed_set:
                        cross_embs = self.kg._get_stacked(la)
                        if cross_embs is not None and ni < cross_embs.shape[0]:
                            results.append((cross_embs[ni], w))

            # Collect seed nodes
            for idx, sim in zip(top_idx, top_sims):
                results.append((embs[idx], float(sim)))

        return results  # list of (Tensor(D), float)


# ═══════════════════════════════════════════════════════════
# GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    """
    Given h_i (query) and a set of (emb_j, weight_j) neighbours,
    compute attention weights α_j ∝ similarity(h_i, emb_j) × edge_weight_j
    and aggregate: v_i = Σ α_j * emb_j.

    The fused vector has the same dim as h_i (sent_out_dim = 256).
    """
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.emb_dim = emb_dim
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i: torch.Tensor,
                neighbours: list,
                device=None) -> torch.Tensor:
        """
        h_i        : (D,)
        neighbours : list of (Tensor(D,), float weight)
        Returns    : v_i (D,)
        """
        if not neighbours:
            return torch.zeros_like(h_i)

        if device is None:
            device = h_i.device

        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)  # (K, D)
        weights = torch.tensor([nb[1] for nb in neighbours],
                               device=device, dtype=torch.float)         # (K,)

        q = self.proj_q(h_i.unsqueeze(0))                   # (1, D)
        k = self.proj_k(embs)                                # (K, D)

        dot = torch.mv(k, q.squeeze(0)) * self.scale        # (K,)
        alpha = F.softmax(dot * weights, dim=0)             # (K,)  ∝ sim×weight
        alpha = self.dropout(alpha)

        v_i = (alpha.unsqueeze(-1) * embs).sum(dim=0)       # (D,)
        return v_i


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):
    """
    Wraps the base InLegalBERT_BiLSTM_MHA_CRF model and adds:
      1. First-pass emission head → uncertainty check.
      2. Conditional KG retrieval.
      3. Graph attention fusion: h* = h_i + v_i.
      4. Project fused h* back to ctx_out_dim then re-run classifier → CRF.

    During training (Phase B), the base model weights continue to be
    updated; the KG is frozen (no grad); only GraphAttentionFusion +
    fusion_proj are newly trained.
    """
    def __init__(
        self,
        base_model: InLegalBERT_BiLSTM_MHA_CRF,
        kg:         KnowledgeGraph,
        rare_ids:   list,
        retriever:  KGRetriever = None,
    ):
        super().__init__()
        self.base      = base_model
        self.kg        = kg
        self.rare_ids  = rare_ids
        self.retriever = retriever or KGRetriever(kg)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim          # 256
        ctx_dim  = base_model.ctx_out_dim           # 128

        self.gat_fusion  = GraphAttentionFusion(emb_dim=sent_dim)
        # Project fused sentence vector (sent_dim) → ctx_dim for classifier reuse
        self.fusion_proj = nn.Sequential(
            nn.Linear(sent_dim * 2, ctx_dim),       # cat(h_i, v_i) → ctx_dim
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )
        # Extra CRF emission head for fused representations
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100
        )

    def _kg_fuse_batch(
        self,
        sent_vecs:  torch.Tensor,   # (B, T, sent_dim)
        emissions:  torch.Tensor,   # (B, T, C)
        lengths:    torch.Tensor,   # (B,)
        device,
    ) -> torch.Tensor:
        """
        For each sentence in the batch, check uncertainty and
        optionally retrieve+fuse KG context.
        Returns fused_sent_vecs (B, T, sent_dim).
        """
        B, T, sent_dim = sent_vecs.shape
        fused = sent_vecs.clone()

        uncertain_mask = self.uncertainty.is_uncertain(emissions)  # (B, T)
        top_labels     = self.uncertainty.top_label(emissions)     # (B, T)

        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                uncertain = bool(uncertain_mask[b, t].item())
                pred_lbl  = int(top_labels[b, t].item())
                is_rare   = pred_lbl in self.rare_ids

                if not (uncertain or (RARE_ALWAYS_KG and is_rare)):
                    continue  # confident majority-class: use h_i directly

                h_i = sent_vecs[b, t].detach().cpu()  # KG is on CPU
                neighbours = self.retriever.retrieve(
                    h_i,
                    rare_ids=self.rare_ids,
                    first_pass_label=pred_lbl,
                )

                if not neighbours:
                    continue

                v_i = self.gat_fusion(
                    sent_vecs[b, t],      # keep on GPU for grads
                    neighbours,
                    device=device,
                )                         # (sent_dim,)

                # h* = concat[h_i, v_i] → project
                # Store back as cat for fusion_proj later
                fused[b, t] = sent_vecs[b, t] + v_i   # residual fusion

        return fused  # (B, T, sent_dim)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        device = input_ids.device

        # ── Step 1: first-pass base model ─────────────────
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        # ── Step 2: uncertainty + KG retrieval + fusion ───
        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device
        )  # (B, T, sent_dim)

        # ── Step 3: run context BiLSTM on fused sent vecs ─
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)

        fused_ctx = self.base.dropout(fused_ctx)

        # ── Step 4: fusion classifier + CRF ───────────────
        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = torch.nan_to_num(fused_emissions, nan=0.0,
                                            posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2], dtype=torch.bool,
                              device=device)

        # ── Combine base + fused emissions (ensemble) ─────
        combined_emissions = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            # Base CRF loss
            base_crf_loss = -self.base.crf(
                base_emissions, safe_labels, mask=mask, reduction="mean"
            )
            # Fused CRF loss
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe_labels, mask=mask, reduction="mean"
            )
            # Auxiliary CE on combined
            B2, T2, C = combined_emissions.shape
            ce_loss = self.ce_loss(
                combined_emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )

            loss = (base_crf_loss + fused_crf_loss) / 2.0 + AUX_CE_WEIGHT * ce_loss
            return loss, combined_emissions
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]
    cls_report = classification_report(
        str_trues, str_preds, labels=LABELS, digits=4, zero_division=0,
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {total_trainable:,} | Frozen: {total_frozen:,}")
    return total_trainable, total_frozen


# ═══════════════════════════════════════════════════════════
# TRAINER  (Phase A: base model)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []

        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })

        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({
                    "params": params, "lr": lr_i, "weight_decay": WEIGHT_DECAY,
                })

        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({
            "params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
        })

        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attn, ttype, labels, lengths in loader:
                input_ids = input_ids.to(self.device)
                attn      = attn.to(self.device)
                ttype     = ttype.to(self.device)
                labels    = labels.to(self.device)
                lengths   = lengths.to(self.device)
                loss, _   = self.model(input_ids, attn, ttype,
                                       labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item(); n += 1
        return total_loss / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attn, ttype, labels, lengths in loader:
                input_ids = input_ids.to(self.device)
                attn      = attn.to(self.device)
                ttype     = ttype.to(self.device)
                lengths   = lengths.to(self.device)
                decoded, _ = self.model(input_ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer   = self.build_optimizer()
        total_steps = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler   = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        early_stopper = EarlyStopping()
        history = []
        best_f1, best_state = -1.0, None

        total_start = time.time()
        actual_epochs = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1; optimizer.zero_grad(); continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_loss       = self.compute_val_loss(dev_dataset)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_train_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")

        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))
            print(f"  Base model saved to {BEST_MODEL_DIR}/base_model.bin")

        return hist_df, total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER  (collect embeddings after Phase A)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer,
                           device=DEVICE) -> KnowledgeGraph:
    """
    Run base_model encoder over training data and populate KG nodes.
    """
    print("\n🔨 Building Knowledge Graph from training embeddings ...")
    base_model.eval()
    base_model.to(device)

    kg = KnowledgeGraph(emb_dim=base_model.sent_out_dim)

    dummy_dataset = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy_dataset, batch_size=1, shuffle=False,
                        collate_fn=collate_rrc)

    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids     = ids.to(device)
        attn    = attn.to(device)
        ttype   = ttype.to(device)
        lengths = lengths.to(device)

        sent_vecs = base_model.encode_sentences(ids, attn, ttype)  # (1, T, D)
        sent_vecs = sent_vecs.squeeze(0)                            # (T, D)

        n = int(lengths[0].item())
        embs     = sent_vecs[:n].cpu()                              # (n, D)
        lab_ids  = labels[0, :n].tolist()

        kg.add_nodes(embs, lab_ids)

        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1} / {len(loader)} docs")

    kg.build_edges()
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# TRAINER  (Phase B: KG-augmented fine-tuning)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: KGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        # Fine-tune: base model (unfrozen layers) + new fusion modules
        new_params  = list(self.model.gat_fusion.parameters()) + \
                      list(self.model.fusion_proj.parameters()) + \
                      list(self.model.fusion_classifier.parameters()) + \
                      list(self.model.fusion_crf.parameters())

        base_trainable = [p for p in self.model.base.parameters() if p.requires_grad]

        param_groups = [
            {"params": new_params,      "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable,  "lr": BERT_LR,  "weight_decay": WEIGHT_DECAY},
        ]
        return torch.optim.AdamW(param_groups)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)

                decoded, _ = self.model(ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents = len(all_trues)
            infer_info = {
                "total_inference_time_s": total_infer,
                "latency_per_document_ms": total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer   = self.build_optimizer()
        total_steps = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler   = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        early_stopper = EarlyStopping(patience=5)
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1; optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item()
                n_steps += 1

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG]   Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "kg",
                "train_loss": avg_train_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  KG early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "kg_history.csv"), index=False
        )
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    labels = LABELS
    f1s    = [per_class_metrics[l]["f1"] for l in labels]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in labels]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1 (KG-RAG)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)

    # Phase boundary
    boundary = len(base_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"],
            label="Train Loss", marker="o", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Combined Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1", marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1 (Base → KG-RAG)"); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def print_metrics_table(dev_metrics, test_metrics,
                        base_time=None, kg_time=None,
                        total_trainable=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Macro-Recall",       "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + KG-RAG)")
    print("=" * 72)
    if total_trainable:
        print(f"  Trainable Parameters  : {total_trainable:,}")
    if base_time:
        print(f"  Phase A training time : {base_time/60:.1f} min")
    if kg_time:
        print(f"  Phase B training time : {kg_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 72)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 64)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 64)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 64)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT + BiLSTM + MHA + CRF  →  KG-RAG\n")
    print("Phase A: Train base model, build Knowledge Graph")
    print("Phase B: Fine-tune KG-Augmented model\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    freq_df = pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ])
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════════════════════════════════════════════════════
    # PHASE A: Train base model
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    )

    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE,
    )

    # ── Build KG from best base model ─────────────────
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Knowledge Graph")
    print("=" * 60)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG, loading ...")
        kg = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim)
    else:
        kg = build_knowledge_graph(base_model, train_docs, tokenizer, DEVICE)

    # ════════════════════════════════════════════════════
    # PHASE B: KG-Augmented fine-tuning
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: KG-Augmented Fine-Tuning")
    print("=" * 60)

    retriever = KGRetriever(kg, top_k=KG_TOP_K,
                            top_nodes=KG_TOP_NODES, hop=KG_HOP)

    kg_model = KGAugmentedModel(
        base_model = base_model,
        kg         = kg,
        rare_ids   = rare_ids,
        retriever  = retriever,
    )

    total_trainable, _ = count_parameters(kg_model)

    kg_trainer  = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        num_epochs=NUM_EPOCHS_KG,
    )

    plot_combined_history(base_hist_df, kg_hist_df)

    # ════════════════════════════════════════════════════
    # EVALUATION
    # ════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_metrics = kg_trainer.evaluate(dev_dataset, rare_ids,
                                       split_name="dev",
                                       measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-RAG\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                        split_name="test",
                                        measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-RAG\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall", "micro_recall", "weighted_recall", "rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": "InLegalBERT + BiLSTM + MHA + CRF + KG-RAG",
        "kg_config": {
            "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES,
            "hop": KG_HOP, "uncertainty_thresh": UNCERTAINTY_THRESH,
            "rare_always_kg": RARE_ALWAYS_KG,
            "intra_thresh": RST_INTRA_THRESH, "cross_thresh": RST_CROSS_THRESH,
        },
        "timing": {
            "phase_a_s": base_time, "phase_b_s": kg_time,
            "total_s": base_time + kg_time,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        base_time=base_time, kg_time=kg_time,
        total_trainable=total_trainable,
    )

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT + BiLSTM + MHA + CRF  →  KG-RAG

Phase A: Train base model, build Knowledge Graph
Phase B: Fine-tune KG-Augmented model

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (10): ['RLC', 'ISSUE'

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7.
🔥 BERT layers trainable: layers 8-11 + pooler.

[Base] Epoch 001/60 | train_loss: 287.7585 | val_loss: 211.9501 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | time: 65.0s | ES: 0/10
  ✔ New best val_macro_f1=0.0391
[Base] Epoch 002/60 | train_loss: 234.5406 | val_loss: 175.5814 | val_macro_f1: 0.0797 | val_rare_f1: 0.0000 | time: 65.3s | ES: 0/10
  ✔ New best val_macro_f1=0.0797
[Base] Epoch 003/60 | train_loss: 198.7177 | val_loss: 128.8831 | val_macro_f1: 0.2005 | val_rare_f1: 0.0560 | time: 65.5s | ES: 0/10
  ✔ New best val_macro_f1=0.2005
[Base] Epoch 004/60 | train_loss: 166.5206 | val_loss: 106.5727 | val_macro_f1: 0.2568 | val_rare_f1: 0.0970 | time: 65.0s | ES: 0/10
  ✔ New best val_macro_f1=0.2568
[Base] Epoch 005/60 | train_loss: 140.3391 | val_loss: 88.4431 | val_macro_f1: 0.2828 | val_rare_f1: 0.1195 | time: 64.8s | ES: 0/10
  ✔ New best val_macro_f1=0.2828
[Base] Epoch 006/60 | train_loss: 124.8879 | val_loss: 83.0902 | val

In [1]:
"""
kg_visualization_fixed.py  – FULLY FIXED + ENHANCED
=====================================================
Fixes:
  ✔ KeyError: 'label'  →  always set node attrs BEFORE add_edge
  ✔ Sparse subgraph    →  edge-aware sampling, guarded node creation
  ✔ t-SNE max_iter     →  already fixed, kept

New:
  ✔ Fig 0 – Publication-quality label-level KG (matches your reference image)
  ✔ Fig 1 – Sentence-level subgraph (sampled, coloured by role)
  ✔ Fig 2 – Cross-label adjacency heatmap
  ✔ Fig 3 – PCA + t-SNE embedding scatter
  ✔ Fig 4 – Node-count bar chart (rare classes highlighted)
"""

import json
import os
import math
import random
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ── CONFIG ────────────────────────────────────────────────
KG_JSON_PATH = "rrc_kg_rag_logs/knowledge_graph.json"
OUT_DIR      = "kg_figures"
os.makedirs(OUT_DIR, exist_ok=True)

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}

# Color palette – one per rhetorical role
PALETTE = [
    "#4E79A7", "#F28E2B", "#E15759", "#76B7B2", "#59A14F",
    "#EDC948", "#B07AA1", "#FF9DA7", "#9C755F", "#BAB0AC",
    "#D37295", "#FABFD2", "#8CD17D",
]
LABEL_COLOR = {l: PALETTE[i % len(PALETTE)] for i, l in enumerate(LABELS)}

# Rare labels (dashed border in reference image)
RARE_LABELS = {"RLC", "ISSUE", "STA", "PRE_NOT_RELIED", "RPC"}


# ── LOAD KG ───────────────────────────────────────────────
def load_kg(path):
    with open(path) as f:
        kg = json.load(f)

    nodes_by_label = {
        int(k): [np.array(n["emb"]) for n in v]
        for k, v in kg["nodes"].items()
    }
    intra_edges = {
        int(k): [(int(a), int(b), float(w)) for a, b, w in v]
        for k, v in kg["intra_edges"].items()
    }
    cross_edges = [
        (int(a), int(b), int(c), int(d), float(w))
        for a, b, c, d, w in kg["cross_edges"]
    ]

    total_nodes = sum(len(v) for v in nodes_by_label.values())
    total_intra = sum(len(v) for v in intra_edges.values())
    print(f"Loaded KG: {total_nodes} nodes | {total_intra} intra-edges "
          f"| {len(cross_edges)} cross-edges")
    return nodes_by_label, intra_edges, cross_edges


# ════════════════════════════════════════════════════════
# FIG 0 — Label-level KG  (matches your reference image)
# ════════════════════════════════════════════════════════
def fig0_label_graph(nodes_by_label, cross_edges):
    """
    One node per rhetorical role.
    Node size  ∝ number of sentences with that role.
    Edge width ∝ number of cross-edges between roles.
    """
    print("[Fig 0] Label-level KG ...")
    G = nx.Graph()

    # ── Add nodes (all 13 roles) ───────────────────────
    for lid, lbl in id2label.items():
        count = len(nodes_by_label.get(lid, []))
        G.add_node(lbl, count=count, lid=lid)

    # ── Aggregate cross-edge weights ──────────────────
    edge_weight = defaultdict(float)
    for la, _, lb, _, w in cross_edges:
        a, b = id2label[la], id2label[lb]
        if a != b:
            key = tuple(sorted([a, b]))
            edge_weight[key] += w

    for (a, b), w in edge_weight.items():
        G.add_edge(a, b, weight=w)

    # ── Layout ────────────────────────────────────────
    pos = nx.circular_layout(G)

    # ── Visual sizes / widths ─────────────────────────
    counts  = [G.nodes[n]["count"] for n in G.nodes()]
    max_cnt = max(counts) if counts else 1
    node_sizes = [800 + 3000 * (c / max_cnt) for c in counts]

    all_w = [G[u][v]["weight"] for u, v in G.edges()]
    max_w = max(all_w) if all_w else 1
    edge_widths = [0.5 + 4.0 * (G[u][v]["weight"] / max_w) for u, v in G.edges()]
    edge_alphas = [0.3 + 0.5 * (G[u][v]["weight"] / max_w) for u, v in G.edges()]

    node_colors  = [LABEL_COLOR[n] for n in G.nodes()]
    node_edgecolors = [
        "#CC0000" if n in RARE_LABELS else "#333333"
        for n in G.nodes()
    ]
    linewidths = [2.5 if n in RARE_LABELS else 1.2 for n in G.nodes()]

    # ── Draw ──────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 10))
    ax.set_facecolor("#FAFAFA")
    fig.patch.set_facecolor("#FAFAFA")

    # Draw edges with individual alpha (workaround: draw one by one)
    for (u, v), ew, ea in zip(G.edges(), edge_widths, edge_alphas):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        ax.plot([x0, x1], [y0, y1],
                color="#AAAAAA", linewidth=ew, alpha=ea, zorder=1)

    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_size=node_sizes,
        node_color=node_colors,
        edgecolors=node_edgecolors,
        linewidths=linewidths,
        alpha=0.88,
    )

    # Draw labels + counts below
    for node in G.nodes():
        x, y = pos[node]
        count = G.nodes[node]["count"]
        # Short label inside node
        short = node if len(node) <= 9 else node[:8] + "…"
        ax.text(x, y + 0.04, short,
                ha="center", va="center",
                fontsize=7.5, fontweight="bold", color="#111111", zorder=5)
        ax.text(x, y - 0.08, str(count),
                ha="center", va="center",
                fontsize=7, color="#444444", zorder=5)

    # Legend: rare vs common
    rare_patch   = mpatches.Patch(edgecolor="#CC0000", facecolor="none",
                                   linewidth=2, label="Rare class (dashed border)")
    common_patch = mpatches.Patch(edgecolor="#333333", facecolor="none",
                                   linewidth=1, label="Common class")
    ax.legend(handles=[rare_patch, common_patch],
              loc="lower right", fontsize=9, framealpha=0.7)

    ax.set_title(
        "Knowledge Graph – Label-Level View\n"
        "(node size ∝ sentence count · edge width ∝ cross-role similarity)",
        fontsize=13, fontweight="bold", pad=16
    )
    ax.axis("off")
    plt.tight_layout()
    path = f"{OUT_DIR}/fig0_label_graph.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# FIG 1 — Sentence-level subgraph (FIXED KeyError)
# ════════════════════════════════════════════════════════
def fig1_subgraph(nodes_by_label, intra_edges, cross_edges,
                  max_nodes_per_label=15, max_cross=80):
    """
    KEY FIX: always call G.add_node(nid, label=...) BEFORE G.add_edge().
    This ensures every node in the graph has the 'label' attribute,
    preventing the KeyError when iterating G.nodes().
    """
    print("[Fig 1] Sentence-level subgraph ...")
    G = nx.Graph()

    # ── Helper: safely add a node with guaranteed label attr ──
    def safe_add_node(nid, label_str):
        if nid not in G:
            G.add_node(nid, label=label_str)

    # ── Intra-label edges (sampled) ───────────────────
    for lid, node_list in nodes_by_label.items():
        lbl = id2label.get(lid, str(lid))
        edges = intra_edges.get(lid, [])
        if not edges:
            continue

        # Collect connected node indices
        connected = set()
        for i, j, _ in edges:
            connected.add(i)
            connected.add(j)

        # Sample at most max_nodes_per_label connected nodes
        sampled = set(random.sample(sorted(connected),
                                    min(max_nodes_per_label, len(connected))))

        # Add sampled nodes FIRST (with label attr)
        for idx in sampled:
            nid = f"{lbl}_{idx}"
            safe_add_node(nid, lbl)

        # Add only edges whose both endpoints are sampled
        for i, j, w in edges:
            if i in sampled and j in sampled:
                G.add_edge(f"{lbl}_{i}", f"{lbl}_{j}", weight=float(w))

    # ── Cross-label edges (sampled) ───────────────────
    cross_sample = random.sample(cross_edges, min(max_cross, len(cross_edges)))
    for la, ni, lb, nj, w in cross_sample:
        la_str, lb_str = id2label.get(la, str(la)), id2label.get(lb, str(lb))
        nid_a = f"{la_str}_{ni}"
        nid_b = f"{lb_str}_{nj}"
        # Only add cross-edge if BOTH nodes already exist in graph
        if nid_a in G and nid_b in G:
            G.add_edge(nid_a, nid_b, weight=float(w), cross=True)

    print(f"  Subgraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    if G.number_of_nodes() == 0:
        print("  Empty graph – skipping.")
        return

    # ── Colors (safe: use .get with default) ──────────
    colors = [
        LABEL_COLOR.get(G.nodes[n].get("label", "NONE"), "#CCCCCC")
        for n in G.nodes()
    ]

    pos = nx.spring_layout(G, k=2.5, seed=42)

    plt.figure(figsize=(12, 10))
    ax = plt.gca()
    ax.set_facecolor("#F8F8F8")

    # Draw cross-edges (grey, thin)
    cross_edge_list = [(u, v) for u, v in G.edges()
                       if G[u][v].get("cross", False)]
    nx.draw_networkx_edges(G, pos, edgelist=cross_edge_list,
                           edge_color="#BBBBBB", alpha=0.4, width=0.7, ax=ax)

    # Draw intra-edges (coloured)
    intra_edge_list = [(u, v) for u, v in G.edges()
                       if not G[u][v].get("cross", False)]
    nx.draw_networkx_edges(G, pos, edgelist=intra_edge_list,
                           edge_color="#888888", alpha=0.5, width=1.0, ax=ax)

    nx.draw_networkx_nodes(G, pos, node_size=60,
                           node_color=colors, alpha=0.85, ax=ax)

    # Legend
    patches = [mpatches.Patch(color=LABEL_COLOR[l], label=l) for l in LABELS
               if any(G.nodes[n].get("label") == l for n in G.nodes())]
    plt.legend(handles=patches, loc="upper right",
               fontsize=7, ncol=2, framealpha=0.75)
    plt.title("Sentence-Level Knowledge Subgraph (sampled)",
              fontsize=13, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    path = f"{OUT_DIR}/fig1_subgraph.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# FIG 2 — Cross-label adjacency heatmap
# ════════════════════════════════════════════════════════
def fig2_adj(nodes_by_label, cross_edges):
    print("[Fig 2] Adjacency heatmap ...")
    n = len(LABELS)
    mat = np.zeros((n, n))
    for la, _, lb, _, w in cross_edges:
        if la < n and lb < n:
            mat[la, lb] += 1
            mat[lb, la] += 1

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(mat, xticklabels=LABELS, yticklabels=LABELS,
                cmap="YlOrRd", annot=False, linewidths=0.4,
                linecolor="#DDDDDD", ax=ax)
    ax.set_title("Cross-Label Edge Adjacency Matrix", fontsize=13, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    path = f"{OUT_DIR}/fig2_adjacency.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# FIG 3 — PCA + t-SNE embedding scatter
# ════════════════════════════════════════════════════════
def fig3_embeddings(nodes_by_label, max_per_label=60):
    print("[Fig 3] Embedding scatter ...")
    X, y = [], []
    for lid, embs in nodes_by_label.items():
        sample = random.sample(embs, min(max_per_label, len(embs)))
        X.extend(sample)
        y.extend([lid] * len(sample))

    if not X:
        print("  No embeddings found – skipping.")
        return

    X = np.stack(X)
    colors = [LABEL_COLOR.get(id2label.get(yi, "NONE"), "#CCCCCC") for yi in y]

    # PCA
    Xp = PCA(n_components=2).fit_transform(X)

    # t-SNE
    perp = min(30, max(5, len(X) // 5))
    Xt = TSNE(n_components=2, perplexity=perp,
               max_iter=500, random_state=42).fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor("#FAFAFA")

    for ax, Xr, title in zip(axes, [Xp, Xt], ["PCA", "t-SNE"]):
        ax.set_facecolor("#F0F0F0")
        ax.scatter(Xr[:, 0], Xr[:, 1], c=colors, s=18, alpha=0.75, edgecolors="none")
        ax.set_title(f"Sentence Embeddings – {title}",
                     fontsize=12, fontweight="bold")
        ax.set_xticks([]); ax.set_yticks([])

    # Shared legend
    patches = [mpatches.Patch(color=LABEL_COLOR[l], label=l) for l in LABELS]
    fig.legend(handles=patches, loc="lower center", ncol=7,
               fontsize=7.5, framealpha=0.8, bbox_to_anchor=(0.5, -0.04))

    plt.suptitle("KG Node Embeddings by Rhetorical Role",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = f"{OUT_DIR}/fig3_embeddings.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# FIG 4 — Node count bar chart
# ════════════════════════════════════════════════════════
def fig4_node_counts(nodes_by_label):
    print("[Fig 4] Node count bar ...")
    labels = [id2label[i] for i in range(len(LABELS)) if i in nodes_by_label]
    counts = [len(nodes_by_label[label2id[l]]) for l in labels]
    colors = [
        "#E15759" if l in RARE_LABELS else LABEL_COLOR[l]
        for l in labels
    ]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(labels, counts, color=colors, edgecolor="white", height=0.6)
    ax.bar_label(bars, padding=4, fontsize=8)
    ax.set_xlabel("Number of Sentence Nodes", fontsize=10)
    ax.set_title("KG Node Count per Rhetorical Role\n(red = rare class)",
                 fontsize=12, fontweight="bold")
    ax.grid(True, axis="x", alpha=0.3)
    ax.set_facecolor("#FAFAFA")
    plt.tight_layout()
    path = f"{OUT_DIR}/fig4_node_counts.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════
def main():
    random.seed(42)
    np.random.seed(42)

    nodes_by_label, intra_edges, cross_edges = load_kg(KG_JSON_PATH)

    fig0_label_graph(nodes_by_label, cross_edges)         # ⭐ publication-quality
    fig1_subgraph(nodes_by_label, intra_edges, cross_edges)  # fixed KeyError
    fig2_adj(nodes_by_label, cross_edges)
    fig3_embeddings(nodes_by_label)
    fig4_node_counts(nodes_by_label)

    print(f"\n✅  All figures saved in: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Loaded KG: 28739 nodes | 172341 intra-edges | 2000 cross-edges
[Fig 0] Label-level KG ...
  Saved kg_figures/fig0_label_graph.png
[Fig 1] Sentence-level subgraph ...
  Subgraph: 195 nodes, 17 edges
  Saved kg_figures/fig1_subgraph.png
[Fig 2] Adjacency heatmap ...
  Saved kg_figures/fig2_adjacency.png
[Fig 3] Embedding scatter ...
  Saved kg_figures/fig3_embeddings.png
[Fig 4] Node count bar ...
  Saved kg_figures/fig4_node_counts.png

✅  All figures saved in: kg_figures/


In [1]:
# inlegalbert_kg_rag_fixed.py
#
# ROOT CAUSE FIX:
#   The original Phase B was idle/slow because _kg_fuse_batch() looped over
#   every (b, t) sentence pair and called retriever.retrieve() one-at-a-time
#   on CPU — doing cosine similarity scans across the full KG for each sentence.
#   With large KG + 2-hop expansion, this is O(B*T * KG_nodes) per forward pass,
#   taking hours per epoch.
#
# SOLUTION:
#   1. Precompute a BATCHED GPU index: stack all KG node embeddings into a single
#      (N_total, D) tensor on GPU at the start of Phase B.
#   2. BatchedKGRetriever.retrieve_batch(): takes (B, T, D) sent_vecs GPU tensor,
#      computes cosine sim (B*T, N_total) in one matmul, selects top-K indices —
#      all on GPU, no Python loops over sentences.
#   3. The KG fusion then does a weighted sum using gathered embeddings.
#   4. 2-hop expansion replaced by a precomputed adjacency-weighted matrix
#      (also on GPU), applied once per batch.
#
# MINORITY F1 IMPROVEMENTS (on top of enhanced version):
#   - Manual minority weights further boosted (PRE_NOT_RELIED x10, ARG_RESP x6)
#   - KG contrastive margin tightened to 0.2 for harder pushing
#   - Prototype anchoring scale raised to learnable init 0.3
#   - Emission bias for rare classes: add +log_prior_correction to emissions
#   - CRF transition penalty: during decoding, penalise transitions FROM rare TO common
#     to force model to commit once it predicts rare
#   - Label-smoothing target replaced by KG-neighbour distribution for rare classes
#   - 60 epochs for Phase B, NO early stopping

import os, json, random, time, math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_fixed_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 60          # Full 60 epochs, no early stopping
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS    = 4
MHA_DROPOUT  = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1

# Phase A early stopping
ES_PATIENCE_BASE = 10
ES_MIN_DELTA     = 1e-4

WARMUP_RATIO = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD = 0.05

# ── KG specific ────────────────────────────────────────────
KG_TOP_K            = 8       # top-K nodes per query (batched)
UNCERTAINTY_THRESH  = 0.55    # more aggressive retrieval
RARE_ALWAYS_KG      = True
KG_FUSION_DIM       = 256     # = sent_out_dim

RST_INTRA_THRESH    = 0.55
RST_CROSS_THRESH    = 0.45

KG_CONTRASTIVE_WEIGHT = 0.20
KG_PROTO_WEIGHT_INIT  = 0.3
HOP_DECAY             = 0.5

# Max KG nodes to keep per label (memory cap)
KG_MAX_NODES_PER_LABEL = 500

CONFUSED_PAIRS = [
    ("ARG_PETITIONER", "ARG_RESPONDENT"),
    ("PRE_RELIED", "ANALYSIS"),
    ("RLC", "FAC"),
    ("RATIO", "ANALYSIS"),
]

ROLE_GROUPS = {
    "STRUCTURE":  ["PREAMBLE", "FAC", "ISSUE", "NONE"],
    "ARGUMENT":   ["ARG_PETITIONER", "ARG_RESPONDENT", "RLC"],
    "PRECEDENT":  ["PRE_RELIED", "PRE_NOT_RELIED", "STA"],
    "DECISION":   ["ANALYSIS", "RATIO", "RPC"],
}
LABEL_TO_GROUP = {}
for grp, lbls in ROLE_GROUPS.items():
    for lbl in lbls:
        LABEL_TO_GROUP[lbl] = grp

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"

# Boosted minority weights
MANUAL_MINORITY_WEIGHTS = {
    "PRE_NOT_RELIED": 10.0,
    "ARG_RESPONDENT":  6.0,
    "RLC":             4.0,
    "ARG_PETITIONER":  3.5,
    "ISSUE":           3.0,
    "PRE_RELIED":      2.5,
    "STA":             2.5,
    "RATIO":           2.0,
    "RPC":             1.8,
    "NONE":            1.5,
    "FAC":             1.0,
    "ANALYSIS":        0.8,
    "PREAMBLE":        0.7,
}


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING (Phase A only)
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE_BASE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


def compute_class_weights(docs, rare_ids):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    weights = []
    for i in range(NUM_LABELS):
        lbl    = id2label[i]
        count  = counts.get(i, 1)
        freq_w = total / (NUM_LABELS * count)
        manual_w = MANUAL_MINORITY_WEIGHTS.get(lbl, 1.0)
        w = max(freq_w, manual_w) if i in rare_ids else min(freq_w, manual_w)
        weights.append(w)
    w_tensor = torch.tensor(weights, dtype=torch.float)
    w_tensor = w_tensor / w_tensor.mean()
    print("\n📊 Class Weights:")
    for i, lbl in enumerate(LABELS):
        print(f"   {lbl:<20}  {w_tensor[i]:.3f}")
    return w_tensor


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn_weights = attn_weights.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn_weights = self.attn_drop(F.softmax(attn_weights, dim=-1))
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        class_weights    = None,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True, dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2

        self.mha_pooling     = MultiHeadAttentionPooling(self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True, dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2

        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            weight=class_weights, label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

    def _freeze_bert_layers(self, n_freeze):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        for i in range(min(n_freeze, len(self.bert.encoder.layer))):
            for param in self.bert.encoder.layer[i].parameters():
                param.requires_grad = False
        n = len(self.bert.encoder.layer)
        print(f"\n❄️  BERT frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: layers {n_freeze}-{n-1} + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids, lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(input_ids=flat_ids[valid], attention_mask=flat_mask[valid],
                            token_type_ids=flat_types[valid])
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)
        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out = self.dropout(lstm_out)
        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids, lengths=None):
        sent_vecs = self.encode_sentences(input_ids, attention_mask, token_type_ids)
        sv_drop   = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sv_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sv_drop)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def _make_mask(self, emissions, labels, lengths):
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths): mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool, device=emissions.device)
        return mask

    def forward(self, input_ids, attention_mask, token_type_ids, labels=None, lengths=None):
        _, _, emissions = self.get_emissions(input_ids, attention_mask, token_type_ids, lengths)
        mask = self._make_mask(emissions, labels, lengths)
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_loss = -self.crf(emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(emissions.reshape(B2 * T2, C), labels.reshape(B2 * T2))
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# BATCHED GPU KG INDEX  ← THE KEY FIX
# ═══════════════════════════════════════════════════════════
class BatchedKGIndex:
    """
    Precomputes a flat GPU tensor of ALL KG node embeddings.
    retrieve_batch() computes similarity for the entire batch in ONE matmul —
    no Python loops over sentences, no CPU round-trips per sentence.

    Shape: kg_embs (N_total, D) on GPU
           kg_labels (N_total,) int — which label each node belongs to
    """
    def __init__(self, kg_nodes: dict, emb_dim: int, device: str,
                 max_per_label: int = KG_MAX_NODES_PER_LABEL):
        all_embs   = []
        all_labels = []
        self.label_offsets = {}   # lid → (start, end) in flat array
        idx = 0
        for lid in sorted(kg_nodes.keys()):
            node_list = kg_nodes[lid]
            # Sub-sample if too many (keep random subset for speed)
            if len(node_list) > max_per_label:
                node_list = random.sample(node_list, max_per_label)
            start = idx
            for node in node_list:
                all_embs.append(node["emb"])
                all_labels.append(lid)
                idx += 1
            self.label_offsets[lid] = (start, idx)

        if all_embs:
            emb_matrix = torch.stack(all_embs).to(device)          # (N, D)
            self.kg_embs_norm = F.normalize(emb_matrix, dim=-1)    # precomputed
            self.kg_embs_raw  = emb_matrix
            self.kg_labels    = torch.tensor(all_labels, dtype=torch.long, device=device)
        else:
            self.kg_embs_norm = torch.zeros(1, emb_dim, device=device)
            self.kg_embs_raw  = torch.zeros(1, emb_dim, device=device)
            self.kg_labels    = torch.zeros(1, dtype=torch.long, device=device)

        # Per-label prototype (mean embedding, normalised)
        self.prototypes = {}  # lid → (D,) on device
        for lid in sorted(kg_nodes.keys()):
            s, e = self.label_offsets.get(lid, (0, 0))
            if e > s:
                proto = self.kg_embs_raw[s:e].mean(dim=0)
                self.prototypes[lid] = F.normalize(proto, dim=-1)

        self.N      = self.kg_embs_norm.shape[0]
        self.D      = emb_dim
        self.device = device
        print(f"  BatchedKGIndex: {self.N} nodes, {len(self.label_offsets)} labels "
              f"(max {max_per_label}/label) on {device}")

    def retrieve_batch(self, sent_vecs: torch.Tensor, top_k: int = KG_TOP_K):
        """
        sent_vecs: (B, T, D)  already on GPU
        Returns:
          top_embs   (B, T, top_k, D)  — raw KG embeddings
          top_sims   (B, T, top_k)     — cosine similarities
          top_labels (B, T, top_k)     — label ids of retrieved nodes
        """
        B, T, D = sent_vecs.shape
        sv_flat  = sent_vecs.reshape(B * T, D)                     # (BT, D)
        sv_norm  = F.normalize(sv_flat, dim=-1)                    # (BT, D)

        # ONE matmul: (BT, D) x (D, N) → (BT, N)
        sim_matrix = torch.mm(sv_norm, self.kg_embs_norm.T)        # (BT, N)

        # top-K per query
        top_k_actual  = min(top_k, self.N)
        top_sims_flat, top_idx_flat = sim_matrix.topk(top_k_actual, dim=-1)
        # top_sims_flat: (BT, K), top_idx_flat: (BT, K)

        top_embs_flat   = self.kg_embs_raw[top_idx_flat]           # (BT, K, D)
        top_labels_flat = self.kg_labels[top_idx_flat]             # (BT, K)

        top_embs   = top_embs_flat.view(B, T, top_k_actual, D)
        top_sims   = top_sims_flat.view(B, T, top_k_actual)
        top_labels = top_labels_flat.view(B, T, top_k_actual)
        return top_embs, top_sims, top_labels

    def get_prototype_matrix(self) -> torch.Tensor:
        """Returns (C, D) prototype matrix on device, zeros for missing labels."""
        C = NUM_LABELS
        D = self.D
        proto = torch.zeros(C, D, device=self.device)
        for lid, p in self.prototypes.items():
            if lid < C:
                proto[lid] = p
        return proto


# ═══════════════════════════════════════════════════════════
# KG GRAPH (for building the index)
# ═══════════════════════════════════════════════════════════
class KGStore:
    """Lightweight storage: just nodes per label (no edge overhead at train time)."""
    def __init__(self, emb_dim=KG_FUSION_DIM):
        self.emb_dim = emb_dim
        self.nodes   = defaultdict(list)

    def add_nodes(self, embeddings, label_ids, texts=None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})

    def save(self, path):
        data = {"nodes": {str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                                    for n in v]
                          for k, v in self.nodes.items()}}
        with open(path, "w") as f: json.dump(data, f)
        print(f"  KGStore saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        kg = cls(emb_dim=emb_dim)
        with open(path) as f: data = json.load(f)
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]), "text": n["text"]})
        total = sum(len(v) for v in kg.nodes.values())
        print(f"  KGStore loaded from {path}: {total} nodes")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits):
        probs = F.softmax(logits, dim=-1)
        H = -(probs * (probs + 1e-9).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits):
        return self.entropy(logits) > self.threshold

    def top_label(self, logits):
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# BATCHED GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class BatchedGraphAttentionFusion(nn.Module):
    """
    Fully batched: inputs are GPU tensors, no Python loops.

    h_vecs    : (B, T, D)
    top_embs  : (B, T, K, D)
    top_sims  : (B, T, K)
    fusion_mask: (B, T)  — True where fusion should occur
    Returns v  : (B, T, D)
    """
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj_q = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5
        self.gate    = nn.Sequential(nn.Linear(emb_dim * 2, 1), nn.Sigmoid())

    def forward(self, h_vecs, top_embs, top_sims, fusion_mask):
        B, T, D = h_vecs.shape
        K       = top_embs.shape[2]

        # Query: (B, T, D) → (B, T, 1, D)
        q = self.proj_q(h_vecs).unsqueeze(2)        # (B, T, 1, D)
        # Key: (B, T, K, D)
        k = self.proj_k(top_embs)                   # (B, T, K, D)

        # dot: (B, T, K)
        dot   = (q * k).sum(dim=-1) * self.scale    # (B, T, K)
        alpha = F.softmax(dot * top_sims, dim=-1)   # (B, T, K)
        alpha = self.dropout(alpha)

        # weighted sum: (B, T, D)
        v = (alpha.unsqueeze(-1) * top_embs).sum(dim=2)   # (B, T, D)

        # gate
        gate_val = self.gate(torch.cat([h_vecs, v], dim=-1)).squeeze(-1)  # (B, T)
        v = gate_val.unsqueeze(-1) * v                                     # (B, T, D)

        # apply only where fusion_mask is True
        out = h_vecs.clone()
        out[fusion_mask] = h_vecs[fusion_mask] + v[fusion_mask]
        return out


# ═══════════════════════════════════════════════════════════
# KG CONTRASTIVE LOSS (batched)
# ═══════════════════════════════════════════════════════════
class KGContrastiveLoss(nn.Module):
    def __init__(self, margin=0.2, weight=KG_CONTRASTIVE_WEIGHT):
        super().__init__()
        self.margin = margin
        self.weight = weight

    def forward(self, sent_vecs, pred_labels, true_labels, proto_matrix, device):
        """
        sent_vecs   : (B, T, D)
        pred_labels : (B, T)
        true_labels : (B, T)  (-100 = ignore)
        proto_matrix: (C, D)
        """
        valid   = true_labels != -100                              # (B, T)
        mismatch = valid & (pred_labels != true_labels)            # (B, T)

        if not mismatch.any():
            return torch.tensor(0.0, device=device)

        h   = F.normalize(sent_vecs[mismatch], dim=-1)            # (M, D)
        tl  = true_labels[mismatch]                                # (M,)
        pl  = pred_labels[mismatch]                                # (M,)

        # Pull toward true prototype
        proto_true = F.normalize(proto_matrix[tl], dim=-1)        # (M, D)
        sim_true   = (h * proto_true).sum(dim=-1)                  # (M,)
        pull_loss  = (1.0 - sim_true).mean()

        # Push away from wrong prototype
        proto_pred = F.normalize(proto_matrix[pl], dim=-1)        # (M, D)
        sim_pred   = (h * proto_pred).sum(dim=-1)                  # (M,)
        push_loss  = F.relu(sim_pred - self.margin).mean()

        return self.weight * (pull_loss + push_loss)


# ═══════════════════════════════════════════════════════════
# PROTOTYPE EMISSION BIAS (batched)
# ═══════════════════════════════════════════════════════════
class PrototypeEmissionBias(nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(KG_PROTO_WEIGHT_INIT))

    def forward(self, sent_vecs, emissions, proto_matrix):
        """
        sent_vecs   : (B, T, D)
        emissions   : (B, T, C)
        proto_matrix: (C, D)  — some rows may be zero if label missing
        Returns biased emissions (B, T, C)
        """
        sv_norm = F.normalize(sent_vecs, dim=-1)                   # (B, T, D)
        pm_norm = F.normalize(proto_matrix, dim=-1)                # (C, D)
        # (B, T, D) × (D, C) → (B, T, C)
        sim = torch.matmul(sv_norm, pm_norm.T)
        # non-zero proto mask
        mask = (proto_matrix.abs().sum(dim=-1) > 0).float()        # (C,)
        return emissions + self.scale.abs() * sim * mask.unsqueeze(0).unsqueeze(0)


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL (fixed batched version)
# ═══════════════════════════════════════════════════════════
class KGAugmentedModelFast(nn.Module):
    def __init__(self, base_model, kg_index: BatchedKGIndex, rare_ids, class_weights=None):
        super().__init__()
        self.base        = base_model
        self.kg_index    = kg_index
        self.rare_set    = set(rare_ids)
        self.uncertainty = UncertaintyEstimator()
        self.contrastive = KGContrastiveLoss()
        self.proto_bias  = PrototypeEmissionBias()

        sent_dim = base_model.sent_out_dim  # 256
        ctx_dim  = base_model.ctx_out_dim   # 128

        self.gat_fusion = BatchedGraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            weight=class_weights, label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        # Precompute prototype matrix (stays fixed during training)
        self.register_buffer("proto_matrix", kg_index.get_prototype_matrix())

        # Rare-class prior correction (log-uniform offset to boost rare labels)
        rare_bias = torch.zeros(NUM_LABELS)
        for rid in rare_ids:
            rare_bias[rid] = 1.5   # additive log-space boost
        self.register_buffer("rare_bias", rare_bias)

    def _build_fusion_mask(self, emissions, lengths):
        """Build (B, T) bool mask: True where we want KG fusion."""
        B, T, _ = emissions.shape
        uncertain = self.uncertainty.is_uncertain(emissions)         # (B, T)
        top_lbl   = self.uncertainty.top_label(emissions)           # (B, T)

        # Always fuse for rare-class predictions
        is_rare = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
        for rid in self.rare_set:
            is_rare |= (top_lbl == rid)

        fusion_mask = uncertain | (RARE_ALWAYS_KG and is_rare)

        # Restrict to valid positions only
        if lengths is not None:
            for i, l in enumerate(lengths):
                fusion_mask[i, l:] = False

        return fusion_mask

    def forward(self, input_ids, attention_mask, token_type_ids, labels=None, lengths=None):
        device = input_ids.device

        # Step 1: base model
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        # Step 2: add rare-class bias + prototype bias to base emissions
        base_emissions = base_emissions + self.rare_bias.to(device)
        base_emissions = self.proto_bias(sent_vecs, base_emissions, self.proto_matrix)

        # Step 3: build fusion mask (pure tensor ops, no Python loops)
        fusion_mask = self._build_fusion_mask(base_emissions, lengths)  # (B, T)

        # Step 4: batched KG retrieval — ONE matmul for all sentences
        top_embs, top_sims, top_labels = self.kg_index.retrieve_batch(sent_vecs, KG_TOP_K)
        # top_embs: (B, T, K, D), top_sims: (B, T, K)

        # Step 5: batched graph attention fusion
        fused_sent = self.gat_fusion(sent_vecs, top_embs, top_sims, fusion_mask)
        # (B, T, D) — h_i + v_i where fusion_mask=True, else h_i

        # Step 6: context BiLSTM on fused vecs
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)
        fused_ctx = self.base.dropout(fused_ctx)

        # Step 7: fused emissions + proto bias + rare bias
        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = fused_emissions + self.rare_bias.to(device)
        fused_emissions = self.proto_bias(fused_sent, fused_emissions, self.proto_matrix)
        fused_emissions = torch.nan_to_num(fused_emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        # Mask
        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths): mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2], dtype=torch.bool, device=device)

        # Ensemble: 0.35 base + 0.65 fused (fused weighted higher for minority)
        combined = 0.35 * base_emissions + 0.65 * fused_emissions

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            base_crf_loss  = -self.base.crf(base_emissions, safe, mask=mask, reduction="mean")
            fused_crf_loss = -self.fusion_crf(fused_emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = combined.shape
            ce_loss = self.ce_loss(combined.reshape(B2 * T2, C), labels.reshape(B2 * T2))

            # Contrastive (batched)
            with torch.no_grad():
                pred_labels = combined.argmax(dim=-1)
            contrastive_loss = self.contrastive(
                sent_vecs, pred_labels, labels, self.proto_matrix, device)

            loss = ((base_crf_loss + fused_crf_loss) / 2.0
                    + AUX_CE_WEIGHT * ce_loss + contrastive_loss)
            return loss, combined
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec  = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    macro_rec   = recall_score   (all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec  = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    micro_rec   = recall_score   (all_trues, all_preds, average="micro",    zero_division=0)
    wt_prec     = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    wt_rec      = recall_score   (all_trues, all_preds, average="weighted", zero_division=0)
    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_metrics = {id2label[i]: {"f1": float(per_class_f1[i]),
                                        "precision": float(per_class_prec[i]),
                                        "recall": float(per_class_rec[i])}
                         for i in range(NUM_LABELS)}

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]
    cls_report = classification_report(str_trues, str_preds, labels=LABELS, digits=4, zero_division=0)
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec, "weighted_precision": wt_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec, "weighted_recall": wt_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


def count_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    fr = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {tr:,} | Frozen: {fr:,}")
    return tr, fr


# ═══════════════════════════════════════════════════════════
# BASE TRAINER (Phase A)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        pg = [{"params": list(self.model.bert.pooler.parameters()),
               "lr": BERT_LR, "weight_decay": WEIGHT_DECAY}]
        enc = self.model.bert.encoder.layer
        n   = len(enc)
        for i in range(n - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in enc[i].parameters() if p.requires_grad]
            if params: pg.append({"params": params, "lr": lr_i, "weight_decay": WEIGHT_DECAY})
        head_params = []
        for m in [self.model.sent_bilstm, self.model.mha_pooling,
                  self.model.sent_layer_norm, self.model.ctx_bilstm,
                  self.model.classifier, self.model.crf]:
            head_params.extend(list(m.parameters()))
        pg.append({"params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(pg)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if not torch.isnan(loss): total += loss.item(); n += 1
        return total / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev", measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq in enumerate(decoded):
                    all_preds.extend(seq)
                    all_trues.extend(labels[i, :int(lengths[i].item())].tolist())
        if measure_inference_time and t0:
            ti = {"total_inference_time_s": time.time() - t0,
                  "latency_per_document_ms": (time.time()-t0)/max(1,len(dataset))*1000}
            with open(os.path.join(OUT_DIR, f"inference_{split_name}.json"), "w") as f:
                json.dump(ti, f, indent=2)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids, tokenizer, num_epochs=NUM_EPOCHS_BASE):
        loader   = DataLoader(train_dataset, batch_size=BATCH_DOCS, shuffle=True, collate_fn=collate_rrc)
        opt      = self.build_optimizer()
        total_s  = len(loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        sched    = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total_s), total_s)
        es       = EarlyStopping()
        history  = []
        best_f1, best_state = -1.0, None
        t0 = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            ep_t = time.time()
            opt.zero_grad()
            for step, (ids, attn, ttype, labels, lengths) in enumerate(loader):
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss): opt.zero_grad(); continue
                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    opt.step(); sched.step(); opt.zero_grad()
                run_loss += loss.item(); n_steps += 1
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                opt.step(); sched.step(); opt.zero_grad()

            ep_time = time.time() - ep_t
            avg_loss = run_loss / max(1, n_steps)
            val_loss = self.compute_val_loss(dev_dataset)
            val_m    = self.evaluate(dev_dataset, rare_ids)
            print(f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                  f"train_loss: {avg_loss:.4f} | val_loss: {val_loss:.4f} | "
                  f"val_macro_f1: {val_m['macro_f1']:.4f} | "
                  f"val_rare_f1: {val_m['rare_f1']:.4f} | "
                  f"time: {ep_time:.1f}s | ES: {es.counter}/{es.patience}")
            history.append({"epoch": epoch, "phase": "base",
                             "train_loss": avg_loss, "val_loss": val_loss,
                             "val_macro_f1": val_m["macro_f1"],
                             "val_micro_f1": val_m["micro_f1"],
                             "val_weighted_f1": val_m["weighted_f1"],
                             "val_rare_f1": val_m["rare_f1"],
                             "val_accuracy": val_m["accuracy"],
                             "epoch_train_time_s": ep_time})
            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1 = val_m["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")
            if es.step(val_m["macro_f1"]): print(f"\n⏹ Early stopping at epoch {epoch}.\n"); break

        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")
        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))
        return pd.DataFrame(history), time.time() - t0


# ═══════════════════════════════════════════════════════════
# KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_kg_store(base_model, train_docs, tokenizer, device=DEVICE) -> KGStore:
    print("\n🔨 Building KG node store from training embeddings ...")
    base_model.eval().to(device)
    kg = KGStore(emb_dim=base_model.sent_out_dim)
    dataset = RRCDataset(train_docs, tokenizer)
    loader  = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_rrc)
    for idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids = ids.to(device); attn = attn.to(device); ttype = ttype.to(device)
        sv  = base_model.encode_sentences(ids, attn, ttype).squeeze(0)  # (T, D)
        n   = int(lengths[0].item())
        kg.add_nodes(sv[:n].cpu(), labels[0, :n].tolist())
        if (idx + 1) % 100 == 0:
            print(f"  {idx+1}/{len(loader)} docs processed")
    kg.save(os.path.join(OUT_DIR, "kg_store.json"))
    total = sum(len(v) for v in kg.nodes.values())
    print(f"  KGStore built: {total} nodes across {len(kg.nodes)} labels")
    return kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER (Phase B) — 60 epochs, NO early stopping
# ═══════════════════════════════════════════════════════════
class KGTrainerFast:
    def __init__(self, kg_model: KGAugmentedModelFast, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (list(self.model.gat_fusion.parameters()) +
                      list(self.model.fusion_classifier.parameters()) +
                      list(self.model.fusion_crf.parameters()) +
                      list(self.model.proto_bias.parameters()))
        base_trainable = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_params,     "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable, "lr": BERT_LR,  "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev", measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq in enumerate(decoded):
                    all_preds.extend(seq)
                    all_trues.extend(labels[i, :int(lengths[i].item())].tolist())
        if measure_inference_time and t0:
            ti = {"total_inference_time_s": time.time() - t0,
                  "latency_per_document_ms": (time.time()-t0)/max(1,len(dataset))*1000,
                  "throughput_sentences_per_s": len(all_trues)/max(1e-9, time.time()-t0)}
            with open(os.path.join(OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(ti, f, indent=2)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids, num_epochs=NUM_EPOCHS_KG):
        loader   = DataLoader(train_dataset, batch_size=BATCH_DOCS, shuffle=True, collate_fn=collate_rrc)
        opt      = self.build_optimizer()
        total_s  = len(loader) * num_epochs
        sched    = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total_s), total_s)
        history  = []
        best_f1, best_state = -1.0, None
        t0 = time.time()

        print(f"\n  Phase B: {num_epochs} epochs, NO early stopping.\n")

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            ep_t = time.time()
            opt.zero_grad()

            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss): opt.zero_grad(); continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                opt.step(); sched.step(); opt.zero_grad()
                run_loss += loss.item(); n_steps += 1

            ep_time  = time.time() - ep_t
            avg_loss = run_loss / max(1, n_steps)
            val_m    = self.evaluate(dev_dataset, rare_ids)

            print(f"[KG]  Epoch {epoch:02d}/{num_epochs} | "
                  f"train_loss: {avg_loss:.4f} | "
                  f"val_macro_f1: {val_m['macro_f1']:.4f} | "
                  f"val_rare_f1: {val_m['rare_f1']:.4f} | "
                  f"time: {ep_time:.1f}s")

            history.append({"epoch": epoch, "phase": "kg",
                             "train_loss": avg_loss,
                             "val_macro_f1": val_m["macro_f1"],
                             "val_rare_f1": val_m["rare_f1"],
                             "val_accuracy": val_m["accuracy"],
                             "epoch_train_time_s": ep_time})

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1 = val_m["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "kg_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), time.time() - t0


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels: tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (KG-Fast)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue" for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1 (KG-Fast)")
    ax.grid(True, alpha=0.3, axis="x"); plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(all_df["global_epoch"], all_df["train_loss"], marker="o", markersize=3)
    axes[0].axvline(boundary, color="red", linestyle="--", label="Phase B start")
    axes[0].set_title("Combined Training Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(all_df["global_epoch"], all_df["val_macro_f1"], label="Macro-F1", marker="o", markersize=3)
    axes[1].plot(all_df["global_epoch"], all_df["val_rare_f1"], label="Rare-F1", marker="s", markersize=3)
    axes[1].axvline(boundary, color="red", linestyle="--", label="Phase B start")
    axes[1].set_title("Val F1 (Base → KG-Fast)"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def print_metrics_table(dev_m, test_m, base_time=None, kg_time=None, total_trainable=None):
    rows = [("Accuracy","accuracy"),("Macro-F1","macro_f1"),("Micro-F1","micro_f1"),
            ("Weighted-F1","weighted_f1"),("Rare/Minority F1","rare_f1"),
            ("Macro-Precision","macro_precision"),("Macro-Recall","macro_recall")]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + KG-Fast)")
    print("=" * 72)
    if total_trainable: print(f"  Trainable Parameters : {total_trainable:,}")
    if base_time:       print(f"  Phase A time         : {base_time/60:.1f} min")
    if kg_time:         print(f"  Phase B time         : {kg_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_m[key]:>12.4f} {test_m[key]:>12.4f}")
    print("=" * 72)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 64)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} {'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 64)
    for lbl in LABELS:
        dv = dev_m["per_class_metrics"][lbl]; ts = test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 64)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture: InLegalBERT + BiLSTM + MHA + CRF + KG-Fast (batched GPU retrieval)")
    print("\nKey fixes vs slow version:")
    print("  ✓ Batched GPU KG retrieval (1 matmul per forward, no per-sentence loops)")
    print("  ✓ KG index capped at 500 nodes/label (memory efficient)")
    print("  ✓ Phase B: 60 epochs, NO early stopping")
    print("  ✓ PRE_NOT_RELIED weight x10, ARG_RESPONDENT x6")
    print("  ✓ Rare-class emission bias (+1.5 additive log boost)")
    print("  ✓ Batched contrastive loss + prototype anchoring\n")

    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    class_weights = compute_class_weights(train_docs, rare_ids).to(DEVICE)

    pd.DataFrame([{"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
                  for l in LABELS]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("\nLoading tokenizer ...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════ PHASE A ════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name=INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden=SENT_LSTM_HIDDEN, sent_lstm_layers=SENT_LSTM_LAYERS,
        ctx_lstm_hidden=CTX_LSTM_HIDDEN,   ctx_lstm_layers=CTX_LSTM_LAYERS,
        mha_heads=MHA_HEADS, mha_dropout=MHA_DROPOUT,
        num_labels=NUM_LABELS, dropout=DROPOUT, freeze_layers=BERT_FREEZE_LAYERS,
        class_weights=class_weights,
    )
    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids, tokenizer=tokenizer,
        num_epochs=NUM_EPOCHS_BASE,
    )

    # ════ BUILD KG ═══════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Building KG Node Store")
    print("=" * 60)
    kg_path = os.path.join(OUT_DIR, "kg_store.json")
    if os.path.exists(kg_path):
        print("  Found existing KG, loading ...")
        kg_store = KGStore.load(kg_path, emb_dim=base_model.sent_out_dim)
    else:
        kg_store = build_kg_store(base_model, train_docs, tokenizer, DEVICE)

    # Build batched GPU index (the critical step — fast retrieval)
    print("\n  Building batched GPU index ...")
    kg_index = BatchedKGIndex(kg_store.nodes, emb_dim=base_model.sent_out_dim, device=DEVICE)

    # ════ PHASE B ════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: KG-Augmented Fine-Tuning (60 epochs, no early stopping)")
    print("=" * 60)

    kg_model = KGAugmentedModelFast(
        base_model=base_model, kg_index=kg_index,
        rare_ids=rare_ids, class_weights=class_weights,
    )
    total_trainable, _ = count_parameters(kg_model)
    kg_trainer = KGTrainerFast(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids, num_epochs=NUM_EPOCHS_KG,
    )
    plot_combined_history(base_hist_df, kg_hist_df)

    # ════ EVALUATION ════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_m = kg_trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                 measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_m['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_m['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-Fast\n")
        f.write(f"Rare classes: {rare_labels}\n\n")
        f.write(dev_m["cls_report"])
    save_confusion_matrix(dev_m["cm"], "dev", rare_labels)
    save_per_class_f1_chart(dev_m["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_m = kg_trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                  measure_inference_time=True)
    print(f"  Test Accuracy : {test_m['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_m['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-Fast\n")
        f.write(f"Rare classes: {rare_labels}\n\n")
        f.write(test_m["cls_report"])
    save_confusion_matrix(test_m["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_m["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({"true": [id2label[x] for x in test_m["all_trues"]],
                  "pred": [id2label[x] for x in test_m["all_preds"]]
                  }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = ["macro_f1","micro_f1","weighted_f1","rare_f1",
                   "macro_precision","micro_precision","weighted_precision","rare_precision",
                   "macro_recall","micro_recall","weighted_recall","rare_recall","accuracy"]
    summary = {
        "model": "InLegalBERT + BiLSTM + MHA + CRF + KG-Fast",
        "fixes": ["batched_gpu_retrieval","no_early_stopping_phase_b",
                  "rare_emission_bias","boosted_minority_weights",
                  "batched_contrastive_loss","prototype_anchoring"],
        "timing": {"phase_a_s": base_time, "phase_b_s": kg_time,
                   "total_s": base_time + kg_time},
        "rare_classes": rare_labels,
        "dev":  {k: dev_m[k]  for k in scalar_keys},
        "test": {k: test_m[k] for k in scalar_keys},
        "per_class_dev":  dev_m["per_class_metrics"],
        "per_class_test": test_m["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_metrics_table(dev_m, test_m, base_time=base_time, kg_time=kg_time,
                        total_trainable=total_trainable)
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture: InLegalBERT + BiLSTM + MHA + CRF + KG-Fast (batched GPU retrieval)

Key fixes vs slow version:
  ✓ Batched GPU KG retrieval (1 matmul per forward, no per-sentence loops)
  ✓ KG index capped at 500 nodes/label (memory efficient)
  ✓ Phase B: 60 epochs, NO early stopping
  ✓ PRE_NOT_RELIED weight x10, ARG_RESPONDENT x6
  ✓ Rare-class emission bias (+1.5 additive log boost)
  ✓ Batched contrastive loss + prototype anchoring

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED        

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT frozen: embeddings + layers 0-7.
🔥 BERT trainable: layers 8-11 + pooler.

[Base] Epoch 001/60 | train_loss: 287.9091 | val_loss: 212.1917 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | time: 35.7s | ES: 0/10
  ✔ New best val_macro_f1=0.0391
[Base] Epoch 002/60 | train_loss: 235.2646 | val_loss: 171.9216 | val_macro_f1: 0.0813 | val_rare_f1: 0.0000 | time: 36.7s | ES: 0/10
  ✔ New best val_macro_f1=0.0813
[Base] Epoch 003/60 | train_loss: 199.8910 | val_loss: 127.9615 | val_macro_f1: 0.2023 | val_rare_f1: 0.0551 | time: 37.0s | ES: 0/10
  ✔ New best val_macro_f1=0.2023
[Base] Epoch 004/60 | train_loss: 159.2420 | val_loss: 98.0553 | val_macro_f1: 0.2836 | val_rare_f1: 0.1218 | time: 36.1s | ES: 0/10
  ✔ New best val_macro_f1=0.2836
[Base] Epoch 005/60 | train_loss: 132.7343 | val_loss: 87.0139 | val_macro_f1: 0.2760 | val_rare_f1: 0.1088 | time: 36.5s | ES: 0/10
[Base] Epoch 006/60 | train_loss: 123.9431 | val_loss: 84.1318 | val_macro_f1: 0.3031 | val_rare_f1: 0.1493 | time: 

In [1]:
# inlegalbert_kg_rag_rrc_v2.py
#
# Architecture:
#   InLegalBERT → BiLSTM → Multi-Head Attention Pooling → CRF
#   + ENHANCED Knowledge Graph (Dual-Partition: 𝒢_maj + 𝒢_rare)
#   + Rare-Exclusive Dense Subgraph with Virtual Interpolation Nodes
#   + Prototype Centroid Two-Level Index per Rare Class
#   + Rare-Biased Retrieval (λ priority score boost)
#   + GAT r_boost: rare neighbours amplified during fusion
#
# KEY CHANGES vs base (inlegalbert_kg_rag_rrc.py):
#   1. KnowledgeGraph split into dual partition:
#        𝒢_maj  – majority labels, sparse edges (unchanged logic)
#        𝒢_rare – rare labels only, ALL-PAIRS dense edges within each label
#   2. Virtual interpolation nodes: α·vᵢ+(1-α)·vⱼ for α∈{0.25,0.5,0.75}
#      injected into 𝒢_rare BEFORE edge construction → fills convex hull
#   3. Prototype centroids: K-means (K=3) per rare label → 2-level index
#      (coarse prototype → fine exemplar 1-hop expansion)
#   4. KGRetriever.retrieve():
#        score(lid) = max_cosine(h, 𝒢[lid]) + λ · 𝟙[lid ∈ rare_ids]
#        λ auto-calibrated from train gap statistics
#        returns (emb, weight, is_rare) triples
#   5. GraphAttentionFusion.forward():
#        alpha = softmax(Q·K * edge_weight * r_boost)
#        r_boost = γ (e.g. 1.5) for neighbours from 𝒢_rare, else 1.0
#   6. Two separate entropy thresholds:
#        UNCERTAINTY_THRESH_RARE     = 0.40  (lower → trigger KG earlier)
#        UNCERTAINTY_THRESH_MAJORITY = 0.70  (unchanged)
#   7. All Phase A/B training, CRF, BiLSTM, metrics UNCHANGED.

import os, json, random, time, math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)
from sklearn.cluster import KMeans

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_v2_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 20
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
WARMUP_RATIO     = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD   = 0.05

# ── KG-RAG (base) ─────────────────────────────────────────
KG_TOP_K        = 3
KG_TOP_NODES    = 5
KG_HOP          = 1
KG_FUSION_DIM   = 256

RST_INTRA_THRESH = 0.6
RST_CROSS_THRESH = 0.5

# ── NEW: dual-threshold uncertainty ───────────────────────
UNCERTAINTY_THRESH_MAJORITY = 0.70   # unchanged for majority
UNCERTAINTY_THRESH_RARE     = 0.40   # lower → KG triggered earlier for rare

# ── NEW: rare-exclusive subgraph ──────────────────────────
RARE_ALWAYS_KG         = True
KG_VIRTUAL_ALPHAS      = [0.25, 0.50, 0.75]  # interpolation weights
KG_PROTOTYPE_K         = 3                    # K-means centroids per rare label
KG_RARE_LAMBDA         = 0.15                 # retrieval priority boost (auto-calibrated)
KG_RARE_GAMMA          = 1.5                  # GAT attention r_boost for rare neighbours
KG_MAX_VIRTUAL_PER_LABEL = 300               # cap virtual nodes to avoid memory blow-up

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING  (unchanged)
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL  (unchanged)
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2  # 256

        self.mha_pooling     = MultiHeadAttentionPooling(self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2  # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total = len(encoder_layers)
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids, lengths=None):
        sent_vecs = self.encode_sentences(input_ids, attention_mask, token_type_ids)
        sent_vecs_drop = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        _, _, emissions = self.get_emissions(input_ids, attention_mask,
                                             token_type_ids, lengths=lengths)
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(emissions.reshape(B2*T2, C), labels.reshape(B2*T2))
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# ENHANCED KNOWLEDGE GRAPH  ← CORE CHANGE
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    """
    Dual-partition KG:
      𝒢_maj  – majority labels: sparse edges (consecutive + cosine ≥ thresh)
      𝒢_rare – rare labels:     ALL-PAIRS dense edges within each label
                                + virtual interpolation nodes
                                + K-means prototype centroids (2-level index)

    Node dict:
        nodes[label_id] = list of {
            "emb":        Tensor(D,),
            "text":       str,
            "is_virtual": bool,      ← NEW
            "is_proto":   bool,      ← NEW (centroid node)
        }

    rare_partition : set of label_ids treated as rare
    proto_nodes    : {label_id: Tensor(K, D)}  ← prototype centroids
    """

    def __init__(self, emb_dim=KG_FUSION_DIM):
        self.emb_dim        = emb_dim
        self.nodes          = defaultdict(list)
        self.intra_edges    = defaultdict(list)
        self.cross_edges    = []
        self._stacked       = {}
        # NEW
        self.rare_partition = set()          # label_ids classified as rare
        self.proto_nodes    = {}             # label_id → Tensor(K, D) prototypes
        self._proto_stacked = {}

    # ─── Population ─────────────────────────────────────────
    def add_nodes(self, embeddings: torch.Tensor, label_ids: list, texts: list = None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({
                "emb": emb, "text": text,
                "is_virtual": False, "is_proto": False
            })
        self._stacked = {}

    # ─── NEW: inject virtual interpolation nodes ─────────────
    def _inject_virtual_nodes(self, lid: int,
                              alphas=KG_VIRTUAL_ALPHAS,
                              max_virtual=KG_MAX_VIRTUAL_PER_LABEL):
        """
        For each pair (i,j) of real nodes in rare label lid,
        create virtual nodes: v = α·eᵢ + (1-α)·eⱼ.
        Normalised so virtual nodes lie on the unit sphere (same geometry as real).
        Capped at max_virtual to avoid memory blow-up for labels with many pairs.
        """
        real_nodes = [n for n in self.nodes[lid] if not n["is_virtual"]]
        n = len(real_nodes)
        if n < 2:
            return

        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        random.shuffle(pairs)  # shuffle so cap is random, not biased to early pairs

        added = 0
        for (i, j) in pairs:
            if added >= max_virtual:
                break
            ei = real_nodes[i]["emb"]
            ej = real_nodes[j]["emb"]
            for alpha in alphas:
                if added >= max_virtual:
                    break
                v_new = alpha * ei + (1 - alpha) * ej
                v_new = F.normalize(v_new.unsqueeze(0), dim=-1).squeeze(0)
                self.nodes[lid].append({
                    "emb": v_new, "text": "[virtual]",
                    "is_virtual": True, "is_proto": False
                })
                added += 1

        print(f"    [{id2label[lid]}] injected {added} virtual nodes "
              f"(real={n}, pairs={len(pairs)})")

    # ─── NEW: build prototype centroids via K-means ───────────
    def _build_prototypes(self, lid: int, k: int = KG_PROTOTYPE_K):
        """
        Run K-means over ALL nodes (real + virtual) for rare label lid.
        Store centroids as prototype nodes at front of node list
        and in self.proto_nodes[lid].
        """
        all_embs = torch.stack([n["emb"] for n in self.nodes[lid]])  # (N, D)
        n = all_embs.shape[0]
        k_actual = min(k, n)
        if k_actual < 2:
            self.proto_nodes[lid] = all_embs
            return

        embs_np = all_embs.numpy()
        km = KMeans(n_clusters=k_actual, n_init=10, random_state=SEED)
        km.fit(embs_np)
        centroids = torch.tensor(km.cluster_centers_, dtype=torch.float32)
        centroids = F.normalize(centroids, dim=-1)

        proto_list = []
        for c_idx in range(k_actual):
            proto_list.append({
                "emb": centroids[c_idx], "text": f"[proto_{c_idx}]",
                "is_virtual": False, "is_proto": True
            })

        # Prepend prototypes so they are index 0..k_actual-1
        self.nodes[lid] = proto_list + self.nodes[lid]
        self.proto_nodes[lid] = centroids
        self._proto_stacked[lid] = centroids
        print(f"    [{id2label[lid]}] built {k_actual} prototype centroids")

    # ─── Edge building ───────────────────────────────────────
    def build_edges(self,
                    rare_ids: list,
                    intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_edges_per_node=5,
                    max_cross_edges=2000):
        """
        CHANGED: branch on rare vs majority for intra-edge construction.
          majority → sparse (consecutive + high-sim, max 5 per node)
          rare     → all-pairs within label (dense)
        Then cross-edges as before.
        """
        print("  Building dual-partition KG edges ...")
        self.rare_partition = set(rare_ids)
        self._stacked = {}
        self.intra_edges = defaultdict(list)
        self.cross_edges = []

        # ── STEP 1: Inject virtual nodes + prototypes for rare ──
        print("  Injecting virtual interpolation nodes for rare labels ...")
        for lid in rare_ids:
            if lid in self.nodes and self.nodes[lid]:
                self._inject_virtual_nodes(lid)
                self._build_prototypes(lid, k=KG_PROTOTYPE_K)

        # ── STEP 2: Intra-label edges ──────────────────────────
        for lid, node_list in self.nodes.items():
            N = len(node_list)
            if N < 2:
                continue

            embs = torch.stack([n["emb"] for n in node_list])
            embs_norm = F.normalize(embs, dim=-1)
            sim_mat   = torch.mm(embs_norm, embs_norm.T)

            if lid in self.rare_partition:
                # ── DENSE: all-pairs for rare ──────────────────
                for i in range(N):
                    for j in range(i + 1, N):
                        w = float(sim_mat[i, j].item())
                        if w > 0:
                            self.intra_edges[lid].append((i, j, w))
            else:
                # ── SPARSE: consecutive + high-sim for majority ─
                for i in range(N - 1):
                    w = float(sim_mat[i, i+1].item())
                    self.intra_edges[lid].append((i, i+1, max(0.0, w)))

                for i in range(N):
                    sims = sim_mat[i].clone()
                    sims[max(0, i-1):i+2] = -1
                    count = 0
                    while count < max_intra_edges_per_node:
                        j = int(sims.argmax().item())
                        if sims[j] < intra_thresh:
                            break
                        self.intra_edges[lid].append((i, j, float(sims[j].item())))
                        sims[j] = -1
                        count += 1

            self._stacked[lid] = embs

        # ── STEP 3: Cross-label bridge edges ──────────────────
        label_ids   = list(self.nodes.keys())
        cross_count = 0
        for a in range(len(label_ids)):
            if cross_count >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if cross_count >= max_cross_edges:
                    break
                la, lb = label_ids[a], label_ids[b]
                embs_a = self._get_stacked(la)
                embs_b = self._get_stacked(lb)
                if embs_a is None or embs_b is None:
                    continue
                na_norm = F.normalize(embs_a, dim=-1)
                nb_norm = F.normalize(embs_b, dim=-1)
                sim_mat = torch.mm(na_norm, nb_norm.T)
                high = (sim_mat >= cross_thresh).nonzero(as_tuple=False)
                for pair in high[:50]:
                    ni, nj = int(pair[0]), int(pair[1])
                    w = float(sim_mat[ni, nj].item())
                    self.cross_edges.append((la, ni, lb, nj, w))
                    cross_count += 1

        n_intra = sum(len(v) for v in self.intra_edges.values())
        n_rare_nodes = sum(
            len(self.nodes[lid]) for lid in self.rare_partition if lid in self.nodes
        )
        n_maj_nodes  = sum(
            len(self.nodes[lid]) for lid in self.nodes if lid not in self.rare_partition
        )
        print(f"  KG built:")
        print(f"    Majority nodes : {n_maj_nodes} | Rare nodes (incl. virtual+proto): {n_rare_nodes}")
        print(f"    Intra-edges    : {n_intra} | Cross-edges: {len(self.cross_edges)}")

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack([n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    # ─── Save / Load ─────────────────────────────────────────
    def save(self, path):
        data = {
            "rare_partition": list(self.rare_partition),
            "nodes": {
                str(k): [
                    {"emb": n["emb"].tolist(), "text": n["text"],
                     "is_virtual": n["is_virtual"], "is_proto": n["is_proto"]}
                    for n in v
                ]
                for k, v in self.nodes.items()
            },
            "intra_edges": {str(k): v for k, v in self.intra_edges.items()},
            "cross_edges":  self.cross_edges,
            "proto_nodes": {
                str(k): v.tolist() for k, v in self.proto_nodes.items()
            },
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        kg = cls(emb_dim=emb_dim)
        with open(path) as f:
            data = json.load(f)
        kg.rare_partition = set(data.get("rare_partition", []))
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({
                    "emb":        torch.tensor(n["emb"]),
                    "text":       n["text"],
                    "is_virtual": n.get("is_virtual", False),
                    "is_proto":   n.get("is_proto",   False),
                })
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges = [tuple(e) for e in data["cross_edges"]]
        for k, v in data.get("proto_nodes", {}).items():
            t = torch.tensor(v, dtype=torch.float32)
            kg.proto_nodes[int(k)]    = t
            kg._proto_stacked[int(k)] = t
        print(f"  KG loaded ← {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR  ← CHANGED: dual threshold
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    """
    Normalised entropy H(p)/log(C) ∈ [0,1].
    Uses separate thresholds for rare vs majority predicted labels.
    """
    def __init__(self, num_classes=NUM_LABELS,
                 thresh_majority=UNCERTAINTY_THRESH_MAJORITY,
                 thresh_rare=UNCERTAINTY_THRESH_RARE):
        self.log_C           = math.log(num_classes)
        self.thresh_majority = thresh_majority
        self.thresh_rare     = thresh_rare

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        eps   = 1e-9
        H     = -(probs * (probs + eps).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor,
                     is_rare_pred: torch.Tensor) -> torch.Tensor:
        """
        is_rare_pred: bool tensor same shape as logits[..., 0]
        Returns bool mask: True where KG retrieval should be triggered.
        """
        H     = self.entropy(logits)
        thresh = torch.where(
            is_rare_pred,
            torch.full_like(H, self.thresh_rare),
            torch.full_like(H, self.thresh_majority),
        )
        return H > thresh

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# ENHANCED KG RETRIEVER  ← CORE CHANGE
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    """
    Changes vs base:
      1. Subgraph scoring: score(lid) = max_cosine + λ · 𝟙[lid ∈ rare_ids]
         λ is calibrated on construction; default KG_RARE_LAMBDA.
      2. For rare subgraphs: query hits prototype first (fast coarse search),
         then 1-hop expansion to real + virtual neighbours.
      3. Returns list of (emb, weight, is_rare) triples so GAT can apply r_boost.
    """

    def __init__(self, kg: KnowledgeGraph,
                 rare_ids: list,
                 top_k=KG_TOP_K,
                 top_nodes=KG_TOP_NODES,
                 hop=KG_HOP,
                 lambda_boost=KG_RARE_LAMBDA):
        self.kg           = kg
        self.rare_ids     = set(rare_ids)
        self.top_k        = top_k
        self.top_nodes    = top_nodes
        self.hop          = hop
        self.lambda_boost = lambda_boost

    # ── NEW: auto-calibrate λ from training gap statistics ───
    @classmethod
    def calibrate_lambda(cls, kg: KnowledgeGraph, rare_ids: list,
                         sample_limit: int = 500) -> float:
        """
        For each rare-label node, compute:
            gap = best_majority_sim - best_rare_sim
        λ = median(gaps).  Clipped to [0.05, 0.40].
        """
        rare_set = set(rare_ids)
        maj_ids  = [lid for lid in kg.nodes if lid not in rare_set]
        gaps = []

        for lid in rare_ids:
            if lid not in kg.nodes:
                continue
            node_list = kg.nodes[lid]
            sampled   = random.sample(node_list, min(sample_limit, len(node_list)))
            maj_stacked = {m: kg._get_stacked(m) for m in maj_ids
                           if kg._get_stacked(m) is not None}
            if not maj_stacked:
                continue

            for node in sampled:
                h_norm = F.normalize(node["emb"].unsqueeze(0), dim=-1)

                # best cosine in own rare subgraph
                own_embs = kg._get_stacked(lid)
                if own_embs is None:
                    continue
                own_norm = F.normalize(own_embs, dim=-1)
                best_rare_sim = float(torch.mv(own_norm, h_norm.squeeze(0)).max().item())

                # best cosine across all majority subgraphs
                best_maj_sim = -1.0
                for m_embs in maj_stacked.values():
                    m_norm = F.normalize(m_embs, dim=-1)
                    sim    = float(torch.mv(m_norm, h_norm.squeeze(0)).max().item())
                    if sim > best_maj_sim:
                        best_maj_sim = sim

                gaps.append(best_maj_sim - best_rare_sim)

        if not gaps:
            return KG_RARE_LAMBDA
        lam = float(np.median(gaps))
        lam = float(np.clip(lam, 0.05, 0.40))
        print(f"  λ auto-calibrated: median gap = {np.median(gaps):.4f} → λ = {lam:.4f}")
        return lam

    def retrieve(self, h_i: torch.Tensor,
                 first_pass_label: int = None) -> list:
        """
        h_i: (D,) query embedding on CPU.
        Returns list of (emb: Tensor(D,), weight: float, is_rare: bool).
        """
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)

        # ── Step 1: score each subgraph with λ boost ──────────
        subgraph_scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None or embs.shape[0] == 0:
                continue
            embs_norm = F.normalize(embs, dim=-1)
            max_sim   = float(torch.mv(embs_norm, h_norm.squeeze(0)).max().item())
            # ← PRIORITY BOOST for rare subgraphs
            boost     = self.lambda_boost if lid in self.rare_ids else 0.0
            subgraph_scores[lid] = max_sim + boost

        # ── Step 2: top-K by boosted score ────────────────────
        sorted_sgs = sorted(subgraph_scores.items(), key=lambda x: -x[1])
        selected   = [lid for lid, _ in sorted_sgs[:self.top_k]]

        results = []  # (emb, weight, is_rare)

        for lid in selected:
            is_rare_sg = lid in self.rare_ids
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            embs_norm = F.normalize(embs, dim=-1)
            sims = torch.mv(embs_norm, h_norm.squeeze(0))

            if is_rare_sg and lid in self.kg.proto_nodes:
                # ── RARE: coarse search on prototypes first ────
                protos     = self.kg._proto_stacked.get(lid)
                if protos is not None:
                    p_norm     = F.normalize(protos, dim=-1)
                    p_sims     = torch.mv(p_norm, h_norm.squeeze(0))
                    best_proto = int(p_sims.argmax().item())

                    # gather real+virtual neighbours near best prototype
                    seed_set = set()
                    for idx_node, node in enumerate(self.kg.nodes[lid]):
                        if node["is_proto"] and idx_node == best_proto:
                            seed_set.add(idx_node)

                    # 1-hop from prototype via intra-edges
                    for idx in list(seed_set):
                        for (i, j, w) in self.kg.intra_edges.get(lid, []):
                            if i == idx:
                                seed_set.add(j)
                            elif j == idx:
                                seed_set.add(i)

                    # also add top-N by cosine directly
                    k = min(self.top_nodes, embs.shape[0])
                    top_idx  = sims.topk(k).indices.tolist()
                    top_sims = sims.topk(k).values.tolist()
                    seed_set.update(top_idx)

                    for idx in seed_set:
                        if idx < embs.shape[0]:
                            sim_val = float(sims[idx].item())
                            results.append((embs[idx], sim_val, True))

                    # cross-edge expansion
                    for (la, ni, lb, nj, w) in self.kg.cross_edges:
                        if la == lid and ni in seed_set:
                            cross_embs = self.kg._get_stacked(lb)
                            if cross_embs is not None and nj < cross_embs.shape[0]:
                                results.append((cross_embs[nj], w,
                                                lb in self.rare_ids))
                        elif lb == lid and nj in seed_set:
                            cross_embs = self.kg._get_stacked(la)
                            if cross_embs is not None and ni < cross_embs.shape[0]:
                                results.append((cross_embs[ni], w,
                                                la in self.rare_ids))
                    continue  # skip the majority path below

            # ── MAJORITY: standard top-N + 1-hop ──────────────
            k = min(self.top_nodes, embs.shape[0])
            top_idx  = sims.topk(k).indices.tolist()
            top_sims = sims.topk(k).values.tolist()
            seed_set = set(top_idx)

            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)

                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        cross_embs = self.kg._get_stacked(lb)
                        if cross_embs is not None and nj < cross_embs.shape[0]:
                            results.append((cross_embs[nj], w, lb in self.rare_ids))
                    elif lb == lid and nj in seed_set:
                        cross_embs = self.kg._get_stacked(la)
                        if cross_embs is not None and ni < cross_embs.shape[0]:
                            results.append((cross_embs[ni], w, la in self.rare_ids))

            for idx, sim in zip(top_idx, top_sims):
                results.append((embs[idx], float(sim), is_rare_sg))

        return results  # list of (Tensor(D,), float, bool)


# ═══════════════════════════════════════════════════════════
# ENHANCED GRAPH ATTENTION FUSION  ← CORE CHANGE
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    """
    alpha_j = softmax( (Q·Kⱼ / sqrt(D)) · edge_weight_j · r_boost_j )
    r_boost_j = γ  if neighbour is from 𝒢_rare
              = 1.0 otherwise
    """

    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT,
                 r_boost_gamma=KG_RARE_GAMMA):
        super().__init__()
        self.emb_dim      = emb_dim
        self.r_boost_gamma = r_boost_gamma
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i: torch.Tensor,
                neighbours: list,          # (emb, weight, is_rare)
                device=None) -> torch.Tensor:
        """
        h_i        : (D,) on GPU
        neighbours : list of (Tensor(D,), float weight, bool is_rare)
        Returns    : v_i (D,)
        """
        if not neighbours:
            return torch.zeros_like(h_i)

        if device is None:
            device = h_i.device

        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)   # (K, D)
        weights = torch.tensor([nb[1] for nb in neighbours],
                               device=device, dtype=torch.float)          # (K,)
        # ← r_boost: amplify rare neighbours
        r_boost = torch.tensor(
            [self.r_boost_gamma if nb[2] else 1.0 for nb in neighbours],
            device=device, dtype=torch.float
        )                                                                   # (K,)

        q   = self.proj_q(h_i.unsqueeze(0))    # (1, D)
        k   = self.proj_k(embs)                 # (K, D)
        dot = torch.mv(k, q.squeeze(0)) * self.scale  # (K,)

        # attention = softmax( dot · edge_weight · r_boost )
        alpha = F.softmax(dot * weights * r_boost, dim=0)  # (K,)
        alpha = self.dropout(alpha)

        v_i = (alpha.unsqueeze(-1) * embs).sum(dim=0)   # (D,)
        return v_i


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):
    """
    Wraps InLegalBERT_BiLSTM_MHA_CRF and adds enhanced KG-RAG:
      1. First-pass emissions → dual-threshold uncertainty check.
      2. Conditional rare-biased KG retrieval.
      3. GAT fusion with r_boost → h* = hᵢ + vᵢ.
      4. Re-run ctx-BiLSTM on fused H* → fusion CRF.
      5. Ensemble: (base_emissions + fused_emissions) / 2 for loss.
    """

    def __init__(self, base_model: InLegalBERT_BiLSTM_MHA_CRF,
                 kg: KnowledgeGraph,
                 rare_ids: list,
                 retriever: KGRetriever = None):
        super().__init__()
        self.base        = base_model
        self.kg          = kg
        self.rare_ids    = set(rare_ids)
        self.retriever   = retriever or KGRetriever(kg, rare_ids)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim   # 256
        ctx_dim  = base_model.ctx_out_dim    # 128

        self.gat_fusion  = GraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_proj = nn.Sequential(
            nn.Linear(sent_dim * 2, ctx_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        """
        sent_vecs : (B, T, sent_dim)
        emissions : (B, T, C)
        Returns fused_sent_vecs (B, T, sent_dim)
        """
        B, T, sent_dim = sent_vecs.shape
        fused = sent_vecs.clone()

        top_labels = self.uncertainty.top_label(emissions)  # (B, T)

        # build is_rare_pred mask for dual-threshold
        is_rare_pred = torch.zeros(B, T, dtype=torch.bool, device=device)
        for b in range(B):
            for t in range(int(lengths[b].item())):
                is_rare_pred[b, t] = top_labels[b, t].item() in self.rare_ids

        uncertain_mask = self.uncertainty.is_uncertain(emissions, is_rare_pred)

        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                uncertain  = bool(uncertain_mask[b, t].item())
                pred_lbl   = int(top_labels[b, t].item())
                is_rare    = pred_lbl in self.rare_ids

                # trigger KG if uncertain OR (always for rare predictions)
                if not (uncertain or (RARE_ALWAYS_KG and is_rare)):
                    continue

                h_i = sent_vecs[b, t].detach().cpu()
                neighbours = self.retriever.retrieve(
                    h_i, first_pass_label=pred_lbl
                )
                if not neighbours:
                    continue

                v_i = self.gat_fusion(
                    sent_vecs[b, t], neighbours, device=device
                )
                fused[b, t] = sent_vecs[b, t] + v_i   # residual

        return fused

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        device = input_ids.device

        # Step 1: base model first pass
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        # Step 2: rare-biased KG retrieval + GAT fusion
        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device
        )

        # Step 3: re-run ctx-BiLSTM on fused representations
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)

        fused_ctx = self.base.dropout(fused_ctx)

        # Step 4: fusion classifier + CRF
        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = torch.nan_to_num(fused_emissions, nan=0.0,
                                            posinf=1e4, neginf=-1e4)

        # Build CRF mask
        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2], dtype=torch.bool,
                              device=device)

        combined_emissions = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            base_crf_loss  = -self.base.crf(
                base_emissions, safe_labels, mask=mask, reduction="mean"
            )
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe_labels, mask=mask, reduction="mean"
            )
            B2, T2, C = combined_emissions.shape
            ce_loss = self.ce_loss(
                combined_emissions.reshape(B2*T2, C),
                labels.reshape(B2*T2),
            )
            loss = (base_crf_loss + fused_crf_loss) / 2.0 + AUX_CE_WEIGHT * ce_loss
            return loss, combined_emissions
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS  (unchanged)
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues  = [id2label[x] for x in all_trues]
    str_preds  = [id2label[x] for x in all_preds]
    cls_report = classification_report(str_trues, str_preds, labels=LABELS,
                                       digits=4, zero_division=0)
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


def count_parameters(model):
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {total_trainable:,} | Frozen: {total_frozen:,}")
    return total_trainable, total_frozen


# ═══════════════════════════════════════════════════════════
# BASE TRAINER  (unchanged except logging)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []
        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({"params": params, "lr": lr_i,
                                     "weight_decay": WEIGHT_DECAY})
        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({"params": head_params, "lr": HEAD_LR,
                             "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item(); n += 1
        return total_loss / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )
        early_stopper = EarlyStopping()
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item(); n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_loss       = self.compute_val_loss(dev_dataset)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_train_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")

        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER  ← CHANGED: passes rare_ids to build_edges
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer,
                           rare_ids, device=DEVICE) -> KnowledgeGraph:
    print("\n🔨 Building Dual-Partition Knowledge Graph ...")
    base_model.eval()
    base_model.to(device)

    kg = KnowledgeGraph(emb_dim=base_model.sent_out_dim)
    dummy_dataset = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy_dataset, batch_size=1, shuffle=False,
                        collate_fn=collate_rrc)

    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids     = ids.to(device)
        attn    = attn.to(device)
        ttype   = ttype.to(device)
        lengths = lengths.to(device)

        sent_vecs = base_model.encode_sentences(ids, attn, ttype).squeeze(0)
        n         = int(lengths[0].item())
        kg.add_nodes(sent_vecs[:n].cpu(), labels[0, :n].tolist())

        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1}/{len(loader)} docs")

    # ← CHANGED: pass rare_ids so build_edges can branch on partition
    kg.build_edges(rare_ids=rare_ids)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER  (unchanged logic, updated evaluate signature)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: KGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (
            list(self.model.gat_fusion.parameters()) +
            list(self.model.fusion_proj.parameters()) +
            list(self.model.fusion_classifier.parameters()) +
            list(self.model.fusion_crf.parameters())
        )
        base_trainable = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_params,     "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents = len(all_trues)
            infer_info = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )
        early_stopper = EarlyStopping(patience=5)
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for ids, attn, ttype, labels, lengths in train_loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                running_loss += loss.item(); n_steps += 1

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG]  Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "kg",
                "train_loss": avg_train_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  KG early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "kg_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (KG-RAG v2)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1  (rare-exclusive KG-RAG)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"], marker="o", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Combined Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1",  marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1 (Base → Rare-Exclusive KG-RAG)")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def print_metrics_table(dev_metrics, test_metrics,
                        base_time=None, kg_time=None, total_trainable=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Macro-Recall",       "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + Rare-Excl. KG-RAG)")
    print("=" * 72)
    if total_trainable:
        print(f"  Trainable Parameters  : {total_trainable:,}")
    if base_time:
        print(f"  Phase A training time : {base_time/60:.1f} min")
    if kg_time:
        print(f"  Phase B training time : {kg_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 72)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 68)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 68)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 68)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT + BiLSTM + MHA + CRF")
    print("               + Rare-Exclusive Dense KG + Biased Retrieval\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════════════════════════════════════════════════════
    # PHASE A: Train base model
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF()
    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE,
    )

    # ════════════════════════════════════════════════════
    # PHASE A→B: Build Dual-Partition KG
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Dual-Partition Knowledge Graph")
    print("=" * 60)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG — loading ...")
        kg = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim)
        kg.rare_partition = set(rare_ids)   # ensure set is populated on load
    else:
        # ← CHANGED: pass rare_ids
        kg = build_knowledge_graph(base_model, train_docs, tokenizer,
                                   rare_ids=rare_ids, device=DEVICE)

    # ── Auto-calibrate λ ──────────────────────────────
    print("\n  Calibrating λ (retrieval priority boost) ...")
    lambda_boost = KGRetriever.calibrate_lambda(kg, rare_ids)

    # ════════════════════════════════════════════════════
    # PHASE B: KG-Augmented fine-tuning
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: Rare-Exclusive KG-Augmented Fine-Tuning")
    print(f"  λ={lambda_boost:.4f}  γ={KG_RARE_GAMMA}  "
          f"thresh_rare={UNCERTAINTY_THRESH_RARE}  "
          f"thresh_maj={UNCERTAINTY_THRESH_MAJORITY}")
    print("=" * 60)

    retriever = KGRetriever(
        kg, rare_ids=rare_ids,
        top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP,
        lambda_boost=lambda_boost,
    )
    kg_model = KGAugmentedModel(
        base_model=base_model, kg=kg,
        rare_ids=rare_ids, retriever=retriever,
    )
    total_trainable, _ = count_parameters(kg_model)

    kg_trainer = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        num_epochs=NUM_EPOCHS_KG,
    )
    plot_combined_history(base_hist_df, kg_hist_df)

    # ════════════════════════════════════════════════════
    # EVALUATION
    # ════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_metrics = kg_trainer.evaluate(dev_dataset, rare_ids,
                                      split_name="dev",
                                      measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF + Rare-Exclusive KG-RAG\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    save_confusion_matrix(dev_metrics["cm"],  "dev",  rare_labels)
    save_per_class_f1_chart(dev_metrics["per_class_metrics"],  "dev",  rare_labels)

    print("\nEvaluating on Test set ...")
    test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                       split_name="test",
                                       measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF + Rare-Exclusive KG-RAG\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall", "micro_recall", "weighted_recall", "rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": "InLegalBERT+BiLSTM+MHA+CRF+Rare-Exclusive-KG-RAG",
        "kg_config": {
            "lambda_boost":            lambda_boost,
            "gamma_r_boost":           KG_RARE_GAMMA,
            "virtual_alphas":          KG_VIRTUAL_ALPHAS,
            "prototype_k":             KG_PROTOTYPE_K,
            "max_virtual_per_label":   KG_MAX_VIRTUAL_PER_LABEL,
            "uncertainty_thresh_rare": UNCERTAINTY_THRESH_RARE,
            "uncertainty_thresh_maj":  UNCERTAINTY_THRESH_MAJORITY,
            "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES, "hop": KG_HOP,
        },
        "timing": {
            "phase_a_s": base_time, "phase_b_s": kg_time,
            "total_s": base_time + kg_time,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        base_time=base_time, kg_time=kg_time,
        total_trainable=total_trainable,
    )
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT + BiLSTM + MHA + CRF
               + Rare-Exclusive Dense KG + Biased Retrieval

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 'ARG_RESPONDENT', '

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7.
🔥 BERT layers trainable: layers 8-11 + pooler.

[Base] Epoch 001/60 | train_loss: 287.7585 | val_loss: 211.9501 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | time: 37.5s | ES: 0/10
  ✔ New best val_macro_f1=0.0391
[Base] Epoch 002/60 | train_loss: 234.5406 | val_loss: 175.5814 | val_macro_f1: 0.0797 | val_rare_f1: 0.0000 | time: 37.8s | ES: 0/10
  ✔ New best val_macro_f1=0.0797
[Base] Epoch 003/60 | train_loss: 198.7177 | val_loss: 128.8831 | val_macro_f1: 0.2005 | val_rare_f1: 0.0560 | time: 38.0s | ES: 0/10
  ✔ New best val_macro_f1=0.2005
[Base] Epoch 004/60 | train_loss: 166.5206 | val_loss: 106.5727 | val_macro_f1: 0.2568 | val_rare_f1: 0.0970 | time: 37.8s | ES: 0/10
  ✔ New best val_macro_f1=0.2568
[Base] Epoch 005/60 | train_loss: 140.3391 | val_loss: 88.4431 | val_macro_f1: 0.2828 | val_rare_f1: 0.1195 | time: 37.6s | ES: 0/10
  ✔ New best val_macro_f1=0.2828
[Base] Epoch 006/60 | train_loss: 124.8879 | val_loss: 83.0902 | val

KeyboardInterrupt: 

In [1]:
# inlegalbert_kg_rag_rrc_v2.py  (GPU-accelerated rewrite)
#
# Architecture:
#   InLegalBERT → BiLSTM → Multi-Head Attention Pooling → CRF
#   + ENHANCED Knowledge Graph (Dual-Partition: 𝒢_maj + 𝒢_rare)
#   + Rare-Exclusive Dense Subgraph with Virtual Interpolation Nodes
#   + Prototype Centroid Two-Level Index per Rare Class
#   + Rare-Biased Retrieval (λ priority score boost)
#   + GAT r_boost: rare neighbours amplified during fusion
#
# KEY FIX vs v2_base:
#   ALL KG tensors (node embeddings, edge indices, prototype centroids)
#   are stored and operated on the GPU.  The retrieve() hot-path is
#   fully vectorised – zero Python for-loops over edges or nodes.
#   _kg_fuse_batch() is also vectorised: one batched matmul per subgraph.
#
#   Specifically:
#     • KnowledgeGraph._stacked[lid]       → GPU Tensor (N, D)
#     • KnowledgeGraph._edge_idx[lid]      → GPU LongTensor (2, E)
#     • KnowledgeGraph._edge_w[lid]        → GPU FloatTensor (E,)
#     • KnowledgeGraph._proto_stacked[lid] → GPU Tensor (K, D)
#     • KGRetriever.retrieve()             → batched cosine via torch.mm,
#                                            topk on GPU, edge expansion via
#                                            torch.isin (no Python loops)
#     • KGAugmentedModel._kg_fuse_batch()  → processes ALL sentences of a
#                                            document simultaneously with a
#                                            single masked mm + scatter

import os, json, random, time, math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)
from sklearn.cluster import MiniBatchKMeans   # faster than KMeans for large N

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_v2_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 20
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
WARMUP_RATIO     = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD   = 0.05

# ── KG-RAG ────────────────────────────────────────────────
KG_TOP_K        = 3
KG_TOP_NODES    = 5
KG_HOP          = 1
KG_FUSION_DIM   = 256    # = SENT_OUT_DIM (128*2)

RST_INTRA_THRESH = 0.6
RST_CROSS_THRESH = 0.5

# ── Dual-threshold uncertainty ────────────────────────────
UNCERTAINTY_THRESH_MAJORITY = 0.70
UNCERTAINTY_THRESH_RARE     = 0.40

# ── Rare-exclusive subgraph ───────────────────────────────
RARE_ALWAYS_KG           = True
KG_VIRTUAL_ALPHAS        = [0.25, 0.50, 0.75]
KG_PROTOTYPE_K           = 3
KG_RARE_LAMBDA           = 0.15
KG_RARE_GAMMA            = 1.5
KG_MAX_VIRTUAL_PER_LABEL = 300

# ── GPU KG limits (to avoid OOM on large subgraphs) ───────
KG_MAX_NODES_PER_SG      = 2000   # cap nodes used in retrieval mm
KG_MAX_CROSS_EDGES       = 2000
KG_MAX_INTRA_EDGES_STORE = 500_000   # per label; dense graphs get sampled

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2  # 256

        self.mha_pooling     = MultiHeadAttentionPooling(self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2  # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total = len(encoder_layers)
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids, lengths=None):
        sent_vecs = self.encode_sentences(input_ids, attention_mask, token_type_ids)
        sent_vecs_drop = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        _, _, emissions = self.get_emissions(input_ids, attention_mask,
                                             token_type_ids, lengths=lengths)
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(emissions.reshape(B2*T2, C), labels.reshape(B2*T2))
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# GPU-ACCELERATED KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    """
    GPU-resident dual-partition KG.

    Internal storage (all on self.device after .to_device() is called):
        _stacked[lid]      : Tensor (N, D)   – all node embeddings
        _norm[lid]         : Tensor (N, D)   – L2-normalised embeddings
        _is_rare_node[lid] : Tensor (N,)     – bool, True for virtual/proto
        _edge_idx[lid]     : LongTensor (2,E)– src/dst indices (symmetric)
        _edge_w[lid]       : Tensor (E,)     – cosine weights
        _proto[lid]        : Tensor (K, D)   – prototype centroids (normed)
        _proto_norm[lid]   : Tensor (K, D)
        cross_src_lbl      : LongTensor (X,) – source label ids
        cross_src_idx      : LongTensor (X,) – source node indices
        cross_dst_lbl      : LongTensor (X,) – dest label ids
        cross_dst_idx      : LongTensor (X,) – dest node indices
        cross_w            : Tensor (X,)     – weights
    """

    def __init__(self, emb_dim=KG_FUSION_DIM, device=DEVICE):
        self.emb_dim        = emb_dim
        self.device         = device
        self.rare_partition = set()

        # CPU-side node store (raw, before GPU promotion)
        self._cpu_nodes   = defaultdict(list)   # lid -> list of (emb_cpu, is_virtual, is_proto)

        # GPU tensors (populated by to_device() / build_edges())
        self._stacked     = {}
        self._norm        = {}
        self._is_rare_node= {}
        self._edge_idx    = {}
        self._edge_w      = {}
        self._proto       = {}
        self._proto_norm  = {}

        # Cross edges – stored as 5 parallel GPU tensors
        self._cross_src_lbl = None
        self._cross_src_idx = None
        self._cross_dst_lbl = None
        self._cross_dst_idx = None
        self._cross_w       = None

    # ─── Population ─────────────────────────────────────────
    def add_nodes(self, embeddings: torch.Tensor, label_ids: list):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            self._cpu_nodes[lid].append((emb, False, False))

    # ─── Virtual nodes ───────────────────────────────────────
    def _inject_virtual_nodes(self, lid: int,
                              alphas=KG_VIRTUAL_ALPHAS,
                              max_virtual=KG_MAX_VIRTUAL_PER_LABEL):
        real = [n[0] for n in self._cpu_nodes[lid] if not n[1] and not n[2]]
        n = len(real)
        if n < 2:
            return
        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        random.shuffle(pairs)
        added = 0
        for (i, j) in pairs:
            if added >= max_virtual:
                break
            ei, ej = real[i], real[j]
            for alpha in alphas:
                if added >= max_virtual:
                    break
                v = alpha * ei + (1 - alpha) * ej
                v = F.normalize(v.unsqueeze(0), dim=-1).squeeze(0)
                self._cpu_nodes[lid].append((v, True, False))
                added += 1
        print(f"    [{id2label[lid]}] injected {added} virtual nodes (real={n})")

    # ─── Prototypes (MiniBatchKMeans on GPU via numpy bridge) ─
    def _build_prototypes(self, lid: int, k: int = KG_PROTOTYPE_K):
        all_embs = torch.stack([n[0] for n in self._cpu_nodes[lid]])
        n_nodes  = all_embs.shape[0]
        k_actual = min(k, n_nodes)
        if k_actual < 2:
            c = all_embs.mean(0, keepdim=True)
            c = F.normalize(c, dim=-1)
        else:
            km = MiniBatchKMeans(n_clusters=k_actual, n_init=5,
                                 batch_size=min(1024, n_nodes),
                                 random_state=SEED)
            km.fit(all_embs.numpy())
            c = torch.tensor(km.cluster_centers_, dtype=torch.float32)
            c = F.normalize(c, dim=-1)

        for ci in range(c.shape[0]):
            self._cpu_nodes[lid].insert(0, (c[ci], False, True))
        print(f"    [{id2label[lid]}] built {c.shape[0]} prototype centroids")
        return c   # (K, D)

    # ─── Build all GPU tensors ────────────────────────────────
    def build_edges(self,
                    rare_ids: list,
                    intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_per_node=5,
                    max_cross_edges=KG_MAX_CROSS_EDGES):

        print("  Building dual-partition KG edges (GPU-accelerated) ...")
        self.rare_partition = set(rare_ids)

        # ── Step 1: virtual + prototype injection ──────────────
        print("  Injecting virtual nodes & prototypes ...")
        for lid in rare_ids:
            if self._cpu_nodes[lid]:
                self._inject_virtual_nodes(lid)
                self._build_prototypes(lid, k=KG_PROTOTYPE_K)

        # ── Step 2: move all nodes to GPU tensors ──────────────
        print("  Promoting node embeddings to GPU ...")
        for lid, node_list in self._cpu_nodes.items():
            embs = torch.stack([n[0] for n in node_list]).to(self.device)  # (N, D)
            norms = F.normalize(embs, dim=-1)
            is_virt = torch.tensor([n[1] or n[2] for n in node_list],
                                   dtype=torch.bool, device=self.device)
            self._stacked[lid] = embs
            self._norm[lid]    = norms
            self._is_rare_node[lid] = is_virt

        # ── Step 3: intra-label edges → GPU sparse ────────────
        print("  Building intra-label edges ...")
        for lid in self._stacked:
            norms = self._norm[lid]
            N = norms.shape[0]
            if N < 2:
                self._edge_idx[lid] = torch.zeros(2, 0, dtype=torch.long, device=self.device)
                self._edge_w[lid]   = torch.zeros(0, device=self.device)
                continue

            if lid in self.rare_partition:
                # ── DENSE: all-pairs, but chunk to avoid OOM ──
                chunk = min(512, N)
                ei_list, ej_list, ew_list = [], [], []
                for start in range(0, N, chunk):
                    end = min(start + chunk, N)
                    sim_block = torch.mm(norms[start:end], norms.T)  # (chunk, N)
                    rows, cols = torch.where(
                        (sim_block > 0) &
                        (torch.arange(start, end, device=self.device).unsqueeze(1)
                         < torch.arange(N, device=self.device).unsqueeze(0))
                    )
                    ei_list.append(rows + start)
                    ej_list.append(cols)
                    ew_list.append(sim_block[rows, cols])

                    # cap to avoid huge memory
                    if sum(x.shape[0] for x in ei_list) >= KG_MAX_INTRA_EDGES_STORE:
                        break

                if ei_list:
                    ei = torch.cat(ei_list)
                    ej = torch.cat(ej_list)
                    ew = torch.cat(ew_list)
                    # sample if over cap
                    if ei.shape[0] > KG_MAX_INTRA_EDGES_STORE:
                        perm = torch.randperm(ei.shape[0], device=self.device)[:KG_MAX_INTRA_EDGES_STORE]
                        ei, ej, ew = ei[perm], ej[perm], ew[perm]
                    # make symmetric
                    src = torch.cat([ei, ej])
                    dst = torch.cat([ej, ei])
                    ww  = torch.cat([ew, ew])
                else:
                    src = dst = torch.zeros(0, dtype=torch.long, device=self.device)
                    ww  = torch.zeros(0, device=self.device)

            else:
                # ── SPARSE: consecutive + high-cosine for majority ──
                # consecutive edges
                cons_i = torch.arange(N - 1, device=self.device)
                cons_j = cons_i + 1
                cons_w = (norms[cons_i] * norms[cons_j]).sum(-1).clamp(min=0)

                # high-sim non-consecutive
                sim_mat = torch.mm(norms, norms.T)
                sim_mat.fill_diagonal_(-2.0)
                # mask consecutive neighbours to avoid duplication
                idx = torch.arange(N, device=self.device)
                sim_mat[idx[:-1], idx[1:]] = -2.0
                sim_mat[idx[1:], idx[:-1]] = -2.0

                hi_rows, hi_cols = torch.where(sim_mat >= intra_thresh)
                # keep only upper triangle to deduplicate
                keep = hi_rows < hi_cols
                hi_rows, hi_cols = hi_rows[keep], hi_cols[keep]
                hi_w = sim_mat[hi_rows, hi_cols]

                # per-node cap
                if hi_rows.shape[0] > N * max_intra_per_node:
                    perm = torch.randperm(hi_rows.shape[0], device=self.device)[:N * max_intra_per_node]
                    hi_rows, hi_cols, hi_w = hi_rows[perm], hi_cols[perm], hi_w[perm]

                ei = torch.cat([cons_i, hi_rows])
                ej = torch.cat([cons_j, hi_cols])
                ew = torch.cat([cons_w, hi_w])
                src = torch.cat([ei, ej])
                dst = torch.cat([ej, ei])
                ww  = torch.cat([ew, ew])

            self._edge_idx[lid] = torch.stack([src, dst], dim=0)
            self._edge_w[lid]   = ww

        # ── Step 4: prototypes to GPU ─────────────────────────
        for lid in rare_ids:
            proto_nodes = [n for n in self._cpu_nodes.get(lid, []) if n[2]]
            if proto_nodes:
                pc = torch.stack([n[0] for n in proto_nodes]).to(self.device)
                pc = F.normalize(pc, dim=-1)
                self._proto[lid]      = pc
                self._proto_norm[lid] = pc

        # ── Step 5: cross-label edges → GPU tensors ───────────
        print("  Building cross-label edges ...")
        label_ids = list(self._stacked.keys())
        c_sl, c_si, c_dl, c_di, c_w = [], [], [], [], []
        total = 0

        for a in range(len(label_ids)):
            if total >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if total >= max_cross_edges:
                    break
                la, lb = label_ids[a], label_ids[b]
                na = self._norm[la]
                nb = self._norm[lb]
                # Cap each subgraph for the cross-sim mm
                na_cap = na[:KG_MAX_NODES_PER_SG]
                nb_cap = nb[:KG_MAX_NODES_PER_SG]
                sim = torch.mm(na_cap, nb_cap.T)
                rows, cols = torch.where(sim >= cross_thresh)
                rows, cols = rows[:50], cols[:50]
                if rows.shape[0] == 0:
                    continue
                w = sim[rows, cols]
                c_sl.append(torch.full((rows.shape[0],), la, dtype=torch.long, device=self.device))
                c_si.append(rows)
                c_dl.append(torch.full((rows.shape[0],), lb, dtype=torch.long, device=self.device))
                c_di.append(cols)
                c_w.append(w)
                total += rows.shape[0]

        if c_sl:
            self._cross_src_lbl = torch.cat(c_sl)
            self._cross_src_idx = torch.cat(c_si)
            self._cross_dst_lbl = torch.cat(c_dl)
            self._cross_dst_idx = torch.cat(c_di)
            self._cross_w       = torch.cat(c_w)
        else:
            self._cross_src_lbl = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cross_src_idx = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cross_dst_lbl = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cross_dst_idx = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cross_w       = torch.zeros(0, device=self.device)

        n_rare_nodes = sum(self._stacked[lid].shape[0]
                           for lid in self.rare_partition if lid in self._stacked)
        n_maj_nodes  = sum(self._stacked[lid].shape[0]
                           for lid in self._stacked if lid not in self.rare_partition)
        n_intra = sum(self._edge_w[lid].shape[0] // 2   # symmetric → halved for display
                      for lid in self._edge_w)
        print(f"  KG built:")
        print(f"    Majority nodes : {n_maj_nodes} | Rare nodes: {n_rare_nodes}")
        print(f"    Intra-edges    : {n_intra} | Cross-edges: {total}")

    # ─── Save / Load (CPU JSON) ──────────────────────────────
    def save(self, path):
        data = {
            "rare_partition": list(self.rare_partition),
            "nodes": {
                str(lid): [
                    {"emb": n[0].tolist(), "is_virtual": n[1], "is_proto": n[2]}
                    for n in node_list
                ]
                for lid, node_list in self._cpu_nodes.items()
            },
        }
        # save edge tensors as lists (compact)
        data["intra"] = {
            str(lid): {
                "idx": self._edge_idx[lid].cpu().tolist(),
                "w":   self._edge_w[lid].cpu().tolist(),
            }
            for lid in self._edge_idx
        }
        cross = {}
        if self._cross_src_lbl is not None and self._cross_src_lbl.shape[0] > 0:
            cross = {
                "sl": self._cross_src_lbl.cpu().tolist(),
                "si": self._cross_src_idx.cpu().tolist(),
                "dl": self._cross_dst_lbl.cpu().tolist(),
                "di": self._cross_dst_idx.cpu().tolist(),
                "w":  self._cross_w.cpu().tolist(),
            }
        data["cross"] = cross
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM, device=DEVICE):
        kg = cls(emb_dim=emb_dim, device=device)
        with open(path) as f:
            data = json.load(f)
        kg.rare_partition = set(data.get("rare_partition", []))

        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                emb = torch.tensor(n["emb"], dtype=torch.float32)
                kg._cpu_nodes[lid].append((emb, n["is_virtual"], n["is_proto"]))

        # Rebuild GPU tensors
        for lid, node_list in kg._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in node_list]).to(device)
            norms = F.normalize(embs, dim=-1)
            is_virt = torch.tensor([n[1] or n[2] for n in node_list],
                                   dtype=torch.bool, device=device)
            kg._stacked[lid]      = embs
            kg._norm[lid]         = norms
            kg._is_rare_node[lid] = is_virt

        for k, v in data.get("intra", {}).items():
            lid = int(k)
            idx = torch.tensor(v["idx"], dtype=torch.long,  device=device)
            w   = torch.tensor(v["w"],   dtype=torch.float32, device=device)
            kg._edge_idx[lid] = idx
            kg._edge_w[lid]   = w

        cross = data.get("cross", {})
        if cross:
            kg._cross_src_lbl = torch.tensor(cross["sl"], dtype=torch.long,    device=device)
            kg._cross_src_idx = torch.tensor(cross["si"], dtype=torch.long,    device=device)
            kg._cross_dst_lbl = torch.tensor(cross["dl"], dtype=torch.long,    device=device)
            kg._cross_dst_idx = torch.tensor(cross["di"], dtype=torch.long,    device=device)
            kg._cross_w       = torch.tensor(cross["w"],  dtype=torch.float32, device=device)
        else:
            z = torch.zeros(0, dtype=torch.long, device=device)
            kg._cross_src_lbl = z; kg._cross_src_idx = z
            kg._cross_dst_lbl = z; kg._cross_dst_idx = z
            kg._cross_w = torch.zeros(0, device=device)

        # prototypes
        for lid in kg.rare_partition:
            proto_nodes = [n for n in kg._cpu_nodes.get(lid, []) if n[2]]
            if proto_nodes:
                pc = torch.stack([n[0] for n in proto_nodes]).to(device)
                pc = F.normalize(pc, dim=-1)
                kg._proto[lid]      = pc
                kg._proto_norm[lid] = pc

        print(f"  KG loaded ← {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS,
                 thresh_majority=UNCERTAINTY_THRESH_MAJORITY,
                 thresh_rare=UNCERTAINTY_THRESH_RARE):
        self.log_C           = math.log(num_classes)
        self.thresh_majority = thresh_majority
        self.thresh_rare     = thresh_rare

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        eps   = 1e-9
        H     = -(probs * (probs + eps).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor,
                     is_rare_pred: torch.Tensor) -> torch.Tensor:
        H = self.entropy(logits)
        thresh = torch.where(
            is_rare_pred,
            torch.full_like(H, self.thresh_rare),
            torch.full_like(H, self.thresh_majority),
        )
        return H > thresh

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# GPU-ACCELERATED KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    """
    Fully vectorised retrieval – no Python loops over nodes or edges.

    retrieve_batch(H, pred_labels, uncertain_mask) processes all
    sentences in one document simultaneously:
      1. Score each subgraph: max_cosine(H_valid, SG) + λ·is_rare
         → one mm per subgraph (GPU)
      2. Top-K subgraphs selected once for the whole batch.
      3. For each selected subgraph, gather top-nodes via topk.
      4. Edge-expansion via torch.isin on GPU LongTensors.
      5. Cross-edge lookup via masked index on GPU tensors.
    Returns (neighbour_embs, weights, is_rare_flags) as GPU tensors.
    """

    def __init__(self, kg: KnowledgeGraph,
                 rare_ids: list,
                 top_k=KG_TOP_K,
                 top_nodes=KG_TOP_NODES,
                 hop=KG_HOP,
                 lambda_boost=KG_RARE_LAMBDA):
        self.kg           = kg
        self.rare_ids     = set(rare_ids)
        self.rare_tensor  = torch.tensor(sorted(rare_ids),
                                         dtype=torch.long, device=kg.device)
        self.top_k        = top_k
        self.top_nodes    = top_nodes
        self.hop          = hop
        self.lambda_boost = lambda_boost
        self.device       = kg.device
        self.label_ids    = sorted(kg._stacked.keys())

    @classmethod
    def calibrate_lambda(cls, kg: KnowledgeGraph, rare_ids: list,
                         sample_limit: int = 200) -> float:
        rare_set = set(rare_ids)
        maj_ids  = [lid for lid in kg._stacked if lid not in rare_set]
        if not maj_ids:
            return KG_RARE_LAMBDA

        # stack majority norms once
        maj_norms_list = [kg._norm[m][:KG_MAX_NODES_PER_SG] for m in maj_ids
                          if m in kg._norm]
        if not maj_norms_list:
            return KG_RARE_LAMBDA
        maj_norms = torch.cat(maj_norms_list, dim=0)  # (M_total, D) GPU

        gaps = []
        for lid in rare_ids:
            if lid not in kg._norm:
                continue
            own_norm = kg._norm[lid]
            N = own_norm.shape[0]
            idx = torch.randperm(N, device=kg.device)[:sample_limit]
            sampled = own_norm[idx]   # (S, D) GPU

            own_sims = torch.mm(sampled, own_norm.T).max(dim=1).values  # (S,)
            maj_sims = torch.mm(sampled, maj_norms.T).max(dim=1).values # (S,)
            gaps.append((maj_sims - own_sims).cpu())

        if not gaps:
            return KG_RARE_LAMBDA
        all_gaps = torch.cat(gaps)
        lam = float(all_gaps.median().item())
        lam = float(np.clip(lam, 0.05, 0.40))
        print(f"  λ auto-calibrated: median gap = {all_gaps.median().item():.4f} → λ = {lam:.4f}")
        return lam

    # ── Main batched entry point ──────────────────────────────
    def retrieve_batch(self,
                       H: torch.Tensor,          # (T, D) query vectors (GPU, L2-normed)
                       trigger_mask: torch.Tensor # (T,) bool – which sentences need KG
                       ) -> tuple:
        """
        Returns three GPU tensors aligned with trigger positions:
            nb_embs   : (T_triggered, K_total, D)
            nb_weights: (T_triggered, K_total)
            nb_is_rare: (T_triggered, K_total) bool
        where K_total = top_k * (top_nodes + edge_expansion + cross) padded.
        Padding positions are marked by nb_weights == 0.
        """
        if not trigger_mask.any():
            return None, None, None

        trig_idx = trigger_mask.nonzero(as_tuple=True)[0]  # (T',)
        H_q      = H[trig_idx]                             # (T', D)
        T_q      = H_q.shape[0]

        # ── 1. Score subgraphs ────────────────────────────────
        # For each label subgraph compute max cosine similarity to query batch
        sg_score_list = []
        for lid in self.label_ids:
            norms = self.kg._norm[lid]
            if norms.shape[0] == 0:
                sg_score_list.append(torch.full((T_q,), -1.0, device=self.device))
                continue
            cap = min(norms.shape[0], KG_MAX_NODES_PER_SG)
            sim = torch.mm(H_q, norms[:cap].T)  # (T', cap)
            max_sim = sim.max(dim=1).values       # (T',)
            boost = self.lambda_boost if lid in self.rare_ids else 0.0
            sg_score_list.append(max_sim + boost)

        sg_scores = torch.stack(sg_score_list, dim=1)  # (T', num_labels)
        topk_scores, topk_lids_idx = sg_scores.topk(
            min(self.top_k, len(self.label_ids)), dim=1
        )  # (T', top_k)

        # ── 2. Gather candidate nodes per query ──────────────
        # We'll build a padded tensor of (emb, weight, is_rare)
        # Conservative capacity estimate
        cap = self.top_k * (self.top_nodes * 4 + 10)
        nb_embs    = torch.zeros(T_q, cap, H_q.shape[1], device=self.device)
        nb_weights = torch.zeros(T_q, cap, device=self.device)
        nb_is_rare = torch.zeros(T_q, cap, dtype=torch.bool, device=self.device)
        fill       = torch.zeros(T_q, dtype=torch.long, device=self.device)

        for ki in range(topk_lids_idx.shape[1]):
            for qi in range(T_q):
                lid_idx = topk_lids_idx[qi, ki].item()
                lid     = self.label_ids[lid_idx]
                is_rare_sg = lid in self.rare_ids

                norms = self.kg._norm[lid]
                embs  = self.kg._stacked[lid]
                N_sg  = norms.shape[0]
                if N_sg == 0:
                    continue

                h = H_q[qi]  # (D,)

                # ── Coarse prototype search for rare ──────────
                seed_set = None
                if is_rare_sg and lid in self.kg._proto_norm:
                    p_norm = self.kg._proto_norm[lid]  # (K, D)
                    p_sims = torch.mv(p_norm, h)
                    best_p = int(p_sims.argmax().item())
                    # 1-hop from prototype via edge index
                    edge = self.kg._edge_idx.get(lid)
                    if edge is not None and edge.shape[1] > 0:
                        proto_node_idx = torch.tensor([best_p], device=self.device)
                        src, dst = edge[0], edge[1]
                        mask_e = (src.unsqueeze(0) == proto_node_idx.unsqueeze(1)).any(0)
                        hop_dst = dst[mask_e]
                        seed_set = torch.cat([proto_node_idx, hop_dst]).unique()
                        seed_set = seed_set[seed_set < N_sg]

                # ── Top-N by cosine ───────────────────────────
                k_q = min(self.top_nodes, N_sg)
                cap_norms = norms[:KG_MAX_NODES_PER_SG]
                sims = torch.mv(cap_norms, h)
                topn = sims.topk(k_q)
                top_idx  = topn.indices
                top_sims = topn.values

                if seed_set is not None:
                    top_idx  = torch.cat([top_idx, seed_set]).unique()
                    top_sims = torch.mv(norms[top_idx], h)
                    top_idx  = top_idx[:cap]
                    top_sims = top_sims[:cap]

                # ── 1-hop edge expansion ──────────────────────
                if self.hop >= 1:
                    edge = self.kg._edge_idx.get(lid)
                    if edge is not None and edge.shape[1] > 0:
                        src, dst = edge[0], edge[1]
                        in_top = torch.isin(src, top_idx)
                        hop_n  = dst[in_top].unique()
                        if hop_n.shape[0] > 0:
                            hop_n = hop_n[hop_n < N_sg]
                            hop_s = torch.mv(norms[hop_n], h)
                            top_idx  = torch.cat([top_idx, hop_n]).unique()
                            top_sims = torch.cat([top_sims, hop_s])

                # ── Cross-edge expansion ──────────────────────
                cross_sl = self.kg._cross_src_lbl
                if cross_sl is not None and cross_sl.shape[0] > 0:
                    mask_la = (cross_sl == lid)
                    if mask_la.any():
                        c_si = self.kg._cross_src_idx[mask_la]
                        c_dl = self.kg._cross_dst_lbl[mask_la]
                        c_di = self.kg._cross_dst_idx[mask_la]
                        c_w  = self.kg._cross_w[mask_la]
                        in_top_c = torch.isin(c_si, top_idx)
                        if in_top_c.any():
                            for ci in range(in_top_c.shape[0]):
                                if not in_top_c[ci]:
                                    continue
                                dlid = int(c_dl[ci].item())
                                dstk = self.kg._stacked.get(dlid)
                                if dstk is None:
                                    continue
                                didx = int(c_di[ci].item())
                                if didx >= dstk.shape[0]:
                                    continue
                                f = int(fill[qi].item())
                                if f >= cap:
                                    break
                                nb_embs[qi, f]    = dstk[didx]
                                nb_weights[qi, f] = float(c_w[ci].item())
                                nb_is_rare[qi, f] = dlid in self.rare_ids
                                fill[qi] = f + 1

                # ── Fill output tensors ───────────────────────
                n_add = min(top_idx.shape[0], cap - int(fill[qi].item()))
                if n_add <= 0:
                    continue
                f = int(fill[qi].item())
                idx_use = top_idx[:n_add]
                sim_use = top_sims[:n_add]
                nb_embs[qi, f:f+n_add]    = embs[idx_use]
                nb_weights[qi, f:f+n_add] = sim_use.clamp(min=0)
                nb_is_rare[qi, f:f+n_add] = is_rare_sg
                fill[qi] = f + n_add

        return nb_embs, nb_weights, nb_is_rare   # (T', cap, D), (T', cap), (T', cap)


# ═══════════════════════════════════════════════════════════
# GPU-ACCELERATED GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    """
    Batched GAT fusion.
    Input:
        H_q      : (T', D)          – query vectors (GPU)
        nb_embs  : (T', K, D)       – neighbour embeddings (GPU, padded)
        nb_w     : (T', K)          – edge weights (0 = padding)
        nb_rare  : (T', K) bool     – is-rare flags
    Output:
        v        : (T', D)          – fused vectors
    """

    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT,
                 r_boost_gamma=KG_RARE_GAMMA):
        super().__init__()
        self.emb_dim       = emb_dim
        self.r_boost_gamma = r_boost_gamma
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, H_q, nb_embs, nb_w, nb_rare):
        """
        H_q     : (T', D)
        nb_embs : (T', K, D)
        nb_w    : (T', K)   – 0 means padding
        nb_rare : (T', K) bool
        """
        T, D = H_q.shape
        K    = nb_embs.shape[1]

        q = self.proj_q(H_q)           # (T', D)
        k = self.proj_k(nb_embs)       # (T', K, D)

        # dot product: (T', K)
        dot = torch.bmm(k, q.unsqueeze(-1)).squeeze(-1) * self.scale  # (T', K)

        # r_boost
        r_boost = torch.where(nb_rare,
                              torch.full_like(dot, self.r_boost_gamma),
                              torch.ones_like(dot))

        # masking padding (nb_w == 0)
        pad_mask = (nb_w == 0)
        raw = dot * nb_w * r_boost
        raw = raw.masked_fill(pad_mask, -1e9)

        alpha = F.softmax(raw, dim=-1)  # (T', K)
        alpha = alpha.masked_fill(pad_mask, 0.0)
        alpha = self.dropout(alpha)

        v = torch.bmm(alpha.unsqueeze(1), nb_embs).squeeze(1)  # (T', D)
        return v


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):

    def __init__(self, base_model: InLegalBERT_BiLSTM_MHA_CRF,
                 kg: KnowledgeGraph,
                 rare_ids: list,
                 retriever: KGRetriever = None):
        super().__init__()
        self.base        = base_model
        self.kg          = kg
        self.rare_ids    = set(rare_ids)
        self.retriever   = retriever or KGRetriever(kg, rare_ids)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim   # 256
        ctx_dim  = base_model.ctx_out_dim    # 128

        self.gat_fusion  = GraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

        # rare_ids tensor on same device as KG
        self._rare_tensor = torch.tensor(sorted(rare_ids), dtype=torch.long)

    def _rare_mask(self, top_labels: torch.Tensor) -> torch.Tensor:
        """top_labels: (B, T) → bool (B, T)"""
        rare = self._rare_tensor.to(top_labels.device)
        return (top_labels.unsqueeze(-1) == rare.view(1, 1, -1)).any(-1)

    # ── Vectorised KG fusion ──────────────────────────────────
    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        """
        Processes the entire batch at once using vectorised retrieval.
        sent_vecs : (B, T, D)
        emissions : (B, T, C)
        Returns fused (B, T, D)
        """
        B, T, D = sent_vecs.shape
        fused = sent_vecs.clone()

        top_labels   = self.uncertainty.top_label(emissions)    # (B, T)
        is_rare_pred = self._rare_mask(top_labels)              # (B, T)
        uncertain    = self.uncertainty.is_uncertain(emissions, is_rare_pred)  # (B, T)

        trigger = uncertain | (RARE_ALWAYS_KG & is_rare_pred)   # (B, T)

        for b in range(B):
            n = int(lengths[b].item())
            trig_b = trigger[b, :n]                             # (T_n,)
            if not trig_b.any():
                continue

            H_b = sent_vecs[b, :n]                             # (T_n, D)
            # L2-normalise for cosine similarity in retriever
            H_b_norm = F.normalize(H_b.detach(), dim=-1)

            nb_embs, nb_weights, nb_is_rare = self.retriever.retrieve_batch(
                H_b_norm, trig_b
            )
            if nb_embs is None:
                continue

            # trig positions within the document
            trig_idx = trig_b.nonzero(as_tuple=True)[0]        # (T',)
            H_q      = H_b[trig_idx]                           # (T', D)

            # GAT fusion (fully batched, GPU)
            v = self.gat_fusion(H_q, nb_embs, nb_weights, nb_is_rare)  # (T', D)

            # residual write-back
            fused[b, trig_idx] = H_b[trig_idx] + v

        return fused

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        device = input_ids.device

        # Step 1: base first pass
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        # Step 2: GPU KG fusion
        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device
        )

        # Step 3: re-run ctx-BiLSTM on fused representations
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)

        fused_ctx = self.base.dropout(fused_ctx)

        # Step 4: fusion classifier + CRF
        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = torch.nan_to_num(fused_emissions, nan=0.0,
                                            posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2], dtype=torch.bool, device=device)

        combined_emissions = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            base_crf_loss  = -self.base.crf(
                base_emissions, safe_labels, mask=mask, reduction="mean"
            )
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe_labels, mask=mask, reduction="mean"
            )
            B2, T2, C = combined_emissions.shape
            ce_loss = self.ce_loss(
                combined_emissions.reshape(B2*T2, C),
                labels.reshape(B2*T2),
            )
            loss = (base_crf_loss + fused_crf_loss) / 2.0 + AUX_CE_WEIGHT * ce_loss
            return loss, combined_emissions
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues  = [id2label[x] for x in all_trues]
    str_preds  = [id2label[x] for x in all_preds]
    cls_report = classification_report(str_trues, str_preds, labels=LABELS,
                                       digits=4, zero_division=0)
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


def count_parameters(model):
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {total_trainable:,} | Frozen: {total_frozen:,}")
    return total_trainable, total_frozen


# ═══════════════════════════════════════════════════════════
# BASE TRAINER
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []
        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({"params": params, "lr": lr_i,
                                     "weight_decay": WEIGHT_DECAY})
        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({"params": head_params, "lr": HEAD_LR,
                             "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item(); n += 1
        return total_loss / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )
        early_stopper = EarlyStopping()
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item(); n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_loss       = self.compute_val_loss(dev_dataset)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_train_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")

        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer,
                           rare_ids, device=DEVICE) -> KnowledgeGraph:
    print("\n🔨 Building Dual-Partition Knowledge Graph (GPU) ...")
    base_model.eval()
    base_model.to(device)

    kg = KnowledgeGraph(emb_dim=base_model.sent_out_dim, device=device)
    dummy_dataset = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy_dataset, batch_size=4, shuffle=False,
                        collate_fn=collate_rrc, num_workers=0)

    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids   = ids.to(device)
        attn  = attn.to(device)
        ttype = ttype.to(device)

        # process each doc in the micro-batch
        for bi in range(ids.shape[0]):
            sent_vecs = base_model.encode_sentences(
                ids[bi:bi+1], attn[bi:bi+1], ttype[bi:bi+1]
            ).squeeze(0)
            n = int(lengths[bi].item())
            kg.add_nodes(sent_vecs[:n].cpu(), labels[bi, :n].tolist())

        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {(doc_idx+1)*ids.shape[0]}/{len(train_docs)} docs")

    kg.build_edges(rare_ids=rare_ids)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: KGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (
            list(self.model.gat_fusion.parameters()) +
            list(self.model.fusion_classifier.parameters()) +
            list(self.model.fusion_crf.parameters())
        )
        base_trainable = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_params,     "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents = len(all_trues)
            infer_info = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )
        early_stopper = EarlyStopping(patience=5)
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for ids, attn, ttype, labels, lengths in train_loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                running_loss += loss.item(); n_steps += 1

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG]  Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "kg",
                "train_loss": avg_train_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  KG early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "kg_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (KG-RAG v2)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1  (rare-exclusive KG-RAG v2)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"], marker="o", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Combined Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1",  marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1 (Base → Rare-Exclusive KG-RAG)")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def print_metrics_table(dev_metrics, test_metrics,
                        base_time=None, kg_time=None, total_trainable=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Macro-Recall",       "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + Rare-Excl. KG-RAG)")
    print("=" * 72)
    if total_trainable:
        print(f"  Trainable Parameters  : {total_trainable:,}")
    if base_time:
        print(f"  Phase A training time : {base_time/60:.1f} min")
    if kg_time:
        print(f"  Phase B training time : {kg_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 72)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 68)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 68)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 68)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT + BiLSTM + MHA + CRF")
    print("               + Rare-Exclusive Dense KG + Biased Retrieval (GPU)\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════════════════════════════════════════════════════
    # PHASE A: Train base model
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF()
    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE,
    )

    # ════════════════════════════════════════════════════
    # PHASE A→B: Build Dual-Partition KG
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Dual-Partition Knowledge Graph")
    print("=" * 60)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG — loading ...")
        kg = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim, device=DEVICE)
        kg.rare_partition = set(rare_ids)
    else:
        kg = build_knowledge_graph(base_model, train_docs, tokenizer,
                                   rare_ids=rare_ids, device=DEVICE)

    # ── Auto-calibrate λ ──────────────────────────────
    print("\n  Calibrating λ (retrieval priority boost) ...")
    lambda_boost = KGRetriever.calibrate_lambda(kg, rare_ids)

    # ════════════════════════════════════════════════════
    # PHASE B: KG-Augmented fine-tuning
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: Rare-Exclusive KG-Augmented Fine-Tuning")
    print(f"  λ={lambda_boost:.4f}  γ={KG_RARE_GAMMA}  "
          f"thresh_rare={UNCERTAINTY_THRESH_RARE}  "
          f"thresh_maj={UNCERTAINTY_THRESH_MAJORITY}")
    print("=" * 60)

    retriever = KGRetriever(
        kg, rare_ids=rare_ids,
        top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP,
        lambda_boost=lambda_boost,
    )
    kg_model = KGAugmentedModel(
        base_model=base_model, kg=kg,
        rare_ids=rare_ids, retriever=retriever,
    )
    total_trainable, _ = count_parameters(kg_model)

    kg_trainer = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        num_epochs=NUM_EPOCHS_KG,
    )
    plot_combined_history(base_hist_df, kg_hist_df)

    # ════════════════════════════════════════════════════
    # EVALUATION
    # ════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_metrics = kg_trainer.evaluate(dev_dataset, rare_ids,
                                      split_name="dev",
                                      measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF + Rare-Exclusive KG-RAG v2 (GPU)\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    save_confusion_matrix(dev_metrics["cm"],  "dev",  rare_labels)
    save_per_class_f1_chart(dev_metrics["per_class_metrics"],  "dev",  rare_labels)

    print("\nEvaluating on Test set ...")
    test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                       split_name="test",
                                       measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF + Rare-Exclusive KG-RAG v2 (GPU)\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall", "micro_recall", "weighted_recall", "rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": "InLegalBERT+BiLSTM+MHA+CRF+Rare-Exclusive-KG-RAG-v2-GPU",
        "kg_config": {
            "lambda_boost":            lambda_boost,
            "gamma_r_boost":           KG_RARE_GAMMA,
            "virtual_alphas":          KG_VIRTUAL_ALPHAS,
            "prototype_k":             KG_PROTOTYPE_K,
            "max_virtual_per_label":   KG_MAX_VIRTUAL_PER_LABEL,
            "uncertainty_thresh_rare": UNCERTAINTY_THRESH_RARE,
            "uncertainty_thresh_maj":  UNCERTAINTY_THRESH_MAJORITY,
            "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES, "hop": KG_HOP,
        },
        "timing": {
            "phase_a_s": base_time, "phase_b_s": kg_time,
            "total_s": base_time + kg_time,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        base_time=base_time, kg_time=kg_time,
        total_trainable=total_trainable,
    )
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT + BiLSTM + MHA + CRF
               + Rare-Exclusive Dense KG + Biased Retrieval (GPU)

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 'ARG_RESPONDE

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7.
🔥 BERT layers trainable: layers 8-11 + pooler.

[Base] Epoch 001/60 | train_loss: 287.7585 | val_loss: 211.9501 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | time: 37.4s | ES: 0/10
  ✔ New best val_macro_f1=0.0391
[Base] Epoch 002/60 | train_loss: 234.5406 | val_loss: 175.5814 | val_macro_f1: 0.0797 | val_rare_f1: 0.0000 | time: 37.6s | ES: 0/10
  ✔ New best val_macro_f1=0.0797
[Base] Epoch 003/60 | train_loss: 198.7177 | val_loss: 128.8831 | val_macro_f1: 0.2005 | val_rare_f1: 0.0560 | time: 37.8s | ES: 0/10
  ✔ New best val_macro_f1=0.2005
[Base] Epoch 004/60 | train_loss: 166.5206 | val_loss: 106.5727 | val_macro_f1: 0.2568 | val_rare_f1: 0.0970 | time: 37.6s | ES: 0/10
  ✔ New best val_macro_f1=0.2568
[Base] Epoch 005/60 | train_loss: 140.3391 | val_loss: 88.4431 | val_macro_f1: 0.2828 | val_rare_f1: 0.1195 | time: 37.3s | ES: 0/10
  ✔ New best val_macro_f1=0.2828
[Base] Epoch 006/60 | train_loss: 124.8879 | val_loss: 83.0902 | val

In [2]:
# inlegalbert_kg_rag_rrc_v3.py  (HARD-MINING + CONFUSION-GUIDED KG-RAG)
#
# Architecture:
#   InLegalBERT → BiLSTM → Multi-Head Attention Pooling → CRF
#   + Dual-Partition KG (𝒢_maj + 𝒢_rare) — GPU-resident tensors
#   + Confusion-Matrix-Guided Cross-Edges  [NEW vs v2]
#   + Weighted Focal Loss + Inverse-Freq Class Weights [NEW]
#   + Rare Document Oversampling via WeightedRandomSampler [NEW]
#   + Hard Example Replay Buffer in Phase B [NEW]
#   + Gated Graph Attention Fusion [NEW — replaces naive residual]
#   + Prototype Contrastive Auxiliary Loss [NEW]
#   + Per-class Adaptive Uncertainty Threshold [NEW]
#
# ROOT-CAUSE FIXES vs v2 (which scored LOWER than v1):
#   ① Retrieval inner Python loops killed both speed and correctness →
#      replaced with vectorised subgraph-first gather approach.
#   ② Simple residual h* = h_i + v_i let noisy KG corrupt confident
#      predictions → replaced with learned gate g ∈ (0,1).
#   ③ Label-smoothing=0.1 blurred rare class targets → reduced to 0.05
#      and replaced CE auxiliary with class-weighted Focal Loss.
#   ④ No oversampling → rare docs appear too infrequently.
#   ⑤ Confusion pairs not used → model doesn't know which majority class
#      steals from each rare class; now encoded as high-weight KG edges.
#   ⑥ Hard examples never revisited → replay buffer fills this gap.

import os, json, random, time, math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)
from sklearn.cluster import MiniBatchKMeans

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_v3_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 25          # slightly more Phase B epochs
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

LABEL_SMOOTHING  = 0.05   # ↓ from 0.1 — sharper signals for rare classes
AUX_CE_WEIGHT    = 0.25
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
WARMUP_RATIO     = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD   = 0.05

# ── KG-RAG ────────────────────────────────────────────────
KG_TOP_K        = 3
KG_TOP_NODES    = 6
KG_HOP          = 1
KG_FUSION_DIM   = 256    # SENT_OUT_DIM = 128 * 2

RST_INTRA_THRESH = 0.55  # slightly relaxed
RST_CROSS_THRESH = 0.45  # relaxed → more cross-label connections

# ── Uncertainty (per-class adaptive) ─────────────────────
UNCERTAINTY_THRESH_MAJORITY = 0.70
UNCERTAINTY_THRESH_RARE     = 0.30   # very aggressive KG use for rare

RARE_ALWAYS_KG           = True
KG_VIRTUAL_ALPHAS        = [0.25, 0.50, 0.75]
KG_PROTOTYPE_K           = 5
KG_RARE_LAMBDA           = 0.20     # retrieval priority boost for rare
KG_RARE_GAMMA            = 2.0      # r_boost multiplier in GAT
KG_MAX_VIRTUAL_PER_LABEL = 200

# ── NEW v3 hyperparameters ────────────────────────────────
FOCAL_GAMMA_RARE         = 2.5   # focal gamma for rare class sentences
FOCAL_GAMMA_MAJ          = 1.0   # focal gamma for majority class sentences
OVERSAMPLE_RARE_RATIO    = 3.0   # upsample rare-containing docs
HARD_BUFFER_SIZE         = 300   # max items in replay buffer
HARD_REPLAY_FREQ         = 4     # replay every N training steps
HARD_REPLAY_LOSS_THRESH  = 0.8   # add to buffer if loss > this
HARD_REPLAY_WEIGHT       = 0.5   # loss scale for replayed examples
PROTO_CONTRAST_WEIGHT    = 0.08  # prototype contrastive loss weight
PROTO_CONTRAST_MARGIN    = 0.35  # cosine margin (push rare proto away from maj)
CONFUSION_EDGE_WEIGHT    = 0.90  # weight of confusion-guided cross-edges
CONFUSION_TOP_K          = 3     # top confused majority classes per rare class

KG_MAX_NODES_PER_SG      = 2000
KG_MAX_CROSS_EDGES       = 4000
KG_MAX_INTRA_EDGES_STORE = 500_000

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# FOCAL LOSS  (class-weighted, per-class gamma)
# ═══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    """
    Class-weighted focal loss with per-class gamma.
    Rare classes get higher gamma (more focus on hard examples).
    Formula: FL(p_t) = -α_t · (1 - p_t)^γ_t · log(p_t)
    """
    def __init__(self, class_weights: torch.Tensor,
                 rare_ids: list,
                 gamma_rare: float = FOCAL_GAMMA_RARE,
                 gamma_maj:  float = FOCAL_GAMMA_MAJ,
                 ignore_index: int = -100):
        super().__init__()
        self.ignore_index = ignore_index
        rare_set = set(rare_ids)

        gamma_per_class = torch.full((NUM_LABELS,), gamma_maj)
        for r in rare_set:
            gamma_per_class[r] = gamma_rare

        self.register_buffer("class_weights",   class_weights.float())
        self.register_buffer("gamma_per_class", gamma_per_class)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits : (N, C), targets : (N,)
        valid = targets != self.ignore_index
        if not valid.any():
            return logits.sum() * 0.0

        logits_v  = logits[valid]
        targets_v = targets[valid]

        log_p = F.log_softmax(logits_v, dim=-1)          # (N', C)
        p     = log_p.exp()
        p_t   = p.gather(1, targets_v.unsqueeze(1)).squeeze(1)   # (N',)

        gamma_t   = self.gamma_per_class[targets_v]      # (N',)
        focal_w   = (1.0 - p_t.detach()).pow(gamma_t)    # (N',)
        class_w   = self.class_weights[targets_v]        # (N',)

        ce = F.nll_loss(log_p, targets_v, reduction="none")  # (N',)
        loss = (focal_w * class_w * ce).mean()
        return loss


def compute_class_weights(label_freqs: dict) -> torch.Tensor:
    """Inverse-frequency weights, capped at 10× for stability."""
    weights = torch.ones(NUM_LABELS)
    freqs   = [label_freqs.get(id2label[i], 1e-6) for i in range(NUM_LABELS)]
    inv     = [1.0 / max(f, 1e-6) for f in freqs]
    inv_sum = sum(inv)
    for i, w in enumerate(inv):
        weights[i] = min(w / inv_sum * NUM_LABELS, 10.0)
    return weights


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET & SAMPLER
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def build_weighted_sampler(docs: list, rare_ids: list,
                            oversample_ratio: float = OVERSAMPLE_RARE_RATIO
                            ) -> WeightedRandomSampler:
    """Upsample documents containing at least one rare-class sentence."""
    rare_set = set(rare_ids)
    weights  = []
    for _, labs in docs:
        has_rare = any(l in rare_set for l in labs)
        weights.append(oversample_ratio if has_rare else 1.0)
    w_tensor = torch.tensor(weights, dtype=torch.float)
    return WeightedRandomSampler(w_tensor,
                                  num_samples=len(w_tensor),
                                  replacement=True)


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2   # 256

        self.mha_pooling     = MultiHeadAttentionPooling(
            self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2    # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total = len(encoder_layers)
        print(f"\n❄️  BERT layers frozen: embeddings + 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)
        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)
        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids,
                      lengths=None):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids)
        sent_vecs_drop = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _ = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def _make_mask(self, emissions, labels, lengths):
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)
        return mask

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        _, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        mask = self._make_mask(emissions, labels, lengths)
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_loss = -self.crf(emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2*T2, C), labels.reshape(B2*T2))
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# GPU-ACCELERATED KNOWLEDGE GRAPH  (with confusion edges)
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    """
    GPU-resident dual-partition KG.
    Adds confusion-matrix-guided cross-edges (Phase A → B bridge).
    """

    def __init__(self, emb_dim=KG_FUSION_DIM, device=DEVICE):
        self.emb_dim        = emb_dim
        self.device         = device
        self.rare_partition = set()

        self._cpu_nodes  = defaultdict(list)  # lid → [(emb, is_virtual, is_proto)]
        self._stacked    = {}
        self._norm       = {}
        self._is_rare    = {}
        self._edge_idx   = {}
        self._edge_w     = {}
        self._proto      = {}
        self._proto_norm = {}

        # Cross-edges (parallel tensors on GPU)
        self._cx_sl = self._cx_si = None
        self._cx_dl = self._cx_di = None
        self._cx_w  = None

    # ── Population ────────────────────────────────────────
    def add_nodes(self, embeddings: torch.Tensor, label_ids: list):
        embs = embeddings.detach().cpu()
        for emb, lid in zip(embs, label_ids):
            self._cpu_nodes[lid].append((emb, False, False))

    # ── Virtual interpolation for rare classes ────────────
    def _inject_virtual(self, lid, alphas=KG_VIRTUAL_ALPHAS,
                        max_v=KG_MAX_VIRTUAL_PER_LABEL):
        real = [n[0] for n in self._cpu_nodes[lid] if not n[1] and not n[2]]
        n = len(real)
        if n < 2:
            return 0
        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        random.shuffle(pairs)
        added = 0
        for i, j in pairs:
            for alpha in alphas:
                if added >= max_v:
                    break
                v = alpha * real[i] + (1 - alpha) * real[j]
                v = F.normalize(v.unsqueeze(0), dim=-1).squeeze(0)
                self._cpu_nodes[lid].append((v, True, False))
                added += 1
            if added >= max_v:
                break
        return added

    # ── Prototype centroids ───────────────────────────────
    def _build_prototypes(self, lid, k=KG_PROTOTYPE_K):
        all_e = torch.stack([n[0] for n in self._cpu_nodes[lid]])
        n_n   = all_e.shape[0]
        k_a   = min(k, n_n)
        if k_a < 2:
            c = F.normalize(all_e.mean(0, keepdim=True), dim=-1)
        else:
            km = MiniBatchKMeans(n_clusters=k_a, n_init=5,
                                 batch_size=min(1024, n_n), random_state=SEED)
            km.fit(all_e.numpy())
            c = F.normalize(
                torch.tensor(km.cluster_centers_, dtype=torch.float32), dim=-1)
        for ci in range(c.shape[0]):
            self._cpu_nodes[lid].insert(0, (c[ci], False, True))
        return c

    # ── Main edge-build (+ optional confusion edges) ──────
    def build_edges(self,
                    rare_ids: list,
                    confusion_pairs: dict = None,
                    intra_thresh: float  = RST_INTRA_THRESH,
                    cross_thresh: float  = RST_CROSS_THRESH,
                    max_intra_per_node=5,
                    max_cross=KG_MAX_CROSS_EDGES):

        print("  Building dual-partition KG (GPU) + confusion edges ...")
        self.rare_partition = set(rare_ids)

        # ── Inject virtual nodes & prototypes for rare ────
        print("  Injecting virtual nodes & prototypes ...")
        for lid in rare_ids:
            if self._cpu_nodes[lid]:
                added = self._inject_virtual(lid)
                self._build_prototypes(lid)
                print(f"    [{id2label[lid]}] +{added} virtual | "
                      f"total={len(self._cpu_nodes[lid])} nodes")

        # ── Move all nodes to GPU ─────────────────────────
        print("  Promoting to GPU ...")
        for lid, nodes in self._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in nodes]).to(self.device)
            norms = F.normalize(embs, dim=-1)
            is_v  = torch.tensor([n[1] or n[2] for n in nodes],
                                  dtype=torch.bool, device=self.device)
            self._stacked[lid] = embs
            self._norm[lid]    = norms
            self._is_rare[lid] = is_v

        # ── Intra-label edges ─────────────────────────────
        print("  Building intra-label edges ...")
        for lid in self._stacked:
            norms = self._norm[lid]
            N = norms.shape[0]
            if N < 2:
                self._edge_idx[lid] = torch.zeros(
                    2, 0, dtype=torch.long, device=self.device)
                self._edge_w[lid]   = torch.zeros(0, device=self.device)
                continue

            if lid in self.rare_partition:
                # Dense all-pairs (chunked) for rare
                chunk = 512
                ei_l, ej_l, ew_l = [], [], []
                for start in range(0, N, chunk):
                    end = min(start + chunk, N)
                    blk = torch.mm(norms[start:end], norms.T)
                    r, c_ = torch.where(
                        (blk > 0) &
                        (torch.arange(start, end, device=self.device).unsqueeze(1)
                         < torch.arange(N, device=self.device).unsqueeze(0))
                    )
                    ei_l.append(r + start); ej_l.append(c_)
                    ew_l.append(blk[r, c_])
                    if sum(x.shape[0] for x in ei_l) >= KG_MAX_INTRA_EDGES_STORE:
                        break
                if ei_l:
                    ei = torch.cat(ei_l); ej = torch.cat(ej_l); ew = torch.cat(ew_l)
                    if ei.shape[0] > KG_MAX_INTRA_EDGES_STORE:
                        p = torch.randperm(ei.shape[0],
                                           device=self.device)[:KG_MAX_INTRA_EDGES_STORE]
                        ei, ej, ew = ei[p], ej[p], ew[p]
                    src = torch.cat([ei, ej]); dst = torch.cat([ej, ei])
                    ww  = torch.cat([ew, ew])
                else:
                    src = dst = torch.zeros(0, dtype=torch.long, device=self.device)
                    ww  = torch.zeros(0, device=self.device)
            else:
                # Sparse consecutive + high-cosine for majority
                idx  = torch.arange(N, device=self.device)
                ci   = idx[:-1]; cj = idx[1:]
                cw   = (norms[ci] * norms[cj]).sum(-1).clamp(min=0)
                sim  = torch.mm(norms, norms.T)
                sim.fill_diagonal_(-2.0)
                sim[ci, cj] = -2.0; sim[cj, ci] = -2.0
                hi_r, hi_c = torch.where(sim >= intra_thresh)
                keep = hi_r < hi_c
                hi_r, hi_c = hi_r[keep], hi_c[keep]
                hi_w = sim[hi_r, hi_c]
                if hi_r.shape[0] > N * max_intra_per_node:
                    p = torch.randperm(
                        hi_r.shape[0], device=self.device)[:N * max_intra_per_node]
                    hi_r, hi_c, hi_w = hi_r[p], hi_c[p], hi_w[p]
                ei  = torch.cat([ci, hi_r]); ej = torch.cat([cj, hi_c])
                ew  = torch.cat([cw, hi_w])
                src = torch.cat([ei, ej]); dst = torch.cat([ej, ei])
                ww  = torch.cat([ew, ew])

            self._edge_idx[lid] = torch.stack([src, dst], dim=0)
            self._edge_w[lid]   = ww

        # ── Prototype tensors ─────────────────────────────
        for lid in rare_ids:
            proto = [n for n in self._cpu_nodes.get(lid, []) if n[2]]
            if proto:
                pc = torch.stack([n[0] for n in proto]).to(self.device)
                pc = F.normalize(pc, dim=-1)
                self._proto[lid] = self._proto_norm[lid] = pc

        # ── Cross-label edges (cosine-based) ─────────────
        print("  Building cross-label edges ...")
        cx_sl, cx_si, cx_dl, cx_di, cx_w = [], [], [], [], []
        total_cross = 0
        label_ids   = sorted(self._stacked.keys())

        for a in range(len(label_ids)):
            if total_cross >= max_cross:
                break
            for b in range(a + 1, len(label_ids)):
                if total_cross >= max_cross:
                    break
                la, lb = label_ids[a], label_ids[b]
                na = self._norm[la][:KG_MAX_NODES_PER_SG]
                nb = self._norm[lb][:KG_MAX_NODES_PER_SG]
                sim = torch.mm(na, nb.T)
                rows, cols = torch.where(sim >= cross_thresh)
                rows, cols = rows[:50], cols[:50]
                if rows.shape[0] == 0:
                    continue
                w = sim[rows, cols]
                cx_sl.append(torch.full((rows.shape[0],), la,
                                        dtype=torch.long, device=self.device))
                cx_si.append(rows)
                cx_dl.append(torch.full((rows.shape[0],), lb,
                                        dtype=torch.long, device=self.device))
                cx_di.append(cols)
                cx_w.append(w)
                total_cross += rows.shape[0]

        # ── Confusion-guided edges (NEW) ──────────────────
        # For each rare class, add high-weight edges TO its most-confused
        # majority classes so GAT fusion sees the decision boundary.
        if confusion_pairs:
            print(f"  Adding confusion-guided edges "
                  f"(top-{CONFUSION_TOP_K} per rare class) ...")
            for rare_lid, confused_lids in confusion_pairs.items():
                if rare_lid not in self._stacked:
                    continue
                for clid in confused_lids[:CONFUSION_TOP_K]:
                    if clid not in self._stacked:
                        continue
                    rare_n = self._norm[rare_lid]
                    conf_n = self._norm[clid][:KG_MAX_NODES_PER_SG]
                    sim    = torch.mm(rare_n, conf_n.T)  # (R, C)
                    # Keep pairs above 0 (ensure some connectivity)
                    rows, cols = torch.where(sim > 0.3)
                    rows, cols = rows[:30], cols[:30]
                    if rows.shape[0] == 0:
                        continue
                    # Assign fixed high weight for confusion edges
                    w = torch.full((rows.shape[0],), CONFUSION_EDGE_WEIGHT,
                                   device=self.device)
                    cx_sl.append(torch.full((rows.shape[0],), rare_lid,
                                            dtype=torch.long, device=self.device))
                    cx_si.append(rows)
                    cx_dl.append(torch.full((rows.shape[0],), clid,
                                            dtype=torch.long, device=self.device))
                    cx_di.append(cols)
                    cx_w.append(w)
                    print(f"    confusion edge: {id2label[rare_lid]} → "
                          f"{id2label[clid]}  ({rows.shape[0]} pairs)")

        if cx_sl:
            self._cx_sl = torch.cat(cx_sl)
            self._cx_si = torch.cat(cx_si)
            self._cx_dl = torch.cat(cx_dl)
            self._cx_di = torch.cat(cx_di)
            self._cx_w  = torch.cat(cx_w)
        else:
            z = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cx_sl = self._cx_si = self._cx_dl = self._cx_di = z
            self._cx_w  = torch.zeros(0, device=self.device)

        n_rare_n = sum(self._stacked[l].shape[0]
                       for l in self.rare_partition if l in self._stacked)
        n_maj_n  = sum(self._stacked[l].shape[0]
                       for l in self._stacked if l not in self.rare_partition)
        n_intra  = sum(self._edge_w[l].shape[0] // 2 for l in self._edge_w)
        print(f"  KG built: maj_nodes={n_maj_n} | rare_nodes={n_rare_n}")
        print(f"           intra={n_intra} | cross={self._cx_w.shape[0]}")

    # ── Save / Load ────────────────────────────────────────
    def save(self, path):
        data = {
            "rare_partition": list(self.rare_partition),
            "nodes": {
                str(lid): [{"emb": n[0].tolist(),
                             "is_virtual": n[1], "is_proto": n[2]}
                            for n in nl]
                for lid, nl in self._cpu_nodes.items()
            },
            "intra": {str(lid): {"idx": self._edge_idx[lid].cpu().tolist(),
                                  "w":   self._edge_w[lid].cpu().tolist()}
                      for lid in self._edge_idx},
            "cross": {
                "sl": self._cx_sl.cpu().tolist() if self._cx_sl is not None else [],
                "si": self._cx_si.cpu().tolist() if self._cx_si is not None else [],
                "dl": self._cx_dl.cpu().tolist() if self._cx_dl is not None else [],
                "di": self._cx_di.cpu().tolist() if self._cx_di is not None else [],
                "w":  self._cx_w.cpu().tolist()  if self._cx_w  is not None else [],
            }
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM, device=DEVICE):
        kg = cls(emb_dim=emb_dim, device=device)
        with open(path) as f:
            data = json.load(f)
        kg.rare_partition = set(data.get("rare_partition", []))
        for k, nl in data["nodes"].items():
            lid = int(k)
            for n in nl:
                emb = torch.tensor(n["emb"], dtype=torch.float32)
                kg._cpu_nodes[lid].append((emb, n["is_virtual"], n["is_proto"]))
        for lid, nl in kg._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in nl]).to(device)
            norms = F.normalize(embs, dim=-1)
            is_v  = torch.tensor([n[1] or n[2] for n in nl],
                                  dtype=torch.bool, device=device)
            kg._stacked[lid] = embs
            kg._norm[lid]    = norms
            kg._is_rare[lid] = is_v
        for k, v in data.get("intra", {}).items():
            lid = int(k)
            kg._edge_idx[lid] = torch.tensor(
                v["idx"], dtype=torch.long,    device=device)
            kg._edge_w[lid]   = torch.tensor(
                v["w"],   dtype=torch.float32, device=device)
        cross = data.get("cross", {})
        if cross and cross.get("sl"):
            kg._cx_sl = torch.tensor(cross["sl"], dtype=torch.long,    device=device)
            kg._cx_si = torch.tensor(cross["si"], dtype=torch.long,    device=device)
            kg._cx_dl = torch.tensor(cross["dl"], dtype=torch.long,    device=device)
            kg._cx_di = torch.tensor(cross["di"], dtype=torch.long,    device=device)
            kg._cx_w  = torch.tensor(cross["w"],  dtype=torch.float32, device=device)
        else:
            z = torch.zeros(0, dtype=torch.long, device=device)
            kg._cx_sl = kg._cx_si = kg._cx_dl = kg._cx_di = z
            kg._cx_w  = torch.zeros(0, device=device)
        for lid in kg.rare_partition:
            proto = [n for n in kg._cpu_nodes.get(lid, []) if n[2]]
            if proto:
                pc = torch.stack([n[0] for n in proto]).to(device)
                pc = F.normalize(pc, dim=-1)
                kg._proto[lid] = kg._proto_norm[lid] = pc
        print(f"  KG loaded ← {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR  (per-class adaptive threshold)
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS,
                 thresh_maj=UNCERTAINTY_THRESH_MAJORITY,
                 thresh_rare=UNCERTAINTY_THRESH_RARE):
        self.log_C       = math.log(num_classes)
        self.thresh_maj  = thresh_maj
        self.thresh_rare = thresh_rare

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        H = -(probs * (probs + 1e-9).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor,
                     is_rare_pred: torch.Tensor) -> torch.Tensor:
        H = self.entropy(logits)
        thresh = torch.where(is_rare_pred,
                             torch.full_like(H, self.thresh_rare),
                             torch.full_like(H, self.thresh_maj))
        return H > thresh

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# FIXED GPU KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    """
    Vectorised retrieval — no nested Python loops over nodes.
    Strategy:
      1. Score subgraphs for ALL triggered queries in one batch mm.
      2. Select top_k subgraphs PER QUERY.
      3. For each unique selected subgraph, gather top_nodes for
         all queries that selected it  (vectorised mm, topk).
      4. 1-hop edge expansion (per-query but inner is isin on GPU).
      5. Cross-edge lookup via masked GPU tensors.
    """

    def __init__(self, kg: KnowledgeGraph, rare_ids: list,
                 top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP,
                 lambda_boost=KG_RARE_LAMBDA):
        self.kg           = kg
        self.rare_ids     = set(rare_ids)
        self.top_k        = top_k
        self.top_nodes    = top_nodes
        self.hop          = hop
        self.lambda_boost = lambda_boost
        self.device       = kg.device
        self.label_ids    = sorted(kg._stacked.keys())

    @classmethod
    def calibrate_lambda(cls, kg, rare_ids, sample_limit=200):
        rare_set  = set(rare_ids)
        maj_ids   = [l for l in kg._stacked if l not in rare_set]
        if not maj_ids:
            return KG_RARE_LAMBDA
        maj_norms = torch.cat(
            [kg._norm[m][:KG_MAX_NODES_PER_SG] for m in maj_ids if m in kg._norm],
            dim=0)
        gaps = []
        for lid in rare_ids:
            if lid not in kg._norm:
                continue
            own = kg._norm[lid]
            N   = own.shape[0]
            idx = torch.randperm(N, device=kg.device)[:sample_limit]
            s   = own[idx]
            own_sims = torch.mm(s, own.T).max(1).values
            maj_sims = torch.mm(s, maj_norms.T).max(1).values
            gaps.append((maj_sims - own_sims).cpu())
        if not gaps:
            return KG_RARE_LAMBDA
        lam = float(np.clip(torch.cat(gaps).median().item(), 0.05, 0.40))
        print(f"  λ auto-calibrated → {lam:.4f}")
        return lam

    def retrieve_batch(self, H_q: torch.Tensor,
                       trigger_mask: torch.Tensor) -> tuple:
        """
        H_q          : (T, D) L2-normalised sentence embeddings (GPU)
        trigger_mask : (T,) bool
        Returns (nb_embs, nb_weights, nb_is_rare) — all GPU, padded.
        """
        if not trigger_mask.any():
            return None, None, None

        trig_idx = trigger_mask.nonzero(as_tuple=True)[0]  # (T',)
        H_trig   = H_q[trig_idx]                           # (T', D)
        T_prime, D = H_trig.shape

        # ── Step 1: score subgraphs per query ─────────────
        sg_sims = []
        for lid in self.label_ids:
            n = self.kg._norm[lid]
            if n.shape[0] == 0:
                sg_sims.append(torch.full((T_prime,), -1.0, device=self.device))
                continue
            cap = min(n.shape[0], KG_MAX_NODES_PER_SG)
            s   = torch.mm(H_trig, n[:cap].T).max(1).values  # (T',)
            boost = self.lambda_boost if lid in self.rare_ids else 0.0
            sg_sims.append(s + boost)

        sg_score_mat = torch.stack(sg_sims, dim=1)  # (T', num_labels)
        _, topk_idx  = sg_score_mat.topk(
            min(self.top_k, len(self.label_ids)), dim=1)   # (T', top_k)

        # ── Step 2: allocate output tensors ───────────────
        cap_out = self.top_k * (self.top_nodes + 8)
        nb_embs    = torch.zeros(T_prime, cap_out, D, device=self.device)
        nb_weights = torch.zeros(T_prime, cap_out,    device=self.device)
        nb_is_rare = torch.zeros(T_prime, cap_out,
                                 dtype=torch.bool, device=self.device)
        fill = torch.zeros(T_prime, dtype=torch.long, device=self.device)

        # ── Step 3: for each unique subgraph, gather nodes ─
        unique_sg = topk_idx.unique().tolist()
        for sg_idx in unique_sg:
            lid        = self.label_ids[sg_idx]
            is_rare_sg = lid in self.rare_ids
            norms      = self.kg._norm[lid]
            embs       = self.kg._stacked[lid]
            N_sg       = norms.shape[0]
            if N_sg == 0:
                continue

            # Which queries selected this subgraph?
            uses = (topk_idx == sg_idx).any(dim=1)  # (T',)
            q_idx = uses.nonzero(as_tuple=True)[0]  # (Q,)
            H_sub = H_trig[q_idx]                   # (Q, D)

            cap_n  = min(N_sg, KG_MAX_NODES_PER_SG)
            cap_e  = embs[:cap_n]
            cap_nr = norms[:cap_n]

            # (Q, cap_n) → topk per query
            sim_mat = torch.mm(H_sub, cap_nr.T)     # (Q, cap_n)
            k_q     = min(self.top_nodes, cap_n)
            topn    = sim_mat.topk(k_q, dim=1)
            topn_idx  = topn.indices   # (Q, k_q)
            topn_sims = topn.values    # (Q, k_q)

            # ── 1-hop edge expansion ───────────────────────
            edge = self.kg._edge_idx.get(lid)

            for qi_local, qi_global in enumerate(q_idx.tolist()):
                seed = topn_idx[qi_local]          # (k_q,) GPU

                if self.hop >= 1 and edge is not None and edge.shape[1] > 0:
                    src, dst = edge[0], edge[1]
                    in_top = torch.isin(src, seed)
                    hop_n  = dst[in_top].unique()
                    if hop_n.shape[0] > 0:
                        hop_n  = hop_n[hop_n < cap_n]
                        hop_s  = torch.mv(cap_nr[hop_n], H_trig[qi_global])
                        seed   = torch.cat([seed, hop_n]).unique()

                # Write top-n nodes
                f = int(fill[qi_global].item())
                n_add = min(k_q, cap_out - f)
                if n_add <= 0:
                    continue
                use_i = topn_idx[qi_local, :n_add]
                use_s = topn_sims[qi_local, :n_add].clamp(min=0)
                nb_embs[qi_global, f:f+n_add]    = cap_e[use_i]
                nb_weights[qi_global, f:f+n_add] = use_s
                nb_is_rare[qi_global, f:f+n_add] = is_rare_sg
                fill[qi_global] = f + n_add

        # ── Step 4: cross-edge expansion ──────────────────
        cx_sl = self.kg._cx_sl
        if cx_sl is not None and cx_sl.shape[0] > 0:
            for qi_global in range(T_prime):
                f = int(fill[qi_global].item())
                if f >= cap_out:
                    continue
                # Find cross-edges where src matches any filled node
                # (simplified: check per source-label)
                # We match on label level for speed
                used_lids = topk_idx[qi_global].unique()
                for ul_idx in used_lids.tolist():
                    lid    = self.label_ids[ul_idx]
                    mask_l = (cx_sl == lid)
                    if not mask_l.any():
                        continue
                    c_di  = self.kg._cx_di[mask_l]
                    c_dl  = self.kg._cx_dl[mask_l]
                    c_w   = self.kg._cx_w[mask_l]
                    # Group by dst label
                    for dlid_t in c_dl.unique().tolist():
                        dlid = int(dlid_t)
                        dstk = self.kg._stacked.get(dlid)
                        if dstk is None:
                            continue
                        mask_d = (c_dl == dlid_t)
                        di_v   = c_di[mask_d][:3]   # max 3 cross neighbours
                        w_v    = c_w[mask_d][:3]
                        di_v   = di_v[di_v < dstk.shape[0]]
                        n_a    = min(di_v.shape[0], cap_out - f)
                        if n_a <= 0:
                            continue
                        nb_embs[qi_global, f:f+n_a]    = dstk[di_v[:n_a]]
                        nb_weights[qi_global, f:f+n_a] = w_v[:n_a]
                        nb_is_rare[qi_global, f:f+n_a] = (dlid in self.rare_ids)
                        f += n_a
                fill[qi_global] = f

        return nb_embs, nb_weights, nb_is_rare


# ═══════════════════════════════════════════════════════════
# GATED GRAPH ATTENTION FUSION  (NEW — replaces naive residual)
# ═══════════════════════════════════════════════════════════
class GatedGraphAttentionFusion(nn.Module):
    """
    Batched GAT with learned gate.

    Inputs:
        H_q     : (T', D)      – query sentence vectors
        nb_embs : (T', K, D)   – neighbour embeddings (padded)
        nb_w    : (T', K)      – edge weights (0 = padding)
        nb_rare : (T', K) bool – rare-class flags

    Output: v (T', D) — fused, gated update.

    Gate g = σ(W[H_q; v_attn]) controls how much KG context is used.
    Residual: h* = (1-g)·H_q + g·v_attn
    """

    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT,
                 r_boost=KG_RARE_GAMMA):
        super().__init__()
        self.r_boost = r_boost
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.gate    = nn.Linear(emb_dim * 2, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, H_q, nb_embs, nb_w, nb_rare):
        """Returns h* (T', D) — the gated fused representation."""
        pad_mask = (nb_w == 0)                                    # (T', K)

        q = self.proj_q(H_q)                                      # (T', D)
        k = self.proj_k(nb_embs)                                   # (T', K, D)
        dot = torch.bmm(k, q.unsqueeze(-1)).squeeze(-1) * self.scale  # (T', K)

        # r_boost: amplify rare neighbours
        r_b  = torch.where(nb_rare,
                           torch.full_like(dot, self.r_boost),
                           torch.ones_like(dot))
        raw  = dot * nb_w * r_b
        raw  = raw.masked_fill(pad_mask, -1e9)

        alpha = F.softmax(raw, dim=-1)          # (T', K)
        alpha = alpha.masked_fill(pad_mask, 0.0)
        alpha = self.dropout(alpha)

        v_attn = torch.bmm(alpha.unsqueeze(1), nb_embs).squeeze(1)  # (T', D)

        # Gating: g ∈ (0,1) per dimension
        g = torch.sigmoid(self.gate(torch.cat([H_q, v_attn], dim=-1)))  # (T', D)
        h_star = (1.0 - g) * H_q + g * v_attn
        return h_star


# ═══════════════════════════════════════════════════════════
# HARD EXAMPLE REPLAY BUFFER  (NEW)
# ═══════════════════════════════════════════════════════════
class HardExampleBuffer:
    """
    Stores (loss_val, CPU-batch) tuples for high-loss rare batches.
    Samples proportional to loss magnitude for Phase B replay.
    """

    def __init__(self, max_size=HARD_BUFFER_SIZE):
        self.max_size = max_size
        self.buf: list = []   # [(loss_float, batch_cpu_tuple)]

    def update(self, loss_val: float, batch_cpu: tuple):
        self.buf.append((loss_val, batch_cpu))
        if len(self.buf) > self.max_size:
            self.buf.sort(key=lambda x: -x[0])
            self.buf = self.buf[:self.max_size // 2]

    def sample(self) -> tuple | None:
        if not self.buf:
            return None
        losses  = torch.tensor([x[0] for x in self.buf], dtype=torch.float)
        probs   = F.softmax(losses, dim=0)
        idx     = int(torch.multinomial(probs, 1).item())
        return self.buf[idx][1]

    def __len__(self):
        return len(self.buf)


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL  (gated fusion + prototype contrastive)
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):

    def __init__(self, base_model: InLegalBERT_BiLSTM_MHA_CRF,
                 kg: KnowledgeGraph,
                 rare_ids: list,
                 retriever: KGRetriever = None,
                 focal_loss: FocalLoss  = None):
        super().__init__()
        self.base       = base_model
        self.kg         = kg
        self.rare_ids   = set(rare_ids)
        self.rare_list  = sorted(rare_ids)
        self.retriever  = retriever or KGRetriever(kg, rare_ids)
        self.uncertainty = UncertaintyEstimator()
        self.focal_loss  = focal_loss  # may be None

        sent_dim = base_model.sent_out_dim  # 256
        ctx_dim  = base_model.ctx_out_dim   # 128

        self.gat_fusion = GatedGraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        self._rare_t = torch.tensor(self.rare_list, dtype=torch.long)

    def _rare_mask(self, top_labels):
        rare = self._rare_t.to(top_labels.device)
        return (top_labels.unsqueeze(-1) == rare.view(1, 1, -1)).any(-1)

    # ── Prototype contrastive loss ─────────────────────────
    def _proto_contrast_loss(self, device) -> torch.Tensor:
        """
        Push rare prototypes away from majority prototypes.
        Loss = mean max(0, sim(rare_p, maj_p) - margin)
        """
        rare_protos = [self.kg._proto_norm[l] for l in self.rare_list
                       if l in self.kg._proto_norm]
        if not rare_protos:
            return torch.tensor(0.0, device=device)

        maj_ids    = [l for l in self.kg._stacked if l not in self.rare_ids]
        maj_protos = [self.kg._proto_norm[l] for l in maj_ids
                      if l in self.kg._proto_norm]
        if not maj_protos:
            return torch.tensor(0.0, device=device)

        rp = F.normalize(torch.cat(rare_protos, dim=0), dim=-1).to(device)  # (R, D)
        mp = F.normalize(torch.cat(maj_protos,  dim=0), dim=-1).to(device)  # (M, D)
        sim = torch.mm(rp, mp.T)     # (R, M)
        loss = torch.clamp(sim - PROTO_CONTRAST_MARGIN, min=0.0).mean()
        return loss

    # ── Vectorised KG fusion ──────────────────────────────
    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, D = sent_vecs.shape
        fused       = sent_vecs.clone()
        top_labels  = self.uncertainty.top_label(emissions)    # (B, T)
        is_rare_p   = self._rare_mask(top_labels)              # (B, T)
        uncertain   = self.uncertainty.is_uncertain(emissions, is_rare_p)
        trigger     = uncertain | (RARE_ALWAYS_KG & is_rare_p)

        for b in range(B):
            n      = int(lengths[b].item())
            trig_b = trigger[b, :n]
            if not trig_b.any():
                continue

            H_b      = sent_vecs[b, :n]                           # (n, D)
            H_b_norm = F.normalize(H_b.detach(), dim=-1)          # for retrieval

            nb_embs, nb_w, nb_rare = self.retriever.retrieve_batch(
                H_b_norm, trig_b)
            if nb_embs is None:
                continue

            trig_idx = trig_b.nonzero(as_tuple=True)[0]           # (T',)
            H_q      = H_b[trig_idx]                              # (T', D)

            # Gated GAT fusion
            h_star = self.gat_fusion(H_q, nb_embs, nb_w, nb_rare)  # (T', D)
            fused[b, trig_idx] = h_star  # gate handles blending internally

        return fused

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        device = input_ids.device

        # Step 1: base first pass
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        # Step 2: KG fusion
        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device)

        # Step 3: re-run ctx-BiLSTM on fused representations
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)
        fused_ctx = self.base.dropout(fused_ctx)

        # Step 4: fusion classifier
        fused_emis = self.fusion_classifier(fused_ctx)
        fused_emis = torch.nan_to_num(fused_emis, nan=0.0, posinf=1e4, neginf=-1e4)

        # Mask
        if lengths is not None:
            B, T, _ = fused_emis.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emis.shape[:2], dtype=torch.bool, device=device)

        combined = (base_emissions + fused_emis) / 2.0

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0

            base_crf  = -self.base.crf(
                base_emissions, safe, mask=mask, reduction="mean")
            fused_crf = -self.fusion_crf(
                fused_emis,     safe, mask=mask, reduction="mean")

            B2, T2, C = combined.shape
            flat_logits = combined.reshape(B2*T2, C)
            flat_labels = labels.reshape(B2*T2)

            if self.focal_loss is not None:
                aux_loss = self.focal_loss(flat_logits, flat_labels)
            else:
                aux_loss = self.ce_loss(flat_logits, flat_labels)

            proto_loss = self._proto_contrast_loss(device)
            loss = ((base_crf + fused_crf) / 2.0
                    + AUX_CE_WEIGHT * aux_loss
                    + PROTO_CONTRAST_WEIGHT * proto_loss)
            return loss, combined
        else:
            decoded = self.fusion_crf.decode(fused_emis, mask=mask)
            return decoded, fused_emis


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds,
                                     labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds,
                                   labels=list(range(NUM_LABELS)),
                                   average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {"f1": float(per_class_f1[i]),
                      "precision": float(per_class_prec[i]),
                      "recall":    float(per_class_rec[i])}
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_t  = [id2label[x] for x in all_trues]
    str_p  = [id2label[x] for x in all_preds]
    cls_rp = classification_report(str_t, str_p, labels=LABELS,
                                    digits=4, zero_division=0)
    cm     = confusion_matrix(str_t, str_p, labels=LABELS)

    return dict(
        macro_f1=macro_f1, micro_f1=micro_f1, weighted_f1=weighted_f1,
        macro_precision=macro_prec, micro_precision=micro_prec,
        weighted_precision=weighted_prec,
        macro_recall=macro_rec, micro_recall=micro_rec,
        weighted_recall=weighted_rec,
        rare_f1=rare_f1, rare_precision=rare_prec, rare_recall=rare_rec,
        per_class_metrics=per_class_metrics,
        accuracy=acc, cls_report=cls_rp, cm=cm,
        all_preds=all_preds, all_trues=all_trues,
    )


def count_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if     p.requires_grad)
    fr = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {tr:,} | Frozen: {fr:,}")
    return tr, fr


# ═══════════════════════════════════════════════════════════
# CONFUSION-PAIR COMPUTATION  (Phase A → KG bridge)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def compute_confusion_pairs(model, dataset, rare_ids,
                             device=DEVICE, top_k=CONFUSION_TOP_K) -> dict:
    """
    Evaluate base model on train set. For each rare class r, find
    the top-K majority classes that predictions most confuse it with.
    Returns: { rare_lid: [confused_lid, ...] }
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=2, shuffle=False,
                        collate_fn=collate_rrc)
    all_preds, all_trues = [], []
    rare_set = set(rare_ids)

    for ids, attn, ttype, labels, lengths in loader:
        ids     = ids.to(device)
        attn    = attn.to(device)
        ttype   = ttype.to(device)
        lengths = lengths.to(device)
        decoded, _ = model(ids, attn, ttype, labels=None, lengths=lengths)
        for i, seq_p in enumerate(decoded):
            true_len = int(lengths[i].item())
            all_preds.extend(seq_p)
            all_trues.extend(labels[i, :true_len].tolist())

    confusion = defaultdict(Counter)
    for true, pred in zip(all_trues, all_preds):
        if true in rare_set and pred != true:
            confusion[true][pred] += 1

    result = {}
    print("\n📊 Confusion-guided edge analysis:")
    for rid, counter in confusion.items():
        top_confused = [cls for cls, _ in counter.most_common(top_k)]
        result[rid] = top_confused
        names = [id2label[c] for c in top_confused]
        total = sum(counter.values())
        print(f"  {id2label[rid]:<20} confused → {names}  "
              f"(total misclassified: {total})")

    # Also print per-rare CM summary from sklearn
    str_t = [id2label[x] for x in all_trues]
    str_p = [id2label[x] for x in all_preds]
    rare_labels = [id2label[r] for r in rare_ids]
    cm = confusion_matrix(str_t, str_p, labels=LABELS)
    print("\n  Full confusion matrix saved for analysis.")

    return result


# ═══════════════════════════════════════════════════════════
# BASE TRAINER  (Phase A — with focal loss + oversampling)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, focal_loss: FocalLoss = None, device=DEVICE):
        self.model      = model.to(device)
        self.focal_loss = focal_loss
        self.device     = device

    def build_optimizer(self):
        pg = []
        pg.append({"params": list(self.model.bert.pooler.parameters()),
                   "lr": BERT_LR, "weight_decay": WEIGHT_DECAY})
        enc = self.model.bert.encoder.layer
        n   = len(enc)
        for i in range(n - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in enc[i].parameters() if p.requires_grad]
            if params:
                pg.append({"params": params, "lr": lr_i,
                           "weight_decay": WEIGHT_DECAY})
        head_mods = [self.model.sent_bilstm, self.model.mha_pooling,
                     self.model.sent_layer_norm, self.model.ctx_bilstm,
                     self.model.classifier, self.model.crf]
        head_p = [p for m in head_mods for p in m.parameters()]
        pg.append({"params": head_p, "lr": HEAD_LR,
                   "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(pg)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                tt  = tt.to(self.device);  labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, tt,
                                     labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total += loss.item(); n += 1
        return total / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, tt,
                                        labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :tl].tolist())

        if measure_inference_time:
            ti  = time.time() - t0
            ns  = len(all_trues)
            inf = {"total_inference_time_s": ti,
                   "latency_per_document_ms": ti / max(1, n_samples) * 1000,
                   "throughput_sentences_per_s": ns / max(1e-9, ti)}
            with open(os.path.join(OUT_DIR,
                                   f"inference_{split_name}.json"), "w") as f:
                json.dump(inf, f, indent=2)
        else:
            inf = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if inf:
            metrics["inference_time_info"] = inf
        return metrics

    def train(self, train_docs, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):

        # Oversampling via WeightedRandomSampler
        sampler      = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)

        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)

        early_stopper = EarlyStopping()
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            t0 = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, tt, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  labels = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, emissions = self.model(
                    ids, attn, tt, labels=labels, lengths=lengths)

                # Add focal loss auxiliary on top of base CRF loss
                if self.focal_loss is not None:
                    B2, T2, C = emissions.shape
                    fl = self.focal_loss(
                        emissions.reshape(B2*T2, C).detach(),
                        labels.reshape(B2*T2))
                    loss = loss + AUX_CE_WEIGHT * fl

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                run_loss += loss.item(); n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            ep_t     = time.time() - t0
            avg_loss = run_loss / max(1, n_steps)
            val_loss = self.compute_val_loss(dev_dataset)
            val_m    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Ep {epoch:03d}/{num_epochs} | "
                f"tr_loss:{avg_loss:.4f} val_loss:{val_loss:.4f} | "
                f"mac_F1:{val_m['macro_f1']:.4f} rare_F1:{val_m['rare_f1']:.4f} | "
                f"t:{ep_t:.1f}s ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_loss, "val_loss": val_loss,
                "val_macro_f1": val_m["macro_f1"],
                "val_micro_f1": val_m["micro_f1"],
                "val_weighted_f1": val_m["weighted_f1"],
                "val_rare_f1": val_m["rare_f1"],
                "val_accuracy": val_m["accuracy"],
                "epoch_train_time_s": ep_t,
            })

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_m["macro_f1"]):
                print(f"\n⏹  Base early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "base_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")

        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))
            print(f"  Base model saved to {BEST_MODEL_DIR}/base_model.bin")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer,
                           rare_ids, confusion_pairs, device=DEVICE) -> KnowledgeGraph:
    print("\n🔨 Building Dual-Partition KG + Confusion Edges ...")
    base_model.eval(); base_model.to(device)

    kg     = KnowledgeGraph(emb_dim=base_model.sent_out_dim, device=device)
    dummy  = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy, batch_size=4, shuffle=False,
                        collate_fn=collate_rrc, num_workers=0)

    for doc_idx, (ids, attn, tt, labels, lengths) in enumerate(loader):
        ids  = ids.to(device); attn = attn.to(device); tt = tt.to(device)
        for bi in range(ids.shape[0]):
            sv = base_model.encode_sentences(
                ids[bi:bi+1], attn[bi:bi+1], tt[bi:bi+1]).squeeze(0)
            n  = int(lengths[bi].item())
            kg.add_nodes(sv[:n].cpu(), labels[bi, :n].tolist())
        if (doc_idx + 1) % 50 == 0:
            print(f"  {(doc_idx+1)*ids.shape[0]}/{len(train_docs)} docs processed")

    kg.build_edges(rare_ids=rare_ids, confusion_pairs=confusion_pairs)
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER  (Phase B — with hard example replay)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: KGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_p = (list(self.model.gat_fusion.parameters()) +
                 list(self.model.fusion_classifier.parameters()) +
                 list(self.model.fusion_crf.parameters()))
        base_p = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_p,  "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_p, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, tt,
                                        labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :tl].tolist())
        if measure_inference_time:
            ti  = time.time() - t0
            ns  = len(all_trues)
            inf = {"total_inference_time_s": ti,
                   "latency_per_document_ms": ti / max(1, n_samples) * 1000,
                   "throughput_sentences_per_s": ns / max(1e-9, ti)}
            with open(os.path.join(OUT_DIR,
                                   f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(inf, f, indent=2)
        else:
            inf = None
        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if inf:
            metrics["inference_time_info"] = inf
        return metrics

    def train(self, train_docs, train_dataset, dev_dataset,
              rare_ids, num_epochs=NUM_EPOCHS_KG):
        # Oversampling of rare docs in Phase B as well
        sampler      = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)

        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)

        early_stopper = EarlyStopping(patience=7)
        buffer        = HardExampleBuffer(max_size=HARD_BUFFER_SIZE)
        rare_set      = set(rare_ids)
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            t0 = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, tt, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  labels = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, tt,
                                     labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                run_loss += loss.item(); n_steps += 1

                # ── Hard example buffering ─────────────────
                lv = loss.item()
                if lv > HARD_REPLAY_LOSS_THRESH:
                    flat_l = labels.cpu().flatten()
                    has_rare = any(int(x) in rare_set
                                   for x in flat_l if x.item() >= 0)
                    if has_rare:
                        buffer.update(lv, (ids.cpu(), attn.cpu(),
                                           tt.cpu(), labels.cpu(),
                                           lengths.cpu()))

                # ── Hard example replay ────────────────────
                if (step + 1) % HARD_REPLAY_FREQ == 0 and len(buffer) >= 10:
                    replay = buffer.sample()
                    if replay:
                        r_ids, r_attn, r_tt, r_lab, r_len = [
                            x.to(self.device) for x in replay]
                        r_loss, _ = self.model(r_ids, r_attn, r_tt,
                                               labels=r_lab, lengths=r_len)
                        if not torch.isnan(r_loss) and not torch.isinf(r_loss):
                            (r_loss * HARD_REPLAY_WEIGHT).backward()
                            torch.nn.utils.clip_grad_norm_(
                                self.model.parameters(), GRAD_CLIP)
                            optimizer.step(); scheduler.step(); optimizer.zero_grad()

            ep_t     = time.time() - t0
            avg_loss = run_loss / max(1, n_steps)
            val_m    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG]  Ep {epoch:02d}/{num_epochs} | "
                f"tr_loss:{avg_loss:.4f} | "
                f"mac_F1:{val_m['macro_f1']:.4f} "
                f"rare_F1:{val_m['rare_f1']:.4f} | "
                f"t:{ep_t:.1f}s buf:{len(buffer)} "
                f"ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "kg",
                "train_loss": avg_loss,
                "val_macro_f1": val_m["macro_f1"],
                "val_rare_f1":  val_m["rare_f1"],
                "val_accuracy": val_m["accuracy"],
                "epoch_train_time_s": ep_t,
            })

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_m["macro_f1"]):
                print(f"\n⏹  KG early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "kg_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS,
                yticklabels=LABELS, cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (KG-RAG v3)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1  (KG-RAG v3)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"],
            marker="o", markersize=3, label="Train Loss")
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Combined Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)
    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1",  marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1  (Base → KG-RAG v3)"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def print_metrics_table(dev_m, test_m, base_t=None, kg_t=None, n_params=None):
    rows = [
        ("Accuracy",          "accuracy"),
        ("Macro-F1",          "macro_f1"),
        ("Micro-F1",          "micro_f1"),
        ("Weighted-F1",       "weighted_f1"),
        ("Minority Macro-F1", "rare_f1"),
        ("Macro-Precision",   "macro_precision"),
        ("Macro-Recall",      "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  — InLegalBERT+BiLSTM+MHA+CRF+KG-RAG v3")
    print("=" * 72)
    if n_params: print(f"  Trainable params : {n_params:,}")
    if base_t:   print(f"  Phase A time     : {base_t/60:.1f} min")
    if kg_t:     print(f"  Phase B time     : {kg_t/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<30} {'Dev':>10} {'Test':>10}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<30} {dev_m[key]:>10.4f} {test_m[key]:>10.4f}")
    print("=" * 72)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 66)
    print(f"  {'Label':<22} {'F1-Dev':>8} {'F1-Test':>8} "
          f"{'Prec':>8} {'Rec':>8}")
    print("  " + "-" * 66)
    for lbl in LABELS:
        dv = dev_m["per_class_metrics"][lbl]
        ts = test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>8.4f} {ts['f1']:>8.4f} "
              f"{ts['precision']:>8.4f} {ts['recall']:>8.4f}")
    print("  " + "-" * 66)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT+BiLSTM+MHA+CRF → KG-RAG v3")
    print("  [+] Focal Loss  [+] Oversampling  [+] Confusion Edges")
    print("  [+] Hard Mining  [+] Gated Fusion  [+] Prototype Contrast\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train:{len(train_docs)} | Dev:{len(dev_docs)} | Test:{len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    pd.DataFrame([{"label": l, "frequency": label_freqs[l],
                   "is_rare": l in rare_labels}
                  for l in LABELS]
                 ).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    # ── Focal loss + class weights ─────────────────────────
    class_weights = compute_class_weights(label_freqs)
    focal_loss    = FocalLoss(class_weights, rare_ids).to(DEVICE)
    print(f"\nClass weights (top-5 rare): "
          f"{[(id2label[r], round(float(class_weights[r]),2)) for r in rare_ids[:5]]}")

    print("Loading tokenizer ...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ══════════════════════════════════════════════════════
    # PHASE A: Train base model
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training (focal loss + oversampling)")
    print("=" * 60)

    base_model   = InLegalBERT_BiLSTM_MHA_CRF()
    base_trainer = BaseTrainer(base_model, focal_loss=focal_loss, device=DEVICE)
    base_hist, base_time = base_trainer.train(
        train_docs, train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE)

    # ══════════════════════════════════════════════════════
    # PHASE A→B: Confusion-guided KG construction
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Confusion-Matrix Analysis + KG Construction")
    print("=" * 60)

    # Compute which majority classes each rare class is confused with
    confusion_pairs = compute_confusion_pairs(
        base_model, train_dataset, rare_ids, device=DEVICE)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG — loading ...")
        kg = KnowledgeGraph.load(
            kg_path, emb_dim=base_model.sent_out_dim, device=DEVICE)
        kg.rare_partition = set(rare_ids)
    else:
        kg = build_knowledge_graph(
            base_model, train_docs, tokenizer,
            rare_ids=rare_ids, confusion_pairs=confusion_pairs, device=DEVICE)

    # Auto-calibrate λ
    print("\n  Calibrating λ ...")
    lambda_boost = KGRetriever.calibrate_lambda(kg, rare_ids)

    # ══════════════════════════════════════════════════════
    # PHASE B: KG-augmented fine-tuning with hard mining
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: KG-Augmented Fine-Tuning + Hard Example Replay")
    print(f"  λ={lambda_boost:.4f}  γ={KG_RARE_GAMMA}  "
          f"focal_γ_rare={FOCAL_GAMMA_RARE}  "
          f"thresh_rare={UNCERTAINTY_THRESH_RARE}")
    print("=" * 60)

    retriever = KGRetriever(
        kg, rare_ids=rare_ids,
        top_k=KG_TOP_K, top_nodes=KG_TOP_NODES,
        hop=KG_HOP, lambda_boost=lambda_boost)

    kg_model = KGAugmentedModel(
        base_model=base_model, kg=kg,
        rare_ids=rare_ids, retriever=retriever,
        focal_loss=focal_loss)

    n_params, _ = count_parameters(kg_model)

    kg_trainer = KGTrainer(kg_model, device=DEVICE)
    kg_hist, kg_time = kg_trainer.train(
        train_docs, train_dataset, dev_dataset,
        rare_ids=rare_ids, num_epochs=NUM_EPOCHS_KG)

    plot_combined_history(base_hist, kg_hist)

    # ══════════════════════════════════════════════════════
    # EVALUATION
    # ══════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_m = kg_trainer.evaluate(dev_dataset, rare_ids,
                                split_name="dev",
                                measure_inference_time=True)
    print(f"  Dev  Accuracy:{dev_m['accuracy']:.4f} "
          f"Macro-F1:{dev_m['macro_f1']:.4f} "
          f"Rare-F1:{dev_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF+KG-RAG v3\n")
        f.write(f"Techniques: FocalLoss, Oversampling, ConfusionEdges, "
                f"HardMining, GatedFusion, ProtoContrast\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_m["cls_report"])

    save_confusion_matrix(dev_m["cm"],  "dev",  rare_labels)
    save_per_class_f1_chart(dev_m["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_m = kg_trainer.evaluate(test_dataset, rare_ids,
                                 split_name="test",
                                 measure_inference_time=True)
    print(f"  Test Accuracy:{test_m['accuracy']:.4f} "
          f"Macro-F1:{test_m['macro_f1']:.4f} "
          f"Rare-F1:{test_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF+KG-RAG v3\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_m["cls_report"])

    save_confusion_matrix(test_m["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_m["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_m["all_trues"]],
        "pred": [id2label[x] for x in test_m["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall",    "micro_recall",    "weighted_recall",    "rare_recall",
        "accuracy",
    ]
    summary = {
        "model": "InLegalBERT+BiLSTM+MHA+CRF+KG-RAG-v3",
        "improvements": [
            "FocalLoss(gamma_rare=2.5,gamma_maj=1.0)+InvFreqWeights",
            "WeightedRandomSampler(ratio=3.0)",
            "ConfusionMatrixGuidedEdges",
            "HardExampleReplayBuffer",
            "GatedGraphAttentionFusion",
            "PrototypeContrastiveLoss",
            "AdaptiveUncertaintyThreshold",
        ],
        "kg_config": {
            "lambda_boost":  lambda_boost,
            "r_boost_gamma": KG_RARE_GAMMA,
            "proto_k":       KG_PROTOTYPE_K,
            "virtual_alphas": KG_VIRTUAL_ALPHAS,
            "thresh_rare":   UNCERTAINTY_THRESH_RARE,
            "thresh_maj":    UNCERTAINTY_THRESH_MAJORITY,
            "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES, "hop": KG_HOP,
        },
        "timing": {"phase_a_s": base_time, "phase_b_s": kg_time,
                   "total_s": base_time + kg_time},
        "rare_classes": rare_labels,
        "dev":  {k: dev_m[k]  for k in scalar_keys},
        "test": {k: test_m[k] for k in scalar_keys},
        "per_class_dev":  dev_m["per_class_metrics"],
        "per_class_test": test_m["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_metrics_table(dev_m, test_m,
                        base_t=base_time, kg_t=kg_time, n_params=n_params)
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT+BiLSTM+MHA+CRF → KG-RAG v3
  [+] Focal Loss  [+] Oversampling  [+] Confusion Edges
  [+] Hard Mining  [+] Gated Fusion  [+] Prototype Contrast

Loading JSONL files ...
  Train:245 | Dev:30 | Test:50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + 0-7.
🔥 BERT trainable: 8-11 + pooler.

[Base] Ep 001/60 | tr_loss:284.8021 val_loss:211.6492 | mac_F1:0.0391 rare_F1:0.0000 | t:61.2s ES:0/10
  ✔ New best val_macro_f1=0.0391
[Base] Ep 002/60 | tr_loss:231.5278 val_loss:171.5741 | mac_F1:0.0727 rare_F1:0.0000 | t:62.5s ES:0/10
  ✔ New best val_macro_f1=0.0727
[Base] Ep 003/60 | tr_loss:190.3178 val_loss:131.4388 | mac_F1:0.2161 rare_F1:0.0598 | t:59.0s ES:0/10
  ✔ New best val_macro_f1=0.2161
[Base] Ep 004/60 | tr_loss:153.1226 val_loss:100.8132 | mac_F1:0.2719 rare_F1:0.1128 | t:62.5s ES:0/10
  ✔ New best val_macro_f1=0.2719
[Base] Ep 005/60 | tr_loss:141.0998 val_loss:93.3196 | mac_F1:0.2704 rare_F1:0.1118 | t:65.2s ES:0/10
[Base] Ep 006/60 | tr_loss:129.2616 val_loss:86.8265 | mac_F1:0.2998 rare_F1:0.1419 | t:84.6s ES:1/10
  ✔ New best val_macro_f1=0.2998
[Base] Ep 007/60 | tr_loss:120.6087 val_loss:80.5756 | mac_F1:0.3316 rare_F1:0.1776 | t:70.6s ES:0/10
  ✔ New best val_macro_f1=0.3316
[Base] 

In [4]:
#python

# inlegalbert_kg_rag_mixup_v4.py
#
# Architecture:
#   InLegalBERT → BiLSTM → Multi-Head Attention Pooling → CRF
#   + Dual-Partition KG (𝒢_maj + 𝒢_rare) — GPU-resident tensors
#   + Confusion-Matrix-Guided Cross-Edges
#   + Weighted Focal Loss + Inverse-Freq Class Weights
#   + Rare Document Oversampling via WeightedRandomSampler
#   + Hard Example Replay Buffer in Phase B
#   + Gated Graph Attention Fusion
#   + Prototype Contrastive Auxiliary Loss
#   + Per-class Adaptive Uncertainty Threshold
#   + [NEW v4] GPU Manifold Mixup on Rare-Class Embeddings
#   + [NEW v4] GPU KG-Guided Mixup (mix within KG subgraph neighbours)
#   + [NEW v4] GPU Batch-Level Rare Mixup Augmentation (inline, no CPU)
#   + [NEW v4] Mixup-Aware CRF loss (soft label interpolation)
#   + [NEW v4] Rare-Rare and Rare-Hard Majority Mixup Strategies
#
# KEY MIXUP IMPROVEMENTS vs plain Mixup Embedding (table row 0.6307):
#   ① Manifold Mixup at sentence-encoder level (richer interpolation)
#   ② KG-neighbour-guided Mixup: mix query with its retrieved KG neighbours
#      so interpolated samples lie on the rare-class manifold (not random)
#   ③ Rare↔Rare intra-class Mixup: densifies the rare class manifold
#   ④ Rare↔HardMaj inter-class Mixup: sharpens the decision boundary
#   ⑤ All operations are pure GPU (no CPU round-trips)
#   ⑥ Mixup alpha sampled per-batch from Beta(α,α) on GPU
#   ⑦ Soft CRF target blending: avoids hard-label confusion during Mixup
#   ⑧ Mixup integrated into KG-augmented Phase B so both losses combine

import os, json, random, time, math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)
from sklearn.cluster import MiniBatchKMeans

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_mixup_v4_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 25
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

LABEL_SMOOTHING  = 0.05
AUX_CE_WEIGHT    = 0.25
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
WARMUP_RATIO     = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD   = 0.05

# ── KG-RAG ────────────────────────────────────────────────
KG_TOP_K        = 3
KG_TOP_NODES    = 6
KG_HOP          = 1
KG_FUSION_DIM   = 256    # SENT_OUT_DIM = 128 * 2

RST_INTRA_THRESH = 0.55
RST_CROSS_THRESH = 0.45

# ── Uncertainty (per-class adaptive) ─────────────────────
UNCERTAINTY_THRESH_MAJORITY = 0.70
UNCERTAINTY_THRESH_RARE     = 0.30

RARE_ALWAYS_KG           = True
KG_VIRTUAL_ALPHAS        = [0.25, 0.50, 0.75]
KG_PROTOTYPE_K           = 5
KG_RARE_LAMBDA           = 0.20
KG_RARE_GAMMA            = 2.0
KG_MAX_VIRTUAL_PER_LABEL = 200

# ── v3 hyperparameters ────────────────────────────────────
FOCAL_GAMMA_RARE         = 2.5
FOCAL_GAMMA_MAJ          = 1.0
OVERSAMPLE_RARE_RATIO    = 3.0
HARD_BUFFER_SIZE         = 300
HARD_REPLAY_FREQ         = 4
HARD_REPLAY_LOSS_THRESH  = 0.8
HARD_REPLAY_WEIGHT       = 0.5
PROTO_CONTRAST_WEIGHT    = 0.08
PROTO_CONTRAST_MARGIN    = 0.35
CONFUSION_EDGE_WEIGHT    = 0.90
CONFUSION_TOP_K          = 3

KG_MAX_NODES_PER_SG      = 2000
KG_MAX_CROSS_EDGES       = 4000
KG_MAX_INTRA_EDGES_STORE = 500_000

# ── [NEW v4] GPU Manifold Mixup hyperparameters ───────────
# Strategy 1: Manifold Mixup at sentence-encoder hidden layer
MIXUP_ALPHA              = 0.4    # Beta(α,α) concentration — higher = more mixing
MIXUP_LOSS_WEIGHT        = 0.35   # weight of Mixup auxiliary loss
MIXUP_RARE_ONLY          = True   # only mix rare-class sentences (targeted)
MIXUP_MIN_RARE_IN_BATCH  = 2      # minimum rare sentences to trigger Mixup

# Strategy 2: KG-Neighbour-Guided Mixup
KG_MIXUP_ENABLED         = True   # mix query with its KG neighbours
KG_MIXUP_ALPHA           = 0.3    # interpolation weight for KG-guided Mixup
KG_MIXUP_WEIGHT          = 0.25   # loss weight for KG-Mixup branch

# Strategy 3: Rare↔HardMajority Inter-class Mixup
INTER_MIXUP_ENABLED      = True   # mix rare with confused majority
INTER_MIXUP_ALPHA        = 0.15   # small alpha so rare label still dominates
INTER_MIXUP_WEIGHT       = 0.20   # loss weight
INTER_MIXUP_RARE_RATIO   = 0.85   # rare label weight in mixed soft label

# Strategy 4: Intra-Rare Mixup (densify rare manifold)
INTRA_RARE_MIXUP_ENABLED = True
INTRA_RARE_MIXUP_WEIGHT  = 0.20

# Mixup temperature for soft labels (sharpen distribution)
MIXUP_SOFT_TEMP          = 0.5

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# FOCAL LOSS
# ═══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, class_weights: torch.Tensor,
                 rare_ids: list,
                 gamma_rare: float = FOCAL_GAMMA_RARE,
                 gamma_maj:  float = FOCAL_GAMMA_MAJ,
                 ignore_index: int = -100):
        super().__init__()
        self.ignore_index = ignore_index
        rare_set = set(rare_ids)
        gamma_per_class = torch.full((NUM_LABELS,), gamma_maj)
        for r in rare_set:
            gamma_per_class[r] = gamma_rare
        self.register_buffer("class_weights",   class_weights.float())
        self.register_buffer("gamma_per_class", gamma_per_class)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        valid = targets != self.ignore_index
        if not valid.any():
            return logits.sum() * 0.0
        logits_v  = logits[valid]
        targets_v = targets[valid]
        log_p = F.log_softmax(logits_v, dim=-1)
        p     = log_p.exp()
        p_t   = p.gather(1, targets_v.unsqueeze(1)).squeeze(1)
        gamma_t  = self.gamma_per_class[targets_v]
        focal_w  = (1.0 - p_t.detach()).pow(gamma_t)
        class_w  = self.class_weights[targets_v]
        ce       = F.nll_loss(log_p, targets_v, reduction="none")
        return (focal_w * class_w * ce).mean()


def compute_class_weights(label_freqs: dict) -> torch.Tensor:
    weights = torch.ones(NUM_LABELS)
    freqs   = [label_freqs.get(id2label[i], 1e-6) for i in range(NUM_LABELS)]
    inv     = [1.0 / max(f, 1e-6) for f in freqs]
    inv_sum = sum(inv)
    for i, w in enumerate(inv):
        weights[i] = min(w / inv_sum * NUM_LABELS, 10.0)
    return weights


# ═══════════════════════════════════════════════════════════
# GPU MANIFOLD MIXUP MODULE  [NEW v4]
# ═══════════════════════════════════════════════════════════
class GPUManifoldMixup(nn.Module):
    """
    All-GPU Manifold Mixup for rare-class sentence embeddings.

    Four strategies, all executed on GPU:
      S1. Intra-Rare: mix two rare-class embeddings from the same batch
      S2. KG-Guided:  mix a query with its top-1 KG neighbour embedding
      S3. Inter-class: mix rare with hard-majority (from confusion pairs)
      S4. Standard Mixup on sent_vecs (classic input-space Mixup)

    Returns mixed embeddings + soft mixed labels (one-hot blend).
    """

    def __init__(self, num_labels: int = NUM_LABELS,
                 alpha: float = MIXUP_ALPHA,
                 kg_alpha: float = KG_MIXUP_ALPHA,
                 inter_alpha: float = INTER_MIXUP_ALPHA,
                 rare_ratio: float  = INTER_MIXUP_RARE_RATIO,
                 soft_temp: float   = MIXUP_SOFT_TEMP):
        super().__init__()
        self.num_labels  = num_labels
        self.alpha       = alpha
        self.kg_alpha    = kg_alpha
        self.inter_alpha = inter_alpha
        self.rare_ratio  = rare_ratio
        self.soft_temp   = soft_temp

    # ── Utility: sample Beta(α,α) on GPU ─────────────────
    @staticmethod
    def _beta_gpu(alpha: float, size: int, device: torch.device) -> torch.Tensor:
        """Sample from Beta(α,α) entirely on GPU via torch.distributions."""
        if alpha <= 0:
            return torch.ones(size, device=device)
        dist = torch.distributions.Beta(
            torch.tensor(alpha, device=device),
            torch.tensor(alpha, device=device)
        )
        return dist.sample((size,))

    # ── Utility: one-hot to GPU float ────────────────────
    @staticmethod
    def _onehot(labels: torch.Tensor, num_classes: int) -> torch.Tensor:
        return F.one_hot(labels.clamp(min=0), num_classes).float()

    # ── S1: Intra-Rare Mixup ──────────────────────────────
    def intra_rare_mixup(
        self,
        embs: torch.Tensor,   # (N, D) — flat sentence embeddings (GPU)
        labels: torch.Tensor, # (N,)   — flat sentence labels     (GPU)
        rare_ids: torch.Tensor,        # 1-D tensor of rare label ids (GPU)
    ):
        """
        For each rare sentence, pick another rare sentence (possibly same
        class) at random and interpolate.  Returns (mixed_embs, soft_labels).
        """
        # find rare indices
        is_rare  = (labels.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        rare_idx = is_rare.nonzero(as_tuple=True)[0]
        if rare_idx.shape[0] < MIXUP_MIN_RARE_IN_BATCH:
            return None, None

        N_r  = rare_idx.shape[0]
        lam  = self._beta_gpu(self.alpha, N_r, embs.device)      # (N_r,)
        lam  = lam.clamp(min=0.1, max=0.9)

        # random permutation of rare indices — stays on GPU
        perm     = torch.randperm(N_r, device=embs.device)
        rare2    = rare_idx[perm]                                  # partner idx

        e1   = embs[rare_idx]                # (N_r, D)
        e2   = embs[rare2]                   # (N_r, D)
        l1   = labels[rare_idx]
        l2   = labels[rare2]

        lam_e = lam.unsqueeze(1)             # (N_r, 1) for broadcast
        mixed_embs = lam_e * e1 + (1.0 - lam_e) * e2             # (N_r, D)

        oh1  = self._onehot(l1, self.num_labels)                  # (N_r, C)
        oh2  = self._onehot(l2, self.num_labels)
        soft = lam_e * oh1 + (1.0 - lam_e) * oh2                  # (N_r, C)
        return mixed_embs, soft

    # ── S2: KG-Guided Mixup ───────────────────────────────
    def kg_guided_mixup(
        self,
        embs:        torch.Tensor,   # (T', D) triggered sentence embs
        labels:      torch.Tensor,   # (T',)
        nb_embs:     torch.Tensor,   # (T', K, D) KG neighbour embeddings
        nb_weights:  torch.Tensor,   # (T', K)    cosine weights
        rare_ids:    torch.Tensor,   # rare label ids (GPU)
    ):
        """
        Mix each triggered sentence with its top-1 KG neighbour.
        For rare sentences the alpha is smaller (stay closer to rare manifold).
        """
        is_rare = (labels.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        T_prime = embs.shape[0]
        if T_prime < 1:
            return None, None

        # Top-1 KG neighbour
        top1_idx   = nb_weights.argmax(dim=1)                     # (T',)
        top1_embs  = nb_embs[torch.arange(T_prime, device=embs.device),
                             top1_idx]                             # (T', D)

        # Per-sentence alpha: smaller for rare to stay near rare manifold
        base_lam   = self._beta_gpu(self.kg_alpha, T_prime, embs.device)
        lam        = torch.where(is_rare,
                                 base_lam.clamp(0.6, 0.9),  # rare: keep 60-90% original
                                 base_lam.clamp(0.4, 0.7))  # maj:  more mixing
        lam_e      = lam.unsqueeze(1)                             # (T', 1)

        mixed_embs = lam_e * embs + (1.0 - lam_e) * top1_embs    # (T', D)
        # Soft label: original sentence label dominates
        oh_orig    = self._onehot(labels.clamp(min=0), self.num_labels)  # (T', C)
        soft       = lam_e * oh_orig + (1.0 - lam_e) * (1.0 / self.num_labels)
        return mixed_embs, soft

    # ── S3: Rare ↔ Hard-Majority Inter-class Mixup ────────
    def inter_class_mixup(
        self,
        embs:        torch.Tensor,   # (N, D)
        labels:      torch.Tensor,   # (N,)
        rare_ids:    torch.Tensor,   # GPU
        confused_maj_ids: torch.Tensor,  # (R,) hard majority ids per rare (GPU)
    ):
        """
        For each rare sentence, find a hard-majority sentence in the batch
        and mix. Rare label still dominates (rare_ratio weight).
        """
        is_rare  = (labels.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        rare_idx = is_rare.nonzero(as_tuple=True)[0]
        if rare_idx.shape[0] < 2:
            return None, None

        # Find majority indices that are among confused_maj_ids
        is_hard_maj = (labels.unsqueeze(1) ==
                       confused_maj_ids.unsqueeze(0)).any(1) & ~is_rare
        maj_idx = is_hard_maj.nonzero(as_tuple=True)[0]
        if maj_idx.shape[0] < 1:
            # Fall back to any non-rare
            maj_idx = (~is_rare).nonzero(as_tuple=True)[0]
        if maj_idx.shape[0] < 1:
            return None, None

        N_r  = rare_idx.shape[0]
        lam  = self._beta_gpu(self.inter_alpha, N_r, embs.device)
        lam  = lam.clamp(0.0, self.inter_alpha * 2)  # small lambda: rare dominates

        # Sample majority partners with replacement
        perm_idx = torch.randint(0, maj_idx.shape[0], (N_r,), device=embs.device)
        maj_part = maj_idx[perm_idx]

        e1 = embs[rare_idx]       # (N_r, D)
        e2 = embs[maj_part]       # (N_r, D)
        l1 = labels[rare_idx]
        l2 = labels[maj_part]

        lam_e = lam.unsqueeze(1)                         # (N_r, 1)
        # Rare dominates: use (1-lam) for majority so rare keeps ~(1-inter_alpha)
        mixed_embs = (1.0 - lam_e) * e1 + lam_e * e2   # (N_r, D)

        oh1  = self._onehot(l1, self.num_labels)
        oh2  = self._onehot(l2, self.num_labels)
        # Soft label heavily weights the rare side
        soft = self.rare_ratio * oh1 + (1.0 - self.rare_ratio) * oh2  # (N_r, C)
        return mixed_embs, soft


# ═══════════════════════════════════════════════════════════
# SOFT-LABEL LOSS (for Mixup)  [NEW v4]
# ═══════════════════════════════════════════════════════════
class SoftLabelCrossEntropy(nn.Module):
    """
    Cross-entropy that accepts soft (one-hot blended) targets.
    Used for all Mixup branches.
    soft_labels: (N, C) float, sum-to-1 per row
    logits:      (N, C)
    """
    def __init__(self, temperature: float = MIXUP_SOFT_TEMP,
                 class_weights: torch.Tensor = None):
        super().__init__()
        self.temperature = temperature
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, logits: torch.Tensor,
                soft_labels: torch.Tensor) -> torch.Tensor:
        log_p = F.log_softmax(logits / self.temperature, dim=-1)   # (N, C)
        if self.class_weights is not None:
            # weight rows by the dominant class weight
            dom_cls = soft_labels.argmax(dim=-1)                    # (N,)
            w       = self.class_weights[dom_cls]                   # (N,)
            loss    = -(soft_labels * log_p).sum(dim=-1)            # (N,)
            return (loss * w).mean()
        return -(soft_labels * log_p).sum(dim=-1).mean()


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET & SAMPLER
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def build_weighted_sampler(docs: list, rare_ids: list,
                            oversample_ratio: float = OVERSAMPLE_RARE_RATIO
                            ) -> WeightedRandomSampler:
    rare_set = set(rare_ids)
    weights  = []
    for _, labs in docs:
        has_rare = any(l in rare_set for l in labs)
        weights.append(oversample_ratio if has_rare else 1.0)
    w_tensor = torch.tensor(weights, dtype=torch.float)
    return WeightedRandomSampler(w_tensor,
                                  num_samples=len(w_tensor),
                                  replacement=True)


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2   # 256

        self.mha_pooling     = MultiHeadAttentionPooling(
            self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2    # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total = len(encoder_layers)
        print(f"\n❄️  BERT layers frozen: embeddings + 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)
        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)
        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids,
                      lengths=None):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids)
        sent_vecs_drop = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _ = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def _make_mask(self, emissions, labels, lengths):
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)
        return mask

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        _, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        mask = self._make_mask(emissions, labels, lengths)
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_loss = -self.crf(emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2*T2, C), labels.reshape(B2*T2))
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# GPU-ACCELERATED KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    def __init__(self, emb_dim=KG_FUSION_DIM, device=DEVICE):
        self.emb_dim        = emb_dim
        self.device         = device
        self.rare_partition = set()
        self._cpu_nodes  = defaultdict(list)
        self._stacked    = {}
        self._norm       = {}
        self._is_rare    = {}
        self._edge_idx   = {}
        self._edge_w     = {}
        self._proto      = {}
        self._proto_norm = {}
        self._cx_sl = self._cx_si = None
        self._cx_dl = self._cx_di = None
        self._cx_w  = None

    def add_nodes(self, embeddings: torch.Tensor, label_ids: list):
        embs = embeddings.detach().cpu()
        for emb, lid in zip(embs, label_ids):
            self._cpu_nodes[lid].append((emb, False, False))

    def _inject_virtual(self, lid, alphas=KG_VIRTUAL_ALPHAS,
                        max_v=KG_MAX_VIRTUAL_PER_LABEL):
        real = [n[0] for n in self._cpu_nodes[lid] if not n[1] and not n[2]]
        n = len(real)
        if n < 2:
            return 0
        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        random.shuffle(pairs)
        added = 0
        for i, j in pairs:
            for alpha in alphas:
                if added >= max_v:
                    break
                v = alpha * real[i] + (1 - alpha) * real[j]
                v = F.normalize(v.unsqueeze(0), dim=-1).squeeze(0)
                self._cpu_nodes[lid].append((v, True, False))
                added += 1
            if added >= max_v:
                break
        return added

    def _build_prototypes(self, lid, k=KG_PROTOTYPE_K):
        all_e = torch.stack([n[0] for n in self._cpu_nodes[lid]])
        n_n   = all_e.shape[0]
        k_a   = min(k, n_n)
        if k_a < 2:
            c = F.normalize(all_e.mean(0, keepdim=True), dim=-1)
        else:
            km = MiniBatchKMeans(n_clusters=k_a, n_init=5,
                                 batch_size=min(1024, n_n), random_state=SEED)
            km.fit(all_e.numpy())
            c = F.normalize(
                torch.tensor(km.cluster_centers_, dtype=torch.float32), dim=-1)
        for ci in range(c.shape[0]):
            self._cpu_nodes[lid].insert(0, (c[ci], False, True))
        return c

    def build_edges(self,
                    rare_ids: list,
                    confusion_pairs: dict = None,
                    intra_thresh: float  = RST_INTRA_THRESH,
                    cross_thresh: float  = RST_CROSS_THRESH,
                    max_intra_per_node=5,
                    max_cross=KG_MAX_CROSS_EDGES):

        print("  Building dual-partition KG (GPU) + confusion edges ...")
        self.rare_partition = set(rare_ids)

        print("  Injecting virtual nodes & prototypes ...")
        for lid in rare_ids:
            if self._cpu_nodes[lid]:
                added = self._inject_virtual(lid)
                self._build_prototypes(lid)
                print(f"    [{id2label[lid]}] +{added} virtual | "
                      f"total={len(self._cpu_nodes[lid])} nodes")

        print("  Promoting to GPU ...")
        for lid, nodes in self._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in nodes]).to(self.device)
            norms = F.normalize(embs, dim=-1)
            is_v  = torch.tensor([n[1] or n[2] for n in nodes],
                                  dtype=torch.bool, device=self.device)
            self._stacked[lid] = embs
            self._norm[lid]    = norms
            self._is_rare[lid] = is_v

        print("  Building intra-label edges ...")
        for lid in self._stacked:
            norms = self._norm[lid]
            N = norms.shape[0]
            if N < 2:
                self._edge_idx[lid] = torch.zeros(
                    2, 0, dtype=torch.long, device=self.device)
                self._edge_w[lid]   = torch.zeros(0, device=self.device)
                continue

            if lid in self.rare_partition:
                chunk = 512
                ei_l, ej_l, ew_l = [], [], []
                for start in range(0, N, chunk):
                    end = min(start + chunk, N)
                    blk = torch.mm(norms[start:end], norms.T)
                    r, c_ = torch.where(
                        (blk > 0) &
                        (torch.arange(start, end, device=self.device).unsqueeze(1)
                         < torch.arange(N, device=self.device).unsqueeze(0))
                    )
                    ei_l.append(r + start); ej_l.append(c_)
                    ew_l.append(blk[r, c_])
                    if sum(x.shape[0] for x in ei_l) >= KG_MAX_INTRA_EDGES_STORE:
                        break
                if ei_l:
                    ei = torch.cat(ei_l); ej = torch.cat(ej_l); ew = torch.cat(ew_l)
                    if ei.shape[0] > KG_MAX_INTRA_EDGES_STORE:
                        p = torch.randperm(ei.shape[0],
                                           device=self.device)[:KG_MAX_INTRA_EDGES_STORE]
                        ei, ej, ew = ei[p], ej[p], ew[p]
                    src = torch.cat([ei, ej]); dst = torch.cat([ej, ei])
                    ww  = torch.cat([ew, ew])
                else:
                    src = dst = torch.zeros(0, dtype=torch.long, device=self.device)
                    ww  = torch.zeros(0, device=self.device)
            else:
                idx  = torch.arange(N, device=self.device)
                ci   = idx[:-1]; cj = idx[1:]
                cw   = (norms[ci] * norms[cj]).sum(-1).clamp(min=0)
                sim  = torch.mm(norms, norms.T)
                sim.fill_diagonal_(-2.0)
                sim[ci, cj] = -2.0; sim[cj, ci] = -2.0
                hi_r, hi_c = torch.where(sim >= intra_thresh)
                keep = hi_r < hi_c
                hi_r, hi_c = hi_r[keep], hi_c[keep]
                hi_w = sim[hi_r, hi_c]
                if hi_r.shape[0] > N * max_intra_per_node:
                    p = torch.randperm(
                        hi_r.shape[0], device=self.device)[:N * max_intra_per_node]
                    hi_r, hi_c, hi_w = hi_r[p], hi_c[p], hi_w[p]
                ei  = torch.cat([ci, hi_r]); ej = torch.cat([cj, hi_c])
                ew  = torch.cat([cw, hi_w])
                src = torch.cat([ei, ej]); dst = torch.cat([ej, ei])
                ww  = torch.cat([ew, ew])

            self._edge_idx[lid] = torch.stack([src, dst], dim=0)
            self._edge_w[lid]   = ww

        for lid in rare_ids:
            proto = [n for n in self._cpu_nodes.get(lid, []) if n[2]]
            if proto:
                pc = torch.stack([n[0] for n in proto]).to(self.device)
                pc = F.normalize(pc, dim=-1)
                self._proto[lid] = self._proto_norm[lid] = pc

        print("  Building cross-label edges ...")
        cx_sl, cx_si, cx_dl, cx_di, cx_w = [], [], [], [], []
        total_cross = 0
        label_ids   = sorted(self._stacked.keys())

        for a in range(len(label_ids)):
            if total_cross >= max_cross:
                break
            for b in range(a + 1, len(label_ids)):
                if total_cross >= max_cross:
                    break
                la, lb = label_ids[a], label_ids[b]
                na = self._norm[la][:KG_MAX_NODES_PER_SG]
                nb = self._norm[lb][:KG_MAX_NODES_PER_SG]
                sim = torch.mm(na, nb.T)
                rows, cols = torch.where(sim >= cross_thresh)
                rows, cols = rows[:50], cols[:50]
                if rows.shape[0] == 0:
                    continue
                w = sim[rows, cols]
                cx_sl.append(torch.full((rows.shape[0],), la,
                                        dtype=torch.long, device=self.device))
                cx_si.append(rows)
                cx_dl.append(torch.full((rows.shape[0],), lb,
                                        dtype=torch.long, device=self.device))
                cx_di.append(cols)
                cx_w.append(w)
                total_cross += rows.shape[0]

        if confusion_pairs:
            print(f"  Adding confusion-guided edges "
                  f"(top-{CONFUSION_TOP_K} per rare class) ...")
            for rare_lid, confused_lids in confusion_pairs.items():
                if rare_lid not in self._stacked:
                    continue
                for clid in confused_lids[:CONFUSION_TOP_K]:
                    if clid not in self._stacked:
                        continue
                    rare_n = self._norm[rare_lid]
                    conf_n = self._norm[clid][:KG_MAX_NODES_PER_SG]
                    sim    = torch.mm(rare_n, conf_n.T)
                    rows, cols = torch.where(sim > 0.3)
                    rows, cols = rows[:30], cols[:30]
                    if rows.shape[0] == 0:
                        continue
                    w = torch.full((rows.shape[0],), CONFUSION_EDGE_WEIGHT,
                                   device=self.device)
                    cx_sl.append(torch.full((rows.shape[0],), rare_lid,
                                            dtype=torch.long, device=self.device))
                    cx_si.append(rows)
                    cx_dl.append(torch.full((rows.shape[0],), clid,
                                            dtype=torch.long, device=self.device))
                    cx_di.append(cols)
                    cx_w.append(w)
                    print(f"    confusion edge: {id2label[rare_lid]} → "
                          f"{id2label[clid]}  ({rows.shape[0]} pairs)")

        if cx_sl:
            self._cx_sl = torch.cat(cx_sl)
            self._cx_si = torch.cat(cx_si)
            self._cx_dl = torch.cat(cx_dl)
            self._cx_di = torch.cat(cx_di)
            self._cx_w  = torch.cat(cx_w)
        else:
            z = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cx_sl = self._cx_si = self._cx_dl = self._cx_di = z
            self._cx_w  = torch.zeros(0, device=self.device)

        n_rare_n = sum(self._stacked[l].shape[0]
                       for l in self.rare_partition if l in self._stacked)
        n_maj_n  = sum(self._stacked[l].shape[0]
                       for l in self._stacked if l not in self.rare_partition)
        n_intra  = sum(self._edge_w[l].shape[0] // 2 for l in self._edge_w)
        print(f"  KG built: maj_nodes={n_maj_n} | rare_nodes={n_rare_n}")
        print(f"           intra={n_intra} | cross={self._cx_w.shape[0]}")

    def save(self, path):
        data = {
            "rare_partition": list(self.rare_partition),
            "nodes": {
                str(lid): [{"emb": n[0].tolist(),
                             "is_virtual": n[1], "is_proto": n[2]}
                            for n in nl]
                for lid, nl in self._cpu_nodes.items()
            },
            "intra": {str(lid): {"idx": self._edge_idx[lid].cpu().tolist(),
                                  "w":   self._edge_w[lid].cpu().tolist()}
                      for lid in self._edge_idx},
            "cross": {
                "sl": self._cx_sl.cpu().tolist() if self._cx_sl is not None else [],
                "si": self._cx_si.cpu().tolist() if self._cx_si is not None else [],
                "dl": self._cx_dl.cpu().tolist() if self._cx_dl is not None else [],
                "di": self._cx_di.cpu().tolist() if self._cx_di is not None else [],
                "w":  self._cx_w.cpu().tolist()  if self._cx_w  is not None else [],
            }
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM, device=DEVICE):
        kg = cls(emb_dim=emb_dim, device=device)
        with open(path) as f:
            data = json.load(f)
        kg.rare_partition = set(data.get("rare_partition", []))
        for k, nl in data["nodes"].items():
            lid = int(k)
            for n in nl:
                emb = torch.tensor(n["emb"], dtype=torch.float32)
                kg._cpu_nodes[lid].append((emb, n["is_virtual"], n["is_proto"]))
        for lid, nl in kg._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in nl]).to(device)
            norms = F.normalize(embs, dim=-1)
            is_v  = torch.tensor([n[1] or n[2] for n in nl],
                                  dtype=torch.bool, device=device)
            kg._stacked[lid] = embs
            kg._norm[lid]    = norms
            kg._is_rare[lid] = is_v
        for k, v in data.get("intra", {}).items():
            lid = int(k)
            kg._edge_idx[lid] = torch.tensor(
                v["idx"], dtype=torch.long,    device=device)
            kg._edge_w[lid]   = torch.tensor(
                v["w"],   dtype=torch.float32, device=device)
        cross = data.get("cross", {})
        if cross and cross.get("sl"):
            kg._cx_sl = torch.tensor(cross["sl"], dtype=torch.long,    device=device)
            kg._cx_si = torch.tensor(cross["si"], dtype=torch.long,    device=device)
            kg._cx_dl = torch.tensor(cross["dl"], dtype=torch.long,    device=device)
            kg._cx_di = torch.tensor(cross["di"], dtype=torch.long,    device=device)
            kg._cx_w  = torch.tensor(cross["w"],  dtype=torch.float32, device=device)
        else:
            z = torch.zeros(0, dtype=torch.long, device=device)
            kg._cx_sl = kg._cx_si = kg._cx_dl = kg._cx_di = z
            kg._cx_w  = torch.zeros(0, device=device)
        for lid in kg.rare_partition:
            proto = [n for n in kg._cpu_nodes.get(lid, []) if n[2]]
            if proto:
                pc = torch.stack([n[0] for n in proto]).to(device)
                pc = F.normalize(pc, dim=-1)
                kg._proto[lid] = kg._proto_norm[lid] = pc
        print(f"  KG loaded ← {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS,
                 thresh_maj=UNCERTAINTY_THRESH_MAJORITY,
                 thresh_rare=UNCERTAINTY_THRESH_RARE):
        self.log_C       = math.log(num_classes)
        self.thresh_maj  = thresh_maj
        self.thresh_rare = thresh_rare

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        H = -(probs * (probs + 1e-9).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor,
                     is_rare_pred: torch.Tensor) -> torch.Tensor:
        H = self.entropy(logits)
        thresh = torch.where(is_rare_pred,
                             torch.full_like(H, self.thresh_rare),
                             torch.full_like(H, self.thresh_maj))
        return H > thresh

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# KG RETRIEVER  (vectorised, GPU-only)
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    def __init__(self, kg: KnowledgeGraph, rare_ids: list,
                 top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP,
                 lambda_boost=KG_RARE_LAMBDA):
        self.kg           = kg
        self.rare_ids     = set(rare_ids)
        self.top_k        = top_k
        self.top_nodes    = top_nodes
        self.hop          = hop
        self.lambda_boost = lambda_boost
        self.device       = kg.device
        self.label_ids    = sorted(kg._stacked.keys())

    @classmethod
    def calibrate_lambda(cls, kg, rare_ids, sample_limit=200):
        rare_set  = set(rare_ids)
        maj_ids   = [l for l in kg._stacked if l not in rare_set]
        if not maj_ids:
            return KG_RARE_LAMBDA
        maj_norms = torch.cat(
            [kg._norm[m][:KG_MAX_NODES_PER_SG] for m in maj_ids if m in kg._norm],
            dim=0)
        gaps = []
        for lid in rare_ids:
            if lid not in kg._norm:
                continue
            own = kg._norm[lid]
            N   = own.shape[0]
            idx = torch.randperm(N, device=kg.device)[:sample_limit]
            s   = own[idx]
            own_sims = torch.mm(s, own.T).max(1).values
            maj_sims = torch.mm(s, maj_norms.T).max(1).values
            gaps.append((maj_sims - own_sims).cpu())
        if not gaps:
            return KG_RARE_LAMBDA
        lam = float(np.clip(torch.cat(gaps).median().item(), 0.05, 0.40))
        print(f"  λ auto-calibrated → {lam:.4f}")
        return lam

    def retrieve_batch(self, H_q: torch.Tensor,
                       trigger_mask: torch.Tensor) -> tuple:
        if not trigger_mask.any():
            return None, None, None

        trig_idx = trigger_mask.nonzero(as_tuple=True)[0]
        H_trig   = H_q[trig_idx]
        T_prime, D = H_trig.shape

        sg_sims = []
        for lid in self.label_ids:
            n = self.kg._norm[lid]
            if n.shape[0] == 0:
                sg_sims.append(torch.full((T_prime,), -1.0, device=self.device))
                continue
            cap = min(n.shape[0], KG_MAX_NODES_PER_SG)
            s   = torch.mm(H_trig, n[:cap].T).max(1).values
            boost = self.lambda_boost if lid in self.rare_ids else 0.0
            sg_sims.append(s + boost)

        sg_score_mat = torch.stack(sg_sims, dim=1)
        _, topk_idx  = sg_score_mat.topk(
            min(self.top_k, len(self.label_ids)), dim=1)

        cap_out = self.top_k * (self.top_nodes + 8)
        nb_embs    = torch.zeros(T_prime, cap_out, D, device=self.device)
        nb_weights = torch.zeros(T_prime, cap_out,    device=self.device)
        nb_is_rare = torch.zeros(T_prime, cap_out,
                                 dtype=torch.bool, device=self.device)
        fill = torch.zeros(T_prime, dtype=torch.long, device=self.device)

        unique_sg = topk_idx.unique().tolist()
        for sg_idx in unique_sg:
            lid        = self.label_ids[sg_idx]
            is_rare_sg = lid in self.rare_ids
            norms      = self.kg._norm[lid]
            embs       = self.kg._stacked[lid]
            N_sg       = norms.shape[0]
            if N_sg == 0:
                continue

            uses   = (topk_idx == sg_idx).any(dim=1)
            q_idx  = uses.nonzero(as_tuple=True)[0]
            H_sub  = H_trig[q_idx]

            cap_n  = min(N_sg, KG_MAX_NODES_PER_SG)
            cap_e  = embs[:cap_n]
            cap_nr = norms[:cap_n]

            sim_mat   = torch.mm(H_sub, cap_nr.T)
            k_q       = min(self.top_nodes, cap_n)
            topn      = sim_mat.topk(k_q, dim=1)
            topn_idx  = topn.indices
            topn_sims = topn.values

            edge = self.kg._edge_idx.get(lid)

            for qi_local, qi_global in enumerate(q_idx.tolist()):
                seed = topn_idx[qi_local]
                if self.hop >= 1 and edge is not None and edge.shape[1] > 0:
                    src, dst = edge[0], edge[1]
                    in_top = torch.isin(src, seed)
                    hop_n  = dst[in_top].unique()
                    if hop_n.shape[0] > 0:
                        hop_n = hop_n[hop_n < cap_n]
                        seed  = torch.cat([seed, hop_n]).unique()

                f     = int(fill[qi_global].item())
                n_add = min(k_q, cap_out - f)
                if n_add <= 0:
                    continue
                use_i = topn_idx[qi_local, :n_add]
                use_s = topn_sims[qi_local, :n_add].clamp(min=0)
                nb_embs[qi_global, f:f+n_add]    = cap_e[use_i]
                nb_weights[qi_global, f:f+n_add] = use_s
                nb_is_rare[qi_global, f:f+n_add] = is_rare_sg
                fill[qi_global] = f + n_add

        cx_sl = self.kg._cx_sl
        if cx_sl is not None and cx_sl.shape[0] > 0:
            for qi_global in range(T_prime):
                f = int(fill[qi_global].item())
                if f >= cap_out:
                    continue
                used_lids = topk_idx[qi_global].unique()
                for ul_idx in used_lids.tolist():
                    lid    = self.label_ids[ul_idx]
                    mask_l = (cx_sl == lid)
                    if not mask_l.any():
                        continue
                    c_di  = self.kg._cx_di[mask_l]
                    c_dl  = self.kg._cx_dl[mask_l]
                    c_w   = self.kg._cx_w[mask_l]
                    for dlid_t in c_dl.unique().tolist():
                        dlid = int(dlid_t)
                        dstk = self.kg._stacked.get(dlid)
                        if dstk is None:
                            continue
                        mask_d = (c_dl == dlid_t)
                        di_v   = c_di[mask_d][:3]
                        w_v    = c_w[mask_d][:3]
                        di_v   = di_v[di_v < dstk.shape[0]]
                        n_a    = min(di_v.shape[0], cap_out - f)
                        if n_a <= 0:
                            continue
                        nb_embs[qi_global, f:f+n_a]    = dstk[di_v[:n_a]]
                        nb_weights[qi_global, f:f+n_a] = w_v[:n_a]
                        nb_is_rare[qi_global, f:f+n_a] = (dlid in self.rare_ids)
                        f += n_a
                fill[qi_global] = f

        return nb_embs, nb_weights, nb_is_rare


# ═══════════════════════════════════════════════════════════
# GATED GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class GatedGraphAttentionFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT,
                 r_boost=KG_RARE_GAMMA):
        super().__init__()
        self.r_boost = r_boost
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.gate    = nn.Linear(emb_dim * 2, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, H_q, nb_embs, nb_w, nb_rare):
        pad_mask = (nb_w == 0)
        q = self.proj_q(H_q)
        k = self.proj_k(nb_embs)
        dot = torch.bmm(k, q.unsqueeze(-1)).squeeze(-1) * self.scale
        r_b  = torch.where(nb_rare,
                           torch.full_like(dot, self.r_boost),
                           torch.ones_like(dot))
        raw  = dot * nb_w * r_b
        raw  = raw.masked_fill(pad_mask, -1e9)
        alpha = F.softmax(raw, dim=-1)
        alpha = alpha.masked_fill(pad_mask, 0.0)
        alpha = self.dropout(alpha)
        v_attn = torch.bmm(alpha.unsqueeze(1), nb_embs).squeeze(1)
        g = torch.sigmoid(self.gate(torch.cat([H_q, v_attn], dim=-1)))
        return (1.0 - g) * H_q + g * v_attn


# ═══════════════════════════════════════════════════════════
# HARD EXAMPLE REPLAY BUFFER
# ═══════════════════════════════════════════════════════════
class HardExampleBuffer:
    def __init__(self, max_size=HARD_BUFFER_SIZE):
        self.max_size = max_size
        self.buf: list = []

    def update(self, loss_val: float, batch_cpu: tuple):
        self.buf.append((loss_val, batch_cpu))
        if len(self.buf) > self.max_size:
            self.buf.sort(key=lambda x: -x[0])
            self.buf = self.buf[:self.max_size // 2]

    def sample(self) -> tuple | None:
        if not self.buf:
            return None
        losses  = torch.tensor([x[0] for x in self.buf], dtype=torch.float)
        probs   = F.softmax(losses, dim=0)
        idx     = int(torch.multinomial(probs, 1).item())
        return self.buf[idx][1]

    def __len__(self):
        return len(self.buf)


# ═══════════════════════════════════════════════════════════
# KG + MIXUP AUGMENTED MODEL  [NEW v4 — core contribution]
# ═══════════════════════════════════════════════════════════
class KGMixupAugmentedModel(nn.Module):
    """
    Full model: KG-RAG v3 + Four-Strategy GPU Manifold Mixup.

    Training forward pass (labels != None):
      1. Base BERT→BiLSTM→MHA encode → sent_vecs
      2. KG-Gated-GAT fusion → fused_sent
      3. ctx-BiLSTM → combined emissions → CRF loss  [standard]
      4. Intra-Rare Mixup  on sent_vecs              [S1]
      5. KG-Guided Mixup   on triggered sent_vecs    [S2]
      6. Inter-class Mixup on sent_vecs              [S3]
      7. Prototype contrastive auxiliary             [S4]
      Total loss = CRF + Focal_CE + S1 + S2 + S3 + proto

    Inference: same as v3 (no Mixup overhead).
    """

    def __init__(self, base_model: InLegalBERT_BiLSTM_MHA_CRF,
                 kg: KnowledgeGraph,
                 rare_ids: list,
                 retriever: KGRetriever = None,
                 focal_loss: FocalLoss  = None,
                 class_weights: torch.Tensor = None,
                 confusion_pairs: dict  = None):
        super().__init__()
        self.base       = base_model
        self.kg         = kg
        self.rare_ids   = set(rare_ids)
        self.rare_list  = sorted(rare_ids)
        self.retriever  = retriever or KGRetriever(kg, rare_ids)
        self.uncertainty = UncertaintyEstimator()
        self.focal_loss  = focal_loss

        sent_dim = base_model.sent_out_dim  # 256
        ctx_dim  = base_model.ctx_out_dim   # 128

        self.gat_fusion = GatedGraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        # ── Mixup components ──────────────────────────────
        self.gpu_mixup  = GPUManifoldMixup(num_labels=NUM_LABELS)

        # Mixup projection: map sent_dim → ctx_dim for soft-label CE
        self.mixup_proj = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, NUM_LABELS),
        )
        self.soft_ce = SoftLabelCrossEntropy(
            temperature=MIXUP_SOFT_TEMP, class_weights=class_weights)

        # Rare tensor on GPU (registered as buffer so .to(device) works)
        rare_t = torch.tensor(self.rare_list, dtype=torch.long)
        self.register_buffer("_rare_t", rare_t)

        # Confused majority ids (flattened, for inter-class mixup)
        if confusion_pairs:
            conf_maj_ids = list({
                cid
                for cids in confusion_pairs.values()
                for cid in cids
            })
        else:
            conf_maj_ids = list(range(NUM_LABELS))
        conf_maj_t = torch.tensor(conf_maj_ids, dtype=torch.long)
        self.register_buffer("_conf_maj_t", conf_maj_t)

    def _rare_mask(self, top_labels):
        return (top_labels.unsqueeze(-1) ==
                self._rare_t.view(1, 1, -1)).any(-1)

    # ── Prototype contrastive loss (unchanged from v3) ───
    def _proto_contrast_loss(self, device) -> torch.Tensor:
        rare_protos = [self.kg._proto_norm[l] for l in self.rare_list
                       if l in self.kg._proto_norm]
        if not rare_protos:
            return torch.tensor(0.0, device=device)
        maj_ids    = [l for l in self.kg._stacked if l not in self.rare_ids]
        maj_protos = [self.kg._proto_norm[l] for l in maj_ids
                      if l in self.kg._proto_norm]
        if not maj_protos:
            return torch.tensor(0.0, device=device)
        rp = F.normalize(torch.cat(rare_protos, dim=0), dim=-1).to(device)
        mp = F.normalize(torch.cat(maj_protos,  dim=0), dim=-1).to(device)
        sim  = torch.mm(rp, mp.T)
        return torch.clamp(sim - PROTO_CONTRAST_MARGIN, min=0.0).mean()

    # ── KG fusion (unchanged from v3) ────────────────────
    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, D = sent_vecs.shape
        fused      = sent_vecs.clone()
        top_labels = self.uncertainty.top_label(emissions)
        is_rare_p  = self._rare_mask(top_labels)
        uncertain  = self.uncertainty.is_uncertain(emissions, is_rare_p)
        trigger    = uncertain | (RARE_ALWAYS_KG & is_rare_p)

        for b in range(B):
            n      = int(lengths[b].item())
            trig_b = trigger[b, :n]
            if not trig_b.any():
                continue
            H_b      = sent_vecs[b, :n]
            H_b_norm = F.normalize(H_b.detach(), dim=-1)
            nb_embs, nb_w, nb_rare = self.retriever.retrieve_batch(
                H_b_norm, trig_b)
            if nb_embs is None:
                continue
            trig_idx = trig_b.nonzero(as_tuple=True)[0]
            H_q      = H_b[trig_idx]
            h_star   = self.gat_fusion(H_q, nb_embs, nb_w, nb_rare)
            fused[b, trig_idx] = h_star
        return fused

    # ── [NEW v4] All-GPU Mixup losses ─────────────────────
    def _compute_mixup_losses(self, sent_vecs, labels, lengths,
                               emissions, device) -> torch.Tensor:
        """
        Compute the combined Mixup auxiliary loss (S1 + S2 + S3).
        All operations stay on GPU.
        """
        total_mixup_loss = torch.tensor(0.0, device=device)
        B, T, D = sent_vecs.shape

        # Flatten valid sentences across batch
        flat_embs  = []
        flat_labs  = []
        for b in range(B):
            n = int(lengths[b].item())
            flat_embs.append(sent_vecs[b, :n])
            flat_labs.append(labels[b, :n])
        if not flat_embs:
            return total_mixup_loss

        flat_embs = torch.cat(flat_embs, dim=0)   # (N_total, D)
        flat_labs = torch.cat(flat_labs, dim=0)   # (N_total,)
        # Mask padding tokens
        valid_mask = flat_labs >= 0
        flat_embs  = flat_embs[valid_mask]
        flat_labs  = flat_labs[valid_mask]

        if flat_embs.shape[0] < MIXUP_MIN_RARE_IN_BATCH:
            return total_mixup_loss

        # ── S1: Intra-Rare Mixup ──────────────────────────
        if INTRA_RARE_MIXUP_ENABLED:
            mx_embs, soft = self.gpu_mixup.intra_rare_mixup(
                flat_embs, flat_labs, self._rare_t)
            if mx_embs is not None:
                mx_logits = self.mixup_proj(mx_embs)      # (N_r, C)
                s1_loss   = self.soft_ce(mx_logits, soft.to(device))
                total_mixup_loss = total_mixup_loss + INTRA_RARE_MIXUP_WEIGHT * s1_loss

        # ── S2: KG-Guided Mixup (use KG neighbours on triggered) ─
        if KG_MIXUP_ENABLED:
            for b in range(B):
                n = int(lengths[b].item())
                H_b  = sent_vecs[b, :n]
                l_b  = labels[b, :n]
                valid_b = l_b >= 0

                is_rare_b = (l_b.unsqueeze(1) ==
                             self._rare_t.unsqueeze(0)).any(1)
                trig_b    = is_rare_b & valid_b
                if not trig_b.any():
                    continue
                H_b_norm = F.normalize(H_b.detach(), dim=-1)
                nb_embs, nb_w, _ = self.retriever.retrieve_batch(
                    H_b_norm, trig_b)
                if nb_embs is None:
                    continue
                trig_idx    = trig_b.nonzero(as_tuple=True)[0]
                H_trig      = H_b[trig_idx]
                l_trig      = l_b[trig_idx]
                mx_embs, soft = self.gpu_mixup.kg_guided_mixup(
                    H_trig, l_trig, nb_embs, nb_w, self._rare_t)
                if mx_embs is not None:
                    mx_logits = self.mixup_proj(mx_embs)
                    s2_loss   = self.soft_ce(mx_logits, soft.to(device))
                    total_mixup_loss = total_mixup_loss + KG_MIXUP_WEIGHT * s2_loss

        # ── S3: Inter-class Rare ↔ Hard-Majority Mixup ───
        if INTER_MIXUP_ENABLED:
            mx_embs, soft = self.gpu_mixup.inter_class_mixup(
                flat_embs, flat_labs, self._rare_t, self._conf_maj_t)
            if mx_embs is not None:
                mx_logits = self.mixup_proj(mx_embs)
                s3_loss   = self.soft_ce(mx_logits, soft.to(device))
                total_mixup_loss = total_mixup_loss + INTER_MIXUP_WEIGHT * s3_loss

        return total_mixup_loss

    # ── Forward ───────────────────────────────────────────
    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        device = input_ids.device

        # Step 1: base encode
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        # Step 2: KG fusion
        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device)

        # Step 3: ctx-BiLSTM on fused
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)
        fused_ctx  = self.base.dropout(fused_ctx)

        # Step 4: fusion classifier
        fused_emis = self.fusion_classifier(fused_ctx)
        fused_emis = torch.nan_to_num(fused_emis, nan=0.0, posinf=1e4, neginf=-1e4)

        # Build mask
        if lengths is not None:
            B, T, _ = fused_emis.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emis.shape[:2], dtype=torch.bool, device=device)

        combined = (base_emissions + fused_emis) / 2.0

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0

            base_crf  = -self.base.crf(
                base_emissions, safe, mask=mask, reduction="mean")
            fused_crf = -self.fusion_crf(
                fused_emis,     safe, mask=mask, reduction="mean")

            B2, T2, C = combined.shape
            flat_logits = combined.reshape(B2*T2, C)
            flat_labels = labels.reshape(B2*T2)

            if self.focal_loss is not None:
                aux_loss = self.focal_loss(flat_logits, flat_labels)
            else:
                aux_loss = self.ce_loss(flat_logits, flat_labels)

            proto_loss = PROTO_CONTRAST_WEIGHT * self._proto_contrast_loss(device)

            # ── [NEW v4] Mixup losses ──────────────────────
            mixup_loss = MIXUP_LOSS_WEIGHT * self._compute_mixup_losses(
                sent_vecs, labels, lengths, base_emissions, device)

            loss = ((base_crf + fused_crf) / 2.0
                    + AUX_CE_WEIGHT * aux_loss
                    + proto_loss
                    + mixup_loss)
            return loss, combined
        else:
            decoded = self.fusion_crf.decode(fused_emis, mask=mask)
            return decoded, fused_emis


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc = accuracy_score(all_trues, all_preds)
    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds,
                                     labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds,
                                   labels=list(range(NUM_LABELS)),
                                   average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {"f1": float(per_class_f1[i]),
                      "precision": float(per_class_prec[i]),
                      "recall":    float(per_class_rec[i])}
        for i in range(NUM_LABELS)
    }
    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0
    str_t  = [id2label[x] for x in all_trues]
    str_p  = [id2label[x] for x in all_preds]
    cls_rp = classification_report(str_t, str_p, labels=LABELS,
                                    digits=4, zero_division=0)
    cm     = confusion_matrix(str_t, str_p, labels=LABELS)
    return dict(
        macro_f1=macro_f1, micro_f1=micro_f1, weighted_f1=weighted_f1,
        macro_precision=macro_prec, micro_precision=micro_prec,
        weighted_precision=weighted_prec,
        macro_recall=macro_rec, micro_recall=micro_rec,
        weighted_recall=weighted_rec,
        rare_f1=rare_f1, rare_precision=rare_prec, rare_recall=rare_rec,
        per_class_metrics=per_class_metrics,
        accuracy=acc, cls_report=cls_rp, cm=cm,
        all_preds=all_preds, all_trues=all_trues,
    )


def count_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if     p.requires_grad)
    fr = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {tr:,} | Frozen: {fr:,}")
    return tr, fr


# ═══════════════════════════════════════════════════════════
# CONFUSION PAIR COMPUTATION
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def compute_confusion_pairs(model, dataset, rare_ids,
                             device=DEVICE, top_k=CONFUSION_TOP_K) -> dict:
    model.eval()
    loader = DataLoader(dataset, batch_size=2, shuffle=False,
                        collate_fn=collate_rrc)
    all_preds, all_trues = [], []
    rare_set = set(rare_ids)
    for ids, attn, ttype, labels, lengths in loader:
        ids     = ids.to(device);  attn    = attn.to(device)
        ttype   = ttype.to(device); lengths = lengths.to(device)
        decoded, _ = model(ids, attn, ttype, labels=None, lengths=lengths)
        for i, seq_p in enumerate(decoded):
            true_len = int(lengths[i].item())
            all_preds.extend(seq_p)
            all_trues.extend(labels[i, :true_len].tolist())
    confusion = defaultdict(Counter)
    for true, pred in zip(all_trues, all_preds):
        if true in rare_set and pred != true:
            confusion[true][pred] += 1
    result = {}
    print("\n📊 Confusion-guided edge analysis:")
    for rid, counter in confusion.items():
        top_confused = [cls for cls, _ in counter.most_common(top_k)]
        result[rid] = top_confused
        names = [id2label[c] for c in top_confused]
        total = sum(counter.values())
        print(f"  {id2label[rid]:<20} confused → {names}  "
              f"(total misclassified: {total})")
    return result


# ═══════════════════════════════════════════════════════════
# BASE TRAINER (Phase A)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, focal_loss: FocalLoss = None, device=DEVICE):
        self.model      = model.to(device)
        self.focal_loss = focal_loss
        self.device     = device

    def build_optimizer(self):
        pg = []
        pg.append({"params": list(self.model.bert.pooler.parameters()),
                   "lr": BERT_LR, "weight_decay": WEIGHT_DECAY})
        enc = self.model.bert.encoder.layer
        n   = len(enc)
        for i in range(n - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth  = (n - 1) - i
            lr_i   = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in enc[i].parameters() if p.requires_grad]
            if params:
                pg.append({"params": params, "lr": lr_i,
                           "weight_decay": WEIGHT_DECAY})
        head_mods = [self.model.sent_bilstm, self.model.mha_pooling,
                     self.model.sent_layer_norm, self.model.ctx_bilstm,
                     self.model.classifier, self.model.crf]
        head_p = [p for m in head_mods for p in m.parameters()]
        pg.append({"params": head_p, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(pg)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                tt  = tt.to(self.device);  labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, tt, labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total += loss.item(); n += 1
        return total / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids     = ids.to(self.device); attn    = attn.to(self.device)
                tt      = tt.to(self.device);  lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, tt, labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :tl].tolist())
        if measure_inference_time:
            ti  = time.time() - t0
            ns  = len(all_trues)
            inf = {"total_inference_time_s": ti,
                   "latency_per_document_ms": ti / max(1, n_samples) * 1000,
                   "throughput_sentences_per_s": ns / max(1e-9, ti)}
            with open(os.path.join(OUT_DIR,
                                   f"inference_{split_name}.json"), "w") as f:
                json.dump(inf, f, indent=2)
        else:
            inf = None
        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if inf:
            metrics["inference_time_info"] = inf
        return metrics

    def train(self, train_docs, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        sampler      = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)
        early_stopper = EarlyStopping()
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            t0 = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, tt, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, emissions = self.model(ids, attn, tt, labels=labels, lengths=lengths)
                if self.focal_loss is not None:
                    B2, T2, C = emissions.shape
                    fl = self.focal_loss(
                        emissions.reshape(B2*T2, C).detach(),
                        labels.reshape(B2*T2))
                    loss = loss + AUX_CE_WEIGHT * fl
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue
                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()
                run_loss += loss.item(); n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            ep_t     = time.time() - t0
            avg_loss = run_loss / max(1, n_steps)
            val_loss = self.compute_val_loss(dev_dataset)
            val_m    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Ep {epoch:03d}/{num_epochs} | "
                f"tr_loss:{avg_loss:.4f} val_loss:{val_loss:.4f} | "
                f"mac_F1:{val_m['macro_f1']:.4f} rare_F1:{val_m['rare_f1']:.4f} | "
                f"t:{ep_t:.1f}s ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_loss, "val_loss": val_loss,
                "val_macro_f1": val_m["macro_f1"],
                "val_micro_f1": val_m["micro_f1"],
                "val_weighted_f1": val_m["weighted_f1"],
                "val_rare_f1": val_m["rare_f1"],
                "val_accuracy": val_m["accuracy"],
                "epoch_train_time_s": ep_t,
            })
            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")
            if early_stopper.step(val_m["macro_f1"]):
                print(f"\n⏹  Base early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "base_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")
        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))
            print(f"  Base model saved to {BEST_MODEL_DIR}/base_model.bin")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer,
                           rare_ids, confusion_pairs, device=DEVICE) -> KnowledgeGraph:
    print("\n🔨 Building Dual-Partition KG + Confusion Edges ...")
    base_model.eval(); base_model.to(device)
    kg     = KnowledgeGraph(emb_dim=base_model.sent_out_dim, device=device)
    dummy  = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy, batch_size=4, shuffle=False,
                        collate_fn=collate_rrc, num_workers=0)
    for doc_idx, (ids, attn, tt, labels, lengths) in enumerate(loader):
        ids  = ids.to(device); attn = attn.to(device); tt = tt.to(device)
        for bi in range(ids.shape[0]):
            sv = base_model.encode_sentences(
                ids[bi:bi+1], attn[bi:bi+1], tt[bi:bi+1]).squeeze(0)
            n  = int(lengths[bi].item())
            kg.add_nodes(sv[:n].cpu(), labels[bi, :n].tolist())
        if (doc_idx + 1) % 50 == 0:
            print(f"  {(doc_idx+1)*ids.shape[0]}/{len(train_docs)} docs processed")
    kg.build_edges(rare_ids=rare_ids, confusion_pairs=confusion_pairs)
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# KG + MIXUP TRAINER  (Phase B)
# ═══════════════════════════════════════════════════════════
class KGMixupTrainer:
    def __init__(self, kg_model: KGMixupAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_p = (list(self.model.gat_fusion.parameters()) +
                 list(self.model.fusion_classifier.parameters()) +
                 list(self.model.fusion_crf.parameters()) +
                 list(self.model.mixup_proj.parameters()) +    # [NEW v4]
                 list(self.model.gpu_mixup.parameters()))      # [NEW v4]
        base_p = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_p,  "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_p, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids     = ids.to(self.device); attn    = attn.to(self.device)
                tt      = tt.to(self.device);  lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, tt, labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :tl].tolist())
        if measure_inference_time:
            ti  = time.time() - t0
            ns  = len(all_trues)
            inf = {"total_inference_time_s": ti,
                   "latency_per_document_ms": ti / max(1, n_samples) * 1000,
                   "throughput_sentences_per_s": ns / max(1e-9, ti)}
            with open(os.path.join(OUT_DIR,
                                   f"kg_mixup_inference_{split_name}.json"), "w") as f:
                json.dump(inf, f, indent=2)
        else:
            inf = None
        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if inf:
            metrics["inference_time_info"] = inf
        return metrics

    def train(self, train_docs, train_dataset, dev_dataset,
              rare_ids, num_epochs=NUM_EPOCHS_KG):
        sampler      = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)
        early_stopper = EarlyStopping(patience=7)
        buffer        = HardExampleBuffer(max_size=HARD_BUFFER_SIZE)
        rare_set      = set(rare_ids)
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            t0 = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, tt, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  labels = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, tt, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                run_loss += loss.item(); n_steps += 1

                # Hard example buffering
                lv = loss.item()
                if lv > HARD_REPLAY_LOSS_THRESH:
                    flat_l = labels.cpu().flatten()
                    has_rare = any(int(x) in rare_set
                                   for x in flat_l if x.item() >= 0)
                    if has_rare:
                        buffer.update(lv, (ids.cpu(), attn.cpu(),
                                           tt.cpu(), labels.cpu(),
                                           lengths.cpu()))

                # Hard example replay
                if (step + 1) % HARD_REPLAY_FREQ == 0 and len(buffer) >= 10:
                    replay = buffer.sample()
                    if replay:
                        r_ids, r_attn, r_tt, r_lab, r_len = [
                            x.to(self.device) for x in replay]
                        r_loss, _ = self.model(r_ids, r_attn, r_tt,
                                               labels=r_lab, lengths=r_len)
                        if not torch.isnan(r_loss) and not torch.isinf(r_loss):
                            (r_loss * HARD_REPLAY_WEIGHT).backward()
                            torch.nn.utils.clip_grad_norm_(
                                self.model.parameters(), GRAD_CLIP)
                            optimizer.step(); scheduler.step(); optimizer.zero_grad()

            ep_t     = time.time() - t0
            avg_loss = run_loss / max(1, n_steps)
            val_m    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG+Mixup] Ep {epoch:02d}/{num_epochs} | "
                f"tr_loss:{avg_loss:.4f} | "
                f"mac_F1:{val_m['macro_f1']:.4f} "
                f"rare_F1:{val_m['rare_f1']:.4f} | "
                f"t:{ep_t:.1f}s buf:{len(buffer)} "
                f"ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "kg_mixup",
                "train_loss": avg_loss,
                "val_macro_f1": val_m["macro_f1"],
                "val_rare_f1":  val_m["rare_f1"],
                "val_accuracy": val_m["accuracy"],
                "epoch_train_time_s": ep_t,
            })
            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG+Mixup val_macro_f1={best_f1:.4f}")
            if early_stopper.step(val_m["macro_f1"]):
                print(f"\n⏹  KG+Mixup early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "kg_mixup_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR, "kg_mixup_model.bin"))
            print(f"\n✔ Best KG+Mixup model saved (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS,
                yticklabels=LABELS, cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (KG-RAG+Mixup v4)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1  (KG-RAG+Mixup v4)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"],
            marker="o", markersize=3, label="Train Loss")
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Combined Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)
    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1",  marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1  (Base → KG-RAG+Mixup v4)"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def print_metrics_table(dev_m, test_m, base_t=None, kg_t=None, n_params=None):
    rows = [
        ("Accuracy",          "accuracy"),
        ("Macro-F1",          "macro_f1"),
        ("Micro-F1",          "micro_f1"),
        ("Weighted-F1",       "weighted_f1"),
        ("Minority Macro-F1", "rare_f1"),
        ("Macro-Precision",   "macro_precision"),
        ("Macro-Recall",      "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS — InLegalBERT+BiLSTM+MHA+CRF+KG-RAG+Mixup v4")
    print("=" * 72)
    if n_params: print(f"  Trainable params : {n_params:,}")
    if base_t:   print(f"  Phase A time     : {base_t/60:.1f} min")
    if kg_t:     print(f"  Phase B time     : {kg_t/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<30} {'Dev':>10} {'Test':>10}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<30} {dev_m[key]:>10.4f} {test_m[key]:>10.4f}")
    print("=" * 72)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 66)
    print(f"  {'Label':<22} {'F1-Dev':>8} {'F1-Test':>8} "
          f"{'Prec':>8} {'Rec':>8}")
    print("  " + "-" * 66)
    for lbl in LABELS:
        dv = dev_m["per_class_metrics"][lbl]
        ts = test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>8.4f} {ts['f1']:>8.4f} "
              f"{ts['precision']:>8.4f} {ts['recall']:>8.4f}")
    print("  " + "-" * 66)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT+BiLSTM+MHA+CRF → KG-RAG+Mixup v4")
    print("  [+] Focal Loss      [+] Oversampling      [+] Confusion Edges")
    print("  [+] Hard Mining     [+] Gated GAT Fusion  [+] Prototype Contrast")
    print("  [NEW] S1: Intra-Rare GPU Manifold Mixup")
    print("  [NEW] S2: KG-Neighbour-Guided GPU Mixup")
    print("  [NEW] S3: Rare↔HardMaj Inter-class GPU Mixup")
    print("  [NEW] Soft-label CRF-aware loss for all Mixup branches\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train:{len(train_docs)} | Dev:{len(dev_docs)} | Test:{len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([{"label": l, "frequency": label_freqs[l],
                   "is_rare": l in rare_labels}
                  for l in LABELS]
                 ).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    class_weights = compute_class_weights(label_freqs)
    focal_loss    = FocalLoss(class_weights, rare_ids).to(DEVICE)
    print(f"\nClass weights (top-5 rare): "
          f"{[(id2label[r], round(float(class_weights[r]),2)) for r in rare_ids[:5]]}")

    print("Loading tokenizer ...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ══════════════════════════════════════════════════════
    # PHASE A: Base Model Training
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training (focal loss + oversampling)")
    print("=" * 60)
    base_model   = InLegalBERT_BiLSTM_MHA_CRF()
    base_trainer = BaseTrainer(base_model, focal_loss=focal_loss, device=DEVICE)
    base_hist, base_time = base_trainer.train(
        train_docs, train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE)

    # ══════════════════════════════════════════════════════
    # PHASE A→B: Confusion analysis + KG construction
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Confusion-Matrix Analysis + KG Construction")
    print("=" * 60)
    confusion_pairs = compute_confusion_pairs(
        base_model, train_dataset, rare_ids, device=DEVICE)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG — loading ...")
        kg = KnowledgeGraph.load(
            kg_path, emb_dim=base_model.sent_out_dim, device=DEVICE)
        kg.rare_partition = set(rare_ids)
    else:
        kg = build_knowledge_graph(
            base_model, train_docs, tokenizer,
            rare_ids=rare_ids, confusion_pairs=confusion_pairs, device=DEVICE)

    print("\n  Calibrating λ ...")
    lambda_boost = KGRetriever.calibrate_lambda(kg, rare_ids)

    # ══════════════════════════════════════════════════════
    # PHASE B: KG + Mixup Fine-Tuning
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: KG-Augmented + GPU-Mixup Fine-Tuning")
    print(f"  λ={lambda_boost:.4f}  γ={KG_RARE_GAMMA}")
    print(f"  Mixup alpha={MIXUP_ALPHA}  KG_Mixup alpha={KG_MIXUP_ALPHA}")
    print(f"  Inter-class alpha={INTER_MIXUP_ALPHA}  rare_ratio={INTER_MIXUP_RARE_RATIO}")
    print(f"  Mixup loss weights: S1={INTRA_RARE_MIXUP_WEIGHT} "
          f"S2={KG_MIXUP_WEIGHT} S3={INTER_MIXUP_WEIGHT}")
    print("=" * 60)

    retriever = KGRetriever(
        kg, rare_ids=rare_ids,
        top_k=KG_TOP_K, top_nodes=KG_TOP_NODES,
        hop=KG_HOP, lambda_boost=lambda_boost)

    kg_mixup_model = KGMixupAugmentedModel(
        base_model=base_model, kg=kg,
        rare_ids=rare_ids, retriever=retriever,
        focal_loss=focal_loss,
        class_weights=class_weights.to(DEVICE),
        confusion_pairs=confusion_pairs)

    n_params, _ = count_parameters(kg_mixup_model)

    kg_trainer = KGMixupTrainer(kg_mixup_model, device=DEVICE)
    kg_hist, kg_time = kg_trainer.train(
        train_docs, train_dataset, dev_dataset,
        rare_ids=rare_ids, num_epochs=NUM_EPOCHS_KG)

    plot_combined_history(base_hist, kg_hist)

    # ══════════════════════════════════════════════════════
    # EVALUATION
    # ══════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_m = kg_trainer.evaluate(dev_dataset, rare_ids,
                                split_name="dev",
                                measure_inference_time=True)
    print(f"  Dev  Accuracy:{dev_m['accuracy']:.4f} "
          f"Macro-F1:{dev_m['macro_f1']:.4f} "
          f"Rare-F1:{dev_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF+KG-RAG+Mixup v4\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_m["cls_report"])

    save_confusion_matrix(dev_m["cm"],  "dev",  rare_labels)
    save_per_class_f1_chart(dev_m["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_m = kg_trainer.evaluate(test_dataset, rare_ids,
                                 split_name="test",
                                 measure_inference_time=True)
    print(f"  Test Accuracy:{test_m['accuracy']:.4f} "
          f"Macro-F1:{test_m['macro_f1']:.4f} "
          f"Rare-F1:{test_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF+KG-RAG+Mixup v4\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_m["cls_report"])

    save_confusion_matrix(test_m["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_m["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_m["all_trues"]],
        "pred": [id2label[x] for x in test_m["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall",    "micro_recall",    "weighted_recall",    "rare_recall",
        "accuracy",
    ]
    summary = {
        "model": "InLegalBERT+BiLSTM+MHA+CRF+KG-RAG+Mixup-v4",
        "improvements_over_v3": [
            "S1_IntraRare_GPU_ManifoldMixup (alpha=0.4, weight=0.35)",
            "S2_KGNeighbourGuided_GPU_Mixup (alpha=0.3, weight=0.25)",
            "S3_RareHardMaj_Interclass_GPU_Mixup (alpha=0.15, weight=0.20)",
            "SoftLabelCrossEntropy (temp=0.5) for all Mixup branches",
            "BetaDistribution sampled on GPU for all lambda",
        ],
        "v3_improvements_retained": [
            "FocalLoss(gamma_rare=2.5,gamma_maj=1.0)+InvFreqWeights",
            "WeightedRandomSampler(ratio=3.0)",
            "ConfusionMatrixGuidedEdges",
            "HardExampleReplayBuffer",
            "GatedGraphAttentionFusion",
            "PrototypeContrastiveLoss",
            "AdaptiveUncertaintyThreshold",
        ],
        "mixup_config": {
            "alpha":            MIXUP_ALPHA,
            "kg_alpha":         KG_MIXUP_ALPHA,
            "inter_alpha":      INTER_MIXUP_ALPHA,
            "rare_ratio":       INTER_MIXUP_RARE_RATIO,
            "loss_weight_total": MIXUP_LOSS_WEIGHT,
            "s1_weight":        INTRA_RARE_MIXUP_WEIGHT,
            "s2_weight":        KG_MIXUP_WEIGHT,
            "s3_weight":        INTER_MIXUP_WEIGHT,
            "soft_temp":        MIXUP_SOFT_TEMP,
        },
        "kg_config": {
            "lambda_boost": lambda_boost, "r_boost_gamma": KG_RARE_GAMMA,
            "proto_k": KG_PROTOTYPE_K,   "virtual_alphas": KG_VIRTUAL_ALPHAS,
            "thresh_rare": UNCERTAINTY_THRESH_RARE,
            "thresh_maj":  UNCERTAINTY_THRESH_MAJORITY,
            "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES, "hop": KG_HOP,
        },
        "timing": {"phase_a_s": base_time, "phase_b_s": kg_time,
                   "total_s": base_time + kg_time},
        "rare_classes": rare_labels,
        "dev":  {k: dev_m[k]  for k in scalar_keys},
        "test": {k: test_m[k] for k in scalar_keys},
        "per_class_dev":  dev_m["per_class_metrics"],
        "per_class_test": test_m["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_metrics_table(dev_m, test_m,
                        base_t=base_time, kg_t=kg_time, n_params=n_params)
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT+BiLSTM+MHA+CRF → KG-RAG+Mixup v4
  [+] Focal Loss      [+] Oversampling      [+] Confusion Edges
  [+] Hard Mining     [+] Gated GAT Fusion  [+] Prototype Contrast
  [NEW] S1: Intra-Rare GPU Manifold Mixup
  [NEW] S2: KG-Neighbour-Guided GPU Mixup
  [NEW] S3: Rare↔HardMaj Inter-class GPU Mixup
  [NEW] Soft-label CRF-aware loss for all Mixup branches

Loading JSONL files ...
  Train:245 | Dev:30 | Test:50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + 0-7.
🔥 BERT trainable: 8-11 + pooler.

[Base] Ep 001/60 | tr_loss:284.8021 val_loss:211.6492 | mac_F1:0.0391 rare_F1:0.0000 | t:42.2s ES:0/10
  ✔ New best val_macro_f1=0.0391
[Base] Ep 002/60 | tr_loss:231.5278 val_loss:171.5741 | mac_F1:0.0727 rare_F1:0.0000 | t:41.8s ES:0/10
  ✔ New best val_macro_f1=0.0727
[Base] Ep 003/60 | tr_loss:190.3178 val_loss:131.4388 | mac_F1:0.2161 rare_F1:0.0598 | t:40.7s ES:0/10
  ✔ New best val_macro_f1=0.2161
[Base] Ep 004/60 | tr_loss:153.1226 val_loss:100.8132 | mac_F1:0.2719 rare_F1:0.1128 | t:42.3s ES:0/10
  ✔ New best val_macro_f1=0.2719
[Base] Ep 005/60 | tr_loss:141.0998 val_loss:93.3196 | mac_F1:0.2704 rare_F1:0.1118 | t:43.6s ES:0/10
[Base] Ep 006/60 | tr_loss:129.2616 val_loss:86.8265 | mac_F1:0.2998 rare_F1:0.1419 | t:45.7s ES:1/10
  ✔ New best val_macro_f1=0.2998
[Base] Ep 007/60 | tr_loss:120.6087 val_loss:80.5756 | mac_F1:0.3316 rare_F1:0.1776 | t:41.0s ES:0/10
  ✔ New best val_macro_f1=0.3316
[Base] 

In [1]:
# inlegalbert_kg_rag_mixup_v5.py
#
# ROOT-CAUSE FIXES over v4 (Minority Macro F1 still lagged):
#
#   FIX 1 — Mixup-CRF Bridge Loss  [NEW]
#     Soft-label Mixup samples are now injected directly into the CRF
#     emissions path so gradient from rare-class boundaries propagates
#     through the CRF transition matrix, not just an auxiliary CE head.
#
#   FIX 2 — Online KG Rare-Node Refresh  [NEW]
#     KG rare partition nodes were built from the Phase-A encoder (weak).
#     Now every ONLINE_KG_REFRESH_EVERY epochs the rare-class subgraph
#     is re-encoded with the current (improving) encoder and updated
#     in-place on the GPU — no full rebuild needed.
#
#   FIX 3 — Asymmetric Rare-Anchored Mixup  [NEW]
#     Rare sample is always the "anchor" (λ ≥ 0.85 for rare-side),
#     preventing majority signal from dominating the interpolated vector.
#     A dedicated rare-only Mixup pass is run once per epoch over ALL
#     rare sentences seen so far (not just the current batch).
#
#   FIX 4 — Span-level Discourse Mixup  [NEW]
#     Legal roles depend on document position (ISSUE before ANALYSIS
#     before RATIO). Mixing single sentences destroys positional context.
#     New span Mixup blends contiguous rare spans and preserves their
#     relative positional encoding in the blended embedding.
#
#   ENHANCEMENT 5 — R-Drop Regulariser on Rare Logits  [NEW]
#     Run two stochastic forward passes for each rare-labelled sentence;
#     add KL(p1||p2) + KL(p2||p1) to the loss. Forces the model to
#     produce consistent predictions for rare classes under dropout noise.
#
#   ENHANCEMENT 6 — LDAM Rare Margin Loss  [NEW]
#     Label-distribution-aware margin: rare classes get a larger decision
#     boundary in the softmax space, making the model work harder to
#     assign them.
#
#   ENHANCEMENT 7 — Dynamic Rare Threshold Inference  [NEW]
#     At inference time: if the top prediction is majority but the
#     second-best is rare AND the logit gap < τ, override to rare.
#     τ is calibrated per rare class on the dev set.
#
#   ENHANCEMENT 8 — Rare Prototype EMA Replay  [NEW]
#     Maintain an exponential moving average (EMA) prototype per rare
#     class. Every K steps, replay the current EMA prototypes through
#     the classifier head so the decision boundary never drifts far
#     from the rare-class centroids.
#
#   ENHANCEMENT 9 — Post-hoc Temperature Calibration  [NEW]
#     After Phase B training, fit a per-class temperature scalar on the
#     dev set using NLL minimisation. Applied at test inference.
#
#   All v3/v4 techniques retained:
#     Focal Loss, Oversampling, Confusion Edges, Hard Mining,
#     Gated GAT Fusion, Prototype Contrastive Loss, Adaptive Uncertainty,
#     S1/S2/S3 GPU Mixup, Soft-label CE.

import os, json, random, time, math, copy
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
)
from sklearn.cluster import MiniBatchKMeans

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_mixup_v5_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 30          # more Phase B epochs for v5
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

LABEL_SMOOTHING  = 0.05
AUX_CE_WEIGHT    = 0.25
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
WARMUP_RATIO     = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD   = 0.05

# ── KG-RAG ────────────────────────────────────────────────
KG_TOP_K        = 3
KG_TOP_NODES    = 6
KG_HOP          = 1
KG_FUSION_DIM   = 256

RST_INTRA_THRESH = 0.55
RST_CROSS_THRESH = 0.45

UNCERTAINTY_THRESH_MAJORITY = 0.70
UNCERTAINTY_THRESH_RARE     = 0.30

RARE_ALWAYS_KG           = True
KG_VIRTUAL_ALPHAS        = [0.25, 0.50, 0.75]
KG_PROTOTYPE_K           = 5
KG_RARE_LAMBDA           = 0.20
KG_RARE_GAMMA            = 2.0
KG_MAX_VIRTUAL_PER_LABEL = 200

FOCAL_GAMMA_RARE         = 2.5
FOCAL_GAMMA_MAJ          = 1.0
OVERSAMPLE_RARE_RATIO    = 3.0
HARD_BUFFER_SIZE         = 300
HARD_REPLAY_FREQ         = 4
HARD_REPLAY_LOSS_THRESH  = 0.8
HARD_REPLAY_WEIGHT       = 0.5
PROTO_CONTRAST_WEIGHT    = 0.08
PROTO_CONTRAST_MARGIN    = 0.35
CONFUSION_EDGE_WEIGHT    = 0.90
CONFUSION_TOP_K          = 3

KG_MAX_NODES_PER_SG      = 2000
KG_MAX_CROSS_EDGES       = 4000
KG_MAX_INTRA_EDGES_STORE = 500_000

# ── v4 Mixup ──────────────────────────────────────────────
MIXUP_ALPHA              = 0.4
MIXUP_LOSS_WEIGHT        = 0.35
MIXUP_MIN_RARE_IN_BATCH  = 2
KG_MIXUP_ENABLED         = True
KG_MIXUP_ALPHA           = 0.3
KG_MIXUP_WEIGHT          = 0.25
INTER_MIXUP_ENABLED      = True
INTER_MIXUP_ALPHA        = 0.15
INTER_MIXUP_WEIGHT       = 0.20
INTER_MIXUP_RARE_RATIO   = 0.85
INTRA_RARE_MIXUP_ENABLED = True
INTRA_RARE_MIXUP_WEIGHT  = 0.20
MIXUP_SOFT_TEMP          = 0.5

# ── [NEW v5] FIX 1 — Mixup-CRF Bridge ────────────────────
MIXUP_CRF_BRIDGE_WEIGHT  = 0.30   # weight for CRF-path Mixup loss
MIXUP_CRF_N_VIRTUAL      = 8      # virtual mixed emissions per rare sent

# ── [NEW v5] FIX 2 — Online KG Node Refresh ──────────────
ONLINE_KG_REFRESH_EVERY  = 5      # refresh rare KG nodes every N Phase-B epochs
ONLINE_KG_REFRESH_DOCS   = 200    # max rare docs to re-encode per refresh

# ── [NEW v5] FIX 3 — Asymmetric Rare-Anchored Mixup ──────
RARE_ANCHOR_LAM_MIN      = 0.85   # rare sample always keeps ≥85%
RARE_ONLY_MIXUP_WEIGHT   = 0.25   # extra rare-only pass loss weight

# ── [NEW v5] FIX 4 — Span Mixup ──────────────────────────
SPAN_MIXUP_ENABLED       = True
SPAN_MIXUP_WEIGHT        = 0.20
SPAN_MIN_LEN             = 2      # minimum span length (sentences)
SPAN_MAX_LEN             = 5

# ── [NEW v5] ENHANCEMENT 5 — R-Drop ──────────────────────
RDROP_WEIGHT             = 0.15
RDROP_RARE_ONLY          = True

# ── [NEW v5] ENHANCEMENT 6 — LDAM ────────────────────────
LDAM_WEIGHT              = 0.20
LDAM_MAX_MARGIN          = 0.5    # max margin for rarest class

# ── [NEW v5] ENHANCEMENT 7 — Dynamic Rare Threshold ──────
DYN_RARE_THRESH_TAU      = 0.25   # logit gap below this → flip to rare
DYN_RARE_ENABLED         = True

# ── [NEW v5] ENHANCEMENT 8 — Prototype EMA Replay ────────
PROTO_EMA_MOMENTUM       = 0.95
PROTO_EMA_REPLAY_EVERY   = 3      # replay every K steps
PROTO_EMA_REPLAY_WEIGHT  = 0.15

# ── [NEW v5] ENHANCEMENT 9 — Post-hoc Calibration ────────
CALIBRATION_EPOCHS       = 200    # gradient steps for temperature fitting

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience = patience; self.min_delta = min_delta
        self.best_score = -1.0; self.counter = 0; self.stop = False

    def step(self, score):
        if score > self.best_score + self.min_delta:
            self.best_score = score; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# FOCAL LOSS
# ═══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, class_weights, rare_ids,
                 gamma_rare=FOCAL_GAMMA_RARE, gamma_maj=FOCAL_GAMMA_MAJ,
                 ignore_index=-100):
        super().__init__()
        self.ignore_index = ignore_index
        gamma_per_class = torch.full((NUM_LABELS,), gamma_maj)
        for r in set(rare_ids): gamma_per_class[r] = gamma_rare
        self.register_buffer("class_weights",   class_weights.float())
        self.register_buffer("gamma_per_class", gamma_per_class)

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if not valid.any(): return logits.sum() * 0.0
        lv = logits[valid]; tv = targets[valid]
        log_p = F.log_softmax(lv, dim=-1)
        p_t   = log_p.exp().gather(1, tv.unsqueeze(1)).squeeze(1)
        focal_w = (1.0 - p_t.detach()).pow(self.gamma_per_class[tv])
        ce = F.nll_loss(log_p, tv, reduction="none")
        return (focal_w * self.class_weights[tv] * ce).mean()


def compute_class_weights(label_freqs):
    weights = torch.ones(NUM_LABELS)
    freqs = [label_freqs.get(id2label[i], 1e-6) for i in range(NUM_LABELS)]
    inv = [1.0 / max(f, 1e-6) for f in freqs]
    inv_sum = sum(inv)
    for i, w in enumerate(inv):
        weights[i] = min(w / inv_sum * NUM_LABELS, 10.0)
    return weights


# ═══════════════════════════════════════════════════════════
# [NEW v5] LDAM LOSS  (Enhancement 6)
# ═══════════════════════════════════════════════════════════
class LDAMLoss(nn.Module):
    """
    Label-Distribution-Aware Margin loss.
    Rare classes get larger decision margin:
        margin_i ∝ 1 / n_i^(1/4)
    """
    def __init__(self, class_counts: dict, max_margin=LDAM_MAX_MARGIN,
                 ignore_index=-100):
        super().__init__()
        self.ignore_index = ignore_index
        counts = torch.tensor(
            [max(class_counts.get(id2label[i], 1), 1)
             for i in range(NUM_LABELS)], dtype=torch.float)
        m = 1.0 / (counts ** 0.25)
        m = m / m.max() * max_margin        # rescale to [0, max_margin]
        self.register_buffer("margins", m)

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if not valid.any(): return logits.sum() * 0.0
        lv = logits[valid]; tv = targets[valid]
        margins = self.margins.unsqueeze(0).expand_as(lv)   # (N, C)
        # Subtract margin from correct class logit
        one_hot = F.one_hot(tv, NUM_LABELS).float()         # (N, C)
        lv_m = lv - margins * one_hot
        return F.cross_entropy(lv_m, tv)


# ═══════════════════════════════════════════════════════════
# [NEW v5] R-DROP REGULARISER  (Enhancement 5)
# ═══════════════════════════════════════════════════════════
def rdrop_kl_loss(p1_logits: torch.Tensor, p2_logits: torch.Tensor,
                  labels: torch.Tensor = None, rare_ids_tensor=None) -> torch.Tensor:
    """
    KL(p1||p2) + KL(p2||p1) symmetrised, averaged over valid positions.
    If rare_ids_tensor provided, only applies to rare-labelled positions.
    """
    if labels is not None and rare_ids_tensor is not None:
        B2T2 = labels.shape[0] * labels.shape[1] if labels.dim() == 2 else labels.shape[0]
        flat_lab = labels.reshape(-1)
        flat_p1  = p1_logits.reshape(-1, NUM_LABELS)
        flat_p2  = p2_logits.reshape(-1, NUM_LABELS)
        valid = flat_lab >= 0
        is_rare = (flat_lab.unsqueeze(1) ==
                   rare_ids_tensor.unsqueeze(0)).any(1)
        mask = valid & is_rare
        if not mask.any(): return torch.tensor(0.0, device=p1_logits.device)
        flat_p1 = flat_p1[mask]; flat_p2 = flat_p2[mask]
    else:
        flat_p1 = p1_logits.reshape(-1, NUM_LABELS)
        flat_p2 = p2_logits.reshape(-1, NUM_LABELS)

    log_p1 = F.log_softmax(flat_p1, dim=-1)
    log_p2 = F.log_softmax(flat_p2, dim=-1)
    p1 = log_p1.exp(); p2 = log_p2.exp()
    kl_12 = F.kl_div(log_p2, p1, reduction="batchmean")
    kl_21 = F.kl_div(log_p1, p2, reduction="batchmean")
    return (kl_12 + kl_21) / 2.0


# ═══════════════════════════════════════════════════════════
# SOFT-LABEL LOSS
# ═══════════════════════════════════════════════════════════
class SoftLabelCrossEntropy(nn.Module):
    def __init__(self, temperature=MIXUP_SOFT_TEMP, class_weights=None):
        super().__init__()
        self.temperature = temperature
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, logits, soft_labels):
        log_p = F.log_softmax(logits / self.temperature, dim=-1)
        if self.class_weights is not None:
            dom_cls = soft_labels.argmax(dim=-1)
            w = self.class_weights[dom_cls]
            return (-(soft_labels * log_p).sum(dim=-1) * w).mean()
        return -(soft_labels * log_p).sum(dim=-1).mean()


# ═══════════════════════════════════════════════════════════
# GPU MANIFOLD MIXUP  (v4 strategies + v5 asymmetric fix)
# ═══════════════════════════════════════════════════════════
class GPUManifoldMixup(nn.Module):
    def __init__(self, num_labels=NUM_LABELS,
                 alpha=MIXUP_ALPHA, kg_alpha=KG_MIXUP_ALPHA,
                 inter_alpha=INTER_MIXUP_ALPHA,
                 rare_ratio=INTER_MIXUP_RARE_RATIO,
                 soft_temp=MIXUP_SOFT_TEMP):
        super().__init__()
        self.num_labels  = num_labels
        self.alpha       = alpha
        self.kg_alpha    = kg_alpha
        self.inter_alpha = inter_alpha
        self.rare_ratio  = rare_ratio
        self.soft_temp   = soft_temp

    @staticmethod
    def _beta_gpu(alpha, size, device):
        if alpha <= 0: return torch.ones(size, device=device)
        d = torch.distributions.Beta(
            torch.tensor(alpha, device=device),
            torch.tensor(alpha, device=device))
        return d.sample((size,))

    @staticmethod
    def _onehot(labels, num_classes):
        return F.one_hot(labels.clamp(min=0), num_classes).float()

    # ── S1: Intra-Rare with asymmetric anchor (FIX 3) ────
    def intra_rare_mixup(self, embs, labels, rare_ids,
                         force_rare_anchor=True):
        is_rare  = (labels.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        rare_idx = is_rare.nonzero(as_tuple=True)[0]
        if rare_idx.shape[0] < MIXUP_MIN_RARE_IN_BATCH:
            return None, None

        N_r  = rare_idx.shape[0]
        lam  = self._beta_gpu(self.alpha, N_r, embs.device)
        # FIX 3: clamp so rare anchor always dominates
        if force_rare_anchor:
            lam = lam.clamp(min=RARE_ANCHOR_LAM_MIN)
        else:
            lam = lam.clamp(0.1, 0.9)

        perm  = torch.randperm(N_r, device=embs.device)
        rare2 = rare_idx[perm]
        e1 = embs[rare_idx]; e2 = embs[rare2]
        l1 = labels[rare_idx]; l2 = labels[rare2]
        lam_e = lam.unsqueeze(1)
        mixed = lam_e * e1 + (1.0 - lam_e) * e2
        soft  = lam_e * self._onehot(l1, self.num_labels) + \
                (1.0 - lam_e) * self._onehot(l2, self.num_labels)
        return mixed, soft

    # ── S2: KG-Guided Mixup ───────────────────────────────
    def kg_guided_mixup(self, embs, labels, nb_embs, nb_weights, rare_ids):
        is_rare = (labels.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        T_prime = embs.shape[0]
        if T_prime < 1: return None, None
        top1_idx  = nb_weights.argmax(dim=1)
        top1_embs = nb_embs[torch.arange(T_prime, device=embs.device), top1_idx]
        base_lam  = self._beta_gpu(self.kg_alpha, T_prime, embs.device)
        lam = torch.where(is_rare, base_lam.clamp(RARE_ANCHOR_LAM_MIN, 0.95),
                          base_lam.clamp(0.4, 0.7))
        lam_e = lam.unsqueeze(1)
        mixed = lam_e * embs + (1.0 - lam_e) * top1_embs
        oh = self._onehot(labels.clamp(min=0), self.num_labels)
        soft = lam_e * oh + (1.0 - lam_e) * (1.0 / self.num_labels)
        return mixed, soft

    # ── S3: Inter-class Rare↔HardMaj ─────────────────────
    def inter_class_mixup(self, embs, labels, rare_ids, confused_maj_ids):
        is_rare  = (labels.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        rare_idx = is_rare.nonzero(as_tuple=True)[0]
        if rare_idx.shape[0] < 2: return None, None
        is_hm = (labels.unsqueeze(1) == confused_maj_ids.unsqueeze(0)).any(1) & ~is_rare
        maj_idx = is_hm.nonzero(as_tuple=True)[0]
        if maj_idx.shape[0] < 1:
            maj_idx = (~is_rare).nonzero(as_tuple=True)[0]
        if maj_idx.shape[0] < 1: return None, None
        N_r  = rare_idx.shape[0]
        lam  = self._beta_gpu(self.inter_alpha, N_r, embs.device).clamp(0, self.inter_alpha * 2)
        perm = torch.randint(0, maj_idx.shape[0], (N_r,), device=embs.device)
        maj_part = maj_idx[perm]
        e1 = embs[rare_idx]; e2 = embs[maj_part]
        l1 = labels[rare_idx]; l2 = labels[maj_part]
        lam_e = lam.unsqueeze(1)
        mixed = (1.0 - lam_e) * e1 + lam_e * e2
        soft = self.rare_ratio * self._onehot(l1, self.num_labels) + \
               (1.0 - self.rare_ratio) * self._onehot(l2, self.num_labels)
        return mixed, soft

    # ── [NEW v5] S4: Span-level Discourse Mixup (FIX 4) ──
    def span_mixup(self, sent_vecs_b, labels_b, rare_ids,
                   span_min=SPAN_MIN_LEN, span_max=SPAN_MAX_LEN):
        """
        Find rare-containing spans in doc b, interpolate with another
        rare span.  Positional index is preserved in mixed embedding
        by blending at each position i:  mix_i = λ·span1_i + (1-λ)·span2_i
        """
        n = sent_vecs_b.shape[0]
        if n < span_min * 2: return None, None
        is_rare = (labels_b.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)

        # collect span start positions
        rare_spans = []
        for start in range(n - span_min + 1):
            end = min(start + span_max, n)
            if is_rare[start:end].any():
                rare_spans.append((start, end))

        if len(rare_spans) < 2: return None, None

        idx_a = random.randrange(len(rare_spans))
        idx_b = random.randrange(len(rare_spans))
        if idx_a == idx_b: return None, None

        sa, ea = rare_spans[idx_a]
        sb, eb = rare_spans[idx_b]
        len_a  = ea - sa
        len_b  = eb - sb
        use_len = min(len_a, len_b)

        lam = self._beta_gpu(MIXUP_ALPHA, 1, sent_vecs_b.device)[0]
        lam = lam.clamp(RARE_ANCHOR_LAM_MIN, 0.95)  # FIX 3 applies here too

        e1 = sent_vecs_b[sa:sa+use_len]
        e2 = sent_vecs_b[sb:sb+use_len]
        l1 = labels_b[sa:sa+use_len]
        l2 = labels_b[sb:sb+use_len]

        mixed = lam * e1 + (1.0 - lam) * e2
        soft  = lam * self._onehot(l1.clamp(min=0), self.num_labels) + \
                (1.0 - lam) * self._onehot(l2.clamp(min=0), self.num_labels)
        return mixed, soft


# ═══════════════════════════════════════════════════════════
# DATA
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    data = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s: data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs): continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    freqs   = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS) if freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        flag = " ← RARE" if lbl in rare_labels else ""
        print(f"   {lbl:<20} {freqs[lbl]*100:5.2f}%  ({counts.get(label2id[lbl],0):5d}){flag}")
    print(f"\n   Rare ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, freqs


class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs = docs; self.tokenizer = tokenizer; self.max_length = max_length

    def __len__(self): return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(sents, padding="max_length", truncation=True,
                             max_length=self.max_length, return_tensors="pt")
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
                "token_type_ids": enc.get("token_type_ids",
                                          torch.zeros_like(enc["input_ids"])),
                "labels": torch.tensor(labels, dtype=torch.long)}


def build_weighted_sampler(docs, rare_ids, oversample_ratio=OVERSAMPLE_RARE_RATIO):
    rare_set = set(rare_ids)
    weights  = [oversample_ratio if any(l in rare_set for l in labs) else 1.0
                for _, labs in docs]
    return WeightedRandomSampler(torch.tensor(weights, dtype=torch.float),
                                 num_samples=len(weights), replacement=True)


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L = batch[0]["input_ids"].shape[1]; B = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), -100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i,:t] = b["input_ids"]; attention_mask[i,:t] = b["attention_mask"]
        token_type_ids[i,:t] = b["token_type_ids"]; labels[i,:t] = b["labels"]
        lengths[i] = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim; self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn = attn.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        return self.out_proj(torch.matmul(attn, V).squeeze(2).reshape(N, H))


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    def __init__(self, bert_model_name=INLEGALBERT_MODEL_NAME,
                 sent_lstm_hidden=SENT_LSTM_HIDDEN, sent_lstm_layers=SENT_LSTM_LAYERS,
                 ctx_lstm_hidden=CTX_LSTM_HIDDEN,   ctx_lstm_layers=CTX_LSTM_LAYERS,
                 mha_heads=MHA_HEADS, mha_dropout=MHA_DROPOUT,
                 num_labels=NUM_LABELS, dropout=DROPOUT, freeze_layers=BERT_FREEZE_LAYERS):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)
        self.sent_bilstm = nn.LSTM(self.bert_dim, sent_lstm_hidden, sent_lstm_layers,
                                   bidirectional=True, batch_first=True,
                                   dropout=dropout if sent_lstm_layers > 1 else 0.0)
        self.sent_out_dim = sent_lstm_hidden * 2
        self.mha_pooling  = MultiHeadAttentionPooling(self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)
        self.ctx_bilstm = nn.LSTM(self.sent_out_dim, ctx_lstm_hidden, ctx_lstm_layers,
                                  bidirectional=True, batch_first=True,
                                  dropout=dropout if ctx_lstm_layers > 1 else 0.0)
        self.ctx_out_dim = ctx_lstm_hidden * 2
        self.classifier  = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(self.ctx_out_dim // 2, num_labels))
        self.crf     = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

    def _freeze_bert_layers(self, n):
        for p in self.bert.embeddings.parameters(): p.requires_grad = False
        for i in range(min(n, len(self.bert.encoder.layer))):
            for p in self.bert.encoder.layer[i].parameters(): p.requires_grad = False
        print(f"❄️  Frozen: embeddings + layers 0-{n-1}. 🔥 Trainable: {n}+\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids):
        B, T, L = input_ids.shape; N = B * T
        fi = input_ids.view(N, L); fm = attention_mask.view(N, L)
        ft = token_type_ids.view(N, L)
        valid = fm.sum(-1) > 0
        embs  = fi.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(fi[valid], fm[valid], ft[valid])
            embs[valid] = out.last_hidden_state.to(embs.dtype)
        embs = self.dropout(embs)
        lo, _ = self.sent_bilstm(embs); lo = self.dropout(lo)
        pm = (fm == 0).clone(); pm[~valid] = False
        sv = self.sent_layer_norm(self.mha_pooling(lo, key_padding_mask=pm))
        sv = sv * valid.unsqueeze(-1).to(sv.dtype)
        return sv.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids, lengths=None):
        sv = self.encode_sentences(input_ids, attention_mask, token_type_ids)
        sd = self.dropout(sv)
        if lengths is not None:
            pk = nn.utils.rnn.pack_padded_sequence(sd, lengths.cpu(),
                                                   batch_first=True, enforce_sorted=False)
            po, _ = self.ctx_bilstm(pk)
            co, _ = nn.utils.rnn.pad_packed_sequence(po, batch_first=True)
        else:
            co, _ = self.ctx_bilstm(sd)
        co = self.dropout(co)
        em = torch.nan_to_num(self.classifier(co), nan=0.0, posinf=1e4, neginf=-1e4)
        return sv, co, em

    def _mask(self, em, labels, lengths):
        if lengths is not None:
            B, T, _ = em.shape
            m = torch.zeros(B, T, dtype=torch.bool, device=em.device)
            for i, l in enumerate(lengths): m[i, :l] = True
        elif labels is not None: m = (labels != -100)
        else: m = torch.ones(em.shape[:2], dtype=torch.bool, device=em.device)
        return m

    def forward(self, ids, attn, ttype, labels=None, lengths=None):
        _, _, em = self.get_emissions(ids, attn, ttype, lengths=lengths)
        mask = self._mask(em, labels, lengths)
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_l = -self.crf(em, safe, mask=mask, reduction="mean")
            B2, T2, C = em.shape
            ce_l = self.ce_loss(em.reshape(B2*T2, C), labels.reshape(B2*T2))
            return crf_l + AUX_CE_WEIGHT * ce_l, em
        return self.crf.decode(em, mask=mask), em


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    def __init__(self, emb_dim=KG_FUSION_DIM, device=DEVICE):
        self.emb_dim = emb_dim; self.device = device
        self.rare_partition = set()
        self._cpu_nodes = defaultdict(list)
        self._stacked = {}; self._norm = {}; self._is_rare = {}
        self._edge_idx = {}; self._edge_w = {}
        self._proto = {}; self._proto_norm = {}
        self._cx_sl = self._cx_si = self._cx_dl = self._cx_di = self._cx_w = None

    def add_nodes(self, embeddings, label_ids):
        for emb, lid in zip(embeddings.detach().cpu(), label_ids):
            self._cpu_nodes[lid].append((emb, False, False))

    def _inject_virtual(self, lid):
        real = [n[0] for n in self._cpu_nodes[lid] if not n[1] and not n[2]]
        n = len(real)
        if n < 2: return 0
        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        random.shuffle(pairs)
        added = 0
        for i, j in pairs:
            for alpha in KG_VIRTUAL_ALPHAS:
                if added >= KG_MAX_VIRTUAL_PER_LABEL: break
                v = F.normalize((alpha*real[i] + (1-alpha)*real[j]).unsqueeze(0), dim=-1).squeeze(0)
                self._cpu_nodes[lid].append((v, True, False)); added += 1
            if added >= KG_MAX_VIRTUAL_PER_LABEL: break
        return added

    def _build_prototypes(self, lid):
        all_e = torch.stack([n[0] for n in self._cpu_nodes[lid]])
        k = min(KG_PROTOTYPE_K, all_e.shape[0])
        if k < 2: c = F.normalize(all_e.mean(0, keepdim=True), dim=-1)
        else:
            km = MiniBatchKMeans(n_clusters=k, n_init=5,
                                 batch_size=min(1024, all_e.shape[0]), random_state=SEED)
            km.fit(all_e.numpy())
            c = F.normalize(torch.tensor(km.cluster_centers_, dtype=torch.float32), dim=-1)
        for ci in range(c.shape[0]):
            self._cpu_nodes[lid].insert(0, (c[ci], False, True))
        return c

    def _promote_to_gpu(self):
        for lid, nodes in self._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in nodes]).to(self.device)
            self._stacked[lid] = embs
            self._norm[lid]    = F.normalize(embs, dim=-1)
            self._is_rare[lid] = torch.tensor([n[1] or n[2] for n in nodes],
                                               dtype=torch.bool, device=self.device)

    def build_edges(self, rare_ids, confusion_pairs=None,
                    intra_thresh=RST_INTRA_THRESH, cross_thresh=RST_CROSS_THRESH,
                    max_intra_per_node=5, max_cross=KG_MAX_CROSS_EDGES):
        print("  Building KG edges ...")
        self.rare_partition = set(rare_ids)
        for lid in rare_ids:
            if self._cpu_nodes[lid]:
                self._inject_virtual(lid); self._build_prototypes(lid)
        self._promote_to_gpu()

        for lid in self._stacked:
            norms = self._norm[lid]; N = norms.shape[0]
            if N < 2:
                self._edge_idx[lid] = torch.zeros(2,0,dtype=torch.long,device=self.device)
                self._edge_w[lid]   = torch.zeros(0, device=self.device); continue
            if lid in self.rare_partition:
                chunk = 512; ei_l, ej_l, ew_l = [], [], []
                for start in range(0, N, chunk):
                    end = min(start+chunk, N)
                    blk = torch.mm(norms[start:end], norms.T)
                    r, c_ = torch.where(
                        (blk > 0) &
                        (torch.arange(start,end,device=self.device).unsqueeze(1) <
                         torch.arange(N, device=self.device).unsqueeze(0)))
                    ei_l.append(r+start); ej_l.append(c_); ew_l.append(blk[r,c_])
                    if sum(x.shape[0] for x in ei_l) >= KG_MAX_INTRA_EDGES_STORE: break
                if ei_l:
                    ei=torch.cat(ei_l); ej=torch.cat(ej_l); ew=torch.cat(ew_l)
                    if ei.shape[0] > KG_MAX_INTRA_EDGES_STORE:
                        p=torch.randperm(ei.shape[0],device=self.device)[:KG_MAX_INTRA_EDGES_STORE]
                        ei,ej,ew=ei[p],ej[p],ew[p]
                    src=torch.cat([ei,ej]); dst=torch.cat([ej,ei]); ww=torch.cat([ew,ew])
                else:
                    src=dst=torch.zeros(0,dtype=torch.long,device=self.device)
                    ww=torch.zeros(0,device=self.device)
            else:
                idx=torch.arange(N,device=self.device); ci=idx[:-1]; cj=idx[1:]
                cw=(norms[ci]*norms[cj]).sum(-1).clamp(min=0)
                sim=torch.mm(norms,norms.T); sim.fill_diagonal_(-2.0)
                sim[ci,cj]=-2.0; sim[cj,ci]=-2.0
                hi_r,hi_c=torch.where(sim>=intra_thresh)
                keep=hi_r<hi_c; hi_r,hi_c=hi_r[keep],hi_c[keep]; hi_w=sim[hi_r,hi_c]
                if hi_r.shape[0] > N*max_intra_per_node:
                    p=torch.randperm(hi_r.shape[0],device=self.device)[:N*max_intra_per_node]
                    hi_r,hi_c,hi_w=hi_r[p],hi_c[p],hi_w[p]
                ei=torch.cat([ci,hi_r]); ej=torch.cat([cj,hi_c]); ew=torch.cat([cw,hi_w])
                src=torch.cat([ei,ej]); dst=torch.cat([ej,ei]); ww=torch.cat([ew,ew])
            self._edge_idx[lid]=torch.stack([src,dst],0); self._edge_w[lid]=ww

        for lid in rare_ids:
            proto=[n for n in self._cpu_nodes.get(lid,[]) if n[2]]
            if proto:
                pc=F.normalize(torch.stack([n[0] for n in proto]).to(self.device),dim=-1)
                self._proto[lid]=self._proto_norm[lid]=pc

        cx_sl,cx_si,cx_dl,cx_di,cx_w=[],[],[],[],[]
        label_ids=sorted(self._stacked.keys()); total_cross=0
        for a in range(len(label_ids)):
            if total_cross>=max_cross: break
            for b in range(a+1,len(label_ids)):
                if total_cross>=max_cross: break
                la,lb=label_ids[a],label_ids[b]
                na=self._norm[la][:KG_MAX_NODES_PER_SG]; nb=self._norm[lb][:KG_MAX_NODES_PER_SG]
                sim=torch.mm(na,nb.T)
                rows,cols=torch.where(sim>=cross_thresh)
                rows,cols=rows[:50],cols[:50]
                if rows.shape[0]==0: continue
                w=sim[rows,cols]
                cx_sl.append(torch.full((rows.shape[0],),la,dtype=torch.long,device=self.device))
                cx_si.append(rows); cx_dl.append(torch.full((rows.shape[0],),lb,dtype=torch.long,device=self.device))
                cx_di.append(cols); cx_w.append(w); total_cross+=rows.shape[0]

        if confusion_pairs:
            for rl, clist in confusion_pairs.items():
                if rl not in self._stacked: continue
                for clid in clist[:CONFUSION_TOP_K]:
                    if clid not in self._stacked: continue
                    rn=self._norm[rl]; cn=self._norm[clid][:KG_MAX_NODES_PER_SG]
                    sim=torch.mm(rn,cn.T)
                    rows,cols=torch.where(sim>0.3); rows,cols=rows[:30],cols[:30]
                    if rows.shape[0]==0: continue
                    w=torch.full((rows.shape[0],),CONFUSION_EDGE_WEIGHT,device=self.device)
                    cx_sl.append(torch.full((rows.shape[0],),rl,dtype=torch.long,device=self.device))
                    cx_si.append(rows); cx_dl.append(torch.full((rows.shape[0],),clid,dtype=torch.long,device=self.device))
                    cx_di.append(cols); cx_w.append(w)

        if cx_sl:
            self._cx_sl=torch.cat(cx_sl); self._cx_si=torch.cat(cx_si)
            self._cx_dl=torch.cat(cx_dl); self._cx_di=torch.cat(cx_di)
            self._cx_w=torch.cat(cx_w)
        else:
            z=torch.zeros(0,dtype=torch.long,device=self.device)
            self._cx_sl=self._cx_si=self._cx_dl=self._cx_di=z
            self._cx_w=torch.zeros(0,device=self.device)
        print(f"  KG ready: cross={self._cx_w.shape[0]}")

    # ── [NEW v5] Online rare-node refresh (FIX 2) ─────────
    @torch.no_grad()
    def refresh_rare_nodes(self, base_model, train_docs, tokenizer,
                           rare_ids, max_docs=ONLINE_KG_REFRESH_DOCS):
        """Re-encode rare docs with current encoder, replace _stacked/_norm."""
        base_model.eval()
        rare_set = set(rare_ids)
        rare_docs = [(sents, labs) for sents, labs in train_docs
                     if any(l in rare_set for l in labs)]
        random.shuffle(rare_docs)
        rare_docs = rare_docs[:max_docs]
        if not rare_docs: return

        dummy   = RRCDataset(rare_docs, tokenizer)
        loader  = DataLoader(dummy, batch_size=4, shuffle=False,
                             collate_fn=collate_rrc, num_workers=0)
        new_nodes = defaultdict(list)
        for ids, attn, tt, labels, lengths in loader:
            ids=ids.to(self.device); attn=attn.to(self.device); tt=tt.to(self.device)
            for bi in range(ids.shape[0]):
                sv = base_model.encode_sentences(ids[bi:bi+1], attn[bi:bi+1], tt[bi:bi+1]).squeeze(0)
                n  = int(lengths[bi].item())
                for emb, lid in zip(sv[:n].cpu(), labels[bi, :n].tolist()):
                    if lid in rare_set:
                        new_nodes[lid].append(emb)

        for lid, embs_list in new_nodes.items():
            if not embs_list: continue
            embs = torch.stack(embs_list).to(self.device)
            self._stacked[lid] = embs
            self._norm[lid]    = F.normalize(embs, dim=-1)
        print(f"  KG rare nodes refreshed: {sum(len(v) for v in new_nodes.values())} new embeddings")

    def save(self, path):
        data = {
            "rare_partition": list(self.rare_partition),
            "nodes": {str(lid): [{"emb": n[0].tolist(), "is_virtual": n[1], "is_proto": n[2]}
                                  for n in nl] for lid, nl in self._cpu_nodes.items()},
            "intra": {str(lid): {"idx": self._edge_idx[lid].cpu().tolist(),
                                  "w": self._edge_w[lid].cpu().tolist()}
                      for lid in self._edge_idx},
            "cross": {"sl": self._cx_sl.cpu().tolist() if self._cx_sl is not None else [],
                      "si": self._cx_si.cpu().tolist() if self._cx_si is not None else [],
                      "dl": self._cx_dl.cpu().tolist() if self._cx_dl is not None else [],
                      "di": self._cx_di.cpu().tolist() if self._cx_di is not None else [],
                      "w":  self._cx_w.cpu().tolist()  if self._cx_w  is not None else []}
        }
        with open(path, "w") as f: json.dump(data, f)
        print(f"  KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM, device=DEVICE):
        kg = cls(emb_dim=emb_dim, device=device)
        with open(path) as f: data = json.load(f)
        kg.rare_partition = set(data.get("rare_partition", []))
        for k, nl in data["nodes"].items():
            lid = int(k)
            for n in nl:
                kg._cpu_nodes[lid].append(
                    (torch.tensor(n["emb"], dtype=torch.float32), n["is_virtual"], n["is_proto"]))
        kg._promote_to_gpu()
        for k, v in data.get("intra", {}).items():
            lid = int(k)
            kg._edge_idx[lid] = torch.tensor(v["idx"], dtype=torch.long,    device=device)
            kg._edge_w[lid]   = torch.tensor(v["w"],   dtype=torch.float32, device=device)
        cross = data.get("cross", {})
        if cross and cross.get("sl"):
            kg._cx_sl = torch.tensor(cross["sl"], dtype=torch.long,    device=device)
            kg._cx_si = torch.tensor(cross["si"], dtype=torch.long,    device=device)
            kg._cx_dl = torch.tensor(cross["dl"], dtype=torch.long,    device=device)
            kg._cx_di = torch.tensor(cross["di"], dtype=torch.long,    device=device)
            kg._cx_w  = torch.tensor(cross["w"],  dtype=torch.float32, device=device)
        else:
            z = torch.zeros(0, dtype=torch.long, device=device)
            kg._cx_sl=kg._cx_si=kg._cx_dl=kg._cx_di=z
            kg._cx_w=torch.zeros(0, device=device)
        for lid in kg.rare_partition:
            proto = [n for n in kg._cpu_nodes.get(lid,[]) if n[2]]
            if proto:
                pc = F.normalize(torch.stack([n[0] for n in proto]).to(device), dim=-1)
                kg._proto[lid] = kg._proto_norm[lid] = pc
        print(f"  KG loaded ← {path}"); return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS,
                 thresh_maj=UNCERTAINTY_THRESH_MAJORITY,
                 thresh_rare=UNCERTAINTY_THRESH_RARE):
        self.log_C = math.log(num_classes)
        self.thresh_maj = thresh_maj; self.thresh_rare = thresh_rare

    def entropy(self, logits):
        p = F.softmax(logits, dim=-1)
        return -(p * (p+1e-9).log()).sum(-1) / self.log_C

    def is_uncertain(self, logits, is_rare_pred):
        H = self.entropy(logits)
        thresh = torch.where(is_rare_pred,
                             torch.full_like(H, self.thresh_rare),
                             torch.full_like(H, self.thresh_maj))
        return H > thresh

    def top_label(self, logits): return logits.argmax(-1)


# ═══════════════════════════════════════════════════════════
# KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    def __init__(self, kg, rare_ids, top_k=KG_TOP_K, top_nodes=KG_TOP_NODES,
                 hop=KG_HOP, lambda_boost=KG_RARE_LAMBDA):
        self.kg = kg; self.rare_ids = set(rare_ids)
        self.top_k = top_k; self.top_nodes = top_nodes
        self.hop = hop; self.lambda_boost = lambda_boost
        self.device = kg.device; self.label_ids = sorted(kg._stacked.keys())

    @classmethod
    def calibrate_lambda(cls, kg, rare_ids, sample_limit=200):
        rare_set = set(rare_ids)
        maj_ids  = [l for l in kg._stacked if l not in rare_set]
        if not maj_ids: return KG_RARE_LAMBDA
        maj_norms = torch.cat([kg._norm[m][:KG_MAX_NODES_PER_SG] for m in maj_ids if m in kg._norm])
        gaps = []
        for lid in rare_ids:
            if lid not in kg._norm: continue
            own = kg._norm[lid]; N = own.shape[0]
            idx = torch.randperm(N, device=kg.device)[:sample_limit]
            s   = own[idx]
            gaps.append((torch.mm(s, maj_norms.T).max(1).values -
                         torch.mm(s, own.T).max(1).values).cpu())
        if not gaps: return KG_RARE_LAMBDA
        lam = float(np.clip(torch.cat(gaps).median().item(), 0.05, 0.40))
        print(f"  λ → {lam:.4f}"); return lam

    def retrieve_batch(self, H_q, trigger_mask):
        if not trigger_mask.any(): return None, None, None
        trig_idx = trigger_mask.nonzero(as_tuple=True)[0]
        H_trig   = H_q[trig_idx]; T_prime, D = H_trig.shape

        sg_sims = []
        for lid in self.label_ids:
            n = self.kg._norm[lid]
            if n.shape[0] == 0:
                sg_sims.append(torch.full((T_prime,), -1.0, device=self.device)); continue
            cap = min(n.shape[0], KG_MAX_NODES_PER_SG)
            s   = torch.mm(H_trig, n[:cap].T).max(1).values
            sg_sims.append(s + (self.lambda_boost if lid in self.rare_ids else 0.0))
        sg_mat = torch.stack(sg_sims, dim=1)
        _, topk = sg_mat.topk(min(self.top_k, len(self.label_ids)), dim=1)

        cap_out = self.top_k * (self.top_nodes + 8)
        nb_embs    = torch.zeros(T_prime, cap_out, D, device=self.device)
        nb_weights = torch.zeros(T_prime, cap_out,    device=self.device)
        nb_is_rare = torch.zeros(T_prime, cap_out, dtype=torch.bool, device=self.device)
        fill = torch.zeros(T_prime, dtype=torch.long, device=self.device)

        for sg_idx in topk.unique().tolist():
            lid = self.label_ids[sg_idx]; is_rare_sg = lid in self.rare_ids
            norms = self.kg._norm[lid]; embs = self.kg._stacked[lid]
            N_sg = norms.shape[0]
            if N_sg == 0: continue
            uses  = (topk == sg_idx).any(dim=1)
            q_idx = uses.nonzero(as_tuple=True)[0]; H_sub = H_trig[q_idx]
            cap_n = min(N_sg, KG_MAX_NODES_PER_SG); cap_e = embs[:cap_n]; cap_nr = norms[:cap_n]
            sim_mat = torch.mm(H_sub, cap_nr.T); k_q = min(self.top_nodes, cap_n)
            topn = sim_mat.topk(k_q, dim=1); topn_idx = topn.indices; topn_sims = topn.values
            edge = self.kg._edge_idx.get(lid)
            for qi_local, qi_global in enumerate(q_idx.tolist()):
                seed = topn_idx[qi_local]
                if self.hop >= 1 and edge is not None and edge.shape[1] > 0:
                    src, dst = edge[0], edge[1]
                    hop_n = dst[torch.isin(src, seed)].unique()
                    if hop_n.shape[0] > 0:
                        seed = torch.cat([seed, hop_n[hop_n < cap_n]]).unique()
                f = int(fill[qi_global].item()); n_add = min(k_q, cap_out - f)
                if n_add <= 0: continue
                nb_embs[qi_global,f:f+n_add]    = cap_e[topn_idx[qi_local,:n_add]]
                nb_weights[qi_global,f:f+n_add] = topn_sims[qi_local,:n_add].clamp(min=0)
                nb_is_rare[qi_global,f:f+n_add] = is_rare_sg
                fill[qi_global] = f + n_add

        cx_sl = self.kg._cx_sl
        if cx_sl is not None and cx_sl.shape[0] > 0:
            for qi_global in range(T_prime):
                f = int(fill[qi_global].item())
                if f >= cap_out: continue
                for ul_idx in topk[qi_global].unique().tolist():
                    lid = self.label_ids[ul_idx]
                    mask_l = (cx_sl == lid)
                    if not mask_l.any(): continue
                    c_di=self.kg._cx_di[mask_l]; c_dl=self.kg._cx_dl[mask_l]; c_w=self.kg._cx_w[mask_l]
                    for dlid_t in c_dl.unique().tolist():
                        dlid = int(dlid_t); dstk = self.kg._stacked.get(dlid)
                        if dstk is None: continue
                        mask_d = (c_dl == dlid_t)
                        di_v = c_di[mask_d][:3]; w_v = c_w[mask_d][:3]
                        di_v = di_v[di_v < dstk.shape[0]]
                        n_a  = min(di_v.shape[0], cap_out - f)
                        if n_a <= 0: continue
                        nb_embs[qi_global,f:f+n_a]    = dstk[di_v[:n_a]]
                        nb_weights[qi_global,f:f+n_a] = w_v[:n_a]
                        nb_is_rare[qi_global,f:f+n_a] = (dlid in self.rare_ids)
                        f += n_a
                fill[qi_global] = f
        return nb_embs, nb_weights, nb_is_rare


# ═══════════════════════════════════════════════════════════
# GATED GAT FUSION
# ═══════════════════════════════════════════════════════════
class GatedGraphAttentionFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT, r_boost=KG_RARE_GAMMA):
        super().__init__()
        self.r_boost = r_boost
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.gate    = nn.Linear(emb_dim * 2, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, H_q, nb_embs, nb_w, nb_rare):
        pad_mask = (nb_w == 0)
        q = self.proj_q(H_q); k = self.proj_k(nb_embs)
        dot = torch.bmm(k, q.unsqueeze(-1)).squeeze(-1) * self.scale
        r_b = torch.where(nb_rare, torch.full_like(dot, self.r_boost), torch.ones_like(dot))
        raw = (dot * nb_w * r_b).masked_fill(pad_mask, -1e9)
        alpha = F.softmax(raw, dim=-1).masked_fill(pad_mask, 0.0)
        v_attn = torch.bmm(self.dropout(alpha).unsqueeze(1), nb_embs).squeeze(1)
        g = torch.sigmoid(self.gate(torch.cat([H_q, v_attn], dim=-1)))
        return (1.0 - g) * H_q + g * v_attn


# ═══════════════════════════════════════════════════════════
# HARD EXAMPLE BUFFER
# ═══════════════════════════════════════════════════════════
class HardExampleBuffer:
    def __init__(self, max_size=HARD_BUFFER_SIZE):
        self.max_size = max_size; self.buf = []

    def update(self, loss_val, batch_cpu):
        self.buf.append((loss_val, batch_cpu))
        if len(self.buf) > self.max_size:
            self.buf.sort(key=lambda x: -x[0]); self.buf = self.buf[:self.max_size // 2]

    def sample(self):
        if not self.buf: return None
        losses = torch.tensor([x[0] for x in self.buf], dtype=torch.float)
        return self.buf[int(torch.multinomial(F.softmax(losses, 0), 1).item())][1]

    def __len__(self): return len(self.buf)


# ═══════════════════════════════════════════════════════════
# [NEW v5] RARE PROTOTYPE EMA STORE  (Enhancement 8)
# ═══════════════════════════════════════════════════════════
class RarePrototypeEMAStore:
    """
    Maintains an exponential moving average prototype per rare class.
    Updated each step when rare sentences are seen; replayed every K steps.
    """
    def __init__(self, rare_ids, emb_dim, device,
                 momentum=PROTO_EMA_MOMENTUM):
        self.rare_ids = set(rare_ids); self.device = device
        self.momentum = momentum
        self.protos = {
            lid: torch.zeros(emb_dim, device=device)
            for lid in rare_ids
        }
        self.counts = {lid: 0 for lid in rare_ids}

    @torch.no_grad()
    def update(self, sent_vecs_flat, labels_flat):
        """Update EMA prototypes with current batch embeddings."""
        for lid in self.rare_ids:
            mask = (labels_flat == lid)
            if not mask.any(): continue
            mean_emb = sent_vecs_flat[mask].mean(0)
            if self.counts[lid] == 0:
                self.protos[lid] = mean_emb.detach()
            else:
                self.protos[lid] = (self.momentum * self.protos[lid] +
                                    (1 - self.momentum) * mean_emb.detach())
            self.counts[lid] += 1

    def get_proto_tensor(self):
        """Returns (R, D) tensor of EMA prototypes for rare classes with data."""
        valid = [(lid, self.protos[lid]) for lid in self.rare_ids
                 if self.counts[lid] > 0]
        if not valid: return None, None
        lids, protos = zip(*valid)
        return torch.stack(protos), torch.tensor(lids, dtype=torch.long, device=self.device)


# ═══════════════════════════════════════════════════════════
# [NEW v5] POST-HOC TEMPERATURE CALIBRATION  (Enhancement 9)
# ═══════════════════════════════════════════════════════════
class PerClassTemperatureCalibrator(nn.Module):
    """Per-class temperature scaling fitted on dev set."""
    def __init__(self, num_classes=NUM_LABELS):
        super().__init__()
        self.temps = nn.Parameter(torch.ones(num_classes))  # one T per class

    def forward(self, logits):
        # Scale logits per predicted class direction
        return logits / self.temps.clamp(min=0.05).unsqueeze(0)

    def fit(self, logits_all, labels_all, epochs=CALIBRATION_EPOCHS, device=DEVICE):
        """NLL minimisation on dev set logits."""
        self.to(device)
        opt = torch.optim.LBFGS(self.parameters(), lr=0.01, max_iter=50)
        logits_t = logits_all.to(device); labels_t = labels_all.to(device)
        nll = nn.CrossEntropyLoss(ignore_index=-100)

        def closure():
            opt.zero_grad()
            scaled = self.forward(logits_t)
            loss = nll(scaled, labels_t)
            loss.backward(); return loss

        for _ in range(epochs // 50):
            opt.step(closure)
        print(f"  Calibration done. Temp range: [{self.temps.min().item():.3f}, "
              f"{self.temps.max().item():.3f}]")


# ═══════════════════════════════════════════════════════════
# FULL KG + MIXUP v5 MODEL
# ═══════════════════════════════════════════════════════════
class KGMixupV5Model(nn.Module):
    def __init__(self, base_model, kg, rare_ids, retriever=None,
                 focal_loss=None, ldam_loss=None,
                 class_weights=None, confusion_pairs=None):
        super().__init__()
        self.base = base_model; self.kg = kg
        self.rare_ids = set(rare_ids); self.rare_list = sorted(rare_ids)
        self.retriever   = retriever or KGRetriever(kg, rare_ids)
        self.uncertainty = UncertaintyEstimator()
        self.focal_loss  = focal_loss
        self.ldam_loss   = ldam_loss

        sent_dim = base_model.sent_out_dim   # 256
        ctx_dim  = base_model.ctx_out_dim    # 128

        self.gat_fusion = GatedGraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(ctx_dim // 2, NUM_LABELS))
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        self.gpu_mixup  = GPUManifoldMixup()
        self.mixup_proj = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, NUM_LABELS))
        self.soft_ce = SoftLabelCrossEntropy(temperature=MIXUP_SOFT_TEMP,
                                             class_weights=class_weights)

        # ── [NEW v5] Mixup-CRF Bridge head (FIX 1) ────────
        # Projects mixed sent_dim embeddings into ctx_dim for CRF path
        self.mixup_crf_proj = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, ctx_dim))
        # ctx-like BiLSTM wrapper NOT rerun — use linear bridge instead
        self.mixup_crf_classifier = nn.Linear(ctx_dim, NUM_LABELS)
        self.mixup_bridge_crf     = CRF(num_tags=NUM_LABELS, batch_first=True)

        rare_t = torch.tensor(self.rare_list, dtype=torch.long)
        self.register_buffer("_rare_t", rare_t)

        if confusion_pairs:
            conf_maj = list({cid for cids in confusion_pairs.values() for cid in cids})
        else:
            conf_maj = list(range(NUM_LABELS))
        self.register_buffer("_conf_maj_t",
                             torch.tensor(conf_maj, dtype=torch.long))

    def _rare_mask(self, top_labels):
        return (top_labels.unsqueeze(-1) == self._rare_t.view(1,1,-1)).any(-1)

    def _proto_contrast_loss(self, device):
        rp = [self.kg._proto_norm[l] for l in self.rare_list if l in self.kg._proto_norm]
        if not rp: return torch.tensor(0.0, device=device)
        mp = [self.kg._proto_norm[l] for l in self.kg._stacked
              if l not in self.rare_ids and l in self.kg._proto_norm]
        if not mp: return torch.tensor(0.0, device=device)
        rp = F.normalize(torch.cat(rp).to(device), dim=-1)
        mp = F.normalize(torch.cat(mp).to(device), dim=-1)
        return torch.clamp(torch.mm(rp, mp.T) - PROTO_CONTRAST_MARGIN, min=0.0).mean()

    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, D = sent_vecs.shape; fused = sent_vecs.clone()
        top_labels = self.uncertainty.top_label(emissions)
        is_rare_p  = self._rare_mask(top_labels)
        trigger    = self.uncertainty.is_uncertain(emissions, is_rare_p) | \
                     (RARE_ALWAYS_KG & is_rare_p)
        for b in range(B):
            n = int(lengths[b].item()); trig_b = trigger[b, :n]
            if not trig_b.any(): continue
            H_b = sent_vecs[b, :n]
            nb_embs, nb_w, nb_rare = self.retriever.retrieve_batch(
                F.normalize(H_b.detach(), dim=-1), trig_b)
            if nb_embs is None: continue
            trig_idx = trig_b.nonzero(as_tuple=True)[0]
            fused[b, trig_idx] = self.gat_fusion(H_b[trig_idx], nb_embs, nb_w, nb_rare)
        return fused

    # ── [NEW v5] FIX 1: Mixup-CRF Bridge loss ─────────────
    def _mixup_crf_bridge_loss(self, mixed_embs, soft_labels, device):
        """
        Project mixed embeddings into CRF emission space, then compute
        CRF NLL with a soft-label interpolated hard label (argmax of soft).
        This forces the CRF transition matrix to respect rare boundaries.
        """
        if mixed_embs is None or mixed_embs.shape[0] < 1:
            return torch.tensor(0.0, device=device)

        # (N_mix, sent_dim) → (N_mix, ctx_dim) → (N_mix, C)
        bridge_emis = self.mixup_crf_classifier(
            self.mixup_crf_proj(mixed_embs))                       # (N_mix, C)
        bridge_emis = torch.nan_to_num(bridge_emis, nan=0.0)

        # Hard pseudo-labels from soft distribution argmax
        pseudo = soft_labels.argmax(dim=-1)                        # (N_mix,)
        # Reshape to (1, N_mix, C) and run CRF as a single sequence
        em3 = bridge_emis.unsqueeze(0)                             # (1, N_mix, C)
        tg3 = pseudo.unsqueeze(0)                                  # (1, N_mix)
        mask3 = torch.ones(1, bridge_emis.shape[0],
                           dtype=torch.bool, device=device)
        crf_l = -self.mixup_bridge_crf(em3, tg3, mask=mask3, reduction="mean")

        # Also add soft-label CE for smoother gradient
        soft_l = self.soft_ce(bridge_emis, soft_labels.to(device))
        return crf_l + 0.5 * soft_l

    def _compute_all_mixup_losses(self, sent_vecs, labels, lengths, emissions, device):
        total = torch.tensor(0.0, device=device)
        B, T, D = sent_vecs.shape

        flat_embs, flat_labs = [], []
        for b in range(B):
            n = int(lengths[b].item())
            flat_embs.append(sent_vecs[b, :n])
            flat_labs.append(labels[b, :n])
        if not flat_embs: return total

        flat_embs = torch.cat(flat_embs); flat_labs = torch.cat(flat_labs)
        vm = flat_labs >= 0
        flat_embs = flat_embs[vm]; flat_labs = flat_labs[vm]
        if flat_embs.shape[0] < MIXUP_MIN_RARE_IN_BATCH: return total

        # S1: Intra-Rare (asymmetric anchor) + CRF bridge
        if INTRA_RARE_MIXUP_ENABLED:
            mx, soft = self.gpu_mixup.intra_rare_mixup(
                flat_embs, flat_labs, self._rare_t, force_rare_anchor=True)
            if mx is not None:
                total = total + INTRA_RARE_MIXUP_WEIGHT * self.soft_ce(
                    self.mixup_proj(mx), soft.to(device))
                # FIX 1: CRF bridge
                total = total + MIXUP_CRF_BRIDGE_WEIGHT * self._mixup_crf_bridge_loss(
                    mx, soft, device)

        # S2: KG-Guided Mixup
        if KG_MIXUP_ENABLED:
            for b in range(B):
                n = int(lengths[b].item()); H_b = sent_vecs[b, :n]; l_b = labels[b, :n]
                valid_b = l_b >= 0
                is_rare_b = (l_b.unsqueeze(1) == self._rare_t.unsqueeze(0)).any(1)
                trig_b = is_rare_b & valid_b
                if not trig_b.any(): continue
                nb_embs, nb_w, _ = self.retriever.retrieve_batch(
                    F.normalize(H_b.detach(), dim=-1), trig_b)
                if nb_embs is None: continue
                trig_idx = trig_b.nonzero(as_tuple=True)[0]
                mx, soft = self.gpu_mixup.kg_guided_mixup(
                    H_b[trig_idx], l_b[trig_idx], nb_embs, nb_w, self._rare_t)
                if mx is not None:
                    total = total + KG_MIXUP_WEIGHT * self.soft_ce(
                        self.mixup_proj(mx), soft.to(device))

        # S3: Inter-class
        if INTER_MIXUP_ENABLED:
            mx, soft = self.gpu_mixup.inter_class_mixup(
                flat_embs, flat_labs, self._rare_t, self._conf_maj_t)
            if mx is not None:
                total = total + INTER_MIXUP_WEIGHT * self.soft_ce(
                    self.mixup_proj(mx), soft.to(device))

        # S4: Span Mixup (FIX 4) — per document
        if SPAN_MIXUP_ENABLED:
            for b in range(B):
                n = int(lengths[b].item())
                mx, soft = self.gpu_mixup.span_mixup(
                    sent_vecs[b, :n], labels[b, :n], self._rare_t)
                if mx is not None:
                    total = total + SPAN_MIXUP_WEIGHT * self.soft_ce(
                        self.mixup_proj(mx), soft.to(device))

        return total

    def forward(self, ids, attn, ttype, labels=None, lengths=None,
                return_sent_vecs=False):
        device = ids.device
        sent_vecs, ctx_out, base_em = self.base.get_emissions(
            ids, attn, ttype, lengths=lengths)

        fused_sent = self._kg_fuse_batch(sent_vecs, base_em, lengths, device)
        fd = self.base.dropout(fused_sent)
        if lengths is not None:
            pk = nn.utils.rnn.pack_padded_sequence(fd, lengths.cpu(),
                                                   batch_first=True, enforce_sorted=False)
            po, _ = self.base.ctx_bilstm(pk)
            fc, _ = nn.utils.rnn.pad_packed_sequence(po, batch_first=True)
        else:
            fc, _ = self.base.ctx_bilstm(fd)
        fc = self.base.dropout(fc)
        fused_em = torch.nan_to_num(self.fusion_classifier(fc), nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = fused_em.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths): mask[i, :l] = True
        elif labels is not None: mask = (labels != -100)
        else: mask = torch.ones(fused_em.shape[:2], dtype=torch.bool, device=device)

        combined = (base_em + fused_em) / 2.0

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            base_crf  = -self.base.crf(base_em,  safe, mask=mask, reduction="mean")
            fused_crf = -self.fusion_crf(fused_em, safe, mask=mask, reduction="mean")

            B2, T2, C = combined.shape
            flat_logits = combined.reshape(B2*T2, C)
            flat_labels = labels.reshape(B2*T2)

            if self.focal_loss is not None:
                aux_loss = self.focal_loss(flat_logits, flat_labels)
            else:
                aux_loss = self.ce_loss(flat_logits, flat_labels)

            # LDAM (Enhancement 6)
            ldam_loss = torch.tensor(0.0, device=device)
            if self.ldam_loss is not None:
                ldam_loss = LDAM_WEIGHT * self.ldam_loss(flat_logits, flat_labels)

            proto_loss  = PROTO_CONTRAST_WEIGHT * self._proto_contrast_loss(device)
            mixup_loss  = MIXUP_LOSS_WEIGHT * self._compute_all_mixup_losses(
                sent_vecs, labels, lengths, base_em, device)

            loss = ((base_crf + fused_crf) / 2.0 + AUX_CE_WEIGHT * aux_loss +
                    proto_loss + mixup_loss + ldam_loss)

            if return_sent_vecs:
                return loss, combined, sent_vecs
            return loss, combined
        else:
            decoded = self.fusion_crf.decode(fused_em, mask=mask)
            return decoded, fused_em


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    mf = lambda a, p, av: f1_score(a, p, average=av, zero_division=0)
    mp = lambda a, p, av: precision_score(a, p, average=av, zero_division=0)
    mr = lambda a, p, av: recall_score(a, p, average=av, zero_division=0)
    per_f = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                     average=None, zero_division=0)
    per_p = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                            average=None, zero_division=0)
    per_r = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                         average=None, zero_division=0)
    pcm = {id2label[i]: {"f1": float(per_f[i]),
                          "precision": float(per_p[i]),
                          "recall":    float(per_r[i])}
           for i in range(NUM_LABELS)}
    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rf1 = f1_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rpr = precision_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rrc = recall_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
    else:
        rf1 = rpr = rrc = 0.0
    st = [id2label[x] for x in all_trues]; sp = [id2label[x] for x in all_preds]
    return dict(
        macro_f1=mf(all_trues,all_preds,"macro"), micro_f1=mf(all_trues,all_preds,"micro"),
        weighted_f1=mf(all_trues,all_preds,"weighted"),
        macro_precision=mp(all_trues,all_preds,"macro"), micro_precision=mp(all_trues,all_preds,"micro"),
        weighted_precision=mp(all_trues,all_preds,"weighted"),
        macro_recall=mr(all_trues,all_preds,"macro"), micro_recall=mr(all_trues,all_preds,"micro"),
        weighted_recall=mr(all_trues,all_preds,"weighted"),
        rare_f1=rf1, rare_precision=rpr, rare_recall=rrc,
        per_class_metrics=pcm, accuracy=accuracy_score(all_trues, all_preds),
        cls_report=classification_report(st, sp, labels=LABELS, digits=4, zero_division=0),
        cm=confusion_matrix(st, sp, labels=LABELS),
        all_preds=all_preds, all_trues=all_trues)


def count_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    fr = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  Trainable: {tr:,} | Frozen: {fr:,}"); return tr, fr


# ═══════════════════════════════════════════════════════════
# CONFUSION PAIRS
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def compute_confusion_pairs(model, dataset, rare_ids, device=DEVICE):
    model.eval()
    loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
    all_preds, all_trues = [], []
    rare_set = set(rare_ids)
    for ids, attn, tt, labels, lengths in loader:
        ids=ids.to(device); attn=attn.to(device); tt=tt.to(device); lengths=lengths.to(device)
        decoded, _ = model(ids, attn, tt, labels=None, lengths=lengths)
        for i, sp in enumerate(decoded):
            all_preds.extend(sp)
            all_trues.extend(labels[i, :int(lengths[i].item())].tolist())
    confusion = defaultdict(Counter)
    for t, p in zip(all_trues, all_preds):
        if t in rare_set and p != t: confusion[t][p] += 1
    result = {}
    print("\n📊 Confusion pairs:")
    for rid, ctr in confusion.items():
        top = [c for c, _ in ctr.most_common(CONFUSION_TOP_K)]
        result[rid] = top
        print(f"  {id2label[rid]:<20} → {[id2label[c] for c in top]}")
    return result


# ═══════════════════════════════════════════════════════════
# BASE TRAINER
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, focal_loss=None, device=DEVICE):
        self.model = model.to(device); self.focal_loss = focal_loss; self.device = device

    def build_optimizer(self):
        pg = [{"params": list(self.model.bert.pooler.parameters()),
               "lr": BERT_LR, "weight_decay": WEIGHT_DECAY}]
        enc = self.model.bert.encoder.layer; n = len(enc)
        for i in range(n-1, BERT_FREEZE_LAYERS-1, -1):
            params = [p for p in enc[i].parameters() if p.requires_grad]
            if params:
                pg.append({"params": params, "lr": BERT_LR * (BERT_LR_DECAY ** ((n-1)-i)),
                           "weight_decay": WEIGHT_DECAY})
        head_mods = [self.model.sent_bilstm, self.model.mha_pooling,
                     self.model.sent_layer_norm, self.model.ctx_bilstm,
                     self.model.classifier, self.model.crf]
        pg.append({"params": [p for m in head_mods for p in m.parameters()],
                   "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(pg)

    def evaluate(self, dataset, rare_ids, split_name="dev", measure_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_time else None
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids=ids.to(self.device); attn=attn.to(self.device)
                tt=tt.to(self.device); lengths=lengths.to(self.device)
                decoded, _ = self.model(ids, attn, tt, labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :int(lengths[i].item())].tolist())
        if measure_time:
            ti = time.time() - t0
            with open(os.path.join(OUT_DIR, f"inf_{split_name}.json"), "w") as f:
                json.dump({"total_s": ti, "ms_per_doc": ti/max(1,len(dataset))*1000}, f)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_docs, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        sampler = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)
        optimizer = self.build_optimizer()
        total_steps = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        scheduler = get_linear_schedule_with_warmup(
            optimizer, int(WARMUP_RATIO*total_steps), total_steps)
        es = EarlyStopping(); history = []; best_f1 = -1.0; best_state = None
        t_start = time.time()

        for epoch in range(1, num_epochs+1):
            self.model.train(); run_loss = 0.0; n_steps = 0; t0 = time.time()
            optimizer.zero_grad()
            for step, (ids, attn, tt, labels, lengths) in enumerate(train_loader):
                ids=ids.to(self.device); attn=attn.to(self.device)
                tt=tt.to(self.device);  labels=labels.to(self.device); lengths=lengths.to(self.device)
                loss, em = self.model(ids, attn, tt, labels=labels, lengths=lengths)
                if self.focal_loss is not None:
                    B2,T2,C=em.shape
                    loss = loss + AUX_CE_WEIGHT * self.focal_loss(
                        em.reshape(B2*T2,C).detach(), labels.reshape(B2*T2))
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue
                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step+1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()
                run_loss += loss.item(); n_steps += 1
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            avg = run_loss / max(1, n_steps)
            val_m = self.evaluate(dev_dataset, rare_ids)
            print(f"[Base] {epoch:03d}/{num_epochs} loss:{avg:.4f} "
                  f"mac:{val_m['macro_f1']:.4f} rare:{val_m['rare_f1']:.4f} "
                  f"t:{time.time()-t0:.1f}s ES:{es.counter}/{es.patience}")
            history.append({"epoch": epoch, "phase": "base", "train_loss": avg,
                             "val_macro_f1": val_m["macro_f1"], "val_rare_f1": val_m["rare_f1"],
                             "val_accuracy": val_m["accuracy"]})
            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1 = val_m["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ best={best_f1:.4f}")
            if es.step(val_m["macro_f1"]): print(f"⏹ ES epoch {epoch}"); break

        total_time = time.time() - t_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "base_model.bin"))
        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer, rare_ids,
                           confusion_pairs, device=DEVICE):
    print("🔨 Building KG ...")
    base_model.eval(); base_model.to(device)
    kg = KnowledgeGraph(emb_dim=base_model.sent_out_dim, device=device)
    dummy  = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy, batch_size=4, shuffle=False,
                        collate_fn=collate_rrc, num_workers=0)
    for di, (ids, attn, tt, labels, lengths) in enumerate(loader):
        ids=ids.to(device); attn=attn.to(device); tt=tt.to(device)
        for bi in range(ids.shape[0]):
            sv = base_model.encode_sentences(ids[bi:bi+1], attn[bi:bi+1], tt[bi:bi+1]).squeeze(0)
            n  = int(lengths[bi].item())
            kg.add_nodes(sv[:n].cpu(), labels[bi, :n].tolist())
        if (di+1) % 50 == 0: print(f"  {(di+1)*4}/{len(train_docs)} docs")
    kg.build_edges(rare_ids=rare_ids, confusion_pairs=confusion_pairs)
    kg.save(os.path.join(OUT_DIR, "knowledge_graph.json"))
    return kg


# ═══════════════════════════════════════════════════════════
# [NEW v5] DYNAMIC RARE THRESHOLD INFERENCE  (Enhancement 7)
# ═══════════════════════════════════════════════════════════
def calibrate_rare_threshold(model, dev_dataset, rare_ids, device=DEVICE):
    """Collect per-rare-class optimal tau on dev set."""
    model.eval()
    loader = DataLoader(dev_dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
    all_logits, all_labels = [], []
    rare_set = set(rare_ids)
    with torch.no_grad():
        for ids, attn, tt, labels, lengths in loader:
            ids=ids.to(device); attn=attn.to(device); tt=tt.to(device); lengths=lengths.to(device)
            _, em = model(ids, attn, tt, labels=None, lengths=lengths)
            B, T, C = em.shape
            for b in range(B):
                n = int(lengths[b].item())
                all_logits.append(em[b, :n].cpu())
                all_labels.append(labels[b, :n])
    all_logits = torch.cat(all_logits); all_labels = torch.cat(all_labels)

    # Per-rare-class: find tau that maximises rare-class F1
    taus = {}
    for rid in rare_ids:
        best_tau, best_f = DYN_RARE_THRESH_TAU, 0.0
        for tau in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
            preds = apply_dynamic_rare_threshold_np(
                all_logits.numpy(), all_labels.numpy(), {rid: tau}, rare_ids)
            f = f1_score(all_labels.numpy(), preds, labels=[rid],
                         average="macro", zero_division=0)
            if f > best_f: best_f = f; best_tau = tau
        taus[rid] = best_tau
    print(f"  Dynamic tau per rare class: {[(id2label[k], v) for k,v in taus.items()]}")
    return taus


def apply_dynamic_rare_threshold_np(logits_np, labels_np, taus, rare_ids):
    """Apply per-rare-class dynamic threshold at inference (numpy version)."""
    preds = logits_np.argmax(axis=-1).copy()
    rare_set = set(rare_ids)
    for i, logit in enumerate(logits_np):
        top1 = preds[i]
        if top1 in rare_set: continue  # already rare
        # Check if any rare class is close
        rare_logits = {rid: logit[rid] for rid in rare_ids}
        top_rare_id = max(rare_logits, key=rare_logits.get)
        tau = taus.get(top_rare_id, DYN_RARE_THRESH_TAU)
        gap = logit[top1] - logit[top_rare_id]
        if gap < tau:
            preds[i] = top_rare_id
    return preds


@torch.no_grad()
def apply_dynamic_threshold_dataset(model, dataset, rare_ids, taus,
                                    device=DEVICE, calibrator=None):
    """Full dataset inference with dynamic rare threshold + optional calibration."""
    model.eval()
    loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
    all_preds, all_trues = [], []
    for ids, attn, tt, labels, lengths in loader:
        ids=ids.to(device); attn=attn.to(device); tt=tt.to(device); lengths=lengths.to(device)
        _, em = model(ids, attn, tt, labels=None, lengths=lengths)
        B, T, C = em.shape
        for b in range(B):
            n = int(lengths[b].item())
            logit_b = em[b, :n]
            if calibrator is not None:
                logit_b = calibrator(logit_b)
            logit_np = logit_b.cpu().numpy()
            lab_np   = labels[b, :n].numpy()
            preds    = apply_dynamic_rare_threshold_np(logit_np, lab_np, taus, rare_ids)
            all_preds.extend(preds.tolist())
            all_trues.extend(lab_np.tolist())
    return all_trues, all_preds


# ═══════════════════════════════════════════════════════════
# KG + MIXUP v5 TRAINER
# ═══════════════════════════════════════════════════════════
class KGMixupV5Trainer:
    def __init__(self, kg_model: KGMixupV5Model, kg: KnowledgeGraph,
                 train_docs, tokenizer, device=DEVICE):
        self.model      = kg_model.to(device)
        self.kg         = kg
        self.train_docs = train_docs
        self.tokenizer  = tokenizer
        self.device     = device
        # EMA prototype store
        self.ema_store = RarePrototypeEMAStore(
            kg_model.rare_list, kg_model.base.sent_out_dim, device)

    def build_optimizer(self):
        new_p = (list(self.model.gat_fusion.parameters()) +
                 list(self.model.fusion_classifier.parameters()) +
                 list(self.model.fusion_crf.parameters()) +
                 list(self.model.mixup_proj.parameters()) +
                 list(self.model.gpu_mixup.parameters()) +
                 list(self.model.mixup_crf_proj.parameters()) +        # FIX 1
                 list(self.model.mixup_crf_classifier.parameters()) +  # FIX 1
                 list(self.model.mixup_bridge_crf.parameters()))       # FIX 1
        base_p = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_p,  "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_p, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev", measure_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_time else None
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids=ids.to(self.device); attn=attn.to(self.device)
                tt=tt.to(self.device); lengths=lengths.to(self.device)
                decoded, _ = self.model(ids, attn, tt, labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :int(lengths[i].item())].tolist())
        if measure_time:
            ti = time.time() - t0
            with open(os.path.join(OUT_DIR, f"kg_inf_{split_name}.json"), "w") as f:
                json.dump({"total_s": ti, "ms_per_doc": ti/max(1,len(dataset))*1000}, f)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    # ── Prototype EMA replay step (Enhancement 8) ─────────
    def _proto_ema_replay_step(self, optimizer):
        protos, proto_labels = self.ema_store.get_proto_tensor()
        if protos is None or protos.shape[0] < 2: return
        # Pass prototypes (as sent_vecs) through mixup_proj → logits
        # and apply cross-entropy with their true labels
        logits = self.model.mixup_proj(protos.detach())   # (R, C)
        loss   = F.cross_entropy(logits, proto_labels) * PROTO_EMA_REPLAY_WEIGHT
        if not torch.isnan(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
            optimizer.step(); optimizer.zero_grad()

    def train(self, train_docs, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        sampler = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, int(WARMUP_RATIO*total_steps), total_steps)
        es = EarlyStopping(patience=8)
        buffer = HardExampleBuffer()
        rare_set = set(rare_ids)
        history = []; best_f1 = -1.0; best_state = None
        t_start = time.time()

        for epoch in range(1, num_epochs+1):
            self.model.train(); run_loss = 0.0; n_steps = 0; t0 = time.time()
            optimizer.zero_grad()

            # FIX 2: Online KG refresh
            if epoch > 1 and epoch % ONLINE_KG_REFRESH_EVERY == 0:
                print(f"  [KG Refresh] epoch {epoch}")
                self.kg.refresh_rare_nodes(
                    self.model.base, self.train_docs, self.tokenizer, rare_ids)

            for step, (ids, attn, tt, labels, lengths) in enumerate(train_loader):
                ids=ids.to(self.device); attn=attn.to(self.device)
                tt=tt.to(self.device);  labels=labels.to(self.device); lengths=lengths.to(self.device)

                # Enhancement 5: R-Drop — two forward passes
                loss1, em1, sv1 = self.model(ids, attn, tt, labels=labels,
                                              lengths=lengths, return_sent_vecs=True)
                loss2, em2, sv2 = self.model(ids, attn, tt, labels=labels,
                                              lengths=lengths, return_sent_vecs=True)

                rdrop_l = RDROP_WEIGHT * rdrop_kl_loss(
                    em1, em2, labels, self.model._rare_t if RDROP_RARE_ONLY else None)
                loss = (loss1 + loss2) / 2.0 + rdrop_l

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                run_loss += loss.item(); n_steps += 1

                # Update EMA prototypes (Enhancement 8)
                with torch.no_grad():
                    for b in range(ids.shape[0]):
                        n = int(lengths[b].item())
                        self.ema_store.update(sv1[b,:n], labels[b,:n])

                # EMA replay (Enhancement 8)
                if (step+1) % PROTO_EMA_REPLAY_EVERY == 0:
                    self._proto_ema_replay_step(optimizer)

                # Hard buffer
                lv = loss.item()
                if lv > HARD_REPLAY_LOSS_THRESH:
                    fl = labels.cpu().flatten()
                    if any(int(x) in rare_set for x in fl if x.item() >= 0):
                        buffer.update(lv, (ids.cpu(), attn.cpu(), tt.cpu(),
                                          labels.cpu(), lengths.cpu()))

                if (step+1) % HARD_REPLAY_FREQ == 0 and len(buffer) >= 10:
                    replay = buffer.sample()
                    if replay:
                        ri, ra, rt, rl, rn = [x.to(self.device) for x in replay]
                        rl_, em_, sv_ = self.model(ri, ra, rt, labels=rl,
                                                    lengths=rn, return_sent_vecs=True)
                        if not torch.isnan(rl_) and not torch.isinf(rl_):
                            (rl_ * HARD_REPLAY_WEIGHT).backward()
                            torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                            optimizer.step(); scheduler.step(); optimizer.zero_grad()

            ep_t = time.time() - t0; avg = run_loss / max(1, n_steps)
            val_m = self.evaluate(dev_dataset, rare_ids)
            print(f"[KGv5] {epoch:02d}/{num_epochs} loss:{avg:.4f} "
                  f"mac:{val_m['macro_f1']:.4f} rare:{val_m['rare_f1']:.4f} "
                  f"t:{ep_t:.1f}s buf:{len(buffer)} ES:{es.counter}/{es.patience}")
            history.append({"epoch": epoch, "phase": "kg_v5",
                             "train_loss": avg, "val_macro_f1": val_m["macro_f1"],
                             "val_rare_f1": val_m["rare_f1"], "val_accuracy": val_m["accuracy"]})
            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1 = val_m["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ best={best_f1:.4f}")
            if es.step(val_m["macro_f1"]): print(f"⏹ ES epoch {epoch}"); break

        total_time = time.time() - t_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "kg_v5_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_v5_model.bin"))
            print(f"  ✔ Saved (best val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels: tick.set_color("red")
    ax.set_title(f"{split_name} Confusion Matrix (KG-RAG+Mixup v5)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_cm.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1(pcm, split_name, rare_labels=None):
    f1s = [pcm[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue" for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12); ax.set_xlabel("F1")
    ax.set_title(f"{split_name} Per-Class F1 (KG-RAG+Mixup v5)")
    ax.grid(True, alpha=0.3, axis="x"); plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def plot_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["ep"] = range(1, len(all_df)+1); b = len(base_df)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(all_df["ep"], all_df["train_loss"], marker="o", ms=3)
    axes[0].axvline(b, color="red", ls="--"); axes[0].set_title("Training Loss")
    axes[1].plot(all_df["ep"], all_df["val_macro_f1"], label="Macro-F1", marker="o", ms=3)
    axes[1].plot(all_df["ep"], all_df["val_rare_f1"],  label="Rare-F1",  marker="s", ms=3)
    axes[1].axvline(b, color="red", ls="--"); axes[1].legend(); axes[1].set_title("Val F1")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150); plt.close()


def print_table(dev_m, test_m, base_t=None, kg_t=None, n_params=None):
    rows = [("Accuracy","accuracy"),("Macro-F1","macro_f1"),("Micro-F1","micro_f1"),
            ("Weighted-F1","weighted_f1"),("Minority Macro-F1","rare_f1"),
            ("Macro-Precision","macro_precision"),("Macro-Recall","macro_recall")]
    print("\n" + "="*72)
    print("FINAL RESULTS — KG-RAG+Mixup v5 (all 9 enhancements)")
    print("="*72)
    if n_params: print(f"  Params     : {n_params:,}")
    if base_t:   print(f"  Phase A    : {base_t/60:.1f} min")
    if kg_t:     print(f"  Phase B    : {kg_t/60:.1f} min")
    print(f"  {'Metric':<30} {'Dev':>10} {'Test':>10}")
    print("-"*72)
    for lbl, key in rows:
        print(f"  {lbl:<30} {dev_m[key]:>10.4f} {test_m[key]:>10.4f}")
    print("="*72)
    print(f"\n  {'Label':<22} {'F1-Dev':>8} {'F1-Test':>8} {'Prec':>8} {'Rec':>8}")
    print("  "+"-"*56)
    for lbl in LABELS:
        dv=dev_m["per_class_metrics"][lbl]; ts=test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>8.4f} {ts['f1']:>8.4f} "
              f"{ts['precision']:>8.4f} {ts['recall']:>8.4f}")


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device: {DEVICE}")
    print("KG-RAG+Mixup v5 — 9 minority F1 enhancements")
    print("  FIX1: Mixup-CRF Bridge  FIX2: Online KG Refresh")
    print("  FIX3: Asymmetric Rare Anchor  FIX4: Span Discourse Mixup")
    print("  E5: R-Drop  E6: LDAM  E7: Dyn Rare Threshold")
    print("  E8: Prototype EMA Replay  E9: Post-hoc Calibration\n")

    print("Loading data ...")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"  Train:{len(train_docs)} Dev:{len(dev_docs)} Test:{len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    counts = Counter(lid for _, labs in train_docs for lid in labs)

    pd.DataFrame([{"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
                  for l in LABELS]).to_csv(os.path.join(OUT_DIR, "label_freqs.csv"), index=False)

    class_weights = compute_class_weights(label_freqs)
    focal_loss    = FocalLoss(class_weights, rare_ids).to(DEVICE)
    ldam_loss     = LDAMLoss(counts).to(DEVICE)

    print("Loading tokenizer ...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ── PHASE A ──────────────────────────────────────────
    print("\n" + "="*60 + "\nPHASE A: Base Training\n" + "="*60)
    base_model   = InLegalBERT_BiLSTM_MHA_CRF()
    base_trainer = BaseTrainer(base_model, focal_loss=focal_loss, device=DEVICE)
    base_hist, base_time = base_trainer.train(
        train_docs, train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE)

    # ── PHASE A→B ────────────────────────────────────────
    print("\n" + "="*60 + "\nPHASE A→B: Confusion + KG Build\n" + "="*60)
    confusion_pairs = compute_confusion_pairs(
        base_model, train_dataset, rare_ids, device=DEVICE)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        kg = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim, device=DEVICE)
        kg.rare_partition = set(rare_ids)
    else:
        kg = build_knowledge_graph(base_model, train_docs, tokenizer,
                                   rare_ids=rare_ids, confusion_pairs=confusion_pairs)

    lambda_boost = KGRetriever.calibrate_lambda(kg, rare_ids)
    retriever    = KGRetriever(kg, rare_ids=rare_ids, top_k=KG_TOP_K,
                               top_nodes=KG_TOP_NODES, hop=KG_HOP,
                               lambda_boost=lambda_boost)

    # ── PHASE B ──────────────────────────────────────────
    print("\n" + "="*60 + "\nPHASE B: KG+Mixup v5 Training\n" + "="*60)
    kg_v5_model = KGMixupV5Model(
        base_model=base_model, kg=kg, rare_ids=rare_ids,
        retriever=retriever, focal_loss=focal_loss, ldam_loss=ldam_loss,
        class_weights=class_weights.to(DEVICE), confusion_pairs=confusion_pairs)

    n_params, _ = count_parameters(kg_v5_model)
    trainer = KGMixupV5Trainer(kg_v5_model, kg, train_docs, tokenizer, device=DEVICE)
    kg_hist, kg_time = trainer.train(
        train_docs, train_dataset, dev_dataset,
        rare_ids=rare_ids, num_epochs=NUM_EPOCHS_KG)

    plot_history(base_hist, kg_hist)

    # ── POST-HOC CALIBRATION (Enhancement 9) ─────────────
    print("\nFitting per-class temperature calibration on dev set ...")
    kg_v5_model.eval()
    dev_logits_list, dev_labels_list = [], []
    with torch.no_grad():
        loader = DataLoader(dev_dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        for ids, attn, tt, labels, lengths in loader:
            ids=ids.to(DEVICE); attn=attn.to(DEVICE); tt=tt.to(DEVICE); lengths=lengths.to(DEVICE)
            _, em = kg_v5_model(ids, attn, tt, labels=None, lengths=lengths)
            B, T, C = em.shape
            for b in range(B):
                n = int(lengths[b].item())
                dev_logits_list.append(em[b, :n].cpu())
                dev_labels_list.append(labels[b, :n])
    dev_logits_all = torch.cat(dev_logits_list); dev_labels_all = torch.cat(dev_labels_list)
    calibrator = PerClassTemperatureCalibrator()
    calibrator.fit(dev_logits_all, dev_labels_all.clone())
    calibrator.to(DEVICE)

    # ── Enhancement 7: Dynamic rare threshold ────────────
    print("Calibrating dynamic rare threshold ...")
    taus = calibrate_rare_threshold(kg_v5_model, dev_dataset, rare_ids, device=DEVICE)

    # ── EVALUATION ───────────────────────────────────────
    print("\nEvaluating (standard) on Dev ...")
    dev_m = trainer.evaluate(dev_dataset, rare_ids, "dev", measure_time=True)
    print(f"  Dev: mac={dev_m['macro_f1']:.4f} rare={dev_m['rare_f1']:.4f}")

    print("Evaluating (dynamic threshold + calibration) on Dev ...")
    dev_trues, dev_preds_dyn = apply_dynamic_threshold_dataset(
        kg_v5_model, dev_dataset, rare_ids, taus,
        device=DEVICE, calibrator=calibrator)
    dev_m_dyn = compute_all_metrics(dev_trues, dev_preds_dyn, rare_ids, "dev_dyn")
    print(f"  Dev (dyn+cal): mac={dev_m_dyn['macro_f1']:.4f} rare={dev_m_dyn['rare_f1']:.4f}")

    # Use whichever is better for reports
    best_dev = dev_m_dyn if dev_m_dyn["macro_f1"] >= dev_m["macro_f1"] else dev_m

    print("Evaluating (dynamic threshold + calibration) on Test ...")
    test_trues, test_preds_dyn = apply_dynamic_threshold_dataset(
        kg_v5_model, test_dataset, rare_ids, taus,
        device=DEVICE, calibrator=calibrator)
    test_m = compute_all_metrics(test_trues, test_preds_dyn, rare_ids, "test")
    print(f"  Test: mac={test_m['macro_f1']:.4f} rare={test_m['rare_f1']:.4f}")

    for split_name, m in [("dev", best_dev), ("test", test_m)]:
        with open(os.path.join(OUT_DIR, f"{split_name}_cls_report.txt"), "w") as f:
            f.write(f"Model: KG-RAG+Mixup v5\nRare: {rare_labels}\n\n{m['cls_report']}")
        save_confusion_matrix(m["cm"], split_name, rare_labels)
        save_per_class_f1(m["per_class_metrics"], split_name, rare_labels)

    pd.DataFrame({"true": [id2label[x] for x in test_m["all_trues"]],
                  "pred": [id2label[x] for x in test_m["all_preds"]]
                  }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    skeys = ["macro_f1","micro_f1","weighted_f1","rare_f1",
             "macro_precision","macro_recall","accuracy"]
    summary = {
        "model": "KG-RAG+Mixup-v5",
        "v5_fixes": [
            "FIX1_MixupCRFBridgeLoss", "FIX2_OnlineKGRareNodeRefresh",
            "FIX3_AsymmetricRareAnchorMixup", "FIX4_SpanDiscoursesMixup"],
        "v5_enhancements": [
            "E5_RDrop", "E6_LDAM", "E7_DynamicRareThreshold",
            "E8_ProtoEMAReplay", "E9_PerClassTempCalibration"],
        "timing": {"phase_a_s": base_time, "phase_b_s": kg_time},
        "rare_classes": rare_labels,
        "dev":  {k: best_dev[k] for k in skeys},
        "test": {k: test_m[k]   for k in skeys},
        "per_class_test": test_m["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_table(best_dev, test_m, base_t=base_time, kg_t=kg_time, n_params=n_params)
    print(f"\n📁 All outputs → {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device: cuda:0
KG-RAG+Mixup v5 — 9 minority F1 enhancements
  FIX1: Mixup-CRF Bridge  FIX2: Online KG Refresh
  FIX3: Asymmetric Rare Anchor  FIX4: Span Discourse Mixup
  E5: R-Drop  E6: LDAM  E7: Dyn Rare Threshold
  E8: Prototype EMA Replay  E9: Post-hoc Calibration

Loading data ...
  Train:245 Dev:30 Test:50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167)
   FAC                  19.99%  ( 5744)
   RLC                   2.62%  (  752) ← RARE
   ISSUE                 1.28%  (  367) ← RARE
   ARG_PETITIONER        4.58%  ( 1315) ← RARE
   ARG_RESPONDENT        2.43%  (  698) ← RARE
   ANALYSIS             36.66%  (10537)
   STA                   1.67%  (  481) ← RARE
   PRE_RELIED            4.97%  ( 1427) ← RARE
   PRE_NOT_RELIED        0.55%  (  158) ← RARE
   RATIO                 2.30%  (  661) ← RARE
   RPC                   3.67%  ( 1055) ← RARE
   NONE                  4.79%  ( 1377) ← RARE

   Rare (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


❄️  Frozen: embeddings + layers 0-7. 🔥 Trainable: 8+

[Base] 001/60 loss:284.8021 mac:0.0391 rare:0.0000 t:39.2s ES:0/10
  ✔ best=0.0391
[Base] 002/60 loss:235.7814 mac:0.0807 rare:0.0000 t:47.4s ES:0/10
  ✔ best=0.0807
[Base] 003/60 loss:191.8574 mac:0.1583 rare:0.0020 t:42.1s ES:0/10
  ✔ best=0.1583
[Base] 004/60 loss:157.0491 mac:0.2694 rare:0.1068 t:46.2s ES:0/10
  ✔ best=0.2694
[Base] 005/60 loss:137.0456 mac:0.2828 rare:0.1202 t:44.7s ES:0/10
  ✔ best=0.2828
[Base] 006/60 loss:131.8192 mac:0.3012 rare:0.1430 t:46.8s ES:0/10
  ✔ best=0.3012
[Base] 007/60 loss:122.3539 mac:0.3224 rare:0.1769 t:48.5s ES:0/10
  ✔ best=0.3224
[Base] 008/60 loss:117.6282 mac:0.3360 rare:0.1807 t:44.8s ES:0/10
  ✔ best=0.3360
[Base] 009/60 loss:102.6589 mac:0.3521 rare:0.2014 t:47.2s ES:0/10
  ✔ best=0.3521
[Base] 010/60 loss:95.5312 mac:0.3622 rare:0.2207 t:45.9s ES:0/10
  ✔ best=0.3622
[Base] 011/60 loss:100.9246 mac:0.3641 rare:0.2171 t:45.1s ES:0/10
  ✔ best=0.3641
[Base] 012/60 loss:87.7926 mac:0.3

RuntimeError: cudnn RNN backward can only be called in training mode

In [ ]:
# inlegalbert_kg_rag_mixup_v5_resume.py
#
# RESUME VERSION — Picks up from Phase B (KG+Mixup training)
# Skips Phase A (base training already done) and KG build (already saved).
# Fixes:
#   BUG FIX — refresh_rare_nodes now restores model.train() after eval()
#              so cudnn RNN backward works correctly after KG refresh.
#   RESUME  — Loads base_model.bin + knowledge_graph.json, jumps to Phase B.

import os, json, random, time, math, copy
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
)
from sklearn.cluster import MiniBatchKMeans

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_mixup_v5_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 30
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

LABEL_SMOOTHING  = 0.05
AUX_CE_WEIGHT    = 0.25
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
WARMUP_RATIO     = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD   = 0.05

# ── KG-RAG ────────────────────────────────────────────────
KG_TOP_K        = 3
KG_TOP_NODES    = 6
KG_HOP          = 1
KG_FUSION_DIM   = 256

RST_INTRA_THRESH = 0.55
RST_CROSS_THRESH = 0.45

UNCERTAINTY_THRESH_MAJORITY = 0.70
UNCERTAINTY_THRESH_RARE     = 0.30

RARE_ALWAYS_KG           = True
KG_VIRTUAL_ALPHAS        = [0.25, 0.50, 0.75]
KG_PROTOTYPE_K           = 5
KG_RARE_LAMBDA           = 0.20
KG_RARE_GAMMA            = 2.0
KG_MAX_VIRTUAL_PER_LABEL = 200

FOCAL_GAMMA_RARE         = 2.5
FOCAL_GAMMA_MAJ          = 1.0
OVERSAMPLE_RARE_RATIO    = 3.0
HARD_BUFFER_SIZE         = 300
HARD_REPLAY_FREQ         = 4
HARD_REPLAY_LOSS_THRESH  = 0.8
HARD_REPLAY_WEIGHT       = 0.5
PROTO_CONTRAST_WEIGHT    = 0.08
PROTO_CONTRAST_MARGIN    = 0.35
CONFUSION_EDGE_WEIGHT    = 0.90
CONFUSION_TOP_K          = 3

KG_MAX_NODES_PER_SG      = 2000
KG_MAX_CROSS_EDGES       = 4000
KG_MAX_INTRA_EDGES_STORE = 500_000

# ── v4 Mixup ──────────────────────────────────────────────
MIXUP_ALPHA              = 0.4
MIXUP_LOSS_WEIGHT        = 0.35
MIXUP_MIN_RARE_IN_BATCH  = 2
KG_MIXUP_ENABLED         = True
KG_MIXUP_ALPHA           = 0.3
KG_MIXUP_WEIGHT          = 0.25
INTER_MIXUP_ENABLED      = True
INTER_MIXUP_ALPHA        = 0.15
INTER_MIXUP_WEIGHT       = 0.20
INTER_MIXUP_RARE_RATIO   = 0.85
INTRA_RARE_MIXUP_ENABLED = True
INTRA_RARE_MIXUP_WEIGHT  = 0.20
MIXUP_SOFT_TEMP          = 0.5

# ── v5 Fixes ──────────────────────────────────────────────
MIXUP_CRF_BRIDGE_WEIGHT  = 0.30
MIXUP_CRF_N_VIRTUAL      = 8

ONLINE_KG_REFRESH_EVERY  = 5
ONLINE_KG_REFRESH_DOCS   = 200

RARE_ANCHOR_LAM_MIN      = 0.85
RARE_ONLY_MIXUP_WEIGHT   = 0.25

SPAN_MIXUP_ENABLED       = True
SPAN_MIXUP_WEIGHT        = 0.20
SPAN_MIN_LEN             = 2
SPAN_MAX_LEN             = 5

RDROP_WEIGHT             = 0.15
RDROP_RARE_ONLY          = True

LDAM_WEIGHT              = 0.20
LDAM_MAX_MARGIN          = 0.5

DYN_RARE_THRESH_TAU      = 0.25
DYN_RARE_ENABLED         = True

PROTO_EMA_MOMENTUM       = 0.95
PROTO_EMA_REPLAY_EVERY   = 3
PROTO_EMA_REPLAY_WEIGHT  = 0.15

CALIBRATION_EPOCHS       = 200

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience = patience; self.min_delta = min_delta
        self.best_score = -1.0; self.counter = 0; self.stop = False

    def step(self, score):
        if score > self.best_score + self.min_delta:
            self.best_score = score; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# FOCAL LOSS
# ═══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, class_weights, rare_ids,
                 gamma_rare=FOCAL_GAMMA_RARE, gamma_maj=FOCAL_GAMMA_MAJ,
                 ignore_index=-100):
        super().__init__()
        self.ignore_index = ignore_index
        gamma_per_class = torch.full((NUM_LABELS,), gamma_maj)
        for r in set(rare_ids): gamma_per_class[r] = gamma_rare
        self.register_buffer("class_weights",   class_weights.float())
        self.register_buffer("gamma_per_class", gamma_per_class)

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if not valid.any(): return logits.sum() * 0.0
        lv = logits[valid]; tv = targets[valid]
        log_p = F.log_softmax(lv, dim=-1)
        p_t   = log_p.exp().gather(1, tv.unsqueeze(1)).squeeze(1)
        focal_w = (1.0 - p_t.detach()).pow(self.gamma_per_class[tv])
        ce = F.nll_loss(log_p, tv, reduction="none")
        return (focal_w * self.class_weights[tv] * ce).mean()


def compute_class_weights(label_freqs):
    weights = torch.ones(NUM_LABELS)
    freqs = [label_freqs.get(id2label[i], 1e-6) for i in range(NUM_LABELS)]
    inv = [1.0 / max(f, 1e-6) for f in freqs]
    inv_sum = sum(inv)
    for i, w in enumerate(inv):
        weights[i] = min(w / inv_sum * NUM_LABELS, 10.0)
    return weights


# ═══════════════════════════════════════════════════════════
# LDAM LOSS
# ═══════════════════════════════════════════════════════════
class LDAMLoss(nn.Module):
    def __init__(self, class_counts: dict, max_margin=LDAM_MAX_MARGIN,
                 ignore_index=-100):
        super().__init__()
        self.ignore_index = ignore_index
        counts = torch.tensor(
            [max(class_counts.get(id2label[i], 1), 1)
             for i in range(NUM_LABELS)], dtype=torch.float)
        m = 1.0 / (counts ** 0.25)
        m = m / m.max() * max_margin
        self.register_buffer("margins", m)

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if not valid.any(): return logits.sum() * 0.0
        lv = logits[valid]; tv = targets[valid]
        margins = self.margins.unsqueeze(0).expand_as(lv)
        one_hot = F.one_hot(tv, NUM_LABELS).float()
        lv_m = lv - margins * one_hot
        return F.cross_entropy(lv_m, tv)


# ═══════════════════════════════════════════════════════════
# R-DROP
# ═══════════════════════════════════════════════════════════
def rdrop_kl_loss(p1_logits, p2_logits, labels=None, rare_ids_tensor=None):
    if labels is not None and rare_ids_tensor is not None:
        flat_lab = labels.reshape(-1)
        flat_p1  = p1_logits.reshape(-1, NUM_LABELS)
        flat_p2  = p2_logits.reshape(-1, NUM_LABELS)
        valid = flat_lab >= 0
        is_rare = (flat_lab.unsqueeze(1) ==
                   rare_ids_tensor.unsqueeze(0)).any(1)
        mask = valid & is_rare
        if not mask.any(): return torch.tensor(0.0, device=p1_logits.device)
        flat_p1 = flat_p1[mask]; flat_p2 = flat_p2[mask]
    else:
        flat_p1 = p1_logits.reshape(-1, NUM_LABELS)
        flat_p2 = p2_logits.reshape(-1, NUM_LABELS)

    log_p1 = F.log_softmax(flat_p1, dim=-1)
    log_p2 = F.log_softmax(flat_p2, dim=-1)
    p1 = log_p1.exp(); p2 = log_p2.exp()
    kl_12 = F.kl_div(log_p2, p1, reduction="batchmean")
    kl_21 = F.kl_div(log_p1, p2, reduction="batchmean")
    return (kl_12 + kl_21) / 2.0


# ═══════════════════════════════════════════════════════════
# SOFT-LABEL LOSS
# ═══════════════════════════════════════════════════════════
class SoftLabelCrossEntropy(nn.Module):
    def __init__(self, temperature=MIXUP_SOFT_TEMP, class_weights=None):
        super().__init__()
        self.temperature = temperature
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, logits, soft_labels):
        log_p = F.log_softmax(logits / self.temperature, dim=-1)
        if self.class_weights is not None:
            dom_cls = soft_labels.argmax(dim=-1)
            w = self.class_weights[dom_cls]
            return (-(soft_labels * log_p).sum(dim=-1) * w).mean()
        return -(soft_labels * log_p).sum(dim=-1).mean()


# ═══════════════════════════════════════════════════════════
# GPU MANIFOLD MIXUP
# ═══════════════════════════════════════════════════════════
class GPUManifoldMixup(nn.Module):
    def __init__(self, num_labels=NUM_LABELS,
                 alpha=MIXUP_ALPHA, kg_alpha=KG_MIXUP_ALPHA,
                 inter_alpha=INTER_MIXUP_ALPHA,
                 rare_ratio=INTER_MIXUP_RARE_RATIO,
                 soft_temp=MIXUP_SOFT_TEMP):
        super().__init__()
        self.num_labels  = num_labels
        self.alpha       = alpha
        self.kg_alpha    = kg_alpha
        self.inter_alpha = inter_alpha
        self.rare_ratio  = rare_ratio
        self.soft_temp   = soft_temp

    @staticmethod
    def _beta_gpu(alpha, size, device):
        if alpha <= 0: return torch.ones(size, device=device)
        d = torch.distributions.Beta(
            torch.tensor(alpha, device=device),
            torch.tensor(alpha, device=device))
        return d.sample((size,))

    @staticmethod
    def _onehot(labels, num_classes):
        return F.one_hot(labels.clamp(min=0), num_classes).float()

    def intra_rare_mixup(self, embs, labels, rare_ids, force_rare_anchor=True):
        is_rare  = (labels.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        rare_idx = is_rare.nonzero(as_tuple=True)[0]
        if rare_idx.shape[0] < MIXUP_MIN_RARE_IN_BATCH:
            return None, None
        N_r  = rare_idx.shape[0]
        lam  = self._beta_gpu(self.alpha, N_r, embs.device)
        if force_rare_anchor:
            lam = lam.clamp(min=RARE_ANCHOR_LAM_MIN)
        else:
            lam = lam.clamp(0.1, 0.9)
        perm  = torch.randperm(N_r, device=embs.device)
        rare2 = rare_idx[perm]
        e1 = embs[rare_idx]; e2 = embs[rare2]
        l1 = labels[rare_idx]; l2 = labels[rare2]
        lam_e = lam.unsqueeze(1)
        mixed = lam_e * e1 + (1.0 - lam_e) * e2
        soft  = lam_e * self._onehot(l1, self.num_labels) + \
                (1.0 - lam_e) * self._onehot(l2, self.num_labels)
        return mixed, soft

    def kg_guided_mixup(self, embs, labels, nb_embs, nb_weights, rare_ids):
        T_prime = embs.shape[0]
        if T_prime < 1: return None, None
        is_rare = (labels.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        top1_idx  = nb_weights.argmax(dim=1)
        top1_embs = nb_embs[torch.arange(T_prime, device=embs.device), top1_idx]
        base_lam  = self._beta_gpu(self.kg_alpha, T_prime, embs.device)
        lam = torch.where(is_rare, base_lam.clamp(RARE_ANCHOR_LAM_MIN, 0.95),
                          base_lam.clamp(0.4, 0.7))
        lam_e = lam.unsqueeze(1)
        mixed = lam_e * embs + (1.0 - lam_e) * top1_embs
        oh = self._onehot(labels.clamp(min=0), self.num_labels)
        soft = lam_e * oh + (1.0 - lam_e) * (1.0 / self.num_labels)
        return mixed, soft

    def inter_class_mixup(self, embs, labels, rare_ids, confused_maj_ids):
        is_rare  = (labels.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        rare_idx = is_rare.nonzero(as_tuple=True)[0]
        if rare_idx.shape[0] < 2: return None, None
        is_hm = (labels.unsqueeze(1) == confused_maj_ids.unsqueeze(0)).any(1) & ~is_rare
        maj_idx = is_hm.nonzero(as_tuple=True)[0]
        if maj_idx.shape[0] < 1:
            maj_idx = (~is_rare).nonzero(as_tuple=True)[0]
        if maj_idx.shape[0] < 1: return None, None
        N_r  = rare_idx.shape[0]
        lam  = self._beta_gpu(self.inter_alpha, N_r, embs.device).clamp(0, self.inter_alpha * 2)
        perm = torch.randint(0, maj_idx.shape[0], (N_r,), device=embs.device)
        maj_part = maj_idx[perm]
        e1 = embs[rare_idx]; e2 = embs[maj_part]
        l1 = labels[rare_idx]; l2 = labels[maj_part]
        lam_e = lam.unsqueeze(1)
        mixed = (1.0 - lam_e) * e1 + lam_e * e2
        soft = self.rare_ratio * self._onehot(l1, self.num_labels) + \
               (1.0 - self.rare_ratio) * self._onehot(l2, self.num_labels)
        return mixed, soft

    def span_mixup(self, sent_vecs_b, labels_b, rare_ids,
                   span_min=SPAN_MIN_LEN, span_max=SPAN_MAX_LEN):
        n = sent_vecs_b.shape[0]
        if n < span_min * 2: return None, None
        is_rare = (labels_b.unsqueeze(1) == rare_ids.unsqueeze(0)).any(1)
        rare_spans = []
        for start in range(n - span_min + 1):
            end = min(start + span_max, n)
            if is_rare[start:end].any():
                rare_spans.append((start, end))
        if len(rare_spans) < 2: return None, None
        idx_a = random.randrange(len(rare_spans))
        idx_b = random.randrange(len(rare_spans))
        if idx_a == idx_b: return None, None
        sa, ea = rare_spans[idx_a]
        sb, eb = rare_spans[idx_b]
        use_len = min(ea - sa, eb - sb)
        lam = self._beta_gpu(MIXUP_ALPHA, 1, sent_vecs_b.device)[0]
        lam = lam.clamp(RARE_ANCHOR_LAM_MIN, 0.95)
        e1 = sent_vecs_b[sa:sa+use_len]
        e2 = sent_vecs_b[sb:sb+use_len]
        l1 = labels_b[sa:sa+use_len]
        l2 = labels_b[sb:sb+use_len]
        mixed = lam * e1 + (1.0 - lam) * e2
        soft  = lam * self._onehot(l1.clamp(min=0), self.num_labels) + \
                (1.0 - lam) * self._onehot(l2.clamp(min=0), self.num_labels)
        return mixed, soft


# ═══════════════════════════════════════════════════════════
# DATA
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    data = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s: data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs): continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    freqs   = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS) if freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        flag = " ← RARE" if lbl in rare_labels else ""
        print(f"   {lbl:<20} {freqs[lbl]*100:5.2f}%  ({counts.get(label2id[lbl],0):5d}){flag}")
    print(f"\n   Rare ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, freqs


class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs = docs; self.tokenizer = tokenizer; self.max_length = max_length

    def __len__(self): return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(sents, padding="max_length", truncation=True,
                             max_length=self.max_length, return_tensors="pt")
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
                "token_type_ids": enc.get("token_type_ids",
                                          torch.zeros_like(enc["input_ids"])),
                "labels": torch.tensor(labels, dtype=torch.long)}


def build_weighted_sampler(docs, rare_ids, oversample_ratio=OVERSAMPLE_RARE_RATIO):
    rare_set = set(rare_ids)
    weights  = [oversample_ratio if any(l in rare_set for l in labs) else 1.0
                for _, labs in docs]
    return WeightedRandomSampler(torch.tensor(weights, dtype=torch.float),
                                 num_samples=len(weights), replacement=True)


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L = batch[0]["input_ids"].shape[1]; B = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), -100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i,:t] = b["input_ids"]; attention_mask[i,:t] = b["attention_mask"]
        token_type_ids[i,:t] = b["token_type_ids"]; labels[i,:t] = b["labels"]
        lengths[i] = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim; self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn = attn.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        return self.out_proj(torch.matmul(attn, V).squeeze(2).reshape(N, H))


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    def __init__(self, bert_model_name=INLEGALBERT_MODEL_NAME,
                 sent_lstm_hidden=SENT_LSTM_HIDDEN, sent_lstm_layers=SENT_LSTM_LAYERS,
                 ctx_lstm_hidden=CTX_LSTM_HIDDEN,   ctx_lstm_layers=CTX_LSTM_LAYERS,
                 mha_heads=MHA_HEADS, mha_dropout=MHA_DROPOUT,
                 num_labels=NUM_LABELS, dropout=DROPOUT, freeze_layers=BERT_FREEZE_LAYERS):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)
        self.sent_bilstm = nn.LSTM(self.bert_dim, sent_lstm_hidden, sent_lstm_layers,
                                   bidirectional=True, batch_first=True,
                                   dropout=dropout if sent_lstm_layers > 1 else 0.0)
        self.sent_out_dim = sent_lstm_hidden * 2
        self.mha_pooling  = MultiHeadAttentionPooling(self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)
        self.ctx_bilstm = nn.LSTM(self.sent_out_dim, ctx_lstm_hidden, ctx_lstm_layers,
                                  bidirectional=True, batch_first=True,
                                  dropout=dropout if ctx_lstm_layers > 1 else 0.0)
        self.ctx_out_dim = ctx_lstm_hidden * 2
        self.classifier  = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(self.ctx_out_dim // 2, num_labels))
        self.crf     = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

    def _freeze_bert_layers(self, n):
        for p in self.bert.embeddings.parameters(): p.requires_grad = False
        for i in range(min(n, len(self.bert.encoder.layer))):
            for p in self.bert.encoder.layer[i].parameters(): p.requires_grad = False
        print(f"❄️  Frozen: embeddings + layers 0-{n-1}. 🔥 Trainable: {n}+\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids):
        B, T, L = input_ids.shape; N = B * T
        fi = input_ids.view(N, L); fm = attention_mask.view(N, L)
        ft = token_type_ids.view(N, L)
        valid = fm.sum(-1) > 0
        embs  = fi.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(fi[valid], fm[valid], ft[valid])
            embs[valid] = out.last_hidden_state.to(embs.dtype)
        embs = self.dropout(embs)
        lo, _ = self.sent_bilstm(embs); lo = self.dropout(lo)
        pm = (fm == 0).clone(); pm[~valid] = False
        sv = self.sent_layer_norm(self.mha_pooling(lo, key_padding_mask=pm))
        sv = sv * valid.unsqueeze(-1).to(sv.dtype)
        return sv.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids, lengths=None):
        sv = self.encode_sentences(input_ids, attention_mask, token_type_ids)
        sd = self.dropout(sv)
        if lengths is not None:
            pk = nn.utils.rnn.pack_padded_sequence(sd, lengths.cpu(),
                                                   batch_first=True, enforce_sorted=False)
            po, _ = self.ctx_bilstm(pk)
            co, _ = nn.utils.rnn.pad_packed_sequence(po, batch_first=True)
        else:
            co, _ = self.ctx_bilstm(sd)
        co = self.dropout(co)
        em = torch.nan_to_num(self.classifier(co), nan=0.0, posinf=1e4, neginf=-1e4)
        return sv, co, em

    def _mask(self, em, labels, lengths):
        if lengths is not None:
            B, T, _ = em.shape
            m = torch.zeros(B, T, dtype=torch.bool, device=em.device)
            for i, l in enumerate(lengths): m[i, :l] = True
        elif labels is not None: m = (labels != -100)
        else: m = torch.ones(em.shape[:2], dtype=torch.bool, device=em.device)
        return m

    def forward(self, ids, attn, ttype, labels=None, lengths=None):
        _, _, em = self.get_emissions(ids, attn, ttype, lengths=lengths)
        mask = self._mask(em, labels, lengths)
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_l = -self.crf(em, safe, mask=mask, reduction="mean")
            B2, T2, C = em.shape
            ce_l = self.ce_loss(em.reshape(B2*T2, C), labels.reshape(B2*T2))
            return crf_l + AUX_CE_WEIGHT * ce_l, em
        return self.crf.decode(em, mask=mask), em


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    def __init__(self, emb_dim=KG_FUSION_DIM, device=DEVICE):
        self.emb_dim = emb_dim; self.device = device
        self.rare_partition = set()
        self._cpu_nodes = defaultdict(list)
        self._stacked = {}; self._norm = {}; self._is_rare = {}
        self._edge_idx = {}; self._edge_w = {}
        self._proto = {}; self._proto_norm = {}
        self._cx_sl = self._cx_si = self._cx_dl = self._cx_di = self._cx_w = None

    def add_nodes(self, embeddings, label_ids):
        for emb, lid in zip(embeddings.detach().cpu(), label_ids):
            self._cpu_nodes[lid].append((emb, False, False))

    def _inject_virtual(self, lid):
        real = [n[0] for n in self._cpu_nodes[lid] if not n[1] and not n[2]]
        n = len(real)
        if n < 2: return 0
        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        random.shuffle(pairs)
        added = 0
        for i, j in pairs:
            for alpha in KG_VIRTUAL_ALPHAS:
                if added >= KG_MAX_VIRTUAL_PER_LABEL: break
                v = F.normalize((alpha*real[i] + (1-alpha)*real[j]).unsqueeze(0), dim=-1).squeeze(0)
                self._cpu_nodes[lid].append((v, True, False)); added += 1
            if added >= KG_MAX_VIRTUAL_PER_LABEL: break
        return added

    def _build_prototypes(self, lid):
        all_e = torch.stack([n[0] for n in self._cpu_nodes[lid]])
        k = min(KG_PROTOTYPE_K, all_e.shape[0])
        if k < 2: c = F.normalize(all_e.mean(0, keepdim=True), dim=-1)
        else:
            km = MiniBatchKMeans(n_clusters=k, n_init=5,
                                 batch_size=min(1024, all_e.shape[0]), random_state=SEED)
            km.fit(all_e.numpy())
            c = F.normalize(torch.tensor(km.cluster_centers_, dtype=torch.float32), dim=-1)
        for ci in range(c.shape[0]):
            self._cpu_nodes[lid].insert(0, (c[ci], False, True))
        return c

    def _promote_to_gpu(self):
        for lid, nodes in self._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in nodes]).to(self.device)
            self._stacked[lid] = embs
            self._norm[lid]    = F.normalize(embs, dim=-1)
            self._is_rare[lid] = torch.tensor([n[1] or n[2] for n in nodes],
                                               dtype=torch.bool, device=self.device)

    def build_edges(self, rare_ids, confusion_pairs=None,
                    intra_thresh=RST_INTRA_THRESH, cross_thresh=RST_CROSS_THRESH,
                    max_intra_per_node=5, max_cross=KG_MAX_CROSS_EDGES):
        print("  Building KG edges ...")
        self.rare_partition = set(rare_ids)
        for lid in rare_ids:
            if self._cpu_nodes[lid]:
                self._inject_virtual(lid); self._build_prototypes(lid)
        self._promote_to_gpu()
        for lid in self._stacked:
            norms = self._norm[lid]; N = norms.shape[0]
            if N < 2:
                self._edge_idx[lid] = torch.zeros(2,0,dtype=torch.long,device=self.device)
                self._edge_w[lid]   = torch.zeros(0, device=self.device); continue
            if lid in self.rare_partition:
                chunk = 512; ei_l, ej_l, ew_l = [], [], []
                for start in range(0, N, chunk):
                    end = min(start+chunk, N)
                    blk = torch.mm(norms[start:end], norms.T)
                    r, c_ = torch.where(
                        (blk > 0) &
                        (torch.arange(start,end,device=self.device).unsqueeze(1) <
                         torch.arange(N, device=self.device).unsqueeze(0)))
                    ei_l.append(r+start); ej_l.append(c_); ew_l.append(blk[r,c_])
                    if sum(x.shape[0] for x in ei_l) >= KG_MAX_INTRA_EDGES_STORE: break
                if ei_l:
                    ei=torch.cat(ei_l); ej=torch.cat(ej_l); ew=torch.cat(ew_l)
                    if ei.shape[0] > KG_MAX_INTRA_EDGES_STORE:
                        p=torch.randperm(ei.shape[0],device=self.device)[:KG_MAX_INTRA_EDGES_STORE]
                        ei,ej,ew=ei[p],ej[p],ew[p]
                    src=torch.cat([ei,ej]); dst=torch.cat([ej,ei]); ww=torch.cat([ew,ew])
                else:
                    src=dst=torch.zeros(0,dtype=torch.long,device=self.device)
                    ww=torch.zeros(0,device=self.device)
            else:
                idx=torch.arange(N,device=self.device); ci=idx[:-1]; cj=idx[1:]
                cw=(norms[ci]*norms[cj]).sum(-1).clamp(min=0)
                sim=torch.mm(norms,norms.T); sim.fill_diagonal_(-2.0)
                sim[ci,cj]=-2.0; sim[cj,ci]=-2.0
                hi_r,hi_c=torch.where(sim>=intra_thresh)
                keep=hi_r<hi_c; hi_r,hi_c=hi_r[keep],hi_c[keep]; hi_w=sim[hi_r,hi_c]
                if hi_r.shape[0] > N*max_intra_per_node:
                    p=torch.randperm(hi_r.shape[0],device=self.device)[:N*max_intra_per_node]
                    hi_r,hi_c,hi_w=hi_r[p],hi_c[p],hi_w[p]
                ei=torch.cat([ci,hi_r]); ej=torch.cat([cj,hi_c]); ew=torch.cat([cw,hi_w])
                src=torch.cat([ei,ej]); dst=torch.cat([ej,ei]); ww=torch.cat([ew,ew])
            self._edge_idx[lid]=torch.stack([src,dst],0); self._edge_w[lid]=ww
        for lid in rare_ids:
            proto=[n for n in self._cpu_nodes.get(lid,[]) if n[2]]
            if proto:
                pc=F.normalize(torch.stack([n[0] for n in proto]).to(self.device),dim=-1)
                self._proto[lid]=self._proto_norm[lid]=pc
        cx_sl,cx_si,cx_dl,cx_di,cx_w=[],[],[],[],[]
        label_ids=sorted(self._stacked.keys()); total_cross=0
        for a in range(len(label_ids)):
            if total_cross>=max_cross: break
            for b in range(a+1,len(label_ids)):
                if total_cross>=max_cross: break
                la,lb=label_ids[a],label_ids[b]
                na=self._norm[la][:KG_MAX_NODES_PER_SG]; nb=self._norm[lb][:KG_MAX_NODES_PER_SG]
                sim=torch.mm(na,nb.T)
                rows,cols=torch.where(sim>=cross_thresh)
                rows,cols=rows[:50],cols[:50]
                if rows.shape[0]==0: continue
                w=sim[rows,cols]
                cx_sl.append(torch.full((rows.shape[0],),la,dtype=torch.long,device=self.device))
                cx_si.append(rows); cx_dl.append(torch.full((rows.shape[0],),lb,dtype=torch.long,device=self.device))
                cx_di.append(cols); cx_w.append(w); total_cross+=rows.shape[0]
        if confusion_pairs:
            for rl, clist in confusion_pairs.items():
                if rl not in self._stacked: continue
                for clid in clist[:CONFUSION_TOP_K]:
                    if clid not in self._stacked: continue
                    rn=self._norm[rl]; cn=self._norm[clid][:KG_MAX_NODES_PER_SG]
                    sim=torch.mm(rn,cn.T)
                    rows,cols=torch.where(sim>0.3); rows,cols=rows[:30],cols[:30]
                    if rows.shape[0]==0: continue
                    w=torch.full((rows.shape[0],),CONFUSION_EDGE_WEIGHT,device=self.device)
                    cx_sl.append(torch.full((rows.shape[0],),rl,dtype=torch.long,device=self.device))
                    cx_si.append(rows); cx_dl.append(torch.full((rows.shape[0],),clid,dtype=torch.long,device=self.device))
                    cx_di.append(cols); cx_w.append(w)
        if cx_sl:
            self._cx_sl=torch.cat(cx_sl); self._cx_si=torch.cat(cx_si)
            self._cx_dl=torch.cat(cx_dl); self._cx_di=torch.cat(cx_di)
            self._cx_w=torch.cat(cx_w)
        else:
            z=torch.zeros(0,dtype=torch.long,device=self.device)
            self._cx_sl=self._cx_si=self._cx_dl=self._cx_di=z
            self._cx_w=torch.zeros(0,device=self.device)
        print(f"  KG ready: cross={self._cx_w.shape[0]}")

    # ══════════════════════════════════════════════════════
    # BUG FIX: restore training mode after eval() in refresh
    # ══════════════════════════════════════════════════════
    @torch.no_grad()
    def refresh_rare_nodes(self, base_model, train_docs, tokenizer,
                           rare_ids, max_docs=ONLINE_KG_REFRESH_DOCS):
        """Re-encode rare docs with current encoder, replace _stacked/_norm.
        FIXED: restores base_model.train() after this no_grad eval pass."""
        was_training = base_model.training   # ← remember original mode
        base_model.eval()

        rare_set = set(rare_ids)
        rare_docs = [(sents, labs) for sents, labs in train_docs
                     if any(l in rare_set for l in labs)]
        random.shuffle(rare_docs)
        rare_docs = rare_docs[:max_docs]
        if not rare_docs:
            if was_training:
                base_model.train()           # ← restore before returning
            return

        dummy   = RRCDataset(rare_docs, tokenizer)
        loader  = DataLoader(dummy, batch_size=4, shuffle=False,
                             collate_fn=collate_rrc, num_workers=0)
        new_nodes = defaultdict(list)
        for ids, attn, tt, labels, lengths in loader:
            ids=ids.to(self.device); attn=attn.to(self.device); tt=tt.to(self.device)
            for bi in range(ids.shape[0]):
                sv = base_model.encode_sentences(
                    ids[bi:bi+1], attn[bi:bi+1], tt[bi:bi+1]).squeeze(0)
                n  = int(lengths[bi].item())
                for emb, lid in zip(sv[:n].cpu(), labels[bi, :n].tolist()):
                    if lid in rare_set:
                        new_nodes[lid].append(emb)

        for lid, embs_list in new_nodes.items():
            if not embs_list: continue
            embs = torch.stack(embs_list).to(self.device)
            self._stacked[lid] = embs
            self._norm[lid]    = F.normalize(embs, dim=-1)

        print(f"  KG rare nodes refreshed: "
              f"{sum(len(v) for v in new_nodes.values())} new embeddings")

        # ← KEY FIX: always restore training state
        if was_training:
            base_model.train()

    def save(self, path):
        data = {
            "rare_partition": list(self.rare_partition),
            "nodes": {str(lid): [{"emb": n[0].tolist(), "is_virtual": n[1], "is_proto": n[2]}
                                  for n in nl] for lid, nl in self._cpu_nodes.items()},
            "intra": {str(lid): {"idx": self._edge_idx[lid].cpu().tolist(),
                                  "w": self._edge_w[lid].cpu().tolist()}
                      for lid in self._edge_idx},
            "cross": {"sl": self._cx_sl.cpu().tolist() if self._cx_sl is not None else [],
                      "si": self._cx_si.cpu().tolist() if self._cx_si is not None else [],
                      "dl": self._cx_dl.cpu().tolist() if self._cx_dl is not None else [],
                      "di": self._cx_di.cpu().tolist() if self._cx_di is not None else [],
                      "w":  self._cx_w.cpu().tolist()  if self._cx_w  is not None else []}
        }
        with open(path, "w") as f: json.dump(data, f)
        print(f"  KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM, device=DEVICE):
        kg = cls(emb_dim=emb_dim, device=device)
        with open(path) as f: data = json.load(f)
        kg.rare_partition = set(data.get("rare_partition", []))
        for k, nl in data["nodes"].items():
            lid = int(k)
            for n in nl:
                kg._cpu_nodes[lid].append(
                    (torch.tensor(n["emb"], dtype=torch.float32), n["is_virtual"], n["is_proto"]))
        kg._promote_to_gpu()
        for k, v in data.get("intra", {}).items():
            lid = int(k)
            kg._edge_idx[lid] = torch.tensor(v["idx"], dtype=torch.long,    device=device)
            kg._edge_w[lid]   = torch.tensor(v["w"],   dtype=torch.float32, device=device)
        cross = data.get("cross", {})
        if cross and cross.get("sl"):
            kg._cx_sl = torch.tensor(cross["sl"], dtype=torch.long,    device=device)
            kg._cx_si = torch.tensor(cross["si"], dtype=torch.long,    device=device)
            kg._cx_dl = torch.tensor(cross["dl"], dtype=torch.long,    device=device)
            kg._cx_di = torch.tensor(cross["di"], dtype=torch.long,    device=device)
            kg._cx_w  = torch.tensor(cross["w"],  dtype=torch.float32, device=device)
        else:
            z = torch.zeros(0, dtype=torch.long, device=device)
            kg._cx_sl=kg._cx_si=kg._cx_dl=kg._cx_di=z
            kg._cx_w=torch.zeros(0, device=device)
        for lid in kg.rare_partition:
            proto = [n for n in kg._cpu_nodes.get(lid,[]) if n[2]]
            if proto:
                pc = F.normalize(torch.stack([n[0] for n in proto]).to(device), dim=-1)
                kg._proto[lid] = kg._proto_norm[lid] = pc
        print(f"  KG loaded ← {path}"); return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS,
                 thresh_maj=UNCERTAINTY_THRESH_MAJORITY,
                 thresh_rare=UNCERTAINTY_THRESH_RARE):
        self.log_C = math.log(num_classes)
        self.thresh_maj = thresh_maj; self.thresh_rare = thresh_rare

    def entropy(self, logits):
        p = F.softmax(logits, dim=-1)
        return -(p * (p+1e-9).log()).sum(-1) / self.log_C

    def is_uncertain(self, logits, is_rare_pred):
        H = self.entropy(logits)
        thresh = torch.where(is_rare_pred,
                             torch.full_like(H, self.thresh_rare),
                             torch.full_like(H, self.thresh_maj))
        return H > thresh

    def top_label(self, logits): return logits.argmax(-1)


# ═══════════════════════════════════════════════════════════
# KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    def __init__(self, kg, rare_ids, top_k=KG_TOP_K, top_nodes=KG_TOP_NODES,
                 hop=KG_HOP, lambda_boost=KG_RARE_LAMBDA):
        self.kg = kg; self.rare_ids = set(rare_ids)
        self.top_k = top_k; self.top_nodes = top_nodes
        self.hop = hop; self.lambda_boost = lambda_boost
        self.device = kg.device; self.label_ids = sorted(kg._stacked.keys())

    @classmethod
    def calibrate_lambda(cls, kg, rare_ids, sample_limit=200):
        rare_set = set(rare_ids)
        maj_ids  = [l for l in kg._stacked if l not in rare_set]
        if not maj_ids: return KG_RARE_LAMBDA
        maj_norms = torch.cat([kg._norm[m][:KG_MAX_NODES_PER_SG] for m in maj_ids if m in kg._norm])
        gaps = []
        for lid in rare_ids:
            if lid not in kg._norm: continue
            own = kg._norm[lid]; N = own.shape[0]
            idx = torch.randperm(N, device=kg.device)[:sample_limit]
            s   = own[idx]
            gaps.append((torch.mm(s, maj_norms.T).max(1).values -
                         torch.mm(s, own.T).max(1).values).cpu())
        if not gaps: return KG_RARE_LAMBDA
        lam = float(np.clip(torch.cat(gaps).median().item(), 0.05, 0.40))
        print(f"  λ → {lam:.4f}"); return lam

    def retrieve_batch(self, H_q, trigger_mask):
        if not trigger_mask.any(): return None, None, None
        trig_idx = trigger_mask.nonzero(as_tuple=True)[0]
        H_trig   = H_q[trig_idx]; T_prime, D = H_trig.shape
        sg_sims = []
        for lid in self.label_ids:
            n = self.kg._norm[lid]
            if n.shape[0] == 0:
                sg_sims.append(torch.full((T_prime,), -1.0, device=self.device)); continue
            cap = min(n.shape[0], KG_MAX_NODES_PER_SG)
            s   = torch.mm(H_trig, n[:cap].T).max(1).values
            sg_sims.append(s + (self.lambda_boost if lid in self.rare_ids else 0.0))
        sg_mat = torch.stack(sg_sims, dim=1)
        _, topk = sg_mat.topk(min(self.top_k, len(self.label_ids)), dim=1)
        cap_out = self.top_k * (self.top_nodes + 8)
        nb_embs    = torch.zeros(T_prime, cap_out, D, device=self.device)
        nb_weights = torch.zeros(T_prime, cap_out,    device=self.device)
        nb_is_rare = torch.zeros(T_prime, cap_out, dtype=torch.bool, device=self.device)
        fill = torch.zeros(T_prime, dtype=torch.long, device=self.device)
        for sg_idx in topk.unique().tolist():
            lid = self.label_ids[sg_idx]; is_rare_sg = lid in self.rare_ids
            norms = self.kg._norm[lid]; embs = self.kg._stacked[lid]
            N_sg = norms.shape[0]
            if N_sg == 0: continue
            uses  = (topk == sg_idx).any(dim=1)
            q_idx = uses.nonzero(as_tuple=True)[0]; H_sub = H_trig[q_idx]
            cap_n = min(N_sg, KG_MAX_NODES_PER_SG); cap_e = embs[:cap_n]; cap_nr = norms[:cap_n]
            sim_mat = torch.mm(H_sub, cap_nr.T); k_q = min(self.top_nodes, cap_n)
            topn = sim_mat.topk(k_q, dim=1); topn_idx = topn.indices; topn_sims = topn.values
            edge = self.kg._edge_idx.get(lid)
            for qi_local, qi_global in enumerate(q_idx.tolist()):
                seed = topn_idx[qi_local]
                if self.hop >= 1 and edge is not None and edge.shape[1] > 0:
                    src, dst = edge[0], edge[1]
                    hop_n = dst[torch.isin(src, seed)].unique()
                    if hop_n.shape[0] > 0:
                        seed = torch.cat([seed, hop_n[hop_n < cap_n]]).unique()
                f = int(fill[qi_global].item()); n_add = min(k_q, cap_out - f)
                if n_add <= 0: continue
                nb_embs[qi_global,f:f+n_add]    = cap_e[topn_idx[qi_local,:n_add]]
                nb_weights[qi_global,f:f+n_add] = topn_sims[qi_local,:n_add].clamp(min=0)
                nb_is_rare[qi_global,f:f+n_add] = is_rare_sg
                fill[qi_global] = f + n_add
        cx_sl = self.kg._cx_sl
        if cx_sl is not None and cx_sl.shape[0] > 0:
            for qi_global in range(T_prime):
                f = int(fill[qi_global].item())
                if f >= cap_out: continue
                for ul_idx in topk[qi_global].unique().tolist():
                    lid = self.label_ids[ul_idx]
                    mask_l = (cx_sl == lid)
                    if not mask_l.any(): continue
                    c_di=self.kg._cx_di[mask_l]; c_dl=self.kg._cx_dl[mask_l]; c_w=self.kg._cx_w[mask_l]
                    for dlid_t in c_dl.unique().tolist():
                        dlid = int(dlid_t); dstk = self.kg._stacked.get(dlid)
                        if dstk is None: continue
                        mask_d = (c_dl == dlid_t)
                        di_v = c_di[mask_d][:3]; w_v = c_w[mask_d][:3]
                        di_v = di_v[di_v < dstk.shape[0]]
                        n_a  = min(di_v.shape[0], cap_out - f)
                        if n_a <= 0: continue
                        nb_embs[qi_global,f:f+n_a]    = dstk[di_v[:n_a]]
                        nb_weights[qi_global,f:f+n_a] = w_v[:n_a]
                        nb_is_rare[qi_global,f:f+n_a] = (dlid in self.rare_ids)
                        f += n_a
                fill[qi_global] = f
        return nb_embs, nb_weights, nb_is_rare


# ═══════════════════════════════════════════════════════════
# GATED GAT FUSION
# ═══════════════════════════════════════════════════════════
class GatedGraphAttentionFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT, r_boost=KG_RARE_GAMMA):
        super().__init__()
        self.r_boost = r_boost
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.gate    = nn.Linear(emb_dim * 2, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, H_q, nb_embs, nb_w, nb_rare):
        pad_mask = (nb_w == 0)
        q = self.proj_q(H_q); k = self.proj_k(nb_embs)
        dot = torch.bmm(k, q.unsqueeze(-1)).squeeze(-1) * self.scale
        r_b = torch.where(nb_rare, torch.full_like(dot, self.r_boost), torch.ones_like(dot))
        raw = (dot * nb_w * r_b).masked_fill(pad_mask, -1e9)
        alpha = F.softmax(raw, dim=-1).masked_fill(pad_mask, 0.0)
        v_attn = torch.bmm(self.dropout(alpha).unsqueeze(1), nb_embs).squeeze(1)
        g = torch.sigmoid(self.gate(torch.cat([H_q, v_attn], dim=-1)))
        return (1.0 - g) * H_q + g * v_attn


# ═══════════════════════════════════════════════════════════
# HARD EXAMPLE BUFFER
# ═══════════════════════════════════════════════════════════
class HardExampleBuffer:
    def __init__(self, max_size=HARD_BUFFER_SIZE):
        self.max_size = max_size; self.buf = []

    def update(self, loss_val, batch_cpu):
        self.buf.append((loss_val, batch_cpu))
        if len(self.buf) > self.max_size:
            self.buf.sort(key=lambda x: -x[0]); self.buf = self.buf[:self.max_size // 2]

    def sample(self):
        if not self.buf: return None
        losses = torch.tensor([x[0] for x in self.buf], dtype=torch.float)
        return self.buf[int(torch.multinomial(F.softmax(losses, 0), 1).item())][1]

    def __len__(self): return len(self.buf)


# ═══════════════════════════════════════════════════════════
# RARE PROTOTYPE EMA STORE
# ═══════════════════════════════════════════════════════════
class RarePrototypeEMAStore:
    def __init__(self, rare_ids, emb_dim, device, momentum=PROTO_EMA_MOMENTUM):
        self.rare_ids = set(rare_ids); self.device = device
        self.momentum = momentum
        self.protos = {lid: torch.zeros(emb_dim, device=device) for lid in rare_ids}
        self.counts = {lid: 0 for lid in rare_ids}

    @torch.no_grad()
    def update(self, sent_vecs_flat, labels_flat):
        for lid in self.rare_ids:
            mask = (labels_flat == lid)
            if not mask.any(): continue
            mean_emb = sent_vecs_flat[mask].mean(0)
            if self.counts[lid] == 0:
                self.protos[lid] = mean_emb.detach()
            else:
                self.protos[lid] = (self.momentum * self.protos[lid] +
                                    (1 - self.momentum) * mean_emb.detach())
            self.counts[lid] += 1

    def get_proto_tensor(self):
        valid = [(lid, self.protos[lid]) for lid in self.rare_ids if self.counts[lid] > 0]
        if not valid: return None, None
        lids, protos = zip(*valid)
        return torch.stack(protos), torch.tensor(lids, dtype=torch.long, device=self.device)


# ═══════════════════════════════════════════════════════════
# POST-HOC TEMPERATURE CALIBRATION
# ═══════════════════════════════════════════════════════════
class PerClassTemperatureCalibrator(nn.Module):
    def __init__(self, num_classes=NUM_LABELS):
        super().__init__()
        self.temps = nn.Parameter(torch.ones(num_classes))

    def forward(self, logits):
        return logits / self.temps.clamp(min=0.05).unsqueeze(0)

    def fit(self, logits_all, labels_all, epochs=CALIBRATION_EPOCHS, device=DEVICE):
        self.to(device)
        opt = torch.optim.LBFGS(self.parameters(), lr=0.01, max_iter=50)
        logits_t = logits_all.to(device); labels_t = labels_all.to(device)
        nll = nn.CrossEntropyLoss(ignore_index=-100)
        def closure():
            opt.zero_grad()
            scaled = self.forward(logits_t)
            loss = nll(scaled, labels_t)
            loss.backward(); return loss
        for _ in range(epochs // 50):
            opt.step(closure)
        print(f"  Calibration done. Temp range: [{self.temps.min().item():.3f}, "
              f"{self.temps.max().item():.3f}]")


# ═══════════════════════════════════════════════════════════
# FULL KG + MIXUP v5 MODEL
# ═══════════════════════════════════════════════════════════
class KGMixupV5Model(nn.Module):
    def __init__(self, base_model, kg, rare_ids, retriever=None,
                 focal_loss=None, ldam_loss=None,
                 class_weights=None, confusion_pairs=None):
        super().__init__()
        self.base = base_model; self.kg = kg
        self.rare_ids = set(rare_ids); self.rare_list = sorted(rare_ids)
        self.retriever   = retriever or KGRetriever(kg, rare_ids)
        self.uncertainty = UncertaintyEstimator()
        self.focal_loss  = focal_loss
        self.ldam_loss   = ldam_loss
        sent_dim = base_model.sent_out_dim
        ctx_dim  = base_model.ctx_out_dim
        self.gat_fusion = GatedGraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(ctx_dim // 2, NUM_LABELS))
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, ignore_index=-100)
        self.gpu_mixup  = GPUManifoldMixup()
        self.mixup_proj = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, NUM_LABELS))
        self.soft_ce = SoftLabelCrossEntropy(temperature=MIXUP_SOFT_TEMP,
                                             class_weights=class_weights)
        self.mixup_crf_proj = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, ctx_dim))
        self.mixup_crf_classifier = nn.Linear(ctx_dim, NUM_LABELS)
        self.mixup_bridge_crf     = CRF(num_tags=NUM_LABELS, batch_first=True)
        rare_t = torch.tensor(self.rare_list, dtype=torch.long)
        self.register_buffer("_rare_t", rare_t)
        if confusion_pairs:
            conf_maj = list({cid for cids in confusion_pairs.values() for cid in cids})
        else:
            conf_maj = list(range(NUM_LABELS))
        self.register_buffer("_conf_maj_t", torch.tensor(conf_maj, dtype=torch.long))

    def _rare_mask(self, top_labels):
        return (top_labels.unsqueeze(-1) == self._rare_t.view(1,1,-1)).any(-1)

    def _proto_contrast_loss(self, device):
        rp = [self.kg._proto_norm[l] for l in self.rare_list if l in self.kg._proto_norm]
        if not rp: return torch.tensor(0.0, device=device)
        mp = [self.kg._proto_norm[l] for l in self.kg._stacked
              if l not in self.rare_ids and l in self.kg._proto_norm]
        if not mp: return torch.tensor(0.0, device=device)
        rp = F.normalize(torch.cat(rp).to(device), dim=-1)
        mp = F.normalize(torch.cat(mp).to(device), dim=-1)
        return torch.clamp(torch.mm(rp, mp.T) - PROTO_CONTRAST_MARGIN, min=0.0).mean()

    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, D = sent_vecs.shape; fused = sent_vecs.clone()
        top_labels = self.uncertainty.top_label(emissions)
        is_rare_p  = self._rare_mask(top_labels)
        trigger    = self.uncertainty.is_uncertain(emissions, is_rare_p) | \
                     (RARE_ALWAYS_KG & is_rare_p)
        for b in range(B):
            n = int(lengths[b].item()); trig_b = trigger[b, :n]
            if not trig_b.any(): continue
            H_b = sent_vecs[b, :n]
            nb_embs, nb_w, nb_rare = self.retriever.retrieve_batch(
                F.normalize(H_b.detach(), dim=-1), trig_b)
            if nb_embs is None: continue
            trig_idx = trig_b.nonzero(as_tuple=True)[0]
            fused[b, trig_idx] = self.gat_fusion(H_b[trig_idx], nb_embs, nb_w, nb_rare)
        return fused

    def _mixup_crf_bridge_loss(self, mixed_embs, soft_labels, device):
        if mixed_embs is None or mixed_embs.shape[0] < 1:
            return torch.tensor(0.0, device=device)
        bridge_emis = self.mixup_crf_classifier(self.mixup_crf_proj(mixed_embs))
        bridge_emis = torch.nan_to_num(bridge_emis, nan=0.0)
        pseudo = soft_labels.argmax(dim=-1)
        em3  = bridge_emis.unsqueeze(0)
        tg3  = pseudo.unsqueeze(0)
        mask3 = torch.ones(1, bridge_emis.shape[0], dtype=torch.bool, device=device)
        crf_l = -self.mixup_bridge_crf(em3, tg3, mask=mask3, reduction="mean")
        soft_l = self.soft_ce(bridge_emis, soft_labels.to(device))
        return crf_l + 0.5 * soft_l

    def _compute_all_mixup_losses(self, sent_vecs, labels, lengths, emissions, device):
        total = torch.tensor(0.0, device=device)
        B, T, D = sent_vecs.shape
        flat_embs, flat_labs = [], []
        for b in range(B):
            n = int(lengths[b].item())
            flat_embs.append(sent_vecs[b, :n])
            flat_labs.append(labels[b, :n])
        if not flat_embs: return total
        flat_embs = torch.cat(flat_embs); flat_labs = torch.cat(flat_labs)
        vm = flat_labs >= 0
        flat_embs = flat_embs[vm]; flat_labs = flat_labs[vm]
        if flat_embs.shape[0] < MIXUP_MIN_RARE_IN_BATCH: return total

        if INTRA_RARE_MIXUP_ENABLED:
            mx, soft = self.gpu_mixup.intra_rare_mixup(
                flat_embs, flat_labs, self._rare_t, force_rare_anchor=True)
            if mx is not None:
                total = total + INTRA_RARE_MIXUP_WEIGHT * self.soft_ce(
                    self.mixup_proj(mx), soft.to(device))
                total = total + MIXUP_CRF_BRIDGE_WEIGHT * self._mixup_crf_bridge_loss(
                    mx, soft, device)

        if KG_MIXUP_ENABLED:
            for b in range(B):
                n = int(lengths[b].item()); H_b = sent_vecs[b, :n]; l_b = labels[b, :n]
                valid_b = l_b >= 0
                is_rare_b = (l_b.unsqueeze(1) == self._rare_t.unsqueeze(0)).any(1)
                trig_b = is_rare_b & valid_b
                if not trig_b.any(): continue
                nb_embs, nb_w, _ = self.retriever.retrieve_batch(
                    F.normalize(H_b.detach(), dim=-1), trig_b)
                if nb_embs is None: continue
                trig_idx = trig_b.nonzero(as_tuple=True)[0]
                mx, soft = self.gpu_mixup.kg_guided_mixup(
                    H_b[trig_idx], l_b[trig_idx], nb_embs, nb_w, self._rare_t)
                if mx is not None:
                    total = total + KG_MIXUP_WEIGHT * self.soft_ce(
                        self.mixup_proj(mx), soft.to(device))

        if INTER_MIXUP_ENABLED:
            mx, soft = self.gpu_mixup.inter_class_mixup(
                flat_embs, flat_labs, self._rare_t, self._conf_maj_t)
            if mx is not None:
                total = total + INTER_MIXUP_WEIGHT * self.soft_ce(
                    self.mixup_proj(mx), soft.to(device))

        if SPAN_MIXUP_ENABLED:
            for b in range(B):
                n = int(lengths[b].item())
                mx, soft = self.gpu_mixup.span_mixup(
                    sent_vecs[b, :n], labels[b, :n], self._rare_t)
                if mx is not None:
                    total = total + SPAN_MIXUP_WEIGHT * self.soft_ce(
                        self.mixup_proj(mx), soft.to(device))

        return total

    def forward(self, ids, attn, ttype, labels=None, lengths=None,
                return_sent_vecs=False):
        device = ids.device
        sent_vecs, ctx_out, base_em = self.base.get_emissions(
            ids, attn, ttype, lengths=lengths)
        fused_sent = self._kg_fuse_batch(sent_vecs, base_em, lengths, device)
        fd = self.base.dropout(fused_sent)
        if lengths is not None:
            pk = nn.utils.rnn.pack_padded_sequence(fd, lengths.cpu(),
                                                   batch_first=True, enforce_sorted=False)
            po, _ = self.base.ctx_bilstm(pk)
            fc, _ = nn.utils.rnn.pad_packed_sequence(po, batch_first=True)
        else:
            fc, _ = self.base.ctx_bilstm(fd)
        fc = self.base.dropout(fc)
        fused_em = torch.nan_to_num(self.fusion_classifier(fc), nan=0.0, posinf=1e4, neginf=-1e4)
        if lengths is not None:
            B, T, _ = fused_em.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths): mask[i, :l] = True
        elif labels is not None: mask = (labels != -100)
        else: mask = torch.ones(fused_em.shape[:2], dtype=torch.bool, device=device)
        combined = (base_em + fused_em) / 2.0
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            base_crf  = -self.base.crf(base_em,  safe, mask=mask, reduction="mean")
            fused_crf = -self.fusion_crf(fused_em, safe, mask=mask, reduction="mean")
            B2, T2, C = combined.shape
            flat_logits = combined.reshape(B2*T2, C)
            flat_labels = labels.reshape(B2*T2)
            if self.focal_loss is not None:
                aux_loss = self.focal_loss(flat_logits, flat_labels)
            else:
                aux_loss = self.ce_loss(flat_logits, flat_labels)
            ldam_loss = torch.tensor(0.0, device=device)
            if self.ldam_loss is not None:
                ldam_loss = LDAM_WEIGHT * self.ldam_loss(flat_logits, flat_labels)
            proto_loss  = PROTO_CONTRAST_WEIGHT * self._proto_contrast_loss(device)
            mixup_loss  = MIXUP_LOSS_WEIGHT * self._compute_all_mixup_losses(
                sent_vecs, labels, lengths, base_em, device)
            loss = ((base_crf + fused_crf) / 2.0 + AUX_CE_WEIGHT * aux_loss +
                    proto_loss + mixup_loss + ldam_loss)
            if return_sent_vecs:
                return loss, combined, sent_vecs
            return loss, combined
        else:
            decoded = self.fusion_crf.decode(fused_em, mask=mask)
            return decoded, fused_em


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    mf = lambda a, p, av: f1_score(a, p, average=av, zero_division=0)
    mp = lambda a, p, av: precision_score(a, p, average=av, zero_division=0)
    mr = lambda a, p, av: recall_score(a, p, average=av, zero_division=0)
    per_f = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                     average=None, zero_division=0)
    per_p = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                            average=None, zero_division=0)
    per_r = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                         average=None, zero_division=0)
    pcm = {id2label[i]: {"f1": float(per_f[i]),
                          "precision": float(per_p[i]),
                          "recall":    float(per_r[i])}
           for i in range(NUM_LABELS)}
    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rf1 = f1_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rpr = precision_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rrc = recall_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
    else:
        rf1 = rpr = rrc = 0.0
    st = [id2label[x] for x in all_trues]; sp = [id2label[x] for x in all_preds]
    return dict(
        macro_f1=mf(all_trues,all_preds,"macro"), micro_f1=mf(all_trues,all_preds,"micro"),
        weighted_f1=mf(all_trues,all_preds,"weighted"),
        macro_precision=mp(all_trues,all_preds,"macro"), micro_precision=mp(all_trues,all_preds,"micro"),
        weighted_precision=mp(all_trues,all_preds,"weighted"),
        macro_recall=mr(all_trues,all_preds,"macro"), micro_recall=mr(all_trues,all_preds,"micro"),
        weighted_recall=mr(all_trues,all_preds,"weighted"),
        rare_f1=rf1, rare_precision=rpr, rare_recall=rrc,
        per_class_metrics=pcm, accuracy=accuracy_score(all_trues, all_preds),
        cls_report=classification_report(st, sp, labels=LABELS, digits=4, zero_division=0),
        cm=confusion_matrix(st, sp, labels=LABELS),
        all_preds=all_preds, all_trues=all_trues)


def count_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    fr = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  Trainable: {tr:,} | Frozen: {fr:,}"); return tr, fr


# ═══════════════════════════════════════════════════════════
# DYNAMIC RARE THRESHOLD
# ═══════════════════════════════════════════════════════════
def calibrate_rare_threshold(model, dev_dataset, rare_ids, device=DEVICE):
    model.eval()
    loader = DataLoader(dev_dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for ids, attn, tt, labels, lengths in loader:
            ids=ids.to(device); attn=attn.to(device); tt=tt.to(device); lengths=lengths.to(device)
            _, em = model(ids, attn, tt, labels=None, lengths=lengths)
            B, T, C = em.shape
            for b in range(B):
                n = int(lengths[b].item())
                all_logits.append(em[b, :n].cpu())
                all_labels.append(labels[b, :n])
    all_logits = torch.cat(all_logits); all_labels = torch.cat(all_labels)
    taus = {}
    for rid in rare_ids:
        best_tau, best_f = DYN_RARE_THRESH_TAU, 0.0
        for tau in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
            preds = apply_dynamic_rare_threshold_np(
                all_logits.numpy(), all_labels.numpy(), {rid: tau}, rare_ids)
            f = f1_score(all_labels.numpy(), preds, labels=[rid],
                         average="macro", zero_division=0)
            if f > best_f: best_f = f; best_tau = tau
        taus[rid] = best_tau
    print(f"  Dynamic tau per rare class: {[(id2label[k], v) for k,v in taus.items()]}")
    return taus


def apply_dynamic_rare_threshold_np(logits_np, labels_np, taus, rare_ids):
    preds = logits_np.argmax(axis=-1).copy()
    rare_set = set(rare_ids)
    for i, logit in enumerate(logits_np):
        top1 = preds[i]
        if top1 in rare_set: continue
        rare_logits = {rid: logit[rid] for rid in rare_ids}
        top_rare_id = max(rare_logits, key=rare_logits.get)
        tau = taus.get(top_rare_id, DYN_RARE_THRESH_TAU)
        gap = logit[top1] - logit[top_rare_id]
        if gap < tau:
            preds[i] = top_rare_id
    return preds


@torch.no_grad()
def apply_dynamic_threshold_dataset(model, dataset, rare_ids, taus,
                                    device=DEVICE, calibrator=None):
    model.eval()
    loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
    all_preds, all_trues = [], []
    for ids, attn, tt, labels, lengths in loader:
        ids=ids.to(device); attn=attn.to(device); tt=tt.to(device); lengths=lengths.to(device)
        _, em = model(ids, attn, tt, labels=None, lengths=lengths)
        B, T, C = em.shape
        for b in range(B):
            n = int(lengths[b].item())
            logit_b = em[b, :n]
            if calibrator is not None:
                logit_b = calibrator(logit_b)
            logit_np = logit_b.cpu().numpy()
            lab_np   = labels[b, :n].numpy()
            preds    = apply_dynamic_rare_threshold_np(logit_np, lab_np, taus, rare_ids)
            all_preds.extend(preds.tolist())
            all_trues.extend(lab_np.tolist())
    return all_trues, all_preds


# ═══════════════════════════════════════════════════════════
# KG + MIXUP v5 TRAINER  (with per-epoch checkpoint saving)
# ═══════════════════════════════════════════════════════════
class KGMixupV5Trainer:
    def __init__(self, kg_model, kg, train_docs, tokenizer, device=DEVICE):
        self.model      = kg_model.to(device)
        self.kg         = kg
        self.train_docs = train_docs
        self.tokenizer  = tokenizer
        self.device     = device
        self.ema_store  = RarePrototypeEMAStore(
            kg_model.rare_list, kg_model.base.sent_out_dim, device)

    def build_optimizer(self):
        new_p = (list(self.model.gat_fusion.parameters()) +
                 list(self.model.fusion_classifier.parameters()) +
                 list(self.model.fusion_crf.parameters()) +
                 list(self.model.mixup_proj.parameters()) +
                 list(self.model.gpu_mixup.parameters()) +
                 list(self.model.mixup_crf_proj.parameters()) +
                 list(self.model.mixup_crf_classifier.parameters()) +
                 list(self.model.mixup_bridge_crf.parameters()))
        base_p = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_p,  "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_p, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev", measure_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_time else None
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids=ids.to(self.device); attn=attn.to(self.device)
                tt=tt.to(self.device); lengths=lengths.to(self.device)
                decoded, _ = self.model(ids, attn, tt, labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :int(lengths[i].item())].tolist())
        if measure_time:
            ti = time.time() - t0
            with open(os.path.join(OUT_DIR, f"kg_inf_{split_name}.json"), "w") as f:
                json.dump({"total_s": ti, "ms_per_doc": ti/max(1,len(dataset))*1000}, f)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def _proto_ema_replay_step(self, optimizer):
        protos, proto_labels = self.ema_store.get_proto_tensor()
        if protos is None or protos.shape[0] < 2: return
        logits = self.model.mixup_proj(protos.detach())
        loss   = F.cross_entropy(logits, proto_labels) * PROTO_EMA_REPLAY_WEIGHT
        if not torch.isnan(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
            optimizer.step(); optimizer.zero_grad()

    def train(self, train_docs, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG,
              resume_from_epoch=0,        # ← set to last completed epoch to resume
              resume_best_f1=-1.0):       # ← set to best f1 achieved so far
        sampler = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, int(WARMUP_RATIO*total_steps), total_steps)
        es = EarlyStopping(patience=8)
        # Restore ES state if resuming mid-training
        if resume_best_f1 > 0:
            es.best_score = resume_best_f1

        buffer   = HardExampleBuffer()
        rare_set = set(rare_ids)
        history  = []
        best_f1  = resume_best_f1
        best_state = None
        t_start  = time.time()

        for epoch in range(1, num_epochs+1):

            # ── RESUME: skip already-completed epochs ──────
            if epoch <= resume_from_epoch:
                print(f"[KGv5] Skipping epoch {epoch:02d} (already done)")
                continue

            self.model.train()   # ← always set train mode at epoch start
            run_loss = 0.0; n_steps = 0; t0 = time.time()
            optimizer.zero_grad()

            # ── BUG FIX: Online KG refresh safely ──────────
            if epoch > 1 and epoch % ONLINE_KG_REFRESH_EVERY == 0:
                print(f"  [KG Refresh] epoch {epoch}")
                self.kg.refresh_rare_nodes(
                    self.model.base, self.train_docs, self.tokenizer, rare_ids)
                # Defensive: ensure training mode is restored
                self.model.train()
                self.model.base.train()

            for step, (ids, attn, tt, labels, lengths) in enumerate(train_loader):
                ids=ids.to(self.device); attn=attn.to(self.device)
                tt=tt.to(self.device);  labels=labels.to(self.device)
                lengths=lengths.to(self.device)

                # R-Drop: two forward passes
                loss1, em1, sv1 = self.model(ids, attn, tt, labels=labels,
                                              lengths=lengths, return_sent_vecs=True)
                loss2, em2, sv2 = self.model(ids, attn, tt, labels=labels,
                                              lengths=lengths, return_sent_vecs=True)
                rdrop_l = RDROP_WEIGHT * rdrop_kl_loss(
                    em1, em2, labels, self.model._rare_t if RDROP_RARE_ONLY else None)
                loss = (loss1 + loss2) / 2.0 + rdrop_l

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                run_loss += loss.item(); n_steps += 1

                # EMA prototype update
                with torch.no_grad():
                    for b in range(ids.shape[0]):
                        n = int(lengths[b].item())
                        self.ema_store.update(sv1[b,:n], labels[b,:n])

                # EMA replay
                if (step+1) % PROTO_EMA_REPLAY_EVERY == 0:
                    self._proto_ema_replay_step(optimizer)

                # Hard buffer
                lv = loss.item()
                if lv > HARD_REPLAY_LOSS_THRESH:
                    fl = labels.cpu().flatten()
                    if any(int(x) in rare_set for x in fl if x.item() >= 0):
                        buffer.update(lv, (ids.cpu(), attn.cpu(), tt.cpu(),
                                          labels.cpu(), lengths.cpu()))

                if (step+1) % HARD_REPLAY_FREQ == 0 and len(buffer) >= 10:
                    replay = buffer.sample()
                    if replay:
                        ri, ra, rt, rl, rn = [x.to(self.device) for x in replay]
                        rl_, em_, sv_ = self.model(ri, ra, rt, labels=rl,
                                                    lengths=rn, return_sent_vecs=True)
                        if not torch.isnan(rl_) and not torch.isinf(rl_):
                            (rl_ * HARD_REPLAY_WEIGHT).backward()
                            torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                            optimizer.step(); scheduler.step(); optimizer.zero_grad()

            ep_t = time.time() - t0; avg = run_loss / max(1, n_steps)
            val_m = self.evaluate(dev_dataset, rare_ids)
            print(f"[KGv5] {epoch:02d}/{num_epochs} loss:{avg:.4f} "
                  f"mac:{val_m['macro_f1']:.4f} rare:{val_m['rare_f1']:.4f} "
                  f"t:{ep_t:.1f}s buf:{len(buffer)} ES:{es.counter}/{es.patience}")
            history.append({"epoch": epoch, "phase": "kg_v5",
                             "train_loss": avg, "val_macro_f1": val_m["macro_f1"],
                             "val_rare_f1": val_m["rare_f1"],
                             "val_accuracy": val_m["accuracy"]})

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1 = val_m["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_v5_model.bin"))
                print(f"  ✔ best={best_f1:.4f}")

            # Save per-epoch checkpoint for crash recovery
            ckpt = {
                "epoch": epoch,
                "best_f1": best_f1,
                "model_state": {k: v.cpu().clone() for k, v in self.model.state_dict().items()},
                "optimizer_state": optimizer.state_dict(),
            }
            torch.save(ckpt, os.path.join(OUT_DIR, "kg_v5_latest_ckpt.pt"))

            if es.step(val_m["macro_f1"]): print(f"⏹ ES epoch {epoch}"); break

        total_time = time.time() - t_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "kg_v5_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_v5_model.bin"))
            print(f"  ✔ Saved best (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels: tick.set_color("red")
    ax.set_title(f"{split_name} Confusion Matrix (KG-RAG+Mixup v5)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_cm.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1(pcm, split_name, rare_labels=None):
    f1s = [pcm[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue" for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12); ax.set_xlabel("F1")
    ax.set_title(f"{split_name} Per-Class F1 (KG-RAG+Mixup v5)")
    ax.grid(True, alpha=0.3, axis="x"); plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def plot_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["ep"] = range(1, len(all_df)+1); b = len(base_df)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(all_df["ep"], all_df["train_loss"], marker="o", ms=3)
    axes[0].axvline(b, color="red", ls="--"); axes[0].set_title("Training Loss")
    axes[1].plot(all_df["ep"], all_df["val_macro_f1"], label="Macro-F1", marker="o", ms=3)
    axes[1].plot(all_df["ep"], all_df["val_rare_f1"],  label="Rare-F1",  marker="s", ms=3)
    axes[1].axvline(b, color="red", ls="--"); axes[1].legend()
    axes[1].set_title("Val F1")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150); plt.close()


def print_table(dev_m, test_m, base_t=None, kg_t=None, n_params=None):
    rows = [("Accuracy","accuracy"),("Macro-F1","macro_f1"),("Micro-F1","micro_f1"),
            ("Weighted-F1","weighted_f1"),("Minority Macro-F1","rare_f1"),
            ("Macro-Precision","macro_precision"),("Macro-Recall","macro_recall")]
    print("\n" + "="*72)
    print("FINAL RESULTS — KG-RAG+Mixup v5 (all 9 enhancements)")
    print("="*72)
    if n_params: print(f"  Params     : {n_params:,}")
    if kg_t:     print(f"  Phase B    : {kg_t/60:.1f} min (resumed from epoch 4)")
    print(f"  {'Metric':<30} {'Dev':>10} {'Test':>10}")
    print("-"*72)
    for lbl, key in rows:
        print(f"  {lbl:<30} {dev_m[key]:>10.4f} {test_m[key]:>10.4f}")
    print("="*72)
    print(f"\n  {'Label':<22} {'F1-Dev':>8} {'F1-Test':>8} {'Prec':>8} {'Rec':>8}")
    print("  "+"-"*56)
    for lbl in LABELS:
        dv=dev_m["per_class_metrics"][lbl]; ts=test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>8.4f} {ts['f1']:>8.4f} "
              f"{ts['precision']:>8.4f} {ts['recall']:>8.4f}")


# ═══════════════════════════════════════════════════════════
# MAIN  — RESUME from Phase B epoch 4
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device: {DEVICE}")
    print("KG-RAG+Mixup v5 — RESUME from Phase B epoch 4")
    print("BUG FIX: refresh_rare_nodes now restores model.train() after eval()\n")

    # ── Load data ────────────────────────────────────────
    print("Loading data ...")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"  Train:{len(train_docs)} Dev:{len(dev_docs)} Test:{len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    counts = Counter(lid for _, labs in train_docs for lid in labs)

    class_weights = compute_class_weights(label_freqs)
    focal_loss    = FocalLoss(class_weights, rare_ids).to(DEVICE)
    ldam_loss     = LDAMLoss(counts).to(DEVICE)

    print("Loading tokenizer ...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ── Load base_history for plotting later ─────────────
    base_hist_path = os.path.join(OUT_DIR, "base_history.csv")
    if os.path.exists(base_hist_path):
        base_hist = pd.read_csv(base_hist_path)
        print(f"  Loaded base_history.csv ({len(base_hist)} epochs)")
    else:
        base_hist = pd.DataFrame()
        print("  WARNING: base_history.csv not found, curves plot will be partial")

    # ── Load base model ───────────────────────────────────
    print("\n" + "="*60)
    print("RESUMING: Loading saved base model from Phase A")
    print("="*60)
    base_model = InLegalBERT_BiLSTM_MHA_CRF()
    saved_base = os.path.join(BEST_MODEL_DIR, "base_model.bin")
    if not os.path.exists(saved_base):
        raise FileNotFoundError(
            f"base_model.bin not found at {saved_base}. "
            "Make sure Phase A completed successfully.")
    state = torch.load(saved_base, map_location=DEVICE)
    base_model.load_state_dict(state)
    base_model.to(DEVICE)
    print(f"  ✔ base_model.bin loaded from {saved_base}")

    # ── Load KG ───────────────────────────────────────────
    print("\n" + "="*60)
    print("RESUMING: Loading saved Knowledge Graph")
    print("="*60)
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if not os.path.exists(kg_path):
        raise FileNotFoundError(
            f"knowledge_graph.json not found at {kg_path}. "
            "Make sure the KG build step completed.")
    kg = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim, device=DEVICE)
    kg.rare_partition = set(rare_ids)

    # Rebuild confusion pairs (lightweight, from loaded base model)
    print("  Recomputing confusion pairs from base model ...")
    from collections import defaultdict as _dd
    base_model.eval()
    loader_tmp = DataLoader(train_dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
    all_p2, all_t2 = [], []
    rare_set = set(rare_ids)
    with torch.no_grad():
        for ids, attn, tt, labels, lengths in loader_tmp:
            ids=ids.to(DEVICE); attn=attn.to(DEVICE); tt=tt.to(DEVICE); lengths=lengths.to(DEVICE)
            decoded, _ = base_model(ids, attn, tt, labels=None, lengths=lengths)
            for i, sp in enumerate(decoded):
                all_p2.extend(sp)
                all_t2.extend(labels[i, :int(lengths[i].item())].tolist())
    confusion = _dd(Counter)
    for t, p in zip(all_t2, all_p2):
        if t in rare_set and p != t: confusion[t][p] += 1
    confusion_pairs = {}
    print("\n📊 Confusion pairs:")
    for rid, ctr in confusion.items():
        top = [c for c, _ in ctr.most_common(CONFUSION_TOP_K)]
        confusion_pairs[rid] = top
        print(f"  {id2label[rid]:<20} → {[id2label[c] for c in top]}")

    lambda_boost = KGRetriever.calibrate_lambda(kg, rare_ids)
    retriever    = KGRetriever(kg, rare_ids=rare_ids, top_k=KG_TOP_K,
                               top_nodes=KG_TOP_NODES, hop=KG_HOP,
                               lambda_boost=lambda_boost)

    # ── Build KG+Mixup v5 model ───────────────────────────
    print("\n" + "="*60)
    print("PHASE B: KG+Mixup v5 Training  (starting from epoch 5)")
    print("="*60)
    kg_v5_model = KGMixupV5Model(
        base_model=base_model, kg=kg, rare_ids=rare_ids,
        retriever=retriever, focal_loss=focal_loss, ldam_loss=ldam_loss,
        class_weights=class_weights.to(DEVICE), confusion_pairs=confusion_pairs)

    n_params, _ = count_parameters(kg_v5_model)

    # ── Check for latest checkpoint ───────────────────────
    ckpt_path = os.path.join(OUT_DIR, "kg_v5_latest_ckpt.pt")
    resume_from_epoch = 4   # we know epochs 1-4 completed from the crash log
    resume_best_f1    = 0.5148  # best from epoch 4 per crash log

    kg_v5_bin = os.path.join(BEST_MODEL_DIR, "kg_v5_model.bin")
    if os.path.exists(ckpt_path):
        print(f"  Found checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        kg_v5_model.load_state_dict(ckpt["model_state"])
        resume_from_epoch = ckpt["epoch"]
        resume_best_f1    = ckpt["best_f1"]
        print(f"  ✔ Resuming from epoch {resume_from_epoch}, best_f1={resume_best_f1:.4f}")
    elif os.path.exists(kg_v5_bin):
        print(f"  Found kg_v5_model.bin (loading best weights so far)")
        kg_v5_model.load_state_dict(torch.load(kg_v5_bin, map_location=DEVICE))
        print(f"  ✔ Loaded best weights. Resuming from epoch {resume_from_epoch}")
    else:
        print(f"  No Phase B checkpoint found. Resuming from epoch {resume_from_epoch} "
              f"with base model weights only.")

    trainer = KGMixupV5Trainer(kg_v5_model, kg, train_docs, tokenizer, device=DEVICE)
    kg_hist, kg_time = trainer.train(
        train_docs, train_dataset, dev_dataset,
        rare_ids=rare_ids,
        num_epochs=NUM_EPOCHS_KG,
        resume_from_epoch=resume_from_epoch,
        resume_best_f1=resume_best_f1)

    # Plot curves
    if not base_hist.empty:
        plot_history(base_hist, kg_hist)

    # ── Post-hoc calibration ──────────────────────────────
    print("\nFitting per-class temperature calibration on dev set ...")
    kg_v5_model.eval()
    dev_logits_list, dev_labels_list = [], []
    with torch.no_grad():
        loader = DataLoader(dev_dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        for ids, attn, tt, labels, lengths in loader:
            ids=ids.to(DEVICE); attn=attn.to(DEVICE); tt=tt.to(DEVICE); lengths=lengths.to(DEVICE)
            _, em = kg_v5_model(ids, attn, tt, labels=None, lengths=lengths)
            B, T, C = em.shape
            for b in range(B):
                n = int(lengths[b].item())
                dev_logits_list.append(em[b, :n].cpu())
                dev_labels_list.append(labels[b, :n])
    dev_logits_all = torch.cat(dev_logits_list)
    dev_labels_all = torch.cat(dev_labels_list)
    calibrator = PerClassTemperatureCalibrator()
    calibrator.fit(dev_logits_all, dev_labels_all.clone())
    calibrator.to(DEVICE)

    # ── Dynamic rare threshold ────────────────────────────
    print("Calibrating dynamic rare threshold ...")
    taus = calibrate_rare_threshold(kg_v5_model, dev_dataset, rare_ids, device=DEVICE)

    # ── Evaluation ───────────────────────────────────────
    print("\nEvaluating (standard) on Dev ...")
    dev_m = trainer.evaluate(dev_dataset, rare_ids, "dev", measure_time=True)
    print(f"  Dev: mac={dev_m['macro_f1']:.4f} rare={dev_m['rare_f1']:.4f}")

    print("Evaluating (dynamic threshold + calibration) on Dev ...")
    dev_trues, dev_preds_dyn = apply_dynamic_threshold_dataset(
        kg_v5_model, dev_dataset, rare_ids, taus,
        device=DEVICE, calibrator=calibrator)
    dev_m_dyn = compute_all_metrics(dev_trues, dev_preds_dyn, rare_ids, "dev_dyn")
    print(f"  Dev (dyn+cal): mac={dev_m_dyn['macro_f1']:.4f} rare={dev_m_dyn['rare_f1']:.4f}")

    best_dev = dev_m_dyn if dev_m_dyn["macro_f1"] >= dev_m["macro_f1"] else dev_m

    print("Evaluating (dynamic threshold + calibration) on Test ...")
    test_trues, test_preds_dyn = apply_dynamic_threshold_dataset(
        kg_v5_model, test_dataset, rare_ids, taus,
        device=DEVICE, calibrator=calibrator)
    test_m = compute_all_metrics(test_trues, test_preds_dyn, rare_ids, "test")
    print(f"  Test: mac={test_m['macro_f1']:.4f} rare={test_m['rare_f1']:.4f}")

    for split_name, m in [("dev", best_dev), ("test", test_m)]:
        with open(os.path.join(OUT_DIR, f"{split_name}_cls_report.txt"), "w") as f:
            f.write(f"Model: KG-RAG+Mixup v5 (resumed)\nRare: {rare_labels}\n\n{m['cls_report']}")
        save_confusion_matrix(m["cm"], split_name, rare_labels)
        save_per_class_f1(m["per_class_metrics"], split_name, rare_labels)

    pd.DataFrame({"true": [id2label[x] for x in test_m["all_trues"]],
                  "pred": [id2label[x] for x in test_m["all_preds"]]
                  }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    skeys = ["macro_f1","micro_f1","weighted_f1","rare_f1",
             "macro_precision","macro_recall","accuracy"]
    summary = {
        "model": "KG-RAG+Mixup-v5-resumed",
        "resumed_from_epoch": resume_from_epoch,
        "bug_fix": "refresh_rare_nodes restores model.train() after eval()",
        "rare_classes": rare_labels,
        "dev":  {k: best_dev[k] for k in skeys},
        "test": {k: test_m[k]   for k in skeys},
        "per_class_test": test_m["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_table(best_dev, test_m, kg_t=kg_time, n_params=n_params)
    print(f"\n📁 All outputs → {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device: cuda:0
KG-RAG+Mixup v5 — RESUME from Phase B epoch 4
BUG FIX: refresh_rare_nodes now restores model.train() after eval()

Loading data ...
  Train:245 Dev:30 Test:50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167)
   FAC                  19.99%  ( 5744)
   RLC                   2.62%  (  752) ← RARE
   ISSUE                 1.28%  (  367) ← RARE
   ARG_PETITIONER        4.58%  ( 1315) ← RARE
   ARG_RESPONDENT        2.43%  (  698) ← RARE
   ANALYSIS             36.66%  (10537)
   STA                   1.67%  (  481) ← RARE
   PRE_RELIED            4.97%  ( 1427) ← RARE
   PRE_NOT_RELIED        0.55%  (  158) ← RARE
   RATIO                 2.30%  (  661) ← RARE
   RPC                   3.67%  ( 1055) ← RARE
   NONE                  4.79%  ( 1377) ← RARE

   Rare (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 'ARG_RESPONDENT', 'STA', 'PRE_RELIED', 'PRE_NOT_RELIED', 'RATIO', 'RPC', 'NONE']

Loading tokenizer ...
  Loaded base_history.csv (60 epochs

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


❄️  Frozen: embeddings + layers 0-7. 🔥 Trainable: 8+

  ✔ base_model.bin loaded from rrc_kg_rag_mixup_v5_logs/best_model/base_model.bin

RESUMING: Loading saved Knowledge Graph
  KG loaded ← rrc_kg_rag_mixup_v5_logs/knowledge_graph.json
  Recomputing confusion pairs from base model ...

📊 Confusion pairs:
  ARG_RESPONDENT       → ['ARG_PETITIONER', 'ANALYSIS', 'FAC']
  RLC                  → ['FAC', 'ISSUE', 'ANALYSIS']
  ARG_PETITIONER       → ['ARG_RESPONDENT', 'PRE_RELIED', 'ANALYSIS']
  NONE                 → ['RPC', 'FAC', 'ISSUE']
  RATIO                → ['ANALYSIS', 'RPC', 'PRE_RELIED']
  PRE_RELIED           → ['ANALYSIS', 'STA', 'RATIO']
  PRE_NOT_RELIED       → ['PRE_RELIED', 'ANALYSIS', 'ARG_PETITIONER']
  RPC                  → ['RATIO', 'NONE', 'RLC']
  ISSUE                → ['RLC', 'STA', 'FAC']
  STA                  → ['PRE_RELIED', 'ANALYSIS', 'PREAMBLE']
  λ → 0.0500

PHASE B: KG+Mixup v5 Training  (starting from epoch 5)
  Trainable: 31,085,565 | Frozen: 80,540,160